In [1]:
import sys

print("Python do notebook:")
print(sys.executable)

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm

print("\nReportLab OK")

Python do notebook:
C:\Users\beelt\Documents\collections_case_candidate\.venv\Scripts\python.exe

ReportLab OK


In [2]:
from pathlib import Path
import json
import io
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Table,
    TableStyle,
    PageBreak,
    Image,
)

warnings.filterwarnings("ignore")

## 1. Configuração

Por padrão, o gerador procura os notebooks na mesma pasta deste arquivo ou na pasta anterior.  
Ajuste apenas os caminhos se a estrutura do repositório mudar.

In [3]:
ROOT = Path.cwd()

CANDIDATES = [
    ROOT / "05_bivariate_eda_collections_macro_analysis.ipynb",
    ROOT.parent / "05_bivariate_eda_collections_macro_analysis.ipynb",
    ROOT / "notebooks" / "05_bivariate_eda_collections_macro_analysis.ipynb",
]

BIVARIATE_NOTEBOOK = next((p for p in CANDIDATES if p.exists()), None)

if BIVARIATE_NOTEBOOK is None:
    raise FileNotFoundError(
        "Não encontrei 05_bivariate_eda_collections_macro_analysis.ipynb. "
        "Ajuste BIVARIATE_NOTEBOOK nesta célula."
    )

REPORT_DIR = ROOT / "reports"
CHART_DIR = REPORT_DIR / "collections_report_charts"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
CHART_DIR.mkdir(parents=True, exist_ok=True)

PDF_PATH = REPORT_DIR / "collections_macro_bivariate_analysis_report.pdf"

print("Notebook fonte :", BIVARIATE_NOTEBOOK.resolve())
print("PDF de saída   :", PDF_PATH.resolve())

Notebook fonte : C:\Users\beelt\Documents\collections_case_candidate\notebooks\05_bivariate_eda_collections_macro_analysis.ipynb
PDF de saída   : C:\Users\beelt\Documents\collections_case_candidate\notebooks\reports\collections_macro_bivariate_analysis_report.pdf


In [4]:
pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_DIR = Path("../data")
QUEUE_PATH = DATA_DIR / "raw" / "collections_queue_sep2026.csv"
WA_PATH = DATA_DIR / "raw" / "whatsapp_collections_history.csv"

if not QUEUE_PATH.exists():
    QUEUE_PATH = Path("/mnt/data/collections_queue_sep2026(1).csv")
if not WA_PATH.exists():
    WA_PATH = Path("/mnt/data/whatsapp_collections_history(2).csv")

queue = pd.read_csv(QUEUE_PATH)
wa = pd.read_csv(WA_PATH)

wa["sent_at"] = pd.to_datetime(wa["sent_at"], errors="coerce")
queue["in_collections_since"] = pd.to_datetime(queue["in_collections_since"], errors="coerce")

print("Queue:", queue.shape)
print("WhatsApp:", wa.shape)

Queue: (10658, 10)
WhatsApp: (75406, 17)


## _______________________________________________________________ "" ____________________________________________________________

## _______________________________________________________________ "" ____________________________________________________________

In [5]:
# ============================================================
# 00A — RAW DATA AUDIT
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 100)
print("00A — RAW DATA AUDIT")
print("=" * 100)


# ============================================================
# 1. SHAPE
# ============================================================

print("\nSHAPE")
print("-" * 100)

print(f"Queue rows       : {len(queue):,}")
print(f"Queue columns    : {queue.shape[1]:,}")
print(f"WhatsApp rows    : {len(wa):,}")
print(f"WhatsApp columns : {wa.shape[1]:,}")


# ============================================================
# 2. COLUMN NAMES
# ============================================================

print("\nQUEUE COLUMNS")
print("-" * 100)

for col in queue.columns:
    print(col)


print("\nWHATSAPP COLUMNS")
print("-" * 100)

for col in wa.columns:
    print(col)


# ============================================================
# 3. CUSTOMER COUNTS
# ============================================================

print("\nCUSTOMERS")
print("-" * 100)

print(
    f"Queue unique customers    : "
    f"{queue['customer_id'].nunique():,}"
)

print(
    f"WhatsApp unique customers : "
    f"{wa['customer_id'].nunique():,}"
)


# ============================================================
# 4. CUSTOMER KEY AUDIT
# ============================================================

print("\nCUSTOMER KEY AUDIT")
print("-" * 100)

print(
    f"Queue duplicate customer_id rows : "
    f"{queue['customer_id'].duplicated().sum():,}"
)

print(
    f"WhatsApp duplicate customer_id rows : "
    f"{wa['customer_id'].duplicated().sum():,}"
)


# ============================================================
# 5. WHATSAPP MESSAGE ID AUDIT
# ============================================================

if "message_id" in wa.columns:

    print("\nMESSAGE ID AUDIT")
    print("-" * 100)

    print(
        f"Missing message_id   : "
        f"{wa['message_id'].isna().sum():,}"
    )

    print(
        f"Unique message_id    : "
        f"{wa['message_id'].nunique(dropna=True):,}"
    )

    print(
        f"Duplicated message_id: "
        f"{wa['message_id'].duplicated(keep=False).sum():,}"
    )


# ============================================================
# 6. EXACT DUPLICATES
# ============================================================

print("\nEXACT DUPLICATES")
print("-" * 100)

print(
    f"Queue exact duplicate rows    : "
    f"{queue.duplicated().sum():,}"
)

print(
    f"WhatsApp exact duplicate rows : "
    f"{wa.duplicated().sum():,}"
)


# ============================================================
# 7. TEMPORAL AUDIT
# ============================================================

print("\nTEMPORAL COVERAGE")
print("-" * 100)

print(
    f"WhatsApp min sent_at : "
    f"{wa['sent_at'].min()}"
)

print(
    f"WhatsApp max sent_at : "
    f"{wa['sent_at'].max()}"
)

print(
    f"Missing sent_at      : "
    f"{wa['sent_at'].isna().sum():,}"
)


print(
    f"\nQueue min collections date : "
    f"{queue['in_collections_since'].min()}"
)

print(
    f"Queue max collections date : "
    f"{queue['in_collections_since'].max()}"
)

print(
    f"Missing collections date   : "
    f"{queue['in_collections_since'].isna().sum():,}"
)


# ============================================================
# 8. DUPLICATE SEND TIMING
# ============================================================

print("\nWHATSAPP TIMING DUPLICATES")
print("-" * 100)

same_timestamp = wa.duplicated(
    subset=["customer_id", "sent_at"],
    keep=False
)

print(
    f"Rows sharing customer + exact timestamp : "
    f"{same_timestamp.sum():,}"
)


wa_audit_date = wa["sent_at"].dt.normalize()

same_day = (
    pd.DataFrame({
        "customer_id": wa["customer_id"],
        "sent_date": wa_audit_date,
    })
    .duplicated(
        subset=["customer_id", "sent_date"],
        keep=False
    )
)

print(
    f"Rows sharing customer + calendar day    : "
    f"{same_day.sum():,}"
)


# ============================================================
# 9. MISSINGNESS
# ============================================================

print("\nWHATSAPP MISSINGNESS")
print("-" * 100)

wa_missing = (
    wa.isna()
    .sum()
    .to_frame("missing")
)

wa_missing["pct"] = (
    wa_missing["missing"]
    / len(wa)
    * 100
)

display(
    wa_missing.loc[
        wa_missing["missing"] > 0
    ].sort_values(
        "missing",
        ascending=False
    )
)


print("\nQUEUE MISSINGNESS")
print("-" * 100)

queue_missing = (
    queue.isna()
    .sum()
    .to_frame("missing")
)

queue_missing["pct"] = (
    queue_missing["missing"]
    / len(queue)
    * 100
)

display(
    queue_missing.loc[
        queue_missing["missing"] > 0
    ].sort_values(
        "missing",
        ascending=False
    )
)


# ============================================================
# 10. OVERLAP BETWEEN QUEUE AND WHATSAPP HISTORY
# ============================================================

queue_ids = set(
    queue["customer_id"].dropna()
)

wa_ids = set(
    wa["customer_id"].dropna()
)

common_ids = queue_ids & wa_ids
queue_only_ids = queue_ids - wa_ids
wa_only_ids = wa_ids - queue_ids


print("\nCUSTOMER POPULATION OVERLAP")
print("-" * 100)

print(
    f"Customers in both        : "
    f"{len(common_ids):,}"
)

print(
    f"Queue only               : "
    f"{len(queue_only_ids):,}"
)

print(
    f"WhatsApp history only    : "
    f"{len(wa_only_ids):,}"
)

print(
    f"Queue history coverage   : "
    f"{len(common_ids) / len(queue_ids) * 100:.2f}%"
)


# ============================================================
# 11. RAW OUTCOME DISTRIBUTIONS
# ============================================================

print("\nRAW DELIVERY STATUS")
print("-" * 100)

display(
    wa["delivery_status"]
    .value_counts(dropna=False)
    .to_frame("messages")
)


print("\nRAW INTERACTION")
print("-" * 100)

display(
    wa["interaction"]
    .value_counts(dropna=False)
    .to_frame("messages")
)


print("\nRAW PAYMENT FLAG")
print("-" * 100)

display(
    wa["paid_within_72h"]
    .value_counts(dropna=False)
    .to_frame("messages")
)


# ============================================================
# 12. FINAL
# ============================================================

print("\n" + "=" * 100)
print("00A — RAW AUDIT COMPLETE")
print("=" * 100)

00A — RAW DATA AUDIT

SHAPE
----------------------------------------------------------------------------------------------------
Queue rows       : 10,658
Queue columns    : 10
WhatsApp rows    : 75,406
WhatsApp columns : 17

QUEUE COLUMNS
----------------------------------------------------------------------------------------------------
customer_id
in_collections_since
days_past_due_on_2026-09-01
outstanding_balance_brl
monthly_salary_brl
payday_day_of_month
n_prior_transactions
account_age_months
days_since_last_app_login
state_uf

WHATSAPP COLUMNS
----------------------------------------------------------------------------------------------------
message_id
customer_id
sent_at
template
n_msgs_last_14d
days_past_due
outstanding_balance_brl
monthly_salary_brl
payday_day_of_month
n_prior_transactions
account_age_months
days_since_last_app_login
state_uf
delivery_status
interaction
paid_within_72h
amount_paid_brl

CUSTOMERS
--------------------------------------------------------------

,missing,pct



QUEUE MISSINGNESS
----------------------------------------------------------------------------------------------------


,missing,pct



CUSTOMER POPULATION OVERLAP
----------------------------------------------------------------------------------------------------
Customers in both        : 5,382
Queue only               : 5,276
WhatsApp history only    : 6,342
Queue history coverage   : 50.50%

RAW DELIVERY STATUS
----------------------------------------------------------------------------------------------------


...


RAW INTERACTION
----------------------------------------------------------------------------------------------------


,messages
interaction,
none,45201
read,17505
clicked_link,8145
replied,4555



RAW PAYMENT FLAG
----------------------------------------------------------------------------------------------------


,messages
paid_within_72h,
0,69780
1,5626



00A — RAW AUDIT COMPLETE


In [6]:
# ============================================================
# 00B — CANONICAL WHATSAPP EVENT TABLE
# ============================================================
#
# PURPOSE
# -------
# Build the canonical chronological event table.
#
# GRAIN
# -----
# 1 row = 1 WhatsApp send event
#
# IMPORTANT
# ---------
# No feature engineering here.
# No historical outcomes here.
# No queue merge here.
#
# We only:
#   - preserve source order
#   - enforce chronological ordering
#   - create canonical event identifiers
#   - explicitly separate action / outcome semantics
#   - audit the resulting grain
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 1. CREATE CANONICAL EVENT TABLE
# ============================================================

events = wa.copy()

events["_source_row"] = np.arange(
    len(events),
    dtype=np.int32
)


# ============================================================
# 2. NORMALIZE TYPES
# ============================================================

events["sent_at"] = pd.to_datetime(
    events["sent_at"],
    errors="raise"
)

events["customer_id"] = (
    events["customer_id"]
    .astype("string")
)

events["message_id"] = (
    events["message_id"]
    .astype("string")
)


# ============================================================
# 3. CHRONOLOGICAL ORDER
# ============================================================
#
# message_id is unique, but _source_row gives us a deterministic
# final tie-breaker independent of its formatting.
#
# We already audited that customer + sent_at is unique.
# ============================================================

events = (
    events
    .sort_values(
        [
            "customer_id",
            "sent_at",
            "_source_row",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ============================================================
# 4. EVENT NUMBER WITHIN CUSTOMER
# ============================================================

events["event_number"] = (
    events
    .groupby(
        "customer_id",
        sort=False
    )
    .cumcount()
    .add(1)
    .astype("int16")
)


# ============================================================
# 5. CANONICAL EVENT ID
# ============================================================
#
# We already have message_id as the source-system identifier.
#
# event_id is our modelling/event-table identifier.
# ============================================================

events["event_id"] = (
    events["customer_id"].astype(str)
    + "__"
    + events["event_number"].astype(str)
)


# ============================================================
# 6. CALENDAR DATE
# ============================================================

events["sent_date"] = (
    events["sent_at"]
    .dt.normalize()
)


# ============================================================
# 7. EXPLICIT CURRENT ACTION
# ============================================================
#
# This is NOT an outcome.
#
# For historical-policy modelling it is the action actually taken.
# Later, for NBA scoring, candidate actions will be evaluated
# separately.
# ============================================================

events["current_template"] = (
    events["template"]
    .astype("string")
)


# ============================================================
# 8. EXPLICIT CURRENT OUTCOME NAMES
# ============================================================
#
# These variables are LABELS / post-action information.
#
# Renaming them now prevents accidental confusion later.
# ============================================================

events = events.rename(
    columns={
        "delivery_status":
            "event_delivery_status",

        "interaction":
            "event_interaction",

        "paid_within_72h":
            "event_paid_within_72h",

        "amount_paid_brl":
            "event_amount_paid_brl",
    }
)


# ============================================================
# 9. EVENT-LEVEL AUDITS
# ============================================================

print("=" * 100)
print("00B — CANONICAL WHATSAPP EVENT TABLE")
print("=" * 100)

print(f"\nRows      : {len(events):,}")
print(
    f"Customers : "
    f"{events['customer_id'].nunique():,}"
)
print(
    f"Messages  : "
    f"{events['message_id'].nunique():,}"
)
print(
    f"Events    : "
    f"{events['event_id'].nunique():,}"
)
print(
    f"Columns   : "
    f"{events.shape[1]:,}"
)


# ============================================================
# 10. UNIQUENESS
# ============================================================

print("\nUNIQUENESS")
print("-" * 100)

print(
    "Duplicate message_id :",
    events["message_id"].duplicated().sum()
)

print(
    "Duplicate event_id   :",
    events["event_id"].duplicated().sum()
)

print(
    "Duplicate customer + timestamp :",
    events.duplicated(
        ["customer_id", "sent_at"]
    ).sum()
)


# ============================================================
# 11. CHRONOLOGICAL ORDER AUDIT
# ============================================================

chronological = (
    events
    .groupby(
        "customer_id",
        sort=False
    )["sent_at"]
    .apply(
        lambda s:
            s.is_monotonic_increasing
    )
    .all()
)

print("\nTEMPORAL ORDER")
print("-" * 100)

print(
    f"All customers chronological : "
    f"{chronological}"
)


# ============================================================
# 12. EVENT NUMBER AUDIT
# ============================================================

customer_size = (
    events
    .groupby(
        "customer_id",
        sort=False
    )
    .size()
)

customer_max_event = (
    events
    .groupby(
        "customer_id",
        sort=False
    )["event_number"]
    .max()
)

event_number_valid = (
    customer_size
    .eq(customer_max_event)
    .all()
)

print(
    f"Event numbering valid       : "
    f"{event_number_valid}"
)


# ============================================================
# 13. CUSTOMER JOURNEY SUMMARY
# ============================================================

journey_summary = (
    customer_size
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
)

print("\nMESSAGES PER CUSTOMER")
print("-" * 100)

display(
    journey_summary
    .to_frame("messages_per_customer")
)


# ============================================================
# 14. PERIOD
# ============================================================

print("\nPERIOD")
print("-" * 100)

print(
    "First send :",
    events["sent_at"].min()
)

print(
    "Last send  :",
    events["sent_at"].max()
)


# ============================================================
# 15. FIRST EVENT AUDIT
# ============================================================

first_events = (
    events["event_number"] == 1
)

print("\nFIRST EVENT AUDIT")
print("-" * 100)

print(
    f"First events       : "
    f"{first_events.sum():,}"
)

print(
    f"Expected customers : "
    f"{events['customer_id'].nunique():,}"
)


# ============================================================
# 16. SEMANTIC GROUPS
# ============================================================

identifier_cols = [
    "event_id",
    "message_id",
    "customer_id",
    "event_number",
    "sent_at",
    "sent_date",
    "_source_row",
]

action_cols = [
    "current_template",
]

label_cols = [
    "event_delivery_status",
    "event_interaction",
    "event_paid_within_72h",
    "event_amount_paid_brl",
]

print("\nSEMANTIC GROUPS")
print("-" * 100)

print("Identifiers:")
for col in identifier_cols:
    print("  -", col)

print("\nCurrent action:")
for col in action_cols:
    print("  -", col)

print("\nCurrent outcomes / labels:")
for col in label_cols:
    print("  -", col)


# ============================================================
# 17. HARD ASSERTIONS
# ============================================================

assert len(events) == len(wa)

assert (
    events["message_id"]
    .is_unique
)

assert (
    events["event_id"]
    .is_unique
)

assert (
    events["sent_at"]
    .notna()
    .all()
)

assert chronological

assert event_number_valid

assert (
    first_events.sum()
    == events["customer_id"].nunique()
)


# ============================================================
# 18. SAMPLE
# ============================================================

print("\nSAMPLE")
print("-" * 100)

sample_cols = [
    "event_id",
    "message_id",
    "customer_id",
    "event_number",
    "sent_at",
    "current_template",
    "days_past_due",
    "outstanding_balance_brl",
    "event_delivery_status",
    "event_interaction",
    "event_paid_within_72h",
    "event_amount_paid_brl",
]

display(
    events[
        sample_cols
    ].head(15)
)


print("\n" + "=" * 100)
print("00B — CANONICAL EVENT TABLE COMPLETE")
print("=" * 100)

00B — CANONICAL WHATSAPP EVENT TABLE

Rows      : 75,406
Customers : 11,724
Messages  : 75,406
Events    : 75,406
Columns   : 22

UNIQUENESS
----------------------------------------------------------------------------------------------------
Duplicate message_id : 0
Duplicate event_id   : 0
Duplicate customer + timestamp : 0

TEMPORAL ORDER
----------------------------------------------------------------------------------------------------
All customers chronological : True
Event numbering valid       : True

MESSAGES PER CUSTOMER
----------------------------------------------------------------------------------------------------


,messages_per_customer
count,"11,724.00"
mean,6.43
std,3.75
min,1.00
25%,3.00
50%,6.00
75%,9.00
90%,12.00
95%,13.00
99%,15.00



PERIOD
----------------------------------------------------------------------------------------------------
First send : 2026-06-01 09:18:00
Last send  : 2026-08-31 20:59:00

FIRST EVENT AUDIT
----------------------------------------------------------------------------------------------------
First events       : 11,724
Expected customers : 11,724

SEMANTIC GROUPS
----------------------------------------------------------------------------------------------------
Identifiers:
  - event_id
  - message_id
  - customer_id
  - event_number
  - sent_at
  - sent_date
  - _source_row

Current action:
  - current_template

Current outcomes / labels:
  - event_delivery_status
  - event_interaction
  - event_paid_within_72h
  - event_amount_paid_brl

SAMPLE
----------------------------------------------------------------------------------------------------


,event_id,message_id,customer_id,event_number,sent_at,current_template,days_past_due,outstanding_balance_brl,event_delivery_status,event_interaction,event_paid_within_72h,event_amount_paid_brl
0,C000001__1,M0018748,C000001,1,2026-07-06 13:29:00,pix_link,3,934.58,delivered,none,0,0.00
1,C000001__2,M0019916,C000001,2,2026-07-07 14:48:00,friendly_reminder,4,934.58,delivered,clicked_link,0,0.00
2,C000001__3,M0020536,C000001,3,2026-07-08 10:46:00,pix_link,5,934.58,delivered,none,0,0.00
3,C000001__4,M0023754,C000001,4,2026-07-11 13:11:00,urgent_reminder,8,934.58,delivered,none,0,0.00
4,C000001__5,M0027130,C000001,5,2026-07-15 12:54:00,urgent_reminder,12,934.58,delivered,none,0,0.00
5,C000001__6,M0029418,C000001,6,2026-07-17 12:54:00,urgent_reminder,14,934.58,delivered,none,0,0.00
6,C000001__7,M0040347,C000001,7,2026-07-29 09:14:00,urgent_reminder,26,934.58,delivered,none,1,934.58
7,C000002__1,M0040569,C000002,1,2026-07-29 10:31:00,pix_link,4,"1,143.79",delivered,read,0,0.00
8,C000002__2,M0041668,C000002,2,2026-07-30 09:31:00,friendly_reminder,5,"1,143.79",delivered,none,0,0.00
9,C000002__3,M0044923,C000002,3,2026-08-02 10:54:00,urgent_reminder,8,"1,143.79",delivered,read,0,0.00



00B — CANONICAL EVENT TABLE COMPLETE


In [7]:
# ============================================================
# 01A — CURRENT STATE: CALENDAR & TIME
# ============================================================
#
# GRAIN:
#   1 row = 1 WhatsApp send event
#
# POINT-IN-TIME:
#   All features describe information available at sent_at.
#
# No historical outcomes.
# No future information.
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 0. INITIALIZE FEATURE TABLE — ONCE
# ============================================================

features = events.copy()

cols_before = features.shape[1]


# ============================================================
# 1. BASIC CALENDAR
# ============================================================

features["send_year"] = (
    features["sent_at"].dt.year.astype("int16")
)

features["send_month"] = (
    features["sent_at"].dt.month.astype("int8")
)

features["send_day"] = (
    features["sent_at"].dt.day.astype("int8")
)

features["send_day_of_year"] = (
    features["sent_at"].dt.dayofyear.astype("int16")
)

features["send_weekday"] = (
    features["sent_at"].dt.weekday.astype("int8")
)

features["send_weekday_name"] = (
    features["sent_at"].dt.day_name()
)

features["send_week_of_year"] = (
    features["sent_at"]
    .dt.isocalendar()
    .week
    .astype("int16")
)

features["send_quarter"] = (
    features["sent_at"].dt.quarter.astype("int8")
)


# ============================================================
# 2. WEEKDAY / WEEKEND
# ============================================================

features["is_weekend"] = (
    features["send_weekday"] >= 5
).astype("int8")

features["is_weekday"] = (
    features["send_weekday"] < 5
).astype("int8")


# Explicit weekday flags
weekday_names = {
    0: "monday",
    1: "tuesday",
    2: "wednesday",
    3: "thursday",
    4: "friday",
    5: "saturday",
    6: "sunday",
}

for weekday_num, weekday_name in weekday_names.items():

    features[f"is_{weekday_name}"] = (
        features["send_weekday"] == weekday_num
    ).astype("int8")


# ============================================================
# 3. TIME
# ============================================================

features["send_hour"] = (
    features["sent_at"].dt.hour.astype("int8")
)

features["send_minute"] = (
    features["sent_at"].dt.minute.astype("int8")
)

features["send_minutes_since_midnight"] = (
    features["send_hour"] * 60
    + features["send_minute"]
).astype("int16")


# Fractional hour:
# 13:30 -> 13.5
features["send_hour_decimal"] = (
    features["send_hour"]
    + features["send_minute"] / 60
)


# ============================================================
# 4. TIME-OF-DAY BUCKET
# ============================================================

features["send_time_of_day"] = pd.cut(
    features["send_hour"],
    bins=[
        -1,
        5,
        8,
        11,
        13,
        16,
        19,
        23,
    ],
    labels=[
        "overnight_00_05",
        "early_morning_06_08",
        "morning_09_11",
        "lunch_12_13",
        "afternoon_14_16",
        "late_afternoon_17_19",
        "evening_20_23",
    ],
)


# ============================================================
# 5. BUSINESS-TIME FLAGS
# ============================================================

features["is_business_hours_09_18"] = (
    features["send_hour"]
    .between(9, 18)
).astype("int8")


features["is_morning_09_11"] = (
    features["send_hour"]
    .between(9, 11)
).astype("int8")


features["is_lunch_12_13"] = (
    features["send_hour"]
    .between(12, 13)
).astype("int8")


features["is_afternoon_14_17"] = (
    features["send_hour"]
    .between(14, 17)
).astype("int8")


features["is_evening_18_plus"] = (
    features["send_hour"] >= 18
).astype("int8")


# ============================================================
# 6. CYCLICAL TIME REPRESENTATION
# ============================================================
#
# Useful because:
# Sunday -> Monday
# 23:59 -> 00:00
# should be geometrically close.
# ============================================================

features["send_hour_sin"] = np.sin(
    2 * np.pi
    * features["send_minutes_since_midnight"]
    / 1440
)

features["send_hour_cos"] = np.cos(
    2 * np.pi
    * features["send_minutes_since_midnight"]
    / 1440
)


features["send_weekday_sin"] = np.sin(
    2 * np.pi
    * features["send_weekday"]
    / 7
)

features["send_weekday_cos"] = np.cos(
    2 * np.pi
    * features["send_weekday"]
    / 7
)


# ============================================================
# 7. MONTH POSITION
# ============================================================

features["is_month_start_5d"] = (
    features["send_day"] <= 5
).astype("int8")

features["is_month_start_10d"] = (
    features["send_day"] <= 10
).astype("int8")

features["is_month_end_5d"] = (
    features["sent_at"].dt.days_in_month
    - features["send_day"]
    < 5
).astype("int8")

features["is_month_end_10d"] = (
    features["sent_at"].dt.days_in_month
    - features["send_day"]
    < 10
).astype("int8")


features["days_from_month_start"] = (
    features["send_day"] - 1
).astype("int8")


features["days_to_month_end"] = (
    features["sent_at"].dt.days_in_month
    - features["send_day"]
).astype("int8")


features["month_progress"] = (
    features["send_day"]
    / features["sent_at"].dt.days_in_month
)


# ============================================================
# 8. AUDIT
# ============================================================

new_cols = features.columns[cols_before:]

print("=" * 100)
print("01A — CALENDAR & TIME COMPLETE")
print("=" * 100)

print(f"Rows          : {len(features):,}")
print(f"Customers     : {features['customer_id'].nunique():,}")
print(f"Columns before: {cols_before:,}")
print(f"Columns after : {features.shape[1]:,}")
print(f"New features  : {len(new_cols):,}")

print("\nNew columns:")
for col in new_cols:
    print("  -", col)


# ============================================================
# 9. HARD AUDITS
# ============================================================

assert len(features) == len(events)

assert features["event_id"].is_unique

assert features["sent_at"].notna().all()

assert features["send_weekday"].between(0, 6).all()

assert features["send_hour"].between(0, 23).all()

assert features["send_minute"].between(0, 59).all()

assert features["month_progress"].between(0, 1).all()


print("\nCalendar/time sanity checks: PASSED")


# ============================================================
# 10. DISTRIBUTIONS
# ============================================================

print("\nSEND WEEKDAY")
display(
    features["send_weekday_name"]
    .value_counts()
    .to_frame("messages")
)


print("\nTIME OF DAY")
display(
    features["send_time_of_day"]
    .value_counts(sort=False)
    .to_frame("messages")
)


print("\nWEEKEND")
display(
    features["is_weekend"]
    .value_counts()
    .sort_index()
    .to_frame("messages")
)

01A — CALENDAR & TIME COMPLETE
Rows          : 75,406
Customers     : 11,724
Columns before: 22
Columns after : 60
New features  : 38

New columns:
  - send_year
  - send_month
  - send_day
  - send_day_of_year
  - send_weekday
  - send_weekday_name
  - send_week_of_year
  - send_quarter
  - is_weekend
  - is_weekday
  - is_monday
  - is_tuesday
  - is_wednesday
  - is_thursday
  - is_friday
  - is_saturday
  - is_sunday
  - send_hour
  - send_minute
  - send_minutes_since_midnight
  - send_hour_decimal
  - send_time_of_day
  - is_business_hours_09_18
  - is_morning_09_11
  - is_lunch_12_13
  - is_afternoon_14_17
  - is_evening_18_plus
  - send_hour_sin
  - send_hour_cos
  - send_weekday_sin
  - send_weekday_cos
  - is_month_start_5d
  - is_month_start_10d
  - is_month_end_5d
  - is_month_end_10d
  - days_from_month_start
  - days_to_month_end
  - month_progress

Calendar/time sanity checks: PASSED

SEND WEEKDAY


,messages
send_weekday_name,
Monday,13789
Thursday,12835
Friday,12795
Tuesday,12724
Wednesday,12635
Saturday,6575
Sunday,4053



TIME OF DAY


,messages
send_time_of_day,
overnight_00_05,0
early_morning_06_08,0
morning_09_11,31011
lunch_12_13,13468
afternoon_14_16,15371
late_afternoon_17_19,12408
evening_20_23,3148



WEEKEND


,messages
is_weekend,
0,64778
1,10628


In [8]:
# ============================================================
# 01B — CURRENT STATE: PAYDAY
# ============================================================
#
# PURPOSE
# -------
# Represent the customer's position relative to payday
# at the exact moment of the current WhatsApp send.
#
# POINT-IN-TIME
# -------------
# Uses only:
#   - sent_at
#   - payday_day_of_month
#
# No historical outcomes.
# No future customer behavior.
#
# IMPORTANT
# ---------
# "Next payday" is future CALENDAR information, not future
# behavioral information, so it is valid at decision time.
#
# Handles month boundaries and months where the configured
# payday exceeds the number of calendar days.
# Example: payday_day_of_month = 31 in February -> Feb 28.
# ============================================================


cols_before = features.shape[1]


# ============================================================
# 1. BASIC PAYDAY VALIDATION
# ============================================================

payday = pd.to_numeric(
    features["payday_day_of_month"],
    errors="coerce"
)

assert payday.notna().all()

assert payday.between(1, 31).all()

features["payday_day"] = (
    payday.astype("int8")
)


# ============================================================
# 2. SAME-MONTH PAYDAY DATE
# ============================================================
#
# If payday = 31 but month has only 30 days,
# use the final calendar day of that month.
# ============================================================

send_month_start = (
    features["sent_at"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

days_in_current_month = (
    features["sent_at"]
    .dt.days_in_month
)

effective_payday_current_month = np.minimum(
    features["payday_day"].to_numpy(),
    days_in_current_month.to_numpy()
)

features["effective_payday_day_current_month"] = (
    effective_payday_current_month
    .astype("int8")
)

current_month_payday = (
    send_month_start
    + pd.to_timedelta(
        effective_payday_current_month - 1,
        unit="D"
    )
)


# ============================================================
# 3. PREVIOUS MONTH PAYDAY
# ============================================================

previous_month_start = (
    send_month_start
    - pd.offsets.MonthBegin(1)
)

previous_month_end = (
    send_month_start
    - pd.Timedelta(days=1)
)

days_in_previous_month = (
    previous_month_end.dt.day
)

effective_payday_previous_month = np.minimum(
    features["payday_day"].to_numpy(),
    days_in_previous_month.to_numpy()
)

previous_month_payday = (
    previous_month_start
    + pd.to_timedelta(
        effective_payday_previous_month - 1,
        unit="D"
    )
)


# ============================================================
# 4. NEXT MONTH PAYDAY
# ============================================================

next_month_start = (
    send_month_start
    + pd.offsets.MonthBegin(1)
)

next_month_end = (
    next_month_start
    + pd.offsets.MonthEnd(0)
)

days_in_next_month = (
    next_month_end.dt.day
)

effective_payday_next_month = np.minimum(
    features["payday_day"].to_numpy(),
    days_in_next_month.to_numpy()
)

next_month_payday = (
    next_month_start
    + pd.to_timedelta(
        effective_payday_next_month - 1,
        unit="D"
    )
)


# ============================================================
# 5. PREVIOUS / NEXT PAYDAY RELATIVE TO SEND DATE
# ============================================================
#
# Compare calendar dates, not timestamp hours.
#
# If send_date == payday:
#   previous payday = today
#   next payday     = today
#
# Therefore:
#   days_since_previous_payday = 0
#   days_until_next_payday     = 0
# ============================================================

send_date = features["sent_date"]

current_payday_has_arrived = (
    current_month_payday <= send_date
)

current_payday_not_passed = (
    current_month_payday >= send_date
)


previous_payday_date = (
    current_month_payday
    .where(
        current_payday_has_arrived,
        previous_month_payday
    )
)

next_payday_date = (
    current_month_payday
    .where(
        current_payday_not_passed,
        next_month_payday
    )
)


features["previous_payday_date"] = (
    previous_payday_date
)

features["next_payday_date"] = (
    next_payday_date
)


# ============================================================
# 6. DISTANCES
# ============================================================

features["days_since_previous_payday"] = (
    (
        send_date
        - features["previous_payday_date"]
    )
    .dt.days
    .astype("int8")
)

features["days_until_next_payday"] = (
    (
        features["next_payday_date"]
        - send_date
    )
    .dt.days
    .astype("int8")
)


# ============================================================
# 7. SAME-MONTH SIGNED POSITION
# ============================================================
#
# Negative = before this month's payday
# Zero     = payday
# Positive = after this month's payday
#
# This is useful for interpretation, while the previous/next
# distances above are chronologically correct across months.
# ============================================================

features["days_from_current_month_payday"] = (
    (
        send_date
        - current_month_payday
    )
    .dt.days
    .astype("int8")
)


# ============================================================
# 8. NEAREST PAYDAY
# ============================================================

previous_distance = (
    features["days_since_previous_payday"]
)

next_distance = (
    features["days_until_next_payday"]
)

features["days_to_nearest_payday"] = np.minimum(
    previous_distance,
    next_distance
).astype("int8")


features["nearest_payday_direction"] = np.select(
    [
        previous_distance.eq(0),
        previous_distance < next_distance,
        next_distance < previous_distance,
    ],
    [
        "payday_today",
        "previous_payday",
        "next_payday",
    ],
    default="equidistant",
)


# ============================================================
# 9. BEFORE / AFTER / ON PAYDAY
# ============================================================

features["is_payday"] = (
    features["days_to_nearest_payday"] == 0
).astype("int8")


features["is_before_current_month_payday"] = (
    features["days_from_current_month_payday"] < 0
).astype("int8")


features["is_after_current_month_payday"] = (
    features["days_from_current_month_payday"] > 0
).astype("int8")


# ============================================================
# 10. PAYDAY WINDOWS
# ============================================================
#
# Symmetric proximity.
# ============================================================

for window in [1, 2, 3, 5, 7]:

    features[
        f"is_within_{window}d_nearest_payday"
    ] = (
        features["days_to_nearest_payday"]
        <= window
    ).astype("int8")


# ============================================================
# 11. POST-PAYDAY WINDOWS
# ============================================================
#
# Example:
# 0-3 days after previous payday.
# Includes payday itself.
# ============================================================

for window in [1, 2, 3, 5, 7]:

    features[
        f"is_within_{window}d_after_payday"
    ] = (
        features["days_since_previous_payday"]
        <= window
    ).astype("int8")


# ============================================================
# 12. PRE-PAYDAY WINDOWS
# ============================================================

for window in [1, 2, 3, 5, 7]:

    features[
        f"is_within_{window}d_before_payday"
    ] = (
        features["days_until_next_payday"]
        <= window
    ).astype("int8")


# ============================================================
# 13. PAYDAY PHASE
# ============================================================

features["payday_phase"] = np.select(
    [
        features["is_payday"].eq(1),

        features["days_since_previous_payday"]
        .between(1, 3),

        features["days_since_previous_payday"]
        .between(4, 7),

        features["days_until_next_payday"]
        .between(1, 3),

        features["days_until_next_payday"]
        .between(4, 7),
    ],
    [
        "payday",
        "post_payday_1_3d",
        "post_payday_4_7d",
        "pre_payday_1_3d",
        "pre_payday_4_7d",
    ],
    default="mid_cycle",
)


# ============================================================
# 14. CYCLICAL PAYDAY POSITION
# ============================================================
#
# Position inside the approximate monthly salary cycle.
# ============================================================

features["payday_cycle_sin"] = np.sin(
    2 * np.pi
    * features["days_since_previous_payday"]
    / 31
)

features["payday_cycle_cos"] = np.cos(
    2 * np.pi
    * features["days_since_previous_payday"]
    / 31
)


# ============================================================
# 15. AUDIT
# ============================================================

new_cols = features.columns[cols_before:]

print("=" * 100)
print("01B — PAYDAY STATE COMPLETE")
print("=" * 100)

print(f"Rows          : {len(features):,}")
print(f"Customers     : {features['customer_id'].nunique():,}")
print(f"Columns before: {cols_before:,}")
print(f"Columns after : {features.shape[1]:,}")
print(f"New features  : {len(new_cols):,}")


# ============================================================
# 16. HARD SANITY CHECKS
# ============================================================

assert len(features) == len(events)

assert features["event_id"].is_unique

assert (
    features["days_since_previous_payday"]
    >= 0
).all()

assert (
    features["days_until_next_payday"]
    >= 0
).all()

assert (
    features["days_to_nearest_payday"]
    >= 0
).all()

assert (
    features["days_to_nearest_payday"]
    ==
    features[
        [
            "days_since_previous_payday",
            "days_until_next_payday",
        ]
    ].min(axis=1)
).all()


print("\nPayday temporal sanity checks: PASSED")


# ============================================================
# 17. DISTRIBUTIONS
# ============================================================

print("\nPAYDAY DAY")
display(
    features["payday_day"]
    .value_counts()
    .sort_index()
    .to_frame("messages")
)


print("\nPAYDAY PHASE")
display(
    features["payday_phase"]
    .value_counts()
    .to_frame("messages")
)


print("\nDISTANCE TO NEAREST PAYDAY")
display(
    features[
        "days_to_nearest_payday"
    ]
    .describe(
        percentiles=[
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame()
)


# ============================================================
# 18. SAMPLE AROUND PAYDAY
# ============================================================

sample_payday_cols = [
    "customer_id",
    "sent_at",
    "payday_day",
    "previous_payday_date",
    "next_payday_date",
    "days_since_previous_payday",
    "days_until_next_payday",
    "days_to_nearest_payday",
    "nearest_payday_direction",
    "payday_phase",
]

print("\nSAMPLE — EVENTS CLOSE TO PAYDAY")

display(
    features.loc[
        features[
            "days_to_nearest_payday"
        ] <= 2,
        sample_payday_cols
    ].head(20)
)


print("\n" + "=" * 100)
print("01B — PAYDAY STATE COMPLETE")
print("=" * 100)

01B — PAYDAY STATE COMPLETE
Rows          : 75,406
Customers     : 11,724
Columns before: 60
Columns after : 90
New features  : 30

Payday temporal sanity checks: PASSED

PAYDAY DAY


,messages
payday_day,
1,6136
5,27095
10,8943
15,7442
20,11260
25,7062
30,7468



PAYDAY PHASE


,messages
payday_phase,
mid_cycle,39367
pre_payday_4_7d,10030
post_payday_4_7d,9636
pre_payday_1_3d,7019
post_payday_1_3d,6982
payday,2372



DISTANCE TO NEAREST PAYDAY


,days_to_nearest_payday
count,"75,406.00"
mean,7.76
std,4.42
min,0.00
25%,4.00
50%,8.00
75%,12.00
90%,14.00
95%,15.00
99%,15.00



SAMPLE — EVENTS CLOSE TO PAYDAY


,customer_id,sent_at,payday_day,previous_payday_date,next_payday_date,days_since_previous_payday,days_until_next_payday,days_to_nearest_payday,nearest_payday_direction,payday_phase
0,C000001,2026-07-06 13:29:00,5,2026-07-05,2026-08-05,1,30,1,previous_payday,post_payday_1_3d
1,C000001,2026-07-07 14:48:00,5,2026-07-05,2026-08-05,2,29,2,previous_payday,post_payday_1_3d
25,C000004,2026-07-06 13:42:00,5,2026-07-05,2026-08-05,1,30,1,previous_payday,post_payday_1_3d
28,C000005,2026-08-05 14:36:00,5,2026-08-05,2026-08-05,0,0,0,payday_today,payday
37,C000006,2026-08-03 09:14:00,5,2026-07-05,2026-08-05,29,2,2,next_payday,pre_payday_1_3d
38,C000006,2026-08-04 19:26:00,5,2026-07-05,2026-08-05,30,1,1,next_payday,pre_payday_1_3d
39,C000006,2026-08-07 17:46:00,5,2026-08-05,2026-09-05,2,29,2,previous_payday,post_payday_1_3d
58,C000008,2026-07-03 10:04:00,5,2026-06-05,2026-07-05,28,2,2,next_payday,pre_payday_1_3d
59,C000008,2026-07-05 12:25:00,5,2026-07-05,2026-07-05,0,0,0,payday_today,payday
72,C000009,2026-08-05 09:48:00,5,2026-08-05,2026-08-05,0,0,0,payday_today,payday



01B — PAYDAY STATE COMPLETE


In [9]:
# ============================================================
# 01C — CURRENT STATE: DPD / COLLECTIONS STATE
# ============================================================
#
# GRAIN
# -----
# 1 row = 1 WhatsApp send event
#
# POINT-IN-TIME
# -------------
# Uses current days_past_due only.
#
# No historical outcomes.
# No future information.
# ============================================================

cols_before = features.shape[1]


# ============================================================
# 1. VALIDATE CURRENT DPD
# ============================================================

dpd = pd.to_numeric(
    features["days_past_due"],
    errors="coerce"
)

assert dpd.notna().all()
assert (dpd >= 0).all()

features["current_dpd"] = (
    dpd.astype("int16")
)


# ============================================================
# 2. BASIC DPD TRANSFORMATIONS
# ============================================================

features["current_dpd_log1p"] = np.log1p(
    features["current_dpd"]
)

features["current_dpd_sqrt"] = np.sqrt(
    features["current_dpd"]
)


# ============================================================
# 3. BUSINESS DPD BUCKETS
# ============================================================
#
# Keep both continuous DPD and interpretable collection stages.
# ============================================================

features["dpd_bucket"] = pd.cut(
    features["current_dpd"],
    bins=[
        -1,
        7,
        15,
        30,
        45,
        59,
        89,
        119,
        np.inf,
    ],
    labels=[
        "dpd_0_7",
        "dpd_8_15",
        "dpd_16_30",
        "dpd_31_45",
        "dpd_46_59",
        "dpd_60_89",
        "dpd_90_119",
        "dpd_120_plus",
    ],
)


# ============================================================
# 4. COLLECTIONS STAGE
# ============================================================

features["collections_stage"] = pd.cut(
    features["current_dpd"],
    bins=[
        -1,
        15,
        30,
        59,
        np.inf,
    ],
    labels=[
        "early_0_15",
        "mid_16_30",
        "late_31_59",
        "dpd_60_plus",
    ],
)


# ============================================================
# 5. DPD THRESHOLD FLAGS
# ============================================================

for threshold in [
    7,
    15,
    30,
    45,
    60,
    90,
    120,
]:

    features[
        f"is_dpd_{threshold}_plus"
    ] = (
        features["current_dpd"]
        >= threshold
    ).astype("int8")


# ============================================================
# 6. PROXIMITY TO IMPORTANT THRESHOLDS
# ============================================================
#
# Particularly useful operationally:
# customer approaching DPD30 / DPD60 / DPD90.
#
# These are based exclusively on current DPD.
# ============================================================

for threshold in [30, 60, 90]:

    features[
        f"days_until_dpd_{threshold}"
    ] = np.maximum(
        threshold
        - features["current_dpd"],
        0
    ).astype("int16")

    features[
        f"days_since_dpd_{threshold}"
    ] = np.maximum(
        features["current_dpd"]
        - threshold,
        0
    ).astype("int16")

    features[
        f"is_within_3d_before_dpd_{threshold}"
    ] = (
        features["current_dpd"]
        .between(
            threshold - 3,
            threshold - 1
        )
    ).astype("int8")

    features[
        f"is_within_7d_before_dpd_{threshold}"
    ] = (
        features["current_dpd"]
        .between(
            threshold - 7,
            threshold - 1
        )
    ).astype("int8")


# ============================================================
# 7. IMPLIED COLLECTIONS ENTRY DATE
# ============================================================
#
# Data dictionary:
# days_past_due = days since customer entered collections
# 1 = first day.
#
# Therefore if:
#
#   sent_date = Jul 10
#   DPD       = 4
#
# implied entry date = Jul 7
#
# because Jul 7 corresponds to day 1.
#
# This is reconstructed from CURRENT information only.
# ============================================================

features["implied_collections_entry_date"] = (
    features["sent_date"]
    - pd.to_timedelta(
        features["current_dpd"] - 1,
        unit="D"
    )
)


# ============================================================
# 8. DPD POSITION INSIDE MONTH
# ============================================================
#
# Useful interactions with calendar/payday later.
# ============================================================

features["dpd_week"] = (
    ((features["current_dpd"] - 1) // 7) + 1
).clip(lower=1).astype("int16")


features["dpd_30d_cycle"] = (
    (features["current_dpd"] - 1) % 30
).astype("int8")


# ============================================================
# 9. AUDIT DPD CONSISTENCY WITHIN CUSTOMER
# ============================================================
#
# IMPORTANT:
# This is AUDIT ONLY.
#
# We are NOT yet creating historical features from previous
# messages.
#
# If DPD really means "days since entering collections",
# implied entry date should be stable within customer.
# ============================================================

dpd_audit = pd.DataFrame(
    {
        "customer_id":
            features["customer_id"],

        "implied_entry":
            features[
                "implied_collections_entry_date"
            ],
    }
)

entry_nunique = (
    dpd_audit
    .groupby(
        "customer_id",
        sort=False
    )["implied_entry"]
    .nunique()
)

customers_stable_entry = (
    entry_nunique.eq(1).sum()
)

customers_inconsistent_entry = (
    entry_nunique.gt(1).sum()
)


# Range of reconstructed entry dates per customer
entry_min = (
    dpd_audit
    .groupby(
        "customer_id",
        sort=False
    )["implied_entry"]
    .min()
)

entry_max = (
    dpd_audit
    .groupby(
        "customer_id",
        sort=False
    )["implied_entry"]
    .max()
)

entry_range_days = (
    entry_max - entry_min
).dt.days


# ============================================================
# 10. AUDIT
# ============================================================

new_cols = features.columns[cols_before:]

print("=" * 100)
print("01C — DPD / COLLECTIONS STATE COMPLETE")
print("=" * 100)

print(f"Rows          : {len(features):,}")
print(
    f"Customers     : "
    f"{features['customer_id'].nunique():,}"
)
print(f"Columns before: {cols_before:,}")
print(f"Columns after : {features.shape[1]:,}")
print(f"New features  : {len(new_cols):,}")


# ============================================================
# 11. HARD CHECKS
# ============================================================

assert len(features) == len(events)

assert features["event_id"].is_unique

assert features["current_dpd"].notna().all()

assert (
    features["current_dpd"] >= 0
).all()

assert (
    features["implied_collections_entry_date"]
    <= features["sent_date"]
).all()


print("\nDPD basic sanity checks: PASSED")


# ============================================================
# 12. DPD DISTRIBUTION
# ============================================================

print("\nCURRENT DPD DISTRIBUTION")

display(
    features["current_dpd"]
    .describe(
        percentiles=[
            .01,
            .05,
            .10,
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame()
)


print("\nDPD BUCKET")

display(
    features["dpd_bucket"]
    .value_counts(sort=False)
    .to_frame("messages")
)


print("\nCOLLECTIONS STAGE")

display(
    features["collections_stage"]
    .value_counts(sort=False)
    .to_frame("messages")
)


# ============================================================
# 13. IMPLIED ENTRY DATE CONSISTENCY
# ============================================================

print("\nIMPLIED COLLECTIONS ENTRY DATE AUDIT")
print("-" * 100)

print(
    f"Customers with stable implied entry date : "
    f"{customers_stable_entry:,}"
)

print(
    f"Customers with >1 implied entry date     : "
    f"{customers_inconsistent_entry:,}"
)

print(
    f"Share stable                             : "
    f"{customers_stable_entry / len(entry_nunique) * 100:.2f}%"
)


print("\nENTRY DATE RANGE WITHIN CUSTOMER")

display(
    entry_range_days
    .describe(
        percentiles=[
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame("entry_date_range_days")
)


print("\n" + "=" * 100)
print("01C — DPD / COLLECTIONS STATE COMPLETE")
print("=" * 100)

01C — DPD / COLLECTIONS STATE COMPLETE
Rows          : 75,406
Customers     : 11,724
Columns before: 90
Columns after : 117
New features  : 27

DPD basic sanity checks: PASSED

CURRENT DPD DISTRIBUTION


,current_dpd
count,"75,406.00"
mean,17.03
std,14.36
min,1.00
1%,1.00
5%,1.00
10%,3.00
25%,6.00
50%,12.00
75%,25.00



DPD BUCKET


,messages
dpd_bucket,
dpd_0_7,23724
dpd_8_15,20629
dpd_16_30,18616
dpd_31_45,7474
dpd_46_59,4721
dpd_60_89,242
dpd_90_119,0
dpd_120_plus,0



COLLECTIONS STAGE


,messages
collections_stage,
early_0_15,44353
mid_16_30,18616
late_31_59,12195
dpd_60_plus,242



IMPLIED COLLECTIONS ENTRY DATE AUDIT
----------------------------------------------------------------------------------------------------
Customers with stable implied entry date : 11,724
Customers with >1 implied entry date     : 0
Share stable                             : 100.00%

ENTRY DATE RANGE WITHIN CUSTOMER


,entry_date_range_days
count,"11,724.00"
mean,0.00
std,0.00
min,0.00
50%,0.00
75%,0.00
90%,0.00
95%,0.00
99%,0.00
max,0.00



01C — DPD / COLLECTIONS STATE COMPLETE


In [10]:
# ============================================================
# 01D — CURRENT STATE: BALANCE / FINANCIAL EXPOSURE
# ============================================================
#
# Uses only outstanding_balance_brl known at current send.
#
# No payment outcomes.
# No historical balance trajectory.
# No future information.
# ============================================================

cols_before = features.shape[1]


# ============================================================
# 1. CURRENT BALANCE
# ============================================================

balance = pd.to_numeric(
    features["outstanding_balance_brl"],
    errors="coerce"
)

assert balance.notna().all()

features["current_balance_brl"] = balance


# ============================================================
# 2. BASIC VALIDATION
# ============================================================

features["is_zero_balance"] = (
    features["current_balance_brl"] == 0
).astype("int8")

features["is_positive_balance"] = (
    features["current_balance_brl"] > 0
).astype("int8")

features["is_negative_balance"] = (
    features["current_balance_brl"] < 0
).astype("int8")


# ============================================================
# 3. TRANSFORMATIONS
# ============================================================

features["current_balance_log1p"] = np.where(
    features["current_balance_brl"] >= 0,
    np.log1p(features["current_balance_brl"]),
    np.nan
)

features["current_balance_sqrt"] = np.where(
    features["current_balance_brl"] >= 0,
    np.sqrt(features["current_balance_brl"]),
    np.nan
)


# ============================================================
# 4. BUSINESS BALANCE BUCKETS
# ============================================================
#
# Fixed monetary bands preserve business interpretation.
# ============================================================

features["balance_bucket"] = pd.cut(
    features["current_balance_brl"],
    bins=[
        -np.inf,
        250,
        500,
        750,
        1_000,
        1_500,
        2_000,
        3_000,
        5_000,
        np.inf,
    ],
    labels=[
        "balance_le_250",
        "balance_250_500",
        "balance_500_750",
        "balance_750_1000",
        "balance_1000_1500",
        "balance_1500_2000",
        "balance_2000_3000",
        "balance_3000_5000",
        "balance_5000_plus",
    ],
    include_lowest=True,
)


# ============================================================
# 5. DATA-DRIVEN BALANCE QUANTILE
# ============================================================
#
# Descriptive representation.
# Later, if used in model validation/OOT, quantile cut points
# should be learned on TRAIN only.
# ============================================================

features["balance_decile_descriptive"] = pd.qcut(
    features["current_balance_brl"],
    q=10,
    labels=False,
    duplicates="drop",
)


# ============================================================
# 6. ROUND-NUMBER THRESHOLDS
# ============================================================

for threshold in [
    250,
    500,
    750,
    1_000,
    1_500,
    2_000,
    3_000,
    5_000,
]:

    features[
        f"is_balance_{threshold}_plus"
    ] = (
        features["current_balance_brl"]
        >= threshold
    ).astype("int8")


# ============================================================
# 7. DISCOUNT SETTLEMENT EXPOSURE
# ============================================================
#
# Business rule:
# discount_offer settles debt at 85% of balance.
#
# This is NOT an outcome.
# It is the amount required to settle the CURRENT balance
# if the discount action were applied.
# ============================================================

DISCOUNT_SETTLEMENT_RATE = 0.85

features["discount_settlement_amount_brl"] = (
    features["current_balance_brl"]
    * DISCOUNT_SETTLEMENT_RATE
)

features["discount_amount_brl"] = (
    features["current_balance_brl"]
    - features["discount_settlement_amount_brl"]
)


# ============================================================
# 8. BALANCE PER DPD DAY
# ============================================================
#
# Interaction between exposure and current collections stage.
# Pure current-state feature.
# ============================================================

features["balance_per_dpd_day"] = (
    features["current_balance_brl"]
    / features["current_dpd"].clip(lower=1)
)


features["log_balance_x_log_dpd"] = (
    features["current_balance_log1p"]
    * features["current_dpd_log1p"]
)


# ============================================================
# 9. AUDIT
# ============================================================

new_cols = features.columns[cols_before:]

print("=" * 100)
print("01D — BALANCE / FINANCIAL EXPOSURE COMPLETE")
print("=" * 100)

print(f"Rows          : {len(features):,}")
print(
    f"Customers     : "
    f"{features['customer_id'].nunique():,}"
)
print(f"Columns before: {cols_before:,}")
print(f"Columns after : {features.shape[1]:,}")
print(f"New features  : {len(new_cols):,}")


# ============================================================
# 10. HARD CHECKS
# ============================================================

assert len(features) == len(events)
assert features["event_id"].is_unique

assert (
    features["current_balance_brl"]
    .notna()
    .all()
)

assert (
    features["discount_settlement_amount_brl"]
    <= features["current_balance_brl"]
).all()


print("\nBalance basic sanity checks: PASSED")


# ============================================================
# 11. DISTRIBUTION
# ============================================================

print("\nCURRENT BALANCE DISTRIBUTION")

display(
    features["current_balance_brl"]
    .describe(
        percentiles=[
            .01,
            .05,
            .10,
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame()
)


print("\nBALANCE BUCKET")

display(
    features["balance_bucket"]
    .value_counts(sort=False)
    .to_frame("messages")
)


# ============================================================
# 12. BALANCE VALIDITY
# ============================================================

print("\nBALANCE VALIDITY")
print("-" * 100)

print(
    f"Negative balance : "
    f"{features['is_negative_balance'].sum():,}"
)

print(
    f"Zero balance     : "
    f"{features['is_zero_balance'].sum():,}"
)

print(
    f"Positive balance : "
    f"{features['is_positive_balance'].sum():,}"
)


# ============================================================
# 13. DISCOUNT SETTLEMENT
# ============================================================

print("\nCURRENT DISCOUNT SETTLEMENT EXPOSURE")

display(
    features[
        "discount_settlement_amount_brl"
    ]
    .describe(
        percentiles=[
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame()
)


print("\n" + "=" * 100)
print("01D — BALANCE / FINANCIAL EXPOSURE COMPLETE")
print("=" * 100)

01D — BALANCE / FINANCIAL EXPOSURE COMPLETE
Rows          : 75,406
Customers     : 11,724
Columns before: 117
Columns after : 137
New features  : 20

Balance basic sanity checks: PASSED

CURRENT BALANCE DISTRIBUTION


,current_balance_brl
count,"75,406.00"
mean,823.40
std,486.64
min,51.61
1%,129.22
5%,250.29
10%,250.97
25%,434.21
50%,725.10
75%,"1,107.40"



BALANCE BUCKET


,messages
balance_bucket,
balance_le_250,2337
balance_250_500,20789
balance_500_750,16014
balance_750_1000,13080
balance_1000_1500,14716
balance_1500_2000,8470
balance_2000_3000,0
balance_3000_5000,0
balance_5000_plus,0



BALANCE VALIDITY
----------------------------------------------------------------------------------------------------
Negative balance : 0
Zero balance     : 0
Positive balance : 75,406

CURRENT DISCOUNT SETTLEMENT EXPOSURE


,discount_settlement_amount_brl
count,"75,406.00"
mean,699.89
std,413.64
min,43.87
25%,369.08
50%,616.34
75%,941.29
90%,"1,326.05"
95%,"1,614.95"
99%,"1,700.00"



01D — BALANCE / FINANCIAL EXPOSURE COMPLETE


In [11]:
# ============================================================
# 01E — CURRENT STATE: SALARY & AFFORDABILITY
# ============================================================
#
# PURPOSE
# -------
# Represent current debt burden relative to monthly salary.
#
# POINT-IN-TIME
# -------------
# Uses only information available at the current send:
#
#   - current_balance_brl
#   - monthly_salary_brl
#   - discount settlement amount
#
# No outcomes.
# No historical payment behavior.
# ============================================================

cols_before = features.shape[1]


# ============================================================
# 1. CURRENT SALARY
# ============================================================

salary = pd.to_numeric(
    features["monthly_salary_brl"],
    errors="coerce"
)

assert salary.notna().all()

features["current_monthly_salary_brl"] = salary


# ============================================================
# 2. SALARY VALIDITY
# ============================================================

features["is_zero_salary"] = (
    features["current_monthly_salary_brl"] == 0
).astype("int8")

features["is_positive_salary"] = (
    features["current_monthly_salary_brl"] > 0
).astype("int8")

features["is_negative_salary"] = (
    features["current_monthly_salary_brl"] < 0
).astype("int8")


# ============================================================
# 3. SALARY TRANSFORMATIONS
# ============================================================

features["salary_log1p"] = np.where(
    features["current_monthly_salary_brl"] >= 0,
    np.log1p(
        features["current_monthly_salary_brl"]
    ),
    np.nan
)

features["salary_sqrt"] = np.where(
    features["current_monthly_salary_brl"] >= 0,
    np.sqrt(
        features["current_monthly_salary_brl"]
    ),
    np.nan
)


# ============================================================
# 4. SALARY BUSINESS BUCKET
# ============================================================

features["salary_bucket"] = pd.cut(
    features["current_monthly_salary_brl"],
    bins=[
        -np.inf,
        1_500,
        2_000,
        3_000,
        4_000,
        5_000,
        7_500,
        10_000,
        15_000,
        np.inf,
    ],
    labels=[
        "salary_le_1500",
        "salary_1500_2000",
        "salary_2000_3000",
        "salary_3000_4000",
        "salary_4000_5000",
        "salary_5000_7500",
        "salary_7500_10000",
        "salary_10000_15000",
        "salary_15000_plus",
    ],
    include_lowest=True,
)


# ============================================================
# 5. DESCRIPTIVE SALARY DECILE
# ============================================================
#
# Descriptive only at this stage.
#
# For actual modelling, cut points must later be fitted
# using TRAIN only.
# ============================================================

features["salary_decile_descriptive"] = pd.qcut(
    features["current_monthly_salary_brl"],
    q=10,
    labels=False,
    duplicates="drop",
)


# ============================================================
# 6. DEBT / INCOME BURDEN
# ============================================================
#
# Example:
#
# balance = 500
# salary  = 2,000
#
# debt_to_salary_ratio = 0.25
#
# The outstanding debt corresponds to 25% of one
# monthly salary.
# ============================================================

safe_salary = (
    features["current_monthly_salary_brl"]
    .replace(0, np.nan)
)

features["debt_to_salary_ratio"] = (
    features["current_balance_brl"]
    / safe_salary
)


features["discount_debt_to_salary_ratio"] = (
    features["discount_settlement_amount_brl"]
    / safe_salary
)


# ============================================================
# 7. SALARY / DEBT COVERAGE
# ============================================================
#
# Inverse representation:
#
# salary / balance
#
# Higher values = salary is large relative to debt.
# ============================================================

safe_balance = (
    features["current_balance_brl"]
    .replace(0, np.nan)
)

features["salary_to_debt_ratio"] = (
    features["current_monthly_salary_brl"]
    / safe_balance
)


# ============================================================
# 8. DEBT AS % OF SALARY
# ============================================================

features["debt_pct_of_salary"] = (
    features["debt_to_salary_ratio"]
    * 100
)

features["discount_debt_pct_of_salary"] = (
    features["discount_debt_to_salary_ratio"]
    * 100
)


# ============================================================
# 9. MONTHLY SALARY REMAINING AFTER FULL SETTLEMENT
# ============================================================
#
# This is NOT claiming disposable income.
#
# It is simply:
#
# monthly salary - current debt
#
# Useful as a relative affordability representation.
# ============================================================

features["salary_minus_debt_brl"] = (
    features["current_monthly_salary_brl"]
    - features["current_balance_brl"]
)

features["salary_minus_discount_settlement_brl"] = (
    features["current_monthly_salary_brl"]
    - features["discount_settlement_amount_brl"]
)


# ============================================================
# 10. AFFORDABILITY FLAGS
# ============================================================

features["debt_le_10pct_salary"] = (
    features["debt_to_salary_ratio"] <= 0.10
).astype("int8")

features["debt_le_25pct_salary"] = (
    features["debt_to_salary_ratio"] <= 0.25
).astype("int8")

features["debt_le_50pct_salary"] = (
    features["debt_to_salary_ratio"] <= 0.50
).astype("int8")

features["debt_le_100pct_salary"] = (
    features["debt_to_salary_ratio"] <= 1.00
).astype("int8")


features["debt_gt_50pct_salary"] = (
    features["debt_to_salary_ratio"] > 0.50
).astype("int8")

features["debt_gt_100pct_salary"] = (
    features["debt_to_salary_ratio"] > 1.00
).astype("int8")


# ============================================================
# 11. DISCOUNT AFFORDABILITY FLAGS
# ============================================================

features["discount_debt_le_10pct_salary"] = (
    features["discount_debt_to_salary_ratio"]
    <= 0.10
).astype("int8")

features["discount_debt_le_25pct_salary"] = (
    features["discount_debt_to_salary_ratio"]
    <= 0.25
).astype("int8")

features["discount_debt_le_50pct_salary"] = (
    features["discount_debt_to_salary_ratio"]
    <= 0.50
).astype("int8")


# ============================================================
# 12. AFFORDABILITY BUCKET
# ============================================================

features["debt_to_salary_bucket"] = pd.cut(
    features["debt_to_salary_ratio"],
    bins=[
        -np.inf,
        0.10,
        0.25,
        0.50,
        0.75,
        1.00,
        np.inf,
    ],
    labels=[
        "debt_le_10pct_salary",
        "debt_10_25pct_salary",
        "debt_25_50pct_salary",
        "debt_50_75pct_salary",
        "debt_75_100pct_salary",
        "debt_gt_100pct_salary",
    ],
    include_lowest=True,
)


# ============================================================
# 13. DISCOUNT AFFORDABILITY IMPROVEMENT
# ============================================================
#
# Absolute and relative burden reduction generated by the
# documented 15% discount.
# ============================================================

features["discount_burden_reduction_ratio"] = (
    features["debt_to_salary_ratio"]
    - features["discount_debt_to_salary_ratio"]
)

features["discount_burden_reduction_pct_salary"] = (
    features["discount_burden_reduction_ratio"]
    * 100
)


# ============================================================
# 14. BALANCE × SALARY INTERACTIONS
# ============================================================

features["log_balance_minus_log_salary"] = (
    features["current_balance_log1p"]
    - features["salary_log1p"]
)

features["log_balance_x_log_salary"] = (
    features["current_balance_log1p"]
    * features["salary_log1p"]
)


# ============================================================
# 15. DPD × AFFORDABILITY
# ============================================================
#
# Same debt burden can mean something different at DPD 5
# versus DPD 55.
# ============================================================

features["dpd_x_debt_to_salary"] = (
    features["current_dpd"]
    * features["debt_to_salary_ratio"]
)

features["log_dpd_x_debt_to_salary"] = (
    features["current_dpd_log1p"]
    * features["debt_to_salary_ratio"]
)


# ============================================================
# 16. AUDIT
# ============================================================

new_cols = features.columns[cols_before:]

print("=" * 100)
print("01E — SALARY & AFFORDABILITY COMPLETE")
print("=" * 100)

print(f"Rows          : {len(features):,}")
print(
    f"Customers     : "
    f"{features['customer_id'].nunique():,}"
)
print(f"Columns before: {cols_before:,}")
print(f"Columns after : {features.shape[1]:,}")
print(f"New features  : {len(new_cols):,}")


# ============================================================
# 17. HARD CHECKS
# ============================================================

assert len(features) == len(events)

assert features["event_id"].is_unique

assert (
    features["current_monthly_salary_brl"]
    .notna()
    .all()
)

assert (
    features["discount_debt_to_salary_ratio"]
    <= features["debt_to_salary_ratio"]
).all()


print("\nSalary / affordability sanity checks: PASSED")


# ============================================================
# 18. SALARY DISTRIBUTION
# ============================================================

print("\nMONTHLY SALARY")

display(
    features["current_monthly_salary_brl"]
    .describe(
        percentiles=[
            .01,
            .05,
            .10,
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame()
)


print("\nSALARY VALIDITY")
print("-" * 100)

print(
    f"Negative salary : "
    f"{features['is_negative_salary'].sum():,}"
)

print(
    f"Zero salary     : "
    f"{features['is_zero_salary'].sum():,}"
)

print(
    f"Positive salary : "
    f"{features['is_positive_salary'].sum():,}"
)


# ============================================================
# 19. DEBT BURDEN DISTRIBUTION
# ============================================================

print("\nDEBT / SALARY RATIO")

display(
    features["debt_to_salary_ratio"]
    .describe(
        percentiles=[
            .01,
            .05,
            .10,
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame()
)


print("\nDEBT / SALARY BUCKET")

display(
    features["debt_to_salary_bucket"]
    .value_counts(sort=False)
    .to_frame("messages")
)


# ============================================================
# 20. DISCOUNT IMPACT
# ============================================================

print("\nDISCOUNT BURDEN REDUCTION (% OF SALARY)")

display(
    features[
        "discount_burden_reduction_pct_salary"
    ]
    .describe(
        percentiles=[
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame()
)


print("\n" + "=" * 100)
print("01E — SALARY & AFFORDABILITY COMPLETE")
print("=" * 100)

01E — SALARY & AFFORDABILITY COMPLETE
Rows          : 75,406
Customers     : 11,724
Columns before: 137
Columns after : 168
New features  : 31

Salary / affordability sanity checks: PASSED

MONTHLY SALARY


,current_monthly_salary_brl
count,"75,406.00"
mean,"2,653.51"
std,"1,229.12"
min,"1,200.00"
1%,"1,200.00"
5%,"1,200.00"
10%,"1,340.00"
25%,"1,740.00"
50%,"2,400.00"
75%,"3,250.00"



SALARY VALIDITY
----------------------------------------------------------------------------------------------------
Negative salary : 0
Zero salary     : 0
Positive salary : 75,406

DEBT / SALARY RATIO


,debt_to_salary_ratio
count,"75,406.00"
mean,0.32
std,0.14
min,0.01
1%,0.06
5%,0.11
10%,0.14
25%,0.20
50%,0.31
75%,0.43



DEBT / SALARY BUCKET


,messages
debt_to_salary_bucket,
debt_le_10pct_salary,3331
debt_10_25pct_salary,24014
debt_25_50pct_salary,39218
debt_50_75pct_salary,8804
debt_75_100pct_salary,39
debt_gt_100pct_salary,0



DISCOUNT BURDEN REDUCTION (% OF SALARY)


,discount_burden_reduction_pct_salary
count,"75,406.00"
mean,4.78
std,2.14
min,0.22
25%,3.04
50%,4.65
75%,6.46
90%,7.68
95%,8.30
99%,9.55



01E — SALARY & AFFORDABILITY COMPLETE


In [12]:
# ============================================================
# 01F — CURRENT STATE: RELATIONSHIP / ACCOUNT
# ============================================================
#
# PURPOSE
# -------
# Represent the customer's pre-existing relationship with
# the company at the moment of the current contact.
#
# Uses:
#   - n_prior_transactions
#   - account_age_months
#
# No outcomes.
# No WhatsApp history.
# No future information.
#
# IMPORTANT
# ---------
# relationship_profile uses ordered / hierarchical rules.
# np.select returns the FIRST matching condition, so more
# specific segments must be evaluated before broader ones.
# ============================================================

cols_before = features.shape[1]


# ============================================================
# 1. RAW RELATIONSHIP STATE
# ============================================================

prior_tx = pd.to_numeric(
    features["n_prior_transactions"],
    errors="coerce",
)

account_age = pd.to_numeric(
    features["account_age_months"],
    errors="coerce",
)

assert prior_tx.notna().all()
assert account_age.notna().all()

features["current_n_prior_transactions"] = (
    prior_tx.astype("int16")
)

features["current_account_age_months"] = (
    account_age.astype("int16")
)


# ============================================================
# 2. VALIDITY FLAGS
# ============================================================

features["has_prior_transactions"] = (
    features["current_n_prior_transactions"] > 0
).astype("int8")

features["is_zero_prior_transactions"] = (
    features["current_n_prior_transactions"] == 0
).astype("int8")

features["is_new_account_3m"] = (
    features["current_account_age_months"] <= 3
).astype("int8")

features["is_new_account_6m"] = (
    features["current_account_age_months"] <= 6
).astype("int8")

features["is_established_account_12m"] = (
    features["current_account_age_months"] >= 12
).astype("int8")

features["is_established_account_24m"] = (
    features["current_account_age_months"] >= 24
).astype("int8")


# ============================================================
# 3. TRANSFORMATIONS
# ============================================================

features["prior_transactions_log1p"] = np.log1p(
    features["current_n_prior_transactions"]
)

features["account_age_log1p"] = np.log1p(
    features["current_account_age_months"]
)

features["account_age_years"] = (
    features["current_account_age_months"] / 12
)


# ============================================================
# 4. ACCOUNT AGE BUCKET
# ============================================================

features["account_age_bucket"] = pd.cut(
    features["current_account_age_months"],
    bins=[
        -1,
        3,
        6,
        12,
        24,
        36,
        60,
        np.inf,
    ],
    labels=[
        "age_0_3m",
        "age_4_6m",
        "age_7_12m",
        "age_13_24m",
        "age_25_36m",
        "age_37_60m",
        "age_60m_plus",
    ],
)


# ============================================================
# 5. PRIOR TRANSACTION BUCKET
# ============================================================

features["prior_transactions_bucket"] = pd.cut(
    features["current_n_prior_transactions"],
    bins=[
        -1,
        0,
        1,
        2,
        5,
        10,
        20,
        np.inf,
    ],
    labels=[
        "tx_0",
        "tx_1",
        "tx_2",
        "tx_3_5",
        "tx_6_10",
        "tx_11_20",
        "tx_20_plus",
    ],
)


# ============================================================
# 6. RELATIONSHIP INTENSITY
# ============================================================
#
# Transactions per month/year of account tenure.
#
# These are relationship-depth proxies.
# They should NOT be interpreted as literal transaction
# frequency unless source-system semantics support that.
# ============================================================

safe_account_age = (
    features["current_account_age_months"]
    .clip(lower=1)
)

features["prior_transactions_per_account_month"] = (
    features["current_n_prior_transactions"]
    / safe_account_age
)

features["prior_transactions_per_account_year"] = (
    features["current_n_prior_transactions"]
    / safe_account_age
    * 12
)


# ============================================================
# 7. RELATIONSHIP DEPTH FLAGS
# ============================================================

features["has_3plus_prior_transactions"] = (
    features["current_n_prior_transactions"] >= 3
).astype("int8")

features["has_5plus_prior_transactions"] = (
    features["current_n_prior_transactions"] >= 5
).astype("int8")

features["has_10plus_prior_transactions"] = (
    features["current_n_prior_transactions"] >= 10
).astype("int8")


# ============================================================
# 8. COMBINED RELATIONSHIP PROFILE
# ============================================================
#
# Interpretable, rule-based relationship segments.
#
# IMPORTANT:
# np.select returns the FIRST matching condition.
#
# Therefore:
#
#   long_high_history:
#       age >= 24 AND tx >= 10
#
# MUST be evaluated BEFORE:
#
#   established_active:
#       age >= 12 AND tx >= 5
#
# because every long_high_history customer also satisfies
# established_active.
#
# Hierarchy:
#
#   1. long_high_history
#   2. new_low_history
#   3. established_active
#   4. intermediate
#
# No target/outcome information is used.
# ============================================================

age = features["current_account_age_months"]
tx = features["current_n_prior_transactions"]


features["relationship_profile"] = np.select(
    [
        # ----------------------------------------------------
        # Most specific high-depth relationship FIRST
        # ----------------------------------------------------
        (age >= 24) & (tx >= 10),

        # ----------------------------------------------------
        # Very new relationship with minimal transaction history
        # ----------------------------------------------------
        (age <= 6) & (tx <= 1),

        # ----------------------------------------------------
        # Broader established/active relationship
        # ----------------------------------------------------
        (age >= 12) & (tx >= 5),
    ],
    [
        "long_high_history",
        "new_low_history",
        "established_active",
    ],
    default="intermediate",
)


# ============================================================
# 9. RELATIONSHIP × CURRENT DEBT
# ============================================================

features["balance_per_prior_transaction"] = np.where(
    features["current_n_prior_transactions"] > 0,

    features["current_balance_brl"]
    / features["current_n_prior_transactions"],

    np.nan,
)


features["debt_to_salary_x_prior_transactions_log"] = (
    features["debt_to_salary_ratio"]
    * features["prior_transactions_log1p"]
)


# ============================================================
# 10. RELATIONSHIP × DPD
# ============================================================

features["dpd_per_account_month"] = (
    features["current_dpd"]
    / safe_account_age
)

features["dpd_x_prior_transactions_log"] = (
    features["current_dpd"]
    * features["prior_transactions_log1p"]
)


# ============================================================
# 11. AUDIT
# ============================================================

new_cols = features.columns[cols_before:]

print("=" * 100)
print("01F — RELATIONSHIP / ACCOUNT STATE COMPLETE")
print("=" * 100)

print(f"Rows          : {len(features):,}")

print(
    f"Customers     : "
    f"{features['customer_id'].nunique():,}"
)

print(f"Columns before: {cols_before:,}")
print(f"Columns after : {features.shape[1]:,}")
print(f"New features  : {len(new_cols):,}")


# ============================================================
# 12. HARD CHECKS — STRUCTURE
# ============================================================

assert len(features) == len(events)

assert features["event_id"].is_unique

assert (
    features["current_n_prior_transactions"] >= 0
).all()

assert (
    features["current_account_age_months"] >= 0
).all()

assert np.isfinite(
    features[
        "prior_transactions_per_account_month"
    ]
).all()

assert np.isfinite(
    features[
        "prior_transactions_per_account_year"
    ]
).all()


# ============================================================
# 13. HARD CHECKS — RELATIONSHIP PROFILE
# ============================================================

valid_profiles = {
    "long_high_history",
    "new_low_history",
    "established_active",
    "intermediate",
}


assert (
    features["relationship_profile"]
    .notna()
    .all()
)


assert set(
    features["relationship_profile"].unique()
).issubset(
    valid_profiles
)


# ------------------------------------------------------------
# 13A. LONG HIGH HISTORY
# ------------------------------------------------------------

long_mask = (
    features["relationship_profile"]
    .eq("long_high_history")
)


# This is the check that catches the previous np.select bug.
assert long_mask.any(), (
    "long_high_history is unexpectedly empty. "
    "Check np.select condition priority."
)


assert (
    age.loc[long_mask] >= 24
).all()


assert (
    tx.loc[long_mask] >= 10
).all()


# Every observation satisfying the defining condition
# MUST now be classified as long_high_history.
expected_long_mask = (
    (age >= 24)
    & (tx >= 10)
)


assert (
    long_mask
    ==
    expected_long_mask
).all(), (
    "Some age>=24 & tx>=10 observations were not classified "
    "as long_high_history."
)


# ------------------------------------------------------------
# 13B. NEW LOW HISTORY
# ------------------------------------------------------------

new_low_mask = (
    features["relationship_profile"]
    .eq("new_low_history")
)


assert (
    age.loc[new_low_mask] <= 6
).all()


assert (
    tx.loc[new_low_mask] <= 1
).all()


# ------------------------------------------------------------
# 13C. ESTABLISHED ACTIVE
# ------------------------------------------------------------

established_mask = (
    features["relationship_profile"]
    .eq("established_active")
)


assert (
    age.loc[established_mask] >= 12
).all()


assert (
    tx.loc[established_mask] >= 5
).all()


# Crucial:
# established_active must NOT contain observations that qualify
# for the more specific long_high_history segment.
assert ~(
    (age.loc[established_mask] >= 24)
    &
    (tx.loc[established_mask] >= 10)
).any(), (
    "long_high_history observations leaked into "
    "established_active."
)


print(
    "\nRelationship/account sanity checks: PASSED"
)

print(
    "Relationship profile hierarchy checks: PASSED"
)


# ============================================================
# 14. DISTRIBUTIONS
# ============================================================

print("\nPRIOR TRANSACTIONS")

display(
    features[
        "current_n_prior_transactions"
    ]
    .describe(
        percentiles=[
            .01,
            .05,
            .10,
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame()
)


print("\nACCOUNT AGE — MONTHS")

display(
    features[
        "current_account_age_months"
    ]
    .describe(
        percentiles=[
            .01,
            .05,
            .10,
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame()
)


print("\nPRIOR TRANSACTION BUCKET")

display(
    features[
        "prior_transactions_bucket"
    ]
    .value_counts(
        sort=False
    )
    .to_frame("messages")
)


print("\nACCOUNT AGE BUCKET")

display(
    features[
        "account_age_bucket"
    ]
    .value_counts(
        sort=False
    )
    .to_frame("messages")
)


# ============================================================
# 15. RELATIONSHIP PROFILE DISTRIBUTION
# ============================================================

print("\nRELATIONSHIP PROFILE")

relationship_profile_summary = (
    features[
        "relationship_profile"
    ]
    .value_counts()
    .to_frame("messages")
)


relationship_profile_summary["pct"] = (
    relationship_profile_summary["messages"]
    / len(features)
    * 100
)


display(
    relationship_profile_summary
)


# ============================================================
# 16. CROSS-CHECK PROFILE DEFINITIONS
# ============================================================

print(
    "\nRELATIONSHIP PROFILE — "
    "AGE / TRANSACTION SUMMARY"
)

display(
    features
    .groupby(
        "relationship_profile"
    )
    .agg(
        messages=(
            "event_id",
            "size",
        ),

        customers=(
            "customer_id",
            "nunique",
        ),

        min_account_age_months=(
            "current_account_age_months",
            "min",
        ),

        median_account_age_months=(
            "current_account_age_months",
            "median",
        ),

        max_account_age_months=(
            "current_account_age_months",
            "max",
        ),

        min_prior_transactions=(
            "current_n_prior_transactions",
            "min",
        ),

        median_prior_transactions=(
            "current_n_prior_transactions",
            "median",
        ),

        max_prior_transactions=(
            "current_n_prior_transactions",
            "max",
        ),
    )
    .sort_values(
        "messages",
        ascending=False,
    )
)


# ============================================================
# 17. RELATIONSHIP INTENSITY
# ============================================================

print("\nTRANSACTIONS PER ACCOUNT YEAR")

display(
    features[
        "prior_transactions_per_account_year"
    ]
    .describe(
        percentiles=[
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame()
)


# ============================================================
# 18. FINAL CHECKPOINT
# ============================================================

print("\n" + "=" * 100)
print("01F — RELATIONSHIP / ACCOUNT STATE COMPLETE")
print("=" * 100)

print(
    f"Final relationship profiles: "
    f"{features['relationship_profile'].nunique()}"
)

print(
    "Correct priority:"
)

print(
    "long_high_history > "
    "new_low_history > "
    "established_active > "
    "intermediate"
)

print("=" * 100)

01F — RELATIONSHIP / ACCOUNT STATE COMPLETE
Rows          : 75,406
Customers     : 11,724
Columns before: 168
Columns after : 191
New features  : 23

Relationship/account sanity checks: PASSED
Relationship profile hierarchy checks: PASSED

PRIOR TRANSACTIONS


,current_n_prior_transactions
count,"75,406.00"
mean,8.83
std,8.26
min,1.00
1%,1.00
5%,1.00
10%,1.00
25%,3.00
50%,6.00
75%,12.00



ACCOUNT AGE — MONTHS


,current_account_age_months
count,"75,406.00"
mean,9.15
std,7.09
min,1.00
1%,1.00
5%,1.00
10%,2.00
25%,4.00
50%,7.00
75%,12.00



PRIOR TRANSACTION BUCKET


...


RELATIONSHIP PROFILE — AGE / TRANSACTION SUMMARY


,messages,customers,min_account_age_months,median_account_age_months,max_account_age_months,min_prior_transactions,median_prior_transactions,max_prior_transactions
relationship_profile,,,,,,,,
intermediate,46553,7229,1,6.00,15,1,5.00,22
established_active,17199,2761,12,16.00,23,5,16.00,35
new_low_history,7801,1151,1,2.00,6,1,1.00,1
long_high_history,3853,583,24,28.00,40,15,31.00,40



TRANSACTIONS PER ACCOUNT YEAR


,prior_transactions_per_account_year
count,"75,406.00"
mean,12.66
std,9.07
min,1.09
25%,7.64
50%,12.00
75%,14.53
90%,20.00
95%,26.00
99%,48.00



01F — RELATIONSHIP / ACCOUNT STATE COMPLETE
Final relationship profiles: 4
Correct priority:
long_high_history > new_low_history > established_active > intermediate


In [13]:
# ============================================================
# 01G — CURRENT STATE: DIGITAL / APP ACTIVITY
# ============================================================
#
# PURPOSE
# -------
# Represent how recently the customer was active in the app
# at the moment of the WhatsApp send.
#
# POINT-IN-TIME
# -------------
# Uses only:
#   - days_since_last_app_login
#
# No WhatsApp outcomes.
# No future information.
# ============================================================

cols_before = features.shape[1]


# ============================================================
# 1. CURRENT LOGIN RECENCY
# ============================================================

login_days = pd.to_numeric(
    features["days_since_last_app_login"],
    errors="coerce"
)

assert login_days.notna().all()

features["current_days_since_last_app_login"] = (
    login_days.astype("int16")
)


# ============================================================
# 2. VALIDITY
# ============================================================

features["is_login_today"] = (
    features["current_days_since_last_app_login"] == 0
).astype("int8")

features["is_login_recency_negative"] = (
    features["current_days_since_last_app_login"] < 0
).astype("int8")


# ============================================================
# 3. TRANSFORMATIONS
# ============================================================

features["login_recency_log1p"] = np.where(
    features["current_days_since_last_app_login"] >= 0,
    np.log1p(
        features["current_days_since_last_app_login"]
    ),
    np.nan,
)

features["login_recency_sqrt"] = np.where(
    features["current_days_since_last_app_login"] >= 0,
    np.sqrt(
        features["current_days_since_last_app_login"]
    ),
    np.nan,
)


# ============================================================
# 4. RECENCY FLAGS
# ============================================================

for window in [1, 3, 7, 14, 30, 60, 90]:

    features[
        f"logged_in_within_{window}d"
    ] = (
        features["current_days_since_last_app_login"]
        <= window
    ).astype("int8")


# ============================================================
# 5. INACTIVITY FLAGS
# ============================================================

for threshold in [7, 14, 30, 60, 90]:

    features[
        f"inactive_app_{threshold}d_plus"
    ] = (
        features["current_days_since_last_app_login"]
        >= threshold
    ).astype("int8")


# ============================================================
# 6. LOGIN RECENCY BUCKET
# ============================================================

features["login_recency_bucket"] = pd.cut(
    features["current_days_since_last_app_login"],
    bins=[
        -1,
        0,
        1,
        3,
        7,
        14,
        30,
        60,
        90,
        np.inf,
    ],
    labels=[
        "login_today",
        "login_1d",
        "login_2_3d",
        "login_4_7d",
        "login_8_14d",
        "login_15_30d",
        "login_31_60d",
        "login_61_90d",
        "login_90d_plus",
    ],
)


# ============================================================
# 7. DIGITAL ACTIVITY LEVEL
# ============================================================

features["digital_activity_level"] = pd.cut(
    features["current_days_since_last_app_login"],
    bins=[
        -1,
        3,
        7,
        30,
        90,
        np.inf,
    ],
    labels=[
        "very_recent_0_3d",
        "recent_4_7d",
        "moderate_8_30d",
        "inactive_31_90d",
        "very_inactive_90d_plus",
    ],
)


# ============================================================
# 8. LOGIN RECENCY RELATIVE TO CURRENT DPD
# ============================================================
#
# Example:
#
# DPD = 20
# last login = 3 days ago
#
# Customer has been digitally active during the current
# collections journey.
#
# Conversely:
#
# DPD = 20
# last login = 40 days ago
#
# Last known login predates entry into collections.
# ============================================================

features["login_recency_to_dpd_ratio"] = (
    features["current_days_since_last_app_login"]
    / features["current_dpd"].clip(lower=1)
)


features["login_recency_minus_dpd"] = (
    features["current_days_since_last_app_login"]
    - features["current_dpd"]
)


# ============================================================
# 9. WAS LAST LOGIN DURING CURRENT COLLECTIONS JOURNEY?
# ============================================================
#
# If days_since_login < current_dpd:
# last login happened after collections entry.
#
# If equal:
# approximately on collections entry date.
#
# If greater:
# last login predates current collections journey.
# ============================================================

features["last_login_during_current_collections"] = (
    features["current_days_since_last_app_login"]
    < features["current_dpd"]
).astype("int8")


features["last_login_around_collections_entry"] = (
    features["current_days_since_last_app_login"]
    == features["current_dpd"]
).astype("int8")


features["last_login_before_current_collections"] = (
    features["current_days_since_last_app_login"]
    > features["current_dpd"]
).astype("int8")


# ============================================================
# 10. APPROXIMATE LAST LOGIN DATE
# ============================================================
#
# Reconstructed calendar date from current state.
# Useful for interpretation and later temporal interactions.
# ============================================================

features["approx_last_app_login_date"] = (
    features["sent_date"]
    - pd.to_timedelta(
        features["current_days_since_last_app_login"],
        unit="D",
    )
)


# ============================================================
# 11. DIGITAL × AFFORDABILITY
# ============================================================

features["login_recency_x_debt_to_salary"] = (
    features["current_days_since_last_app_login"]
    * features["debt_to_salary_ratio"]
)


features["login_log_x_debt_to_salary"] = (
    features["login_recency_log1p"]
    * features["debt_to_salary_ratio"]
)


# ============================================================
# 12. DIGITAL × DPD
# ============================================================

features["login_log_x_dpd_log"] = (
    features["login_recency_log1p"]
    * features["current_dpd_log1p"]
)


# ============================================================
# 13. DIGITAL × PAYDAY
# ============================================================
#
# Is the customer digitally active while also close to payday?
# Still purely current state.
# ============================================================

features["recent_login_7d_x_near_payday_3d"] = (
    features["logged_in_within_7d"]
    * features["is_within_3d_nearest_payday"]
).astype("int8")


features["recent_login_3d_x_payday"] = (
    features["logged_in_within_3d"]
    * features["is_payday"]
).astype("int8")


# ============================================================
# 14. AUDIT
# ============================================================

new_cols = features.columns[cols_before:]

print("=" * 100)
print("01G — DIGITAL / APP STATE COMPLETE")
print("=" * 100)

print(f"Rows          : {len(features):,}")
print(
    f"Customers     : "
    f"{features['customer_id'].nunique():,}"
)
print(f"Columns before: {cols_before:,}")
print(f"Columns after : {features.shape[1]:,}")
print(f"New features  : {len(new_cols):,}")


# ============================================================
# 15. HARD CHECKS
# ============================================================

assert len(features) == len(events)
assert features["event_id"].is_unique

assert (
    features["current_days_since_last_app_login"]
    .notna()
    .all()
)


print("\nDigital/app basic sanity checks: PASSED")


# ============================================================
# 16. LOGIN RECENCY DISTRIBUTION
# ============================================================

print("\nDAYS SINCE LAST APP LOGIN")

display(
    features[
        "current_days_since_last_app_login"
    ]
    .describe(
        percentiles=[
            .01,
            .05,
            .10,
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame()
)


print("\nLOGIN RECENCY VALIDITY")
print("-" * 100)

print(
    f"Negative recency : "
    f"{features['is_login_recency_negative'].sum():,}"
)

print(
    f"Login today      : "
    f"{features['is_login_today'].sum():,}"
)


print("\nLOGIN RECENCY BUCKET")

display(
    features["login_recency_bucket"]
    .value_counts(sort=False)
    .to_frame("messages")
)


# ============================================================
# 17. LOGIN RELATIVE TO COLLECTIONS JOURNEY
# ============================================================

print("\nLAST LOGIN RELATIVE TO CURRENT COLLECTIONS JOURNEY")
print("-" * 100)

journey_login = pd.Series(
    np.select(
        [
            features[
                "last_login_during_current_collections"
            ].eq(1),

            features[
                "last_login_around_collections_entry"
            ].eq(1),
        ],
        [
            "during_collections",
            "around_entry",
        ],
        default="before_collections",
    ),
    index=features.index,
    name="login_position",
)

display(
    journey_login
    .value_counts()
    .to_frame("messages")
)


# ============================================================
# 18. SAMPLE
# ============================================================

sample_cols = [
    "customer_id",
    "sent_at",
    "current_dpd",
    "implied_collections_entry_date",
    "current_days_since_last_app_login",
    "approx_last_app_login_date",
    "digital_activity_level",
    "last_login_during_current_collections",
]

print("\nSAMPLE")

display(
    features[
        sample_cols
    ].head(20)
)


print("\n" + "=" * 100)
print("01G — DIGITAL / APP STATE COMPLETE")
print("=" * 100)

01G — DIGITAL / APP STATE COMPLETE
Rows          : 75,406
Customers     : 11,724
Columns before: 191
Columns after : 221
New features  : 30

Digital/app basic sanity checks: PASSED

DAYS SINCE LAST APP LOGIN


,current_days_since_last_app_login
count,"75,406.00"
mean,23.45
std,23.27
min,0.00
1%,1.00
5%,2.00
10%,3.00
25%,7.00
50%,16.00
75%,33.00



LOGIN RECENCY VALIDITY
----------------------------------------------------------------------------------------------------
Negative recency : 0
Login today      : 327

LOGIN RECENCY BUCKET


,messages
login_recency_bucket,
login_today,327
login_1d,3402
login_2_3d,5845
login_4_7d,10870
login_8_14d,14284
login_15_30d,20170
login_31_60d,15032
login_61_90d,3969
login_90d_plus,1507



LAST LOGIN RELATIVE TO CURRENT COLLECTIONS JOURNEY
----------------------------------------------------------------------------------------------------


,messages
login_position,
before_collections,38227
during_collections,35320
around_entry,1859



SAMPLE


,customer_id,sent_at,current_dpd,implied_collections_entry_date,current_days_since_last_app_login,approx_last_app_login_date,digital_activity_level,last_login_during_current_collections
0,C000001,2026-07-06 13:29:00,3,2026-07-04,38,2026-05-29,inactive_31_90d,0
1,C000001,2026-07-07 14:48:00,4,2026-07-04,39,2026-05-29,inactive_31_90d,0
2,C000001,2026-07-08 10:46:00,5,2026-07-04,1,2026-07-07,very_recent_0_3d,1
3,C000001,2026-07-11 13:11:00,8,2026-07-04,4,2026-07-07,recent_4_7d,1
4,C000001,2026-07-15 12:54:00,12,2026-07-04,8,2026-07-07,moderate_8_30d,1
5,C000001,2026-07-17 12:54:00,14,2026-07-04,10,2026-07-07,moderate_8_30d,1
6,C000001,2026-07-29 09:14:00,26,2026-07-04,22,2026-07-07,moderate_8_30d,1
7,C000002,2026-07-29 10:31:00,4,2026-07-26,15,2026-07-14,moderate_8_30d,0
8,C000002,2026-07-30 09:31:00,5,2026-07-26,16,2026-07-14,moderate_8_30d,0
9,C000002,2026-08-02 10:54:00,8,2026-07-26,19,2026-07-14,moderate_8_30d,0



01G — DIGITAL / APP STATE COMPLETE


In [14]:
# ============================================================
# 01H — GEOGRAPHY + CURRENT ACTION CONTEXT
# ============================================================
#
# IMPORTANT SEMANTIC SEPARATION
# -----------------------------
#
# STATE:
#   information describing customer/context before decision
#
# ACTION:
#   action actually chosen by the historical policy
#
# For NBA candidate scoring later:
#
#   STATE stays fixed
#   ACTION changes across candidate templates
#
# No outcomes are used here.
# ============================================================

cols_before = features.shape[1]


# ============================================================
# 1. GEOGRAPHY — STATE
# ============================================================

features["current_state_uf"] = (
    features["state_uf"]
    .astype("string")
    .str.upper()
    .str.strip()
)


features["state_region"] = (
    features["current_state_uf"]
    .map(
        {
            # Southeast
            "SP": "southeast",
            "RJ": "southeast",
            "MG": "southeast",
            "ES": "southeast",

            # South
            "PR": "south",
            "SC": "south",
            "RS": "south",

            # Center-West
            "DF": "center_west",
            "GO": "center_west",
            "MT": "center_west",
            "MS": "center_west",

            # Northeast
            "BA": "northeast",
            "SE": "northeast",
            "AL": "northeast",
            "PE": "northeast",
            "PB": "northeast",
            "RN": "northeast",
            "CE": "northeast",
            "PI": "northeast",
            "MA": "northeast",

            # North
            "AC": "north",
            "AP": "north",
            "AM": "north",
            "PA": "north",
            "RO": "north",
            "RR": "north",
            "TO": "north",
        }
    )
    .fillna("unknown")
)


# ============================================================
# 2. REGION FLAGS — STATE
# ============================================================

for region in [
    "southeast",
    "south",
    "center_west",
    "northeast",
    "north",
]:

    features[
        f"is_region_{region}"
    ] = (
        features["state_region"] == region
    ).astype("int8")


# ============================================================
# 3. CURRENT ACTION
# ============================================================
#
# Already created in 00B:
#
# current_template
#
# We now create explicit action flags.
#
# These are NOT ordinary customer-state predictors.
# They represent the action historically selected.
# ============================================================

template_norm = (
    features["current_template"]
    .astype("string")
    .str.lower()
    .str.strip()
)

features["action_template"] = template_norm


expected_templates = [
    "friendly_reminder",
    "urgent_reminder",
    "discount_offer",
    "pix_link",
]


for template_name in expected_templates:

    features[
        f"action_is_{template_name}"
    ] = (
        features["action_template"]
        == template_name
    ).astype("int8")


# ============================================================
# 4. ACTION CHARACTERISTICS
# ============================================================
#
# These are deterministic properties of the action itself.
# They will later be recomputed for each candidate action.
# ============================================================

features["action_is_discount"] = (
    features["action_template"]
    == "discount_offer"
).astype("int8")


features["action_is_pix_link"] = (
    features["action_template"]
    == "pix_link"
).astype("int8")


features["action_is_reminder"] = (
    features["action_template"]
    .isin(
        [
            "friendly_reminder",
            "urgent_reminder",
        ]
    )
).astype("int8")


features["action_is_urgent"] = (
    features["action_template"]
    == "urgent_reminder"
).astype("int8")


features["action_is_friendly"] = (
    features["action_template"]
    == "friendly_reminder"
).astype("int8")


# ============================================================
# 5. ACTION × CURRENT STATE
# ============================================================
#
# These are ACTION_INTERACTION features.
#
# They are valid for historical modelling, but later NBA
# scoring must recompute them separately for every candidate
# template.
# ============================================================

features["action_discount_x_debt_to_salary"] = (
    features["action_is_discount"]
    * features["debt_to_salary_ratio"]
)


features["action_discount_x_balance"] = (
    features["action_is_discount"]
    * features["current_balance_brl"]
)


features["action_discount_x_discount_amount"] = (
    features["action_is_discount"]
    * features["discount_amount_brl"]
)


features["action_discount_x_near_payday_3d"] = (
    features["action_is_discount"]
    * features["is_within_3d_nearest_payday"]
).astype("int8")


features["action_discount_x_recent_login_7d"] = (
    features["action_is_discount"]
    * features["logged_in_within_7d"]
).astype("int8")


features["action_pix_x_recent_login_7d"] = (
    features["action_is_pix_link"]
    * features["logged_in_within_7d"]
).astype("int8")


features["action_pix_x_near_payday_3d"] = (
    features["action_is_pix_link"]
    * features["is_within_3d_nearest_payday"]
).astype("int8")


features["action_urgent_x_dpd"] = (
    features["action_is_urgent"]
    * features["current_dpd"]
)


features["action_urgent_x_dpd30_plus"] = (
    features["action_is_urgent"]
    * features["is_dpd_30_plus"]
).astype("int8")


features["action_friendly_x_early_collections"] = (
    features["action_is_friendly"]
    * (
        features["current_dpd"] <= 15
    ).astype("int8")
).astype("int8")


# ============================================================
# 6. ACTION × AFFORDABILITY
# ============================================================

features["action_discount_x_affordable_25pct"] = (
    features["action_is_discount"]
    * features["debt_le_25pct_salary"]
).astype("int8")


features["action_discount_x_high_burden_50pct"] = (
    features["action_is_discount"]
    * features["debt_gt_50pct_salary"]
).astype("int8")


# ============================================================
# 7. AUDIT
# ============================================================

new_cols = features.columns[cols_before:]

print("=" * 100)
print("01H — GEOGRAPHY + CURRENT ACTION CONTEXT COMPLETE")
print("=" * 100)

print(f"Rows          : {len(features):,}")
print(
    f"Customers     : "
    f"{features['customer_id'].nunique():,}"
)
print(f"Columns before: {cols_before:,}")
print(f"Columns after : {features.shape[1]:,}")
print(f"New features  : {len(new_cols):,}")


# ============================================================
# 8. HARD CHECKS
# ============================================================

assert len(features) == len(events)

assert features["event_id"].is_unique


# Exactly one expected action per event
action_flag_cols = [
    f"action_is_{x}"
    for x in expected_templates
]

action_flag_sum = (
    features[action_flag_cols]
    .sum(axis=1)
)

assert action_flag_sum.eq(1).all()


# Region mapping completeness
assert (
    features["state_region"]
    .notna()
    .all()
)


print("\nGeography/action sanity checks: PASSED")


# ============================================================
# 9. GEOGRAPHY DISTRIBUTION
# ============================================================

print("\nSTATE UF")

display(
    features["current_state_uf"]
    .value_counts()
    .to_frame("messages")
)


print("\nREGION")

display(
    features["state_region"]
    .value_counts()
    .to_frame("messages")
)


# ============================================================
# 10. ACTION DISTRIBUTION
# ============================================================

print("\nCURRENT HISTORICAL ACTION")

display(
    features["action_template"]
    .value_counts()
    .to_frame("messages")
)


print("\nCURRENT HISTORICAL ACTION — SHARE")

action_share = (
    features["action_template"]
    .value_counts(normalize=True)
    .mul(100)
    .rename("pct_messages")
    .to_frame()
)

display(action_share)


# ============================================================
# 11. ACTION × DPD DESCRIPTIVE AUDIT
# ============================================================
#
# IMPORTANT:
# This describes the HISTORICAL POLICY.
# It does NOT measure template effectiveness.
# ============================================================

print("\nHISTORICAL ACTION × COLLECTIONS STAGE")

action_stage = pd.crosstab(
    features["collections_stage"],
    features["action_template"],
)

display(action_stage)


print("\nHISTORICAL ACTION SHARE WITHIN COLLECTIONS STAGE (%)")

action_stage_pct = pd.crosstab(
    features["collections_stage"],
    features["action_template"],
    normalize="index",
).mul(100)

display(
    action_stage_pct.round(2)
)


# ============================================================
# 12. ACTION × PAYDAY DESCRIPTIVE AUDIT
# ============================================================

print("\nHISTORICAL ACTION × PAYDAY PHASE (%)")

action_payday_pct = pd.crosstab(
    features["payday_phase"],
    features["action_template"],
    normalize="index",
).mul(100)

display(
    action_payday_pct.round(2)
)


print("\n" + "=" * 100)
print("01H — GEOGRAPHY + CURRENT ACTION CONTEXT COMPLETE")
print("=" * 100)

01H — GEOGRAPHY + CURRENT ACTION CONTEXT COMPLETE
Rows          : 75,406
Customers     : 11,724
Columns before: 221
Columns after : 249
New features  : 28

Geography/action sanity checks: PASSED

STATE UF


,messages
current_state_uf,
SP,16347
MG,7592
RJ,5878
BA,5674
RS,4136
PR,4033
CE,3574
PA,3028
PE,2954



REGION


,messages
state_region,
southeast,31003
northeast,20233
south,10701
north,7196
center_west,6273



CURRENT HISTORICAL ACTION


...


HISTORICAL ACTION SHARE WITHIN COLLECTIONS STAGE (%)


action_template,discount_offer,friendly_reminder,pix_link,urgent_reminder
collections_stage,,,,
early_0_15,0.00,46.61,30.10,23.29
mid_16_30,0.00,20.05,29.88,50.07
late_31_59,49.89,0.00,20.37,29.74
dpd_60_plus,55.37,0.00,16.94,27.69



HISTORICAL ACTION × PAYDAY PHASE (%)


action_template,discount_offer,friendly_reminder,pix_link,urgent_reminder
payday_phase,,,,
mid_cycle,8.45,32.13,28.22,31.20
payday,8.52,31.83,27.99,31.66
post_payday_1_3d,7.72,32.64,29.16,30.48
post_payday_4_7d,7.78,33.76,28.46,30.00
pre_payday_1_3d,8.16,32.08,28.19,31.56
pre_payday_4_7d,8.26,32.07,28.96,30.71



01H — GEOGRAPHY + CURRENT ACTION CONTEXT COMPLETE


In [15]:
# ============================================================
# 01I — LAYER 01 FINAL AUDIT
# CURRENT STATE + CURRENT ACTION
# ============================================================
#
# PURPOSE
# -------
# Close Layer 01 before historical feature engineering.
#
# Validate:
#   1. grain preservation
#   2. identifiers
#   3. label isolation
#   4. state/action semantic separation
#   5. missingness / infinity
#   6. constant features
#   7. current-state raw variables
#   8. feature-family inventory
#
# IMPORTANT
# ---------
# This cell creates NO modelling features.
# ============================================================


# ============================================================
# 1. CORE GRAIN AUDIT
# ============================================================

print("=" * 100)
print("01I — LAYER 01 FINAL AUDIT")
print("=" * 100)

print(f"Rows       : {len(features):,}")
print(
    f"Customers  : "
    f"{features['customer_id'].nunique():,}"
)
print(
    f"Events     : "
    f"{features['event_id'].nunique():,}"
)
print(
    f"Messages   : "
    f"{features['message_id'].nunique():,}"
)
print(
    f"Columns    : "
    f"{features.shape[1]:,}"
)


assert len(features) == 75_406
assert features["event_id"].is_unique
assert features["message_id"].is_unique
assert features["customer_id"].nunique() == 11_724

print("\nCore grain checks: PASSED")


# ============================================================
# 2. LABEL / OUTCOME COLUMNS
# ============================================================
#
# These columns are allowed to EXIST in the canonical table.
#
# They must NOT have been used to construct Layer 01 state
# features.
# ============================================================

label_cols = [
    "event_delivery_status",
    "event_interaction",
    "event_paid_within_72h",
    "event_amount_paid_brl",
]

print("\n" + "=" * 100)
print("OUTCOME / LABEL COLUMNS")
print("=" * 100)

for col in label_cols:
    print(
        f"{col:<35} "
        f"{'PRESENT' if col in features.columns else 'MISSING'}"
    )

assert all(
    col in features.columns
    for col in label_cols
)

print("\nOutcome columns preserved separately: PASSED")


# ============================================================
# 3. SEARCH FOR SUSPICIOUS DERIVED COLUMN NAMES
# ============================================================
#
# We expect the ORIGINAL event outcome columns above.
#
# We do NOT expect derived Layer-01 features containing
# payment/delivery/interaction outcome concepts.
# ============================================================

allowed_outcome_cols = set(label_cols)

outcome_terms = [
    "paid",
    "payment",
    "recovered",
    "recovery",
    "delivery",
    "interaction",
    "clicked",
    "click",
    "replied",
    "reply",
    "read",
]

suspicious_outcome_cols = []

for col in features.columns:

    col_lower = col.lower()

    if (
        col not in allowed_outcome_cols
        and any(
            term in col_lower
            for term in outcome_terms
        )
    ):
        suspicious_outcome_cols.append(col)


print("\n" + "=" * 100)
print("SUSPICIOUS OUTCOME-DERIVED COLUMN NAMES")
print("=" * 100)

if suspicious_outcome_cols:

    for col in suspicious_outcome_cols:
        print(col)

else:
    print("None")

assert len(suspicious_outcome_cols) == 0

print("\nNo outcome-derived Layer-01 feature names: PASSED")


# ============================================================
# 4. DEFINE SEMANTIC COLUMN REGISTRY
# ============================================================

identifier_cols = [
    "event_id",
    "message_id",
    "customer_id",
    "event_number",
    "_source_row",
]


action_cols = [
    col
    for col in features.columns
    if (
        col == "current_template"
        or col == "action_template"
        or col.startswith("action_")
    )
]


label_cols = [
    "event_delivery_status",
    "event_interaction",
    "event_paid_within_72h",
    "event_amount_paid_brl",
]


technical_cols = [
    "sent_at",
    "sent_date",
]


# Everything else is current state / raw state representation.
state_cols = [
    col
    for col in features.columns
    if col not in (
        set(identifier_cols)
        | set(action_cols)
        | set(label_cols)
        | set(technical_cols)
    )
]


print("\n" + "=" * 100)
print("SEMANTIC REGISTRY")
print("=" * 100)

print(
    f"Identifiers : {len(identifier_cols):,}"
)

print(
    f"State       : {len(state_cols):,}"
)

print(
    f"Action      : {len(action_cols):,}"
)

print(
    f"Labels      : {len(label_cols):,}"
)

print(
    f"Technical   : {len(technical_cols):,}"
)

registry_total = (
    len(identifier_cols)
    + len(state_cols)
    + len(action_cols)
    + len(label_cols)
    + len(technical_cols)
)

print(
    f"\nRegistry total : {registry_total:,}"
)

assert registry_total == features.shape[1]

print("\nSemantic registry coverage: PASSED")


# ============================================================
# 5. ACTION COLUMN INVENTORY
# ============================================================

print("\n" + "=" * 100)
print("ACTION / ACTION-INTERACTION COLUMNS")
print("=" * 100)

for col in action_cols:
    print(col)


# ============================================================
# 6. NUMERIC FEATURE QUALITY
# ============================================================

numeric_cols = (
    features
    .select_dtypes(include=[np.number])
    .columns
    .tolist()
)


numeric_state_action_cols = [
    col
    for col in numeric_cols
    if col not in (
        set(identifier_cols)
        | set(label_cols)
    )
]


numeric_matrix = features[
    numeric_state_action_cols
]


n_inf = np.isinf(
    numeric_matrix
    .to_numpy(copy=False)
).sum()


print("\n" + "=" * 100)
print("NUMERIC QUALITY")
print("=" * 100)

print(
    f"Numeric state/action columns : "
    f"{len(numeric_state_action_cols):,}"
)

print(
    f"Infinite values              : "
    f"{n_inf:,}"
)

assert n_inf == 0

print("\nNo infinite numeric values: PASSED")


# ============================================================
# 7. MISSINGNESS
# ============================================================
#
# Some NaN can be intentional:
# e.g. balance_per_prior_transaction if prior tx = 0.
#
# We inspect rather than automatically fail.
# ============================================================

missing = (
    features
    .isna()
    .sum()
)

missing = (
    missing[
        missing > 0
    ]
    .sort_values(
        ascending=False
    )
)


print("\n" + "=" * 100)
print("COLUMNS WITH MISSING VALUES")
print("=" * 100)

if len(missing) == 0:

    print("None")

else:

    display(
        missing
        .rename("missing_rows")
        .to_frame()
        .assign(
            missing_pct=lambda x:
                x["missing_rows"]
                / len(features)
                * 100
        )
    )


# ============================================================
# 8. CONSTANT / ZERO-VARIANCE COLUMNS
# ============================================================
#
# Do NOT remove them yet.
#
# We only inventory them for later feature selection.
# ============================================================

constant_cols = []

for col in features.columns:

    if features[col].nunique(
        dropna=False
    ) <= 1:

        constant_cols.append(col)


print("\n" + "=" * 100)
print("CONSTANT / ZERO-VARIANCE COLUMNS")
print("=" * 100)

print(
    f"Count: {len(constant_cols):,}"
)

for col in constant_cols:
    print(col)


# ============================================================
# 9. HIGH-CARDINALITY CATEGORICAL AUDIT
# ============================================================

categorical_cols = (
    features
    .select_dtypes(
        include=[
            "object",
            "string",
            "category",
        ]
    )
    .columns
    .tolist()
)


cat_cardinality = (
    pd.Series(
        {
            col:
                features[col]
                .nunique(
                    dropna=False
                )
            for col in categorical_cols
        }
    )
    .sort_values(
        ascending=False
    )
)


print("\n" + "=" * 100)
print("CATEGORICAL CARDINALITY")
print("=" * 100)

display(
    cat_cardinality
    .rename("n_unique")
    .to_frame()
)


# ============================================================
# 10. CURRENT RAW STATE VARIABLES
# ============================================================
#
# Verify that the original decision-time fields remain intact.
# ============================================================

raw_state_cols = [
    "days_past_due",
    "outstanding_balance_brl",
    "monthly_salary_brl",
    "payday_day_of_month",
    "n_prior_transactions",
    "account_age_months",
    "days_since_last_app_login",
    "state_uf",
]


print("\n" + "=" * 100)
print("RAW CURRENT-STATE VARIABLES")
print("=" * 100)

for col in raw_state_cols:

    print(
        f"{col:<35} "
        f"{'PRESENT' if col in features.columns else 'MISSING'}"
    )


assert all(
    col in features.columns
    for col in raw_state_cols
)

print("\nRaw current-state preservation: PASSED")


# ============================================================
# 11. FEATURE FAMILY INVENTORY
# ============================================================

family_rules = {
    "calendar_time": [
        "send_",
        "is_week",
        "is_month_",
        "days_from_month",
        "days_to_month",
        "month_progress",
    ],

    "payday": [
        "payday",
        "days_since_previous_payday",
        "days_until_next_payday",
        "days_to_nearest_payday",
        "days_from_current_month_payday",
        "nearest_payday",
    ],

    "dpd": [
        "dpd",
        "collections_stage",
        "implied_collections",
    ],

    "balance": [
        "balance",
        "discount_settlement",
        "discount_amount",
    ],

    "salary_affordability": [
        "salary",
        "debt_to_salary",
        "discount_debt",
        "afford",
    ],

    "relationship": [
        "prior_transaction",
        "account_age",
        "relationship",
    ],

    "digital": [
        "login",
        "digital",
        "inactive_app",
    ],

    "geography": [
        "state_",
        "region",
    ],

    "action": [
        "action_",
        "current_template",
    ],
}


family_inventory = {}

for family, patterns in family_rules.items():

    matched = [
        col
        for col in features.columns
        if any(
            pattern in col
            for pattern in patterns
        )
    ]

    family_inventory[family] = len(
        set(matched)
    )


print("\n" + "=" * 100)
print("FEATURE FAMILY INVENTORY")
print("=" * 100)

display(
    pd.Series(
        family_inventory,
        name="n_columns"
    )
    .sort_values(
        ascending=False
    )
    .to_frame()
)


# ============================================================
# 12. FINAL LAYER-01 SUMMARY
# ============================================================

print("\n" + "=" * 100)
print("LAYER 01 — FINAL SUMMARY")
print("=" * 100)

print(
    f"Final rows       : "
    f"{len(features):,}"
)

print(
    f"Final customers  : "
    f"{features['customer_id'].nunique():,}"
)

print(
    f"Final columns     : "
    f"{features.shape[1]:,}"
)

print(
    f"State columns     : "
    f"{len(state_cols):,}"
)

print(
    f"Action columns    : "
    f"{len(action_cols):,}"
)

print(
    f"Outcome columns   : "
    f"{len(label_cols):,}"
)

print(
    f"Constant columns  : "
    f"{len(constant_cols):,}"
)

print(
    f"Columns with NaN  : "
    f"{len(missing):,}"
)

print(
    f"Infinite values   : "
    f"{n_inf:,}"
)


print("\nLayer 01 structural audit: PASSED")

print("\n" + "=" * 100)
print("01I — LAYER 01 FINAL AUDIT COMPLETE")
print("=" * 100)

01I — LAYER 01 FINAL AUDIT
Rows       : 75,406
Customers  : 11,724
Events     : 75,406
Messages   : 75,406
Columns    : 249

Core grain checks: PASSED

OUTCOME / LABEL COLUMNS
event_delivery_status               PRESENT
event_interaction                   PRESENT
event_paid_within_72h               PRESENT
event_amount_paid_brl               PRESENT

Outcome columns preserved separately: PASSED

SUSPICIOUS OUTCOME-DERIVED COLUMN NAMES
None

No outcome-derived Layer-01 feature names: PASSED

SEMANTIC REGISTRY
Identifiers : 5
State       : 216
Action      : 22
Labels      : 4
Technical   : 2

Registry total : 249

Semantic registry coverage: PASSED

ACTION / ACTION-INTERACTION COLUMNS
current_template
action_template
action_is_friendly_reminder
action_is_urgent_reminder
action_is_discount_offer
action_is_pix_link
action_is_discount
action_is_reminder
action_is_urgent
action_is_friendly
action_discount_x_debt_to_salary
action_discount_x_balance
action_discount_x_discount_amount
action_dis

,n_unique
message_id,75406
event_id,75406
customer_id,11724
state_uf,27
current_state_uf,27
login_recency_bucket,9
salary_bucket,8
send_weekday_name,7
dpd_bucket,6
account_age_bucket,6



RAW CURRENT-STATE VARIABLES
days_past_due                       PRESENT
outstanding_balance_brl             PRESENT
monthly_salary_brl                  PRESENT
payday_day_of_month                 PRESENT
n_prior_transactions                PRESENT
account_age_months                  PRESENT
days_since_last_app_login           PRESENT
state_uf                            PRESENT

Raw current-state preservation: PASSED

FEATURE FAMILY INVENTORY


,n_columns
dpd,38
salary_affordability,36
payday,35
balance,27
calendar_time,26
digital,26
action,22
relationship,20
geography,8



LAYER 01 — FINAL SUMMARY
Final rows       : 75,406
Final customers  : 11,724
Final columns     : 249
State columns     : 216
Action columns    : 22
Outcome columns   : 4
Constant columns  : 20
Columns with NaN  : 0
Infinite values   : 0

Layer 01 structural audit: PASSED

01I — LAYER 01 FINAL AUDIT COMPLETE


## ______________________________________________________________________________________________________________________________________________________________________

## ______________________________________________________________________________________________________________________________________________________________________

In [16]:
# ============================================================
# 02A — HISTORICAL CONTACT STATE
# BASIC COUNT + RECENCY
# ============================================================
#
# GRAIN
# -----
# 1 row = current WhatsApp send event
#
# STRICT POINT-IN-TIME
# --------------------
# For event t:
#
#   historical features use ONLY events < t
#
# Current event itself must NEVER contribute to its own
# historical features.
#
# NO outcomes are used here:
#   - no delivery
#   - no interaction
#   - no payment
#
# ============================================================

cols_before = features.shape[1]


# ============================================================
# 1. BASIC PREVIOUS-CONTACT COUNT
# ============================================================
#
# event_number:
#
# first message  -> 1
# second message -> 2
#
# Therefore:
#
# previous messages = event_number - 1
# ============================================================

features["hist_n_previous_messages"] = (
    features["event_number"] - 1
).astype("int16")


features["hist_has_previous_message"] = (
    features["hist_n_previous_messages"] > 0
).astype("int8")


features["hist_is_first_contact"] = (
    features["hist_n_previous_messages"] == 0
).astype("int8")


# ============================================================
# 2. PREVIOUS MESSAGE TIMESTAMP
# ============================================================

features["hist_previous_message_at"] = (
    features
    .groupby(
        "customer_id",
        sort=False
    )["sent_at"]
    .shift(1)
)


# ============================================================
# 3. TIME SINCE PREVIOUS MESSAGE
# ============================================================

previous_gap = (
    features["sent_at"]
    - features["hist_previous_message_at"]
)


features["hist_hours_since_previous_message"] = (
    previous_gap.dt.total_seconds()
    / 3600
)


features["hist_days_since_previous_message"] = (
    previous_gap.dt.total_seconds()
    / 86400
)


# ============================================================
# 4. CALENDAR DAYS SINCE PREVIOUS MESSAGE
# ============================================================
#
# Different from elapsed 24h periods.
#
# Example:
#
# Monday 23:00 -> Tuesday 09:00
#
# elapsed = 10h
# calendar-day difference = 1
# ============================================================

previous_date = (
    features["hist_previous_message_at"]
    .dt.normalize()
)


features["hist_calendar_days_since_previous_message"] = (
    features["sent_date"]
    - previous_date
).dt.days


# ============================================================
# 5. FIRST MESSAGE IN OBSERVED HISTORY
# ============================================================
#
# transform("first") includes current event for first contact.
#
# That is acceptable as a reference timestamp:
# it describes the beginning of the OBSERVED contact journey.
#
# It is NOT an outcome.
# ============================================================

features["hist_first_observed_message_at"] = (
    features
    .groupby(
        "customer_id",
        sort=False
    )["sent_at"]
    .transform("first")
)


# ============================================================
# 6. ELAPSED TIME SINCE FIRST OBSERVED CONTACT
# ============================================================

features["hist_days_since_first_observed_message"] = (
    (
        features["sent_at"]
        - features["hist_first_observed_message_at"]
    )
    .dt.total_seconds()
    / 86400
)


# ============================================================
# 7. CONTACT JOURNEY POSITION
# ============================================================

features["hist_contact_number"] = (
    features["event_number"]
    .astype("int16")
)


features["hist_contact_number_log1p"] = np.log1p(
    features["hist_contact_number"]
)


features["hist_previous_messages_log1p"] = np.log1p(
    features["hist_n_previous_messages"]
)


# ============================================================
# 8. PREVIOUS-CONTACT GAP BUCKET
# ============================================================
#
# First contact remains NaN here intentionally.
# ============================================================

features["hist_previous_contact_gap_bucket"] = pd.cut(
    features["hist_days_since_previous_message"],
    bins=[
        -np.inf,
        1,
        2,
        3,
        7,
        14,
        30,
        np.inf,
    ],
    labels=[
        "gap_le_1d",
        "gap_1_2d",
        "gap_2_3d",
        "gap_3_7d",
        "gap_7_14d",
        "gap_14_30d",
        "gap_30d_plus",
    ],
)


# ============================================================
# 9. PREVIOUS-CONTACT RECENCY FLAGS
# ============================================================
#
# IMPORTANT:
#
# First contacts must be 0, not True due to NaN handling.
# ============================================================

has_prev = (
    features["hist_has_previous_message"]
    .eq(1)
)


for window in [1, 2, 3, 7, 14, 30]:

    features[
        f"hist_previous_message_within_{window}d"
    ] = (
        has_prev
        & (
            features[
                "hist_days_since_previous_message"
            ] <= window
        )
    ).astype("int8")


# ============================================================
# 10. RAPID RECONTACT FLAGS
# ============================================================

features["hist_previous_message_within_24h"] = (
    has_prev
    & (
        features[
            "hist_hours_since_previous_message"
        ] <= 24
    )
).astype("int8")


features["hist_previous_message_within_48h"] = (
    has_prev
    & (
        features[
            "hist_hours_since_previous_message"
        ] <= 48
    )
).astype("int8")


features["hist_previous_message_within_72h"] = (
    has_prev
    & (
        features[
            "hist_hours_since_previous_message"
        ] <= 72
    )
).astype("int8")


# ============================================================
# 11. OBSERVED CONTACT DENSITY
# ============================================================
#
# Number of PREVIOUS messages divided by elapsed observed
# journey time.
#
# +1 day stabilizes first/very-early observations.
# ============================================================

features["hist_previous_messages_per_observed_day"] = (
    features["hist_n_previous_messages"]
    / (
        features[
            "hist_days_since_first_observed_message"
        ] + 1
    )
)


features["hist_previous_messages_per_observed_week"] = (
    features[
        "hist_previous_messages_per_observed_day"
    ]
    * 7
)


# ============================================================
# 12. CONTACT POSITION FLAGS
# ============================================================

features["hist_is_second_contact"] = (
    features["hist_contact_number"] == 2
).astype("int8")


features["hist_is_third_contact"] = (
    features["hist_contact_number"] == 3
).astype("int8")


features["hist_is_5plus_contact"] = (
    features["hist_contact_number"] >= 5
).astype("int8")


features["hist_is_10plus_contact"] = (
    features["hist_contact_number"] >= 10
).astype("int8")


# ============================================================
# 13. STRICT PIT AUDIT
# ============================================================
#
# Every previous timestamp must be STRICTLY earlier than
# current timestamp.
# ============================================================

previous_rows = (
    features["hist_has_previous_message"]
    .eq(1)
)

assert (
    features.loc[
        previous_rows,
        "hist_previous_message_at"
    ]
    <
    features.loc[
        previous_rows,
        "sent_at"
    ]
).all()


# First contacts must have no previous timestamp
first_rows = (
    features["hist_is_first_contact"]
    .eq(1)
)

assert (
    features.loc[
        first_rows,
        "hist_previous_message_at"
    ]
    .isna()
    .all()
)


# Non-first contacts must have previous timestamp
assert (
    features.loc[
        ~first_rows,
        "hist_previous_message_at"
    ]
    .notna()
    .all()
)


# Previous count must match event number exactly
assert (
    features["hist_n_previous_messages"]
    ==
    features["event_number"] - 1
).all()


# ============================================================
# 14. AUDIT OUTPUT
# ============================================================

new_cols = features.columns[cols_before:]

print("=" * 100)
print("02A — HISTORICAL CONTACT COUNT + RECENCY COMPLETE")
print("=" * 100)

print(f"Rows          : {len(features):,}")
print(
    f"Customers     : "
    f"{features['customer_id'].nunique():,}"
)
print(f"Columns before: {cols_before:,}")
print(f"Columns after : {features.shape[1]:,}")
print(f"New features  : {len(new_cols):,}")

print("\nStrict PIT contact-history checks: PASSED")


# ============================================================
# 15. FIRST VS REPEAT CONTACT
# ============================================================

print("\nCONTACT HISTORY AVAILABILITY")
print("-" * 100)

print(
    f"First contacts  : "
    f"{features['hist_is_first_contact'].sum():,}"
)

print(
    f"Repeat contacts : "
    f"{features['hist_has_previous_message'].sum():,}"
)

print(
    f"Expected first contacts (= customers): "
    f"{features['customer_id'].nunique():,}"
)


# ============================================================
# 16. PREVIOUS MESSAGE COUNT DISTRIBUTION
# ============================================================

print("\nNUMBER OF PREVIOUS MESSAGES")

display(
    features[
        "hist_n_previous_messages"
    ]
    .describe(
        percentiles=[
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame()
)


# ============================================================
# 17. PREVIOUS CONTACT GAP
# ============================================================

print("\nDAYS SINCE PREVIOUS MESSAGE — REPEAT CONTACTS ONLY")

display(
    features.loc[
        previous_rows,
        "hist_days_since_previous_message"
    ]
    .describe(
        percentiles=[
            .01,
            .05,
            .10,
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame()
)


print("\nPREVIOUS CONTACT GAP BUCKET")

display(
    features.loc[
        previous_rows,
        "hist_previous_contact_gap_bucket"
    ]
    .value_counts(sort=False)
    .to_frame("messages")
)


# ============================================================
# 18. RAPID RECONTACT
# ============================================================

print("\nRAPID RECONTACT — REPEAT CONTACTS")

repeat_n = previous_rows.sum()

rapid_summary = pd.DataFrame(
    {
        "messages": [
            features.loc[
                previous_rows,
                "hist_previous_message_within_24h"
            ].sum(),

            features.loc[
                previous_rows,
                "hist_previous_message_within_48h"
            ].sum(),

            features.loc[
                previous_rows,
                "hist_previous_message_within_72h"
            ].sum(),
        ]
    },
    index=[
        "within_24h",
        "within_48h",
        "within_72h",
    ],
)

rapid_summary["pct_repeat_contacts"] = (
    rapid_summary["messages"]
    / repeat_n
    * 100
)

display(
    rapid_summary
)


# ============================================================
# 19. SAMPLE — ONE CUSTOMER JOURNEY
# ============================================================

sample_customer = (
    features[
        "customer_id"
    ]
    .value_counts()
    .index[0]
)


sample_cols = [
    "customer_id",
    "event_number",
    "sent_at",
    "current_dpd",
    "action_template",
    "hist_n_previous_messages",
    "hist_previous_message_at",
    "hist_hours_since_previous_message",
    "hist_days_since_previous_message",
    "hist_days_since_first_observed_message",
]


print(
    f"\nSAMPLE JOURNEY — {sample_customer}"
)

display(
    features.loc[
        features["customer_id"]
        == sample_customer,
        sample_cols,
    ]
)


print("\n" + "=" * 100)
print("02A — HISTORICAL CONTACT COUNT + RECENCY COMPLETE")
print("=" * 100)

02A — HISTORICAL CONTACT COUNT + RECENCY COMPLETE
Rows          : 75,406
Customers     : 11,724
Columns before: 249
Columns after : 277
New features  : 28

Strict PIT contact-history checks: PASSED

CONTACT HISTORY AVAILABILITY
----------------------------------------------------------------------------------------------------
First contacts  : 11,724
Repeat contacts : 63,682
Expected first contacts (= customers): 11,724

NUMBER OF PREVIOUS MESSAGES


,hist_n_previous_messages
count,"75,406.00"
mean,3.81
std,3.19
min,0.00
25%,1.00
50%,3.00
75%,6.00
90%,8.00
95%,10.00
99%,13.00



DAYS SINCE PREVIOUS MESSAGE — REPEAT CONTACTS ONLY


,hist_days_since_previous_message
count,"63,682.00"
mean,4.49
std,4.53
min,0.51
1%,0.64
5%,0.82
10%,0.95
25%,1.34
50%,3.02
75%,5.90



PREVIOUS CONTACT GAP BUCKET


...


RAPID RECONTACT — REPEAT CONTACTS


,messages,pct_repeat_contacts
within_24h,8359,13.13
within_48h,22022,34.58
within_72h,31492,49.45



SAMPLE JOURNEY — C006634


,customer_id,event_number,sent_at,current_dpd,action_template,hist_n_previous_messages,hist_previous_message_at,hist_hours_since_previous_message,hist_days_since_previous_message,hist_days_since_first_observed_message
42054,C006634,1,2026-06-14 11:21:00,2,friendly_reminder,0,NaT,NaN,NaN,0.00
42055,C006634,2,2026-06-15 14:39:00,3,friendly_reminder,1,2026-06-14 11:21:00,27.30,1.14,1.14
42056,C006634,3,2026-06-16 11:15:00,4,friendly_reminder,2,2026-06-15 14:39:00,20.60,0.86,2.00
42057,C006634,4,2026-06-17 10:55:00,5,friendly_reminder,3,2026-06-16 11:15:00,23.67,0.99,2.98
42058,C006634,5,2026-06-18 13:52:00,6,pix_link,4,2026-06-17 10:55:00,26.95,1.12,4.10
42059,C006634,6,2026-06-21 17:36:00,9,urgent_reminder,5,2026-06-18 13:52:00,75.73,3.16,7.26
42060,C006634,7,2026-06-22 16:56:00,10,urgent_reminder,6,2026-06-21 17:36:00,23.33,0.97,8.23
42061,C006634,8,2026-06-23 12:11:00,11,urgent_reminder,7,2026-06-22 16:56:00,19.25,0.80,9.03
42062,C006634,9,2026-06-24 20:52:00,12,pix_link,8,2026-06-23 12:11:00,32.68,1.36,10.40
42063,C006634,10,2026-06-25 11:36:00,13,urgent_reminder,9,2026-06-24 20:52:00,14.73,0.61,11.01



02A — HISTORICAL CONTACT COUNT + RECENCY COMPLETE


In [17]:
# ============================================================
# 02B — HISTORICAL CONTACT PRESSURE
# ROLLING TIME WINDOWS — STRICT PIT
# CORRECTED VERSION
# ============================================================
#
# PURPOSE
# -------
# Measure contact pressure BEFORE the current message.
#
# For event t:
#
#   count previous sends occurring inside each lookback window
#
# Window semantics:
#
#   [t - window, t)
#
# Current event is ALWAYS excluded.
#
# NO outcomes.
# NO templates.
#
# ============================================================


# ============================================================
# 0. CLEAN POSSIBLE PARTIAL OUTPUT FROM FAILED RUN
# ============================================================

partial_02b_cols = [
    col
    for col in features.columns
    if (
        col.startswith("hist_n_messages_last_")
        or col.startswith("hist_any_message_last_")
        or col.startswith("hist_3plus_messages_last_")
        or col.startswith("hist_5plus_messages_last_")
        or col.startswith("hist_10plus_messages_last_")
        or col.startswith("hist_share_previous_messages_last_")
        or col.startswith("hist_contact_pressure_")
        or col == "audit_reconstructed_14d_minus_raw"
    )
]

if partial_02b_cols:

    print(
        f"Removing {len(partial_02b_cols)} "
        f"partial 02B columns from failed run..."
    )

    features.drop(
        columns=partial_02b_cols,
        inplace=True,
    )


# ============================================================
# 1. STARTING CHECKPOINT
# ============================================================

cols_before = features.shape[1]

assert cols_before == 277, (
    f"Expected 277 columns before 02B, "
    f"found {cols_before}"
)

assert len(features) == 75_406
assert features["event_id"].is_unique


print("=" * 100)
print("02B — HISTORICAL CONTACT PRESSURE")
print("=" * 100)

print(
    f"Starting shape: "
    f"{features.shape}"
)


# ============================================================
# 2. MINIMAL TEMPORARY TABLE
# ============================================================
#
# Work only with the columns required for temporal counting.
# Avoid rolling over the wide 277-column dataframe.
# ============================================================

contact_tmp = features[
    [
        "event_id",
        "customer_id",
        "sent_at",
    ]
].copy()


# ============================================================
# 3. VERIFY STRICT CHRONOLOGICAL ORDER
# ============================================================

time_diff = (
    contact_tmp
    .groupby(
        "customer_id",
        sort=False
    )["sent_at"]
    .diff()
)


assert (
    time_diff
    .dropna()
    .gt(pd.Timedelta(0))
    .all()
)

print(
    "Chronological ordering: PASSED"
)


# ============================================================
# 4. ROLLING WINDOWS
# ============================================================

rolling_windows = {
    "24h": pd.Timedelta(hours=24),
    "48h": pd.Timedelta(hours=48),
    "72h": pd.Timedelta(hours=72),
    "3d":  pd.Timedelta(days=3),
    "7d":  pd.Timedelta(days=7),
    "14d": pd.Timedelta(days=14),
    "30d": pd.Timedelta(days=30),
}


# ============================================================
# 5. OUTPUT ARRAYS
# ============================================================
#
# One array per rolling window.
#
# Position corresponds exactly to features.index position.
# ============================================================

rolling_arrays = {
    suffix: np.zeros(
        len(contact_tmp),
        dtype=np.int16,
    )
    for suffix in rolling_windows
}


# ============================================================
# 6. CUSTOMER GROUP POSITIONS
# ============================================================
#
# groupby.indices gives row POSITIONS for each customer.
#
# This avoids:
#   - wide dataframe groupby
#   - apply/explode
#   - index-alignment problems
# ============================================================

customer_positions = (
    contact_tmp
    .groupby(
        "customer_id",
        sort=False
    )
    .indices
)


sent_ns_all = (
    contact_tmp["sent_at"]
    .astype("int64")
    .to_numpy(
        copy=False
    )
)


# ============================================================
# 7. STRICT PIT WINDOW COUNTS
# ============================================================
#
# For each customer:
#
# current event position = i
#
# left boundary:
#
#   timestamp >= current_timestamp - window
#
# right boundary:
#
#   i
#
# Because i itself is excluded, only events BEFORE the
# current event are counted.
#
# np.searchsorted(..., side="left") implements:
#
#   [t - window, t)
#
# ============================================================

window_ns = {
    suffix: int(
        window.value
    )
    for suffix, window in rolling_windows.items()
}


for customer_id, positions in customer_positions.items():

    positions = np.asarray(
        positions,
        dtype=np.int64,
    )

    customer_times = (
        sent_ns_all[
            positions
        ]
    )

    # Defensive chronological check
    if len(customer_times) > 1:

        assert np.all(
            np.diff(
                customer_times
            ) > 0
        )

    current_position = np.arange(
        len(customer_times),
        dtype=np.int16,
    )

    for suffix, lookback_ns in window_ns.items():

        lower_bound = (
            customer_times
            - lookback_ns
        )

        left_position = np.searchsorted(
            customer_times,
            lower_bound,
            side="left",
        )

        counts = (
            current_position
            - left_position
        )

        rolling_arrays[
            suffix
        ][positions] = (
            counts.astype(
                np.int16
            )
        )


# ============================================================
# 8. ADD COUNTS TO FEATURE TABLE
# ============================================================

for suffix in rolling_windows:

    features[
        f"hist_n_messages_last_{suffix}"
    ] = rolling_arrays[
        suffix
    ]


# Free temporary arrays we no longer need
del rolling_arrays
del customer_positions
del sent_ns_all


# ============================================================
# 9. ROLLING COLUMN LIST
# ============================================================

rolling_cols = [
    f"hist_n_messages_last_{suffix}"
    for suffix in rolling_windows
]


# ============================================================
# 10. FIRST CONTACT AUDIT
# ============================================================

first_mask = (
    features[
        "hist_is_first_contact"
    ]
    .eq(1)
)


assert (
    features.loc[
        first_mask,
        rolling_cols,
    ]
    .eq(0)
    .all()
    .all()
)


# ============================================================
# 11. COUNTS CANNOT EXCEED TOTAL PREVIOUS MESSAGES
# ============================================================

for col in rolling_cols:

    assert (
        features[col]
        <= features[
            "hist_n_previous_messages"
        ]
    ).all()


# ============================================================
# 12. MONOTONIC WINDOW AUDIT
# ============================================================

assert (
    features[
        "hist_n_messages_last_24h"
    ]
    <=
    features[
        "hist_n_messages_last_48h"
    ]
).all()


assert (
    features[
        "hist_n_messages_last_48h"
    ]
    <=
    features[
        "hist_n_messages_last_72h"
    ]
).all()


# Same exact duration
assert (
    features[
        "hist_n_messages_last_72h"
    ]
    ==
    features[
        "hist_n_messages_last_3d"
    ]
).all()


assert (
    features[
        "hist_n_messages_last_3d"
    ]
    <=
    features[
        "hist_n_messages_last_7d"
    ]
).all()


assert (
    features[
        "hist_n_messages_last_7d"
    ]
    <=
    features[
        "hist_n_messages_last_14d"
    ]
).all()


assert (
    features[
        "hist_n_messages_last_14d"
    ]
    <=
    features[
        "hist_n_messages_last_30d"
    ]
).all()


print(
    "Strict PIT rolling-window checks: PASSED"
)


# ============================================================
# 13. CONTACT-PRESENCE FLAGS
# ============================================================

features["hist_any_message_last_24h"] = (
    features[
        "hist_n_messages_last_24h"
    ] > 0
).astype("int8")


features["hist_any_message_last_72h"] = (
    features[
        "hist_n_messages_last_72h"
    ] > 0
).astype("int8")


features["hist_any_message_last_7d"] = (
    features[
        "hist_n_messages_last_7d"
    ] > 0
).astype("int8")


features["hist_any_message_last_14d"] = (
    features[
        "hist_n_messages_last_14d"
    ] > 0
).astype("int8")


features["hist_any_message_last_30d"] = (
    features[
        "hist_n_messages_last_30d"
    ] > 0
).astype("int8")


# ============================================================
# 14. PRESSURE THRESHOLD FLAGS
# ============================================================

features["hist_3plus_messages_last_7d"] = (
    features[
        "hist_n_messages_last_7d"
    ] >= 3
).astype("int8")


features["hist_5plus_messages_last_7d"] = (
    features[
        "hist_n_messages_last_7d"
    ] >= 5
).astype("int8")


features["hist_3plus_messages_last_14d"] = (
    features[
        "hist_n_messages_last_14d"
    ] >= 3
).astype("int8")


features["hist_5plus_messages_last_14d"] = (
    features[
        "hist_n_messages_last_14d"
    ] >= 5
).astype("int8")


features["hist_10plus_messages_last_30d"] = (
    features[
        "hist_n_messages_last_30d"
    ] >= 10
).astype("int8")


# ============================================================
# 15. SHARE OF PREVIOUS HISTORY CONCENTRATED RECENTLY
# ============================================================
#
# First contacts intentionally remain NaN.
# ============================================================

safe_previous = (
    features[
        "hist_n_previous_messages"
    ]
    .replace(
        0,
        np.nan,
    )
)


features[
    "hist_share_previous_messages_last_7d"
] = (
    features[
        "hist_n_messages_last_7d"
    ]
    / safe_previous
)


features[
    "hist_share_previous_messages_last_14d"
] = (
    features[
        "hist_n_messages_last_14d"
    ]
    / safe_previous
)


features[
    "hist_share_previous_messages_last_30d"
] = (
    features[
        "hist_n_messages_last_30d"
    ]
    / safe_previous
)


# ============================================================
# 16. PRESSURE BUCKET — 7 DAYS
# ============================================================

features[
    "hist_contact_pressure_7d"
] = pd.cut(
    features[
        "hist_n_messages_last_7d"
    ],
    bins=[
        -1,
        0,
        1,
        2,
        4,
        7,
        np.inf,
    ],
    labels=[
        "none",
        "one",
        "two",
        "three_four",
        "five_seven",
        "eight_plus",
    ],
)


# ============================================================
# 17. PRESSURE BUCKET — 14 DAYS
# ============================================================

features[
    "hist_contact_pressure_14d"
] = pd.cut(
    features[
        "hist_n_messages_last_14d"
    ],
    bins=[
        -1,
        0,
        1,
        2,
        4,
        7,
        10,
        np.inf,
    ],
    labels=[
        "none",
        "one",
        "two",
        "three_four",
        "five_seven",
        "eight_ten",
        "eleven_plus",
    ],
)


# ============================================================
# 18. AUDIT AGAINST RAW n_msgs_last_14d
# ============================================================

raw_14d = pd.to_numeric(
    features[
        "n_msgs_last_14d"
    ],
    errors="coerce",
)


assert raw_14d.notna().all()


features[
    "audit_reconstructed_14d_minus_raw"
] = (
    features[
        "hist_n_messages_last_14d"
    ]
    - raw_14d
)


# ============================================================
# 19. FINAL STRUCTURAL AUDIT
# ============================================================

new_cols = (
    features.columns[
        cols_before:
    ]
)


print("\n" + "=" * 100)
print("02B — HISTORICAL CONTACT PRESSURE COMPLETE")
print("=" * 100)

print(
    f"Rows          : "
    f"{len(features):,}"
)

print(
    f"Customers     : "
    f"{features['customer_id'].nunique():,}"
)

print(
    f"Columns before: "
    f"{cols_before:,}"
)

print(
    f"Columns after : "
    f"{features.shape[1]:,}"
)

print(
    f"New features  : "
    f"{len(new_cols):,}"
)


# ============================================================
# 20. ROLLING COUNT DISTRIBUTIONS
# ============================================================

print(
    "\nROLLING CONTACT COUNTS"
)

display(
    features[
        rolling_cols
    ]
    .describe(
        percentiles=[
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .T
)


# ============================================================
# 21. PRESSURE DISTRIBUTIONS
# ============================================================

print(
    "\n7-DAY CONTACT PRESSURE"
)

display(
    features[
        "hist_contact_pressure_7d"
    ]
    .value_counts(
        sort=False
    )
    .to_frame(
        "messages"
    )
)


print(
    "\n14-DAY CONTACT PRESSURE"
)

display(
    features[
        "hist_contact_pressure_14d"
    ]
    .value_counts(
        sort=False
    )
    .to_frame(
        "messages"
    )
)


# ============================================================
# 22. RAW VS RECONSTRUCTED 14D
# ============================================================

comparison = pd.DataFrame(
    {
        "raw_14d":
            raw_14d,

        "reconstructed_14d":
            features[
                "hist_n_messages_last_14d"
            ],
    }
)


comparison["difference"] = (
    comparison[
        "reconstructed_14d"
    ]
    - comparison[
        "raw_14d"
    ]
)


exact_match = (
    comparison[
        "difference"
    ]
    .eq(0)
)


print(
    "\nRAW n_msgs_last_14d "
    "VS RECONSTRUCTED STRICT-PIT 14D"
)

print("-" * 100)

print(
    f"Exact matches : "
    f"{exact_match.sum():,} "
    f"({exact_match.mean():.2%})"
)

print(
    f"Mismatches    : "
    f"{(~exact_match).sum():,} "
    f"({(~exact_match).mean():.2%})"
)


print(
    "\nDIFFERENCE DISTRIBUTION"
)

display(
    comparison[
        "difference"
    ]
    .value_counts()
    .sort_index()
    .to_frame(
        "messages"
    )
)


# ============================================================
# 23. SAMPLE MISMATCHES
# ============================================================

mismatch_mask = (
    comparison[
        "difference"
    ]
    .ne(0)
)


print(
    "\nSAMPLE — RAW / RECONSTRUCTED 14D MISMATCHES"
)

display(
    features.loc[
        mismatch_mask,
        [
            "customer_id",
            "event_number",
            "sent_at",
            "n_msgs_last_14d",
            "hist_n_messages_last_14d",
            "audit_reconstructed_14d_minus_raw",
        ],
    ]
    .head(20)
)


# ============================================================
# 24. MANUAL JOURNEY AUDIT
# ============================================================

sample_customer = (
    features[
        "customer_id"
    ]
    .value_counts()
    .index[0]
)


print(
    f"\nSAMPLE ROLLING JOURNEY — "
    f"{sample_customer}"
)


display(
    features.loc[
        features[
            "customer_id"
        ].eq(
            sample_customer
        ),
        [
            "customer_id",
            "event_number",
            "sent_at",
            "hist_n_previous_messages",
            "hist_n_messages_last_24h",
            "hist_n_messages_last_3d",
            "hist_n_messages_last_7d",
            "hist_n_messages_last_14d",
            "hist_n_messages_last_30d",
            "n_msgs_last_14d",
        ],
    ]
)


print(
    "\n" + "=" * 100
)

print(
    "02B — HISTORICAL CONTACT PRESSURE COMPLETE"
)

print(
    "=" * 100
)

02B — HISTORICAL CONTACT PRESSURE
Starting shape: (75406, 277)
Chronological ordering: PASSED
Strict PIT rolling-window checks: PASSED

02B — HISTORICAL CONTACT PRESSURE COMPLETE
Rows          : 75,406
Customers     : 11,724
Columns before: 277
Columns after : 300
New features  : 23

ROLLING CONTACT COUNTS


,count,mean,std,min,25%,50%,75%,90%,95%,99%,max
hist_n_messages_last_24h,"75,406.00",0.11,0.31,0.00,0.00,0.00,0.00,1.00,1.00,1.00,1.00
hist_n_messages_last_48h,"75,406.00",0.32,0.52,0.00,0.00,0.00,1.00,1.00,1.00,2.00,2.00
hist_n_messages_last_72h,"75,406.00",0.51,0.67,0.00,0.00,0.00,1.00,1.00,2.00,2.00,3.00
hist_n_messages_last_3d,"75,406.00",0.51,0.67,0.00,0.00,0.00,1.00,1.00,2.00,2.00,3.00
hist_n_messages_last_7d,"75,406.00",1.22,1.11,0.00,0.00,1.00,2.00,3.00,3.00,4.00,7.00
hist_n_messages_last_14d,"75,406.00",2.11,1.68,0.00,1.00,2.00,3.00,4.00,5.00,7.00,10.00
hist_n_messages_last_30d,"75,406.00",3.23,2.53,0.00,1.00,3.00,5.00,7.00,8.00,10.00,15.00



7-DAY CONTACT PRESSURE


,messages
hist_contact_pressure_7d,
none,23566
one,25109
two,16694
three_four,9592
five_seven,445
eight_plus,0



14-DAY CONTACT PRESSURE


...


SAMPLE — RAW / RECONSTRUCTED 14D MISMATCHES


,customer_id,event_number,sent_at,n_msgs_last_14d,hist_n_messages_last_14d,audit_reconstructed_14d_minus_raw
32,C000005,5,2026-08-25 19:13:00,2,1,-1
146,C000024,7,2026-08-28 11:55:00,5,4,-1
152,C000025,6,2026-08-05 14:29:00,2,1,-1
183,C000028,7,2026-07-31 10:37:00,5,4,-1
184,C000028,8,2026-08-01 17:24:00,5,4,-1
221,C000036,6,2026-08-19 13:55:00,5,4,-1
228,C000037,3,2026-07-21 16:52:00,2,1,-1
242,C000038,9,2026-07-28 15:39:00,4,3,-1
254,C000040,7,2026-07-16 11:27:00,4,3,-1
255,C000040,8,2026-07-30 11:32:00,1,0,-1



SAMPLE ROLLING JOURNEY — C006634


,customer_id,event_number,sent_at,hist_n_previous_messages,hist_n_messages_last_24h,hist_n_messages_last_3d,hist_n_messages_last_7d,hist_n_messages_last_14d,hist_n_messages_last_30d,n_msgs_last_14d
42054,C006634,1,2026-06-14 11:21:00,0,0,0,0,0,0,0
42055,C006634,2,2026-06-15 14:39:00,1,0,1,1,1,1,1
42056,C006634,3,2026-06-16 11:15:00,2,1,2,2,2,2,2
42057,C006634,4,2026-06-17 10:55:00,3,1,3,3,3,3,3
42058,C006634,5,2026-06-18 13:52:00,4,0,3,4,4,4,4
42059,C006634,6,2026-06-21 17:36:00,5,0,0,4,5,5,5
42060,C006634,7,2026-06-22 16:56:00,6,1,1,4,6,6,6
42061,C006634,8,2026-06-23 12:11:00,7,1,2,4,7,7,7
42062,C006634,9,2026-06-24 20:52:00,8,0,2,4,8,8,8
42063,C006634,10,2026-06-25 11:36:00,9,1,3,5,9,9,9



02B — HISTORICAL CONTACT PRESSURE COMPLETE


In [18]:
# ============================================================
# 02B.1 — INVESTIGATE RAW VS RECONSTRUCTED 14D MISMATCH
# ============================================================
#
# OBSERVED:
#
# reconstructed - raw:
#
#     0  -> 71,276
#    -1  ->  4,130
#
# Goal:
# determine WHY raw n_msgs_last_14d contains exactly one
# additional message for these events.
#
# No features are created here.
# Pure audit.
# ============================================================


print("=" * 100)
print("02B.1 — INVESTIGATE 14D COUNT MISMATCH")
print("=" * 100)


# ============================================================
# 1. DEFINE MISMATCHES
# ============================================================

mismatch = (
    features[
        "audit_reconstructed_14d_minus_raw"
    ]
    .ne(0)
)


print(
    f"Mismatch events    : "
    f"{mismatch.sum():,}"
)

print(
    f"Mismatch customers : "
    f"{features.loc[mismatch, 'customer_id'].nunique():,}"
)

print(
    f"Mismatch rate      : "
    f"{mismatch.mean():.2%}"
)


# ============================================================
# 2. VERIFY DIFFERENCE IS ALWAYS -1
# ============================================================

difference_counts = (
    features.loc[
        mismatch,
        "audit_reconstructed_14d_minus_raw"
    ]
    .value_counts()
    .sort_index()
)


print("\nMISMATCH DIFFERENCES")

display(
    difference_counts
    .to_frame("events")
)


assert (
    features.loc[
        mismatch,
        "audit_reconstructed_14d_minus_raw"
    ]
    .eq(-1)
    .all()
)


# ============================================================
# 3. TIME FROM CURRENT EVENT TO FIRST OBSERVED MESSAGE
# ============================================================

features_audit = features.loc[
    :,
    [
        "customer_id",
        "event_number",
        "sent_at",
        "hist_first_observed_message_at",
        "hist_days_since_first_observed_message",
        "hist_n_previous_messages",
        "n_msgs_last_14d",
        "hist_n_messages_last_14d",
        "audit_reconstructed_14d_minus_raw",
    ]
].copy()


features_audit["is_mismatch"] = mismatch.to_numpy()


# ============================================================
# 4. ARE MISMATCHES NEAR BEGINNING OF OBSERVED HISTORY?
# ============================================================

print("\nEVENT NUMBER — MATCH VS MISMATCH")

event_number_summary = (
    features_audit
    .groupby(
        "is_mismatch"
    )["event_number"]
    .describe(
        percentiles=[
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
)

display(event_number_summary)


# ============================================================
# 5. JOURNEY AGE — MATCH VS MISMATCH
# ============================================================

print("\nDAYS SINCE FIRST OBSERVED MESSAGE — MATCH VS MISMATCH")

journey_age_summary = (
    features_audit
    .groupby(
        "is_mismatch"
    )[
        "hist_days_since_first_observed_message"
    ]
    .describe(
        percentiles=[
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
)

display(journey_age_summary)


# ============================================================
# 6. CHECK IF MISMATCHES OCCUR WITHIN FIRST 14 DAYS
# ============================================================

within_first_14d = (
    features[
        "hist_days_since_first_observed_message"
    ]
    <= 14
)


mismatch_within_first_14d = (
    mismatch
    & within_first_14d
)


mismatch_after_first_14d = (
    mismatch
    & ~within_first_14d
)


print("\nMISMATCH LOCATION IN OBSERVED JOURNEY")
print("-" * 100)

print(
    f"Within first 14d : "
    f"{mismatch_within_first_14d.sum():,} "
    f"({mismatch_within_first_14d.sum() / mismatch.sum():.2%})"
)

print(
    f"After first 14d  : "
    f"{mismatch_after_first_14d.sum():,} "
    f"({mismatch_after_first_14d.sum() / mismatch.sum():.2%})"
)


# ============================================================
# 7. CHECK DATASET LEFT BOUNDARY
# ============================================================
#
# If raw feature was computed using data before Jun/01,
# mismatches may concentrate near beginning of dataset.
# ============================================================

dataset_start = (
    features["sent_at"]
    .min()
)


features_audit["days_since_dataset_start"] = (
    features_audit["sent_at"]
    - dataset_start
).dt.total_seconds() / 86400


print(
    f"\nDataset start: {dataset_start}"
)


print(
    "\nDAYS SINCE DATASET START — MATCH VS MISMATCH"
)

display(
    features_audit
    .groupby(
        "is_mismatch"
    )[
        "days_since_dataset_start"
    ]
    .describe(
        percentiles=[
            .10,
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
)


# ============================================================
# 8. MISMATCH RATE BY SEND MONTH
# ============================================================

features_audit["send_month"] = (
    features_audit["sent_at"]
    .dt.to_period("M")
    .astype(str)
)


month_audit = (
    features_audit
    .groupby(
        "send_month"
    )
    .agg(
        events=(
            "is_mismatch",
            "size",
        ),
        mismatches=(
            "is_mismatch",
            "sum",
        ),
    )
)


month_audit["mismatch_pct"] = (
    month_audit["mismatches"]
    / month_audit["events"]
    * 100
)


print("\nMISMATCH RATE BY MONTH")

display(month_audit)


# ============================================================
# 9. CHECK EXACT 14-DAY BOUNDARY
# ============================================================
#
# For each event, ask:
#
# Was there a previous message approximately 14 days ago?
#
# We'll calculate the gap from every previous message for
# mismatch events using only the narrow contact table.
# ============================================================

mismatch_event_ids = set(
    features.loc[
        mismatch,
        "event_id"
    ]
)


contact_narrow = features[
    [
        "event_id",
        "customer_id",
        "sent_at",
    ]
].copy()


# ============================================================
# 10. FIND CLOSEST PREVIOUS MESSAGE TO 14-DAY BOUNDARY
# ============================================================

boundary_records = []


for customer_id, group in contact_narrow.groupby(
    "customer_id",
    sort=False,
):

    mismatch_group = group[
        group["event_id"].isin(
            mismatch_event_ids
        )
    ]

    if mismatch_group.empty:
        continue

    all_times = (
        group["sent_at"]
        .to_numpy()
    )

    all_event_ids = (
        group["event_id"]
        .to_numpy()
    )

    for row in mismatch_group.itertuples(
        index=False
    ):

        current_time = (
            pd.Timestamp(
                row.sent_at
            )
        )

        previous_mask = (
            all_times
            <
            np.datetime64(
                current_time
            )
        )

        previous_times = (
            all_times[
                previous_mask
            ]
        )

        if len(previous_times) == 0:
            continue

        gaps_days = (
            current_time.to_datetime64()
            - previous_times
        ) / np.timedelta64(
            1,
            "D",
        )

        distance_to_14 = np.abs(
            gaps_days
            - 14
        )

        nearest_idx = np.argmin(
            distance_to_14
        )

        boundary_records.append(
            {
                "event_id":
                    row.event_id,

                "customer_id":
                    customer_id,

                "sent_at":
                    current_time,

                "nearest_gap_to_14d":
                    float(
                        gaps_days[
                            nearest_idx
                        ]
                    ),

                "distance_from_14d":
                    float(
                        distance_to_14[
                            nearest_idx
                        ]
                    ),
            }
        )


boundary_audit = pd.DataFrame(
    boundary_records
)


# ============================================================
# 11. BOUNDARY DISTRIBUTION
# ============================================================

print(
    "\nCLOSEST PREVIOUS MESSAGE TO 14-DAY BOUNDARY"
)

display(
    boundary_audit[
        "nearest_gap_to_14d"
    ]
    .describe(
        percentiles=[
            .01,
            .05,
            .10,
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame()
)


# ============================================================
# 12. HOW MANY ARE VERY CLOSE TO EXACTLY 14 DAYS?
# ============================================================

for tolerance_hours in [
    1,
    6,
    12,
    24,
]:

    tolerance_days = (
        tolerance_hours
        / 24
    )

    count = (
        boundary_audit[
            "distance_from_14d"
        ]
        <= tolerance_days
    ).sum()

    print(
        f"Within ±{tolerance_hours:>2}h "
        f"of 14d boundary : "
        f"{count:,} "
        f"({count / len(boundary_audit):.2%})"
    )


# ============================================================
# 13. MISMATCH BY EVENT NUMBER
# ============================================================

event_number_audit = (
    features_audit
    .groupby(
        "event_number"
    )
    .agg(
        events=(
            "is_mismatch",
            "size",
        ),
        mismatches=(
            "is_mismatch",
            "sum",
        ),
    )
)


event_number_audit["mismatch_pct"] = (
    event_number_audit["mismatches"]
    / event_number_audit["events"]
    * 100
)


print("\nMISMATCH RATE BY EVENT NUMBER")

display(
    event_number_audit
)


# ============================================================
# 14. SAMPLE — MISMATCHES AFTER FIRST 14 OBSERVED DAYS
# ============================================================

print(
    "\nSAMPLE — MISMATCHES AFTER FIRST 14 OBSERVED DAYS"
)

display(
    features.loc[
        mismatch_after_first_14d,
        [
            "customer_id",
            "event_number",
            "sent_at",
            "hist_first_observed_message_at",
            "hist_days_since_first_observed_message",
            "hist_n_previous_messages",
            "n_msgs_last_14d",
            "hist_n_messages_last_14d",
            "audit_reconstructed_14d_minus_raw",
        ],
    ]
    .head(30)
)


print("\n" + "=" * 100)
print("02B.1 — INVESTIGATION COMPLETE")
print("=" * 100)

02B.1 — INVESTIGATE 14D COUNT MISMATCH
Mismatch events    : 4,130
Mismatch customers : 3,153
Mismatch rate      : 5.48%

MISMATCH DIFFERENCES


,events
audit_reconstructed_14d_minus_raw,
-1,4130



EVENT NUMBER — MATCH VS MISMATCH


,count,mean,std,min,25%,50%,75%,90%,95%,99%,max
is_mismatch,,,,,,,,,,,
False,"71,276.00",4.64,3.15,1.00,2.00,4.00,7.00,9.00,11.00,13.00,21.00
True,"4,130.00",7.71,2.51,2.00,6.00,7.00,9.00,11.00,12.00,14.00,20.00



DAYS SINCE FIRST OBSERVED MESSAGE — MATCH VS MISMATCH


...

Within ± 1h of 14d boundary : 780 (18.89%)
Within ± 6h of 14d boundary : 3,177 (76.92%)
Within ±12h of 14d boundary : 4,130 (100.00%)
Within ±24h of 14d boundary : 4,130 (100.00%)

MISMATCH RATE BY EVENT NUMBER


,events,mismatches,mismatch_pct
event_number,,,
1,11724,0,0.00
2,10551,11,0.10
3,9524,86,0.90
4,8547,241,2.82
5,7573,423,5.59
6,6578,649,9.87
7,5598,689,12.31
8,4563,643,14.09
9,3565,476,13.35



SAMPLE — MISMATCHES AFTER FIRST 14 OBSERVED DAYS


,customer_id,event_number,sent_at,hist_first_observed_message_at,hist_days_since_first_observed_message,hist_n_previous_messages,n_msgs_last_14d,hist_n_messages_last_14d,audit_reconstructed_14d_minus_raw
32,C000005,5,2026-08-25 19:13:00,2026-08-05 14:36:00,20.19,4,2,1,-1
146,C000024,7,2026-08-28 11:55:00,2026-08-13 19:59:00,14.66,6,5,4,-1
152,C000025,6,2026-08-05 14:29:00,2026-07-13 10:33:00,23.16,5,2,1,-1
183,C000028,7,2026-07-31 10:37:00,2026-07-16 10:54:00,14.99,6,5,4,-1
184,C000028,8,2026-08-01 17:24:00,2026-07-16 10:54:00,16.27,7,5,4,-1
221,C000036,6,2026-08-19 13:55:00,2026-08-05 11:54:00,14.08,5,5,4,-1
228,C000037,3,2026-07-21 16:52:00,2026-07-07 13:40:00,14.13,2,2,1,-1
242,C000038,9,2026-07-28 15:39:00,2026-06-30 14:36:00,28.04,8,4,3,-1
254,C000040,7,2026-07-16 11:27:00,2026-06-30 13:32:00,15.91,6,4,3,-1
255,C000040,8,2026-07-30 11:32:00,2026-06-30 13:32:00,29.92,7,1,0,-1



02B.1 — INVESTIGATION COMPLETE


In [19]:
# ============================================================
# 02B.2 — RECONSTRUCT RAW 14D USING CALENDAR-DAY SEMANTICS
# ============================================================
#
# Hypothesis:
#
# raw n_msgs_last_14d counts previous messages whose CALENDAR
# DATE is within the previous 14 calendar days.
#
# For current date D:
#
#     D - 14 days <= previous_send_date < D
#
# Note:
# Messages earlier on the SAME current calendar date would NOT
# be counted under this exact hypothesis.
#
# Pure audit. No production feature is created yet.
# ============================================================

print("=" * 100)
print("02B.2 — CALENDAR-DAY 14D RECONSTRUCTION")
print("=" * 100)


# ------------------------------------------------------------
# 1. Narrow event table
# ------------------------------------------------------------

audit_tmp = features[
    [
        "event_id",
        "customer_id",
        "sent_at",
        "n_msgs_last_14d",
    ]
].copy()


audit_tmp["sent_date"] = (
    audit_tmp["sent_at"]
    .dt.normalize()
)


# ------------------------------------------------------------
# 2. Output array
# ------------------------------------------------------------

calendar_14d_count = np.zeros(
    len(audit_tmp),
    dtype=np.int16,
)


# ------------------------------------------------------------
# 3. Group positions
# ------------------------------------------------------------

customer_positions = (
    audit_tmp
    .groupby(
        "customer_id",
        sort=False
    )
    .indices
)


date_ns_all = (
    audit_tmp["sent_date"]
    .astype("int64")
    .to_numpy(copy=False)
)


# ------------------------------------------------------------
# 4. Calendar-day count
# ------------------------------------------------------------

fourteen_days_ns = int(
    pd.Timedelta(days=14).value
)


for customer_id, positions in customer_positions.items():

    positions = np.asarray(
        positions,
        dtype=np.int64,
    )

    dates = (
        date_ns_all[
            positions
        ]
    )

    for i in range(
        len(positions)
    ):

        current_date = dates[i]

        lower_date = (
            current_date
            - fourteen_days_ns
        )

        previous_dates = (
            dates[:i]
        )

        count = np.sum(
            (previous_dates >= lower_date)
            &
            (previous_dates < current_date)
        )

        calendar_14d_count[
            positions[i]
        ] = count


# ------------------------------------------------------------
# 5. Compare with raw
# ------------------------------------------------------------

raw_14d = (
    pd.to_numeric(
        audit_tmp[
            "n_msgs_last_14d"
        ],
        errors="raise",
    )
    .to_numpy()
)


difference = (
    calendar_14d_count
    - raw_14d
)


exact_match = (
    difference == 0
)


# ------------------------------------------------------------
# 6. Results
# ------------------------------------------------------------

print(
    f"Events       : "
    f"{len(audit_tmp):,}"
)

print(
    f"Exact matches: "
    f"{exact_match.sum():,} "
    f"({exact_match.mean():.4%})"
)

print(
    f"Mismatches   : "
    f"{(~exact_match).sum():,} "
    f"({(~exact_match).mean():.4%})"
)


print(
    "\nDIFFERENCE DISTRIBUTION"
)

display(
    pd.Series(
        difference,
        name="calendar_reconstructed_minus_raw",
    )
    .value_counts()
    .sort_index()
    .to_frame("events")
)


# ------------------------------------------------------------
# 7. Compare all three definitions
# ------------------------------------------------------------

comparison_14d = pd.DataFrame(
    {
        "raw_14d":
            raw_14d,

        "exact_14x24h":
            features[
                "hist_n_messages_last_14d"
            ].to_numpy(),

        "calendar_14d":
            calendar_14d_count,
    }
)


comparison_14d[
    "exact_minus_raw"
] = (
    comparison_14d[
        "exact_14x24h"
    ]
    -
    comparison_14d[
        "raw_14d"
    ]
)


comparison_14d[
    "calendar_minus_raw"
] = (
    comparison_14d[
        "calendar_14d"
    ]
    -
    comparison_14d[
        "raw_14d"
    ]
)


print(
    "\nTHREE-WAY COMPARISON"
)

display(
    comparison_14d[
        [
            "raw_14d",
            "exact_14x24h",
            "calendar_14d",
        ]
    ]
    .describe()
    .T
)


# ------------------------------------------------------------
# 8. Sample remaining mismatches, if any
# ------------------------------------------------------------

remaining_mismatch = (
    difference != 0
)


print(
    "\nSAMPLE REMAINING CALENDAR-DAY MISMATCHES"
)

if remaining_mismatch.any():

    sample_idx = np.flatnonzero(
        remaining_mismatch
    )[:30]

    sample = audit_tmp.iloc[
        sample_idx
    ].copy()

    sample[
        "calendar_reconstructed_14d"
    ] = calendar_14d_count[
        sample_idx
    ]

    sample[
        "difference"
    ] = difference[
        sample_idx
    ]

    display(sample)

else:

    print(
        "None — calendar-day hypothesis reproduces "
        "the raw feature exactly."
    )


print("\n" + "=" * 100)
print("02B.2 — AUDIT COMPLETE")
print("=" * 100)

02B.2 — CALENDAR-DAY 14D RECONSTRUCTION
Events       : 75,406
Exact matches: 75,406 (100.0000%)
Mismatches   : 0 (0.0000%)

DIFFERENCE DISTRIBUTION


,events
calendar_reconstructed_minus_raw,
0,75406



THREE-WAY COMPARISON


,count,mean,std,min,25%,50%,75%,max
raw_14d,"75,406.00",2.17,1.72,0.00,1.00,2.00,3.00,10.00
exact_14x24h,"75,406.00",2.11,1.68,0.00,1.00,2.00,3.00,10.00
calendar_14d,"75,406.00",2.17,1.72,0.00,1.00,2.00,3.00,10.00



SAMPLE REMAINING CALENDAR-DAY MISMATCHES
None — calendar-day hypothesis reproduces the raw feature exactly.

02B.2 — AUDIT COMPLETE


In [20]:
# ============================================================
# 02C — HISTORICAL TEMPLATE EXPOSURE
# ============================================================
#
# PURPOSE
# -------
# Represent the customer's historical template exposure
# strictly BEFORE the current WhatsApp action.
#
# Grain:
#   1 row = 1 WhatsApp send event
#
# PIT:
#   X_t = only information available before action t
#
# Includes:
#   - previous template
#   - historical count/share by template
#   - template diversity
#   - historical switching
#   - previous-template streak
#   - historical recency by template
#   - exposure to current action before t
#   - broad action-family history
#
# Does NOT use:
#   - delivery_status
#   - interaction
#   - paid_within_72h
#   - amount_paid_brl
#
# Expected:
#   start = 75,406 × 300
#   end   = 75,406 × 362
# ============================================================


print("=" * 100)
print("02C — HISTORICAL TEMPLATE EXPOSURE")
print("=" * 100)


# ============================================================
# 0. STARTING CHECKPOINT
# ============================================================

rows_before = len(features)
cols_before = features.shape[1]


assert features.shape == (75_406, 300)

assert (
    features["customer_id"]
    .nunique()
    == 11_724
)

assert features["event_id"].is_unique
assert features["message_id"].is_unique


print(
    f"Starting shape: "
    f"{features.shape}"
)


# ============================================================
# 1. TEMPLATE UNIVERSE
# ============================================================

TEMPLATES = [
    "friendly_reminder",
    "urgent_reminder",
    "pix_link",
    "discount_offer",
]


template = (
    features["current_template"]
    .astype(str)
)

customer_key = (
    features["customer_id"]
)

sent_at = (
    features["sent_at"]
)


unexpected_templates = (
    set(template.unique())
    - set(TEMPLATES)
)


assert not unexpected_templates, (
    f"Unexpected templates: "
    f"{unexpected_templates}"
)


print("\nCURRENT TEMPLATE DISTRIBUTION")

display(
    template
    .value_counts()
    .to_frame("messages")
)


# ============================================================
# 2. PREVIOUS TEMPLATE
# ============================================================

features[
    "hist_previous_template"
] = (
    features
    .groupby(
        "customer_id",
        sort=False,
    )["current_template"]
    .shift(1)
)


features[
    "hist_has_previous_template"
] = (
    features[
        "hist_previous_template"
    ]
    .notna()
    .astype("int8")
)


# ============================================================
# 3. HISTORICAL COUNT BY TEMPLATE
# ============================================================
#
# cumsum includes current action.
# Subtracting current flag restores strict PIT.
# ============================================================

for tpl in TEMPLATES:

    current_flag = (
        template
        .eq(tpl)
        .astype("int16")
    )

    cumulative = (
        current_flag
        .groupby(
            customer_key,
            sort=False,
        )
        .cumsum()
    )

    features[
        f"hist_n_previous_{tpl}"
    ] = (
        cumulative
        - current_flag
    ).astype("int16")


# ============================================================
# 4. EVER RECEIVED EACH TEMPLATE
# ============================================================

for tpl in TEMPLATES:

    features[
        f"hist_ever_received_{tpl}"
    ] = (
        features[
            f"hist_n_previous_{tpl}"
        ] > 0
    ).astype("int8")


# ============================================================
# 5. HISTORICAL SHARE BY TEMPLATE
# ============================================================

previous_n = (
    features[
        "hist_n_previous_messages"
    ]
)


for tpl in TEMPLATES:

    features[
        f"hist_share_previous_{tpl}"
    ] = np.where(
        previous_n > 0,

        features[
            f"hist_n_previous_{tpl}"
        ]
        / previous_n,

        np.nan,
    )


# ============================================================
# 6. TEMPLATE DIVERSITY
# ============================================================

ever_cols = [
    f"hist_ever_received_{tpl}"
    for tpl in TEMPLATES
]


features[
    "hist_n_distinct_previous_templates"
] = (
    features[
        ever_cols
    ]
    .sum(axis=1)
    .astype("int8")
)


features[
    "hist_has_template_diversity"
] = (
    features[
        "hist_n_distinct_previous_templates"
    ] >= 2
).astype("int8")


features[
    "hist_seen_all_templates"
] = (
    features[
        "hist_n_distinct_previous_templates"
    ] == 4
).astype("int8")


# ============================================================
# 7. CURRENT ACTION × PREVIOUS ACTION
# ============================================================
#
# These use the current action.
#
# They are NOT pure state features.
# Later governance:
#
#   action × history
#
# They are still PIT-safe because the action being evaluated
# is known at decision time.
# ============================================================

previous_template = (
    features[
        "hist_previous_template"
    ]
)


features[
    "hist_current_same_as_previous_template"
] = (
    previous_template.notna()
    &
    template.eq(previous_template)
).astype("int8")


features[
    "hist_current_diff_from_previous_template"
] = (
    previous_template.notna()
    &
    template.ne(previous_template)
).astype("int8")


# ============================================================
# 8. HISTORICAL TEMPLATE SWITCHES
# ============================================================
#
# current_transition tells whether transition:
#
#   t-1 -> t
#
# changes template.
#
# That transition cannot enter X_t.
#
# Therefore:
#
# cumulative switches including current transition
# minus current transition
#
# = switches strictly before t.
# ============================================================

current_transition = (
    previous_template.notna()
    &
    template.ne(previous_template)
).astype("int16")


switches_including_current = (
    current_transition
    .groupby(
        customer_key,
        sort=False,
    )
    .cumsum()
)


features[
    "hist_n_template_switches"
] = (
    switches_including_current
    - current_transition
).astype("int16")


features[
    "hist_ever_switched_template"
] = (
    features[
        "hist_n_template_switches"
    ] > 0
).astype("int8")


possible_historical_transitions = (
    previous_n - 1
).clip(lower=0)


features[
    "hist_template_switch_rate"
] = np.where(
    possible_historical_transitions > 0,

    features[
        "hist_n_template_switches"
    ]
    / possible_historical_transitions,

    np.nan,
)


# ============================================================
# 9. PREVIOUS-TEMPLATE STREAK
# ============================================================
#
# Number of consecutive identical templates ending at t-1.
#
# Example:
#
# historical sequence:
#
# friendly → urgent → urgent → urgent
#
# before current action:
#
# previous_template_streak = 3
# ============================================================

template_values = (
    template
    .to_numpy(copy=False)
)


streak_before = np.zeros(
    len(features),
    dtype=np.int16,
)


customer_positions = (
    features
    .groupby(
        "customer_id",
        sort=False,
    )
    .indices
)


for positions in customer_positions.values():

    positions = np.asarray(
        positions,
        dtype=np.int64,
    )

    running_streak = 0
    previous_value = None

    for local_i, global_i in enumerate(
        positions
    ):

        # Store historical streak BEFORE current action
        streak_before[
            global_i
        ] = running_streak

        current_value = (
            template_values[
                global_i
            ]
        )

        if local_i == 0:

            running_streak = 1

        elif current_value == previous_value:

            running_streak += 1

        else:

            running_streak = 1

        previous_value = current_value


features[
    "hist_previous_same_template_streak"
] = streak_before


features[
    "hist_previous_template_streak_2plus"
] = (
    features[
        "hist_previous_same_template_streak"
    ] >= 2
).astype("int8")


features[
    "hist_previous_template_streak_3plus"
] = (
    features[
        "hist_previous_same_template_streak"
    ] >= 3
).astype("int8")


features[
    "hist_previous_template_streak_5plus"
] = (
    features[
        "hist_previous_same_template_streak"
    ] >= 5
).astype("int8")


# ============================================================
# 10. CURRENT TEMPLATE HISTORICAL EXPOSURE
# ============================================================
#
# How many times has THIS action already been used before t?
#
# Very useful for NBA:
#
# "Are we repeating something already tried?"
# ============================================================

count_matrix = (
    features[
        [
            "hist_n_previous_friendly_reminder",
            "hist_n_previous_urgent_reminder",
            "hist_n_previous_pix_link",
            "hist_n_previous_discount_offer",
        ]
    ]
    .to_numpy(copy=False)
)


template_to_position = {
    "friendly_reminder": 0,
    "urgent_reminder": 1,
    "pix_link": 2,
    "discount_offer": 3,
}


current_template_position = (
    template
    .map(
        template_to_position
    )
    .to_numpy()
)


row_position = np.arange(
    len(features)
)


current_template_previous_count = (
    count_matrix[
        row_position,
        current_template_position,
    ]
)


features[
    "hist_n_previous_current_template"
] = (
    current_template_previous_count
    .astype("int16")
)


features[
    "hist_current_template_never_used_before"
] = (
    features[
        "hist_n_previous_current_template"
    ] == 0
).astype("int8")


features[
    "hist_current_template_used_before"
] = (
    features[
        "hist_n_previous_current_template"
    ] > 0
).astype("int8")


features[
    "hist_current_template_used_3plus_before"
] = (
    features[
        "hist_n_previous_current_template"
    ] >= 3
).astype("int8")


features[
    "hist_current_template_used_5plus_before"
] = (
    features[
        "hist_n_previous_current_template"
    ] >= 5
).astype("int8")


# ============================================================
# 11. LAST HISTORICAL OCCURRENCE OF EACH TEMPLATE
# ============================================================
#
# Vectorized PIT-safe implementation.
# ============================================================

for tpl in TEMPLATES:

    flagged_time = (
        sent_at.where(
            template.eq(tpl)
        )
    )

    last_including_current = (
        flagged_time
        .groupby(
            customer_key,
            sort=False,
        )
        .ffill()
    )

    last_before_current = (
        last_including_current
        .groupby(
            customer_key,
            sort=False,
        )
        .shift(1)
    )

    features[
        f"hist_last_{tpl}_at"
    ] = last_before_current


    features[
        f"hist_days_since_last_{tpl}"
    ] = (
        sent_at
        - last_before_current
    ).dt.total_seconds() / 86_400


# ============================================================
# 12. RECENT EXPOSURE BY TEMPLATE
# ============================================================

RECENCY_WINDOWS = [
    3,
    7,
    14,
]


for tpl in TEMPLATES:

    recency_col = (
        f"hist_days_since_last_{tpl}"
    )

    for days in RECENCY_WINDOWS:

        features[
            f"hist_{tpl}_seen_last_{days}d"
        ] = (
            features[
                recency_col
            ]
            .le(days)
            .fillna(False)
            .astype("int8")
        )


# ============================================================
# 13. BROADER ACTION FAMILIES
# ============================================================

features[
    "hist_n_previous_reminders"
] = (
    features[
        "hist_n_previous_friendly_reminder"
    ]
    +
    features[
        "hist_n_previous_urgent_reminder"
    ]
).astype("int16")


features[
    "hist_n_previous_payment_link"
] = (
    features[
        "hist_n_previous_pix_link"
    ]
).astype("int16")


features[
    "hist_n_previous_discount"
] = (
    features[
        "hist_n_previous_discount_offer"
    ]
).astype("int16")


features[
    "hist_ever_received_reminder"
] = (
    features[
        "hist_n_previous_reminders"
    ] > 0
).astype("int8")


features[
    "hist_ever_received_pix"
] = (
    features[
        "hist_n_previous_pix_link"
    ] > 0
).astype("int8")


features[
    "hist_ever_received_discount"
] = (
    features[
        "hist_n_previous_discount_offer"
    ] > 0
).astype("int8")


# ============================================================
# 14. HISTORICAL TEMPLATE CONCENTRATION
# ============================================================
#
# HHI-style concentration:
#
# sum(template_share²)
#
# 1.00:
#   all historical messages same template
#
# 0.25:
#   four templates equally represented
#
# First contact:
#   NaN
# ============================================================

share_cols = [
    f"hist_share_previous_{tpl}"
    for tpl in TEMPLATES
]


features[
    "hist_template_concentration"
] = (
    features[
        share_cols
    ]
    .pow(2)
    .sum(
        axis=1,
        min_count=1,
    )
)


features.loc[
    previous_n.eq(0),
    "hist_template_concentration"
] = np.nan


# ============================================================
# 15. DOMINANT HISTORICAL TEMPLATE
# ============================================================

count_cols = [
    f"hist_n_previous_{tpl}"
    for tpl in TEMPLATES
]


historical_counts = (
    features[
        count_cols
    ]
)


max_count = (
    historical_counts
    .max(axis=1)
)


n_at_max = (
    historical_counts
    .eq(
        max_count,
        axis=0,
    )
    .sum(axis=1)
)


features[
    "hist_dominant_template_tie"
] = (
    (previous_n > 0)
    &
    (n_at_max > 1)
).astype("int8")


dominant_template = (
    historical_counts
    .idxmax(axis=1)
    .str.replace(
        "hist_n_previous_",
        "",
        regex=False,
    )
)


features[
    "hist_dominant_previous_template"
] = (
    dominant_template
    .where(
        previous_n > 0
    )
)


# ============================================================
# 16. HARD PIT AUDITS
# ============================================================

first_contact = (
    features[
        "hist_is_first_contact"
    ].eq(1)
)


# ------------------------------------------------------------
# 16A. First contact
# ------------------------------------------------------------

assert (
    first_contact.sum()
    == 11_724
)


assert (
    features.loc[
        first_contact,
        "hist_previous_template",
    ]
    .isna()
    .all()
)


assert (
    features.loc[
        first_contact,
        "hist_previous_same_template_streak",
    ] == 0
).all()


for tpl in TEMPLATES:

    assert (
        features.loc[
            first_contact,
            f"hist_n_previous_{tpl}",
        ] == 0
    ).all()

    assert (
        features.loc[
            first_contact,
            f"hist_ever_received_{tpl}",
        ] == 0
    ).all()

    assert (
        features.loc[
            first_contact,
            f"hist_last_{tpl}_at",
        ]
        .isna()
        .all()
    )

    assert (
        features.loc[
            first_contact,
            f"hist_days_since_last_{tpl}",
        ]
        .isna()
        .all()
    )


# ------------------------------------------------------------
# 16B. Template counts must exactly reconstruct previous msgs
# ------------------------------------------------------------

template_count_sum = (
    features[
        count_cols
    ]
    .sum(axis=1)
)


assert (
    template_count_sum
    ==
    previous_n
).all(), (
    "Historical template counts do not sum to "
    "hist_n_previous_messages."
)


# ------------------------------------------------------------
# 16C. Historical shares
# ------------------------------------------------------------

share_sum = (
    features[
        share_cols
    ]
    .sum(
        axis=1,
        min_count=1,
    )
)


history_mask = (
    previous_n > 0
)


assert np.allclose(
    share_sum.loc[
        history_mask
    ],
    1.0,
)


assert (
    share_sum.loc[
        ~history_mask
    ]
    .isna()
    .all()
)


# ------------------------------------------------------------
# 16D. Distinct template count
# ------------------------------------------------------------

assert (
    features[
        "hist_n_distinct_previous_templates"
    ]
    .between(
        0,
        4,
    )
    .all()
)


assert (
    features.loc[
        first_contact,
        "hist_n_distinct_previous_templates",
    ] == 0
).all()


# ------------------------------------------------------------
# 16E. Streak
# ------------------------------------------------------------

assert (
    features.loc[
        ~first_contact,
        "hist_previous_same_template_streak",
    ] >= 1
).all()


assert (
    features[
        "hist_previous_same_template_streak"
    ]
    <= previous_n
).all()


# ------------------------------------------------------------
# 16F. Historical recencies
# ------------------------------------------------------------

for tpl in TEMPLATES:

    recency = (
        features[
            f"hist_days_since_last_{tpl}"
        ]
        .dropna()
    )

    assert (
        recency >= 0
    ).all()


# ------------------------------------------------------------
# 16G. Switches
# ------------------------------------------------------------

assert (
    features[
        "hist_n_template_switches"
    ]
    <= possible_historical_transitions
).all()


# ------------------------------------------------------------
# 16H. Current-template count
# ------------------------------------------------------------

assert (
    features[
        "hist_n_previous_current_template"
    ]
    <= previous_n
).all()


assert (
    (
        features[
            "hist_current_template_never_used_before"
        ]
        +
        features[
            "hist_current_template_used_before"
        ]
    )
    == 1
).all()


# ============================================================
# 17. FINAL STRUCTURAL AUDIT
# ============================================================

cols_after = (
    features.shape[1]
)

new_features = (
    cols_after
    - cols_before
)


print("\n" + "=" * 100)
print("02C — HISTORICAL TEMPLATE EXPOSURE COMPLETE")
print("=" * 100)

print(
    f"Rows          : "
    f"{len(features):,}"
)

print(
    f"Customers     : "
    f"{features['customer_id'].nunique():,}"
)

print(
    f"Columns before: "
    f"{cols_before:,}"
)

print(
    f"Columns after : "
    f"{cols_after:,}"
)

print(
    f"New features  : "
    f"{new_features:,}"
)


assert len(features) == rows_before
assert features["event_id"].is_unique


# ============================================================
# 18. PRIOR TEMPLATE COUNTS
# ============================================================

print(
    "\nPRIOR TEMPLATE COUNTS"
)

display(
    features[
        count_cols
    ]
    .describe(
        percentiles=[
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .T
)


# ============================================================
# 19. EVER RECEIVED BEFORE CURRENT ACTION
# ============================================================

print(
    "\nEVER RECEIVED BEFORE CURRENT ACTION"
)

ever_summary = pd.DataFrame(
    {
        tpl: {
            "events_with_prior_exposure":
                int(
                    features[
                        f"hist_ever_received_{tpl}"
                    ].sum()
                ),

            "pct_events":
                (
                    features[
                        f"hist_ever_received_{tpl}"
                    ].mean()
                    * 100
                ),
        }

        for tpl in TEMPLATES
    }
).T


display(
    ever_summary
)


# ============================================================
# 20. DISTINCT PREVIOUS TEMPLATES
# ============================================================

print(
    "\nDISTINCT PREVIOUS TEMPLATES"
)

display(
    features[
        "hist_n_distinct_previous_templates"
    ]
    .value_counts()
    .sort_index()
    .to_frame("messages")
)


# ============================================================
# 21. PREVIOUS TEMPLATE
# ============================================================

print(
    "\nPREVIOUS TEMPLATE"
)

display(
    features[
        "hist_previous_template"
    ]
    .value_counts(
        dropna=False
    )
    .to_frame("messages")
)


# ============================================================
# 22. HISTORICAL SWITCHES
# ============================================================

print(
    "\nHISTORICAL TEMPLATE SWITCHES"
)

display(
    features[
        "hist_n_template_switches"
    ]
    .describe(
        percentiles=[
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame()
)


# ============================================================
# 23. PREVIOUS-TEMPLATE STREAK
# ============================================================

print(
    "\nPREVIOUS SAME-TEMPLATE STREAK"
)

display(
    features[
        "hist_previous_same_template_streak"
    ]
    .describe(
        percentiles=[
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame()
)


# ============================================================
# 24. CURRENT TEMPLATE PRIOR EXPOSURE
# ============================================================

print(
    "\nCURRENT TEMPLATE — NUMBER OF PREVIOUS USES"
)

display(
    features[
        "hist_n_previous_current_template"
    ]
    .describe(
        percentiles=[
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame()
)


# ============================================================
# 25. DOMINANT HISTORICAL TEMPLATE
# ============================================================

print(
    "\nDOMINANT HISTORICAL TEMPLATE"
)

display(
    features[
        "hist_dominant_previous_template"
    ]
    .value_counts(
        dropna=False
    )
    .to_frame("messages")
)


print(
    "\nDominant-template ties: "
    f"{features['hist_dominant_template_tie'].sum():,}"
)


# ============================================================
# 26. SAMPLE JOURNEY
# ============================================================

sample_customer = (
    features
    .groupby(
        "customer_id",
        sort=False,
    )
    .size()
    .idxmax()
)


print(
    f"\nSAMPLE TEMPLATE JOURNEY — "
    f"{sample_customer}"
)


display(
    features.loc[
        features[
            "customer_id"
        ].eq(
            sample_customer
        ),
        [
            "customer_id",
            "event_number",
            "sent_at",
            "current_template",
            "hist_previous_template",
            "hist_n_previous_messages",
            "hist_n_previous_friendly_reminder",
            "hist_n_previous_urgent_reminder",
            "hist_n_previous_pix_link",
            "hist_n_previous_discount_offer",
            "hist_n_distinct_previous_templates",
            "hist_n_template_switches",
            "hist_previous_same_template_streak",
            "hist_n_previous_current_template",
        ],
    ]
)


print("\n" + "=" * 100)
print("02C — HISTORICAL TEMPLATE EXPOSURE COMPLETE")
print("=" * 100)

02C — HISTORICAL TEMPLATE EXPOSURE
Starting shape: (75406, 300)

CURRENT TEMPLATE DISTRIBUTION


,messages
current_template,
friendly_reminder,24404
urgent_reminder,23347
pix_link,21437
discount_offer,6218



02C — HISTORICAL TEMPLATE EXPOSURE COMPLETE
Rows          : 75,406
Customers     : 11,724
Columns before: 300
Columns after : 360
New features  : 60

PRIOR TEMPLATE COUNTS


,count,mean,std,min,25%,50%,75%,90%,95%,99%,max
hist_n_previous_friendly_reminder,"75,406.00",1.58,1.37,0.00,0.00,1.00,2.00,3.00,4.00,5.00,9.00
hist_n_previous_urgent_reminder,"75,406.00",1.02,1.44,0.00,0.00,0.00,2.00,3.00,4.00,6.00,11.00
hist_n_previous_pix_link,"75,406.00",1.11,1.27,0.00,0.00,1.00,2.00,3.00,4.00,5.00,12.00
hist_n_previous_discount_offer,"75,406.00",0.10,0.41,0.00,0.00,0.00,0.00,0.00,1.00,2.00,6.00



EVER RECEIVED BEFORE CURRENT ACTION


...


DOMINANT HISTORICAL TEMPLATE


,messages
hist_dominant_previous_template,
friendly_reminder,39962
pix_link,12653
NaN,11724
urgent_reminder,11000
discount_offer,67



Dominant-template ties: 14,201

SAMPLE TEMPLATE JOURNEY — C005915


,customer_id,event_number,sent_at,current_template,hist_previous_template,hist_n_previous_messages,hist_n_previous_friendly_reminder,hist_n_previous_urgent_reminder,hist_n_previous_pix_link,hist_n_previous_discount_offer,hist_n_distinct_previous_templates,hist_n_template_switches,hist_previous_same_template_streak,hist_n_previous_current_template
37464,C005915,1,2026-06-11 10:52:00,pix_link,<NA>,0,0,0,0,0,0,0,0,0
37465,C005915,2,2026-06-12 10:02:00,pix_link,pix_link,1,0,0,1,0,1,0,1,1
37466,C005915,3,2026-06-15 09:45:00,urgent_reminder,pix_link,2,0,0,2,0,1,0,2,0
37467,C005915,4,2026-06-17 16:17:00,pix_link,urgent_reminder,3,0,1,2,0,2,1,1,2
37468,C005915,5,2026-06-19 12:18:00,urgent_reminder,pix_link,4,0,1,3,0,2,2,1,1
37469,C005915,6,2026-06-21 11:52:00,urgent_reminder,urgent_reminder,5,0,2,3,0,2,3,1,2
37470,C005915,7,2026-06-22 16:15:00,pix_link,urgent_reminder,6,0,3,3,0,2,3,2,3
37471,C005915,8,2026-06-25 10:18:00,urgent_reminder,pix_link,7,0,3,4,0,2,4,1,3
37472,C005915,9,2026-06-29 12:35:00,pix_link,urgent_reminder,8,0,4,4,0,2,5,1,4
37473,C005915,10,2026-06-30 14:22:00,urgent_reminder,pix_link,9,0,4,5,0,2,6,1,4



02C — HISTORICAL TEMPLATE EXPOSURE COMPLETE


In [21]:
# ============================================================
# 02D — HISTORICAL CADENCE / SEQUENCE STATE
# ============================================================
#
# PURPOSE
# -------
# Represent HOW the historical contact cadence evolved before
# the current WhatsApp action.
#
# Strict PIT:
#
#   X_t = information from contacts strictly before event t
#
# Examples:
#   - average historical gap
#   - variability of historical gaps
#   - recent gap vs historical average
#   - cadence accelerating / decelerating
#   - number of short-gap contacts
#   - historical burst intensity
#
# NO:
#   payment outcomes
#   delivery outcomes
#   interaction outcomes
#
# Starting checkpoint:
#   75,406 × 360
# ============================================================


print("=" * 100)
print("02D — HISTORICAL CADENCE / SEQUENCE STATE")
print("=" * 100)


# ============================================================
# 0. STARTING CHECKPOINT
# ============================================================

rows_before = len(features)
cols_before = features.shape[1]


assert features.shape == (75_406, 360)

assert (
    features["customer_id"]
    .nunique()
    == 11_724
)

assert features["event_id"].is_unique
assert features["message_id"].is_unique


print(
    f"Starting shape: "
    f"{features.shape}"
)


# ============================================================
# 1. BASE HISTORICAL GAP
# ============================================================
#
# hist_days_since_previous_message already represents:
#
#   current send time - previous send time
#
# Although calculated at event t, it contains no future
# information: the current send time is known when action t
# occurs.
#
# For historical cadence summaries at t, however, we need gaps
# COMPLETED strictly before t.
#
# Example:
#
# msg1 ----- msg2 -------- msg3
#
# at msg3:
#
# current gap       = msg3 - msg2
# historical gaps   = [msg2 - msg1]
#
# Therefore cumulative summaries of the gap column need shift(1).
# ============================================================

customer_key = features["customer_id"]

current_gap_days = (
    features[
        "hist_days_since_previous_message"
    ]
    .astype("float64")
)


# ============================================================
# 2. NUMBER OF COMPLETED HISTORICAL GAPS
# ============================================================
#
# n previous messages:
#
# 0 -> 0 completed gaps
# 1 -> 0 completed gaps
# 2 -> 1 completed gap
# 3 -> 2 completed gaps
# ============================================================

features[
    "hist_n_completed_contact_gaps"
] = (
    features[
        "hist_n_previous_messages"
    ]
    .sub(1)
    .clip(lower=0)
    .astype("int16")
)


# ============================================================
# 3. HISTORICAL GAP — EXPANDING MEAN
# ============================================================
#
# Expanding statistics are calculated including the current
# observed gap and then shifted one event within customer.
#
# This guarantees strict PIT for the historical summary.
# ============================================================

gap_cumsum = (
    current_gap_days
    .fillna(0)
    .groupby(
        customer_key,
        sort=False,
    )
    .cumsum()
)


gap_valid_count = (
    current_gap_days
    .notna()
    .astype("int16")
    .groupby(
        customer_key,
        sort=False,
    )
    .cumsum()
)


historical_gap_sum = (
    gap_cumsum
    .groupby(
        customer_key,
        sort=False,
    )
    .shift(1)
)


historical_gap_count = (
    gap_valid_count
    .groupby(
        customer_key,
        sort=False,
    )
    .shift(1)
    .fillna(0)
)


features[
    "hist_mean_completed_gap_days"
] = np.where(
    historical_gap_count > 0,

    historical_gap_sum
    / historical_gap_count,

    np.nan,
)


# ============================================================
# 4. HISTORICAL GAP — SECOND MOMENT / STD
# ============================================================

gap_squared = (
    current_gap_days
    .fillna(0)
    .pow(2)
)


gap_squared_cumsum = (
    gap_squared
    .groupby(
        customer_key,
        sort=False,
    )
    .cumsum()
)


historical_gap_squared_sum = (
    gap_squared_cumsum
    .groupby(
        customer_key,
        sort=False,
    )
    .shift(1)
)


historical_gap_variance = np.where(
    historical_gap_count > 1,

    (
        historical_gap_squared_sum
        -
        (
            historical_gap_sum.pow(2)
            / historical_gap_count
        )
    )
    /
    (
        historical_gap_count - 1
    ),

    np.nan,
)


# Numerical protection
historical_gap_variance = np.where(
    np.isnan(historical_gap_variance),
    np.nan,
    np.maximum(
        historical_gap_variance,
        0,
    ),
)


features[
    "hist_std_completed_gap_days"
] = np.sqrt(
    historical_gap_variance
)


# ============================================================
# 5. HISTORICAL GAP — MIN / MAX
# ============================================================
#
# cummin / cummax include current gap, therefore shift.
# ============================================================

gap_for_min = (
    current_gap_days
    .fillna(np.inf)
)


gap_for_max = (
    current_gap_days
    .fillna(-np.inf)
)


historical_gap_min = (
    gap_for_min
    .groupby(
        customer_key,
        sort=False,
    )
    .cummin()
    .groupby(
        customer_key,
        sort=False,
    )
    .shift(1)
)


historical_gap_max = (
    gap_for_max
    .groupby(
        customer_key,
        sort=False,
    )
    .cummax()
    .groupby(
        customer_key,
        sort=False,
    )
    .shift(1)
)


features[
    "hist_min_completed_gap_days"
] = (
    historical_gap_min
    .replace(
        [np.inf, -np.inf],
        np.nan,
    )
)


features[
    "hist_max_completed_gap_days"
] = (
    historical_gap_max
    .replace(
        [np.inf, -np.inf],
        np.nan,
    )
)


# ============================================================
# 6. HISTORICAL GAP REGULARITY
# ============================================================
#
# Coefficient of variation:
#
#   std / mean
#
# Low:
#   historically regular cadence
#
# High:
#   historically irregular cadence
# ============================================================

features[
    "hist_gap_coefficient_of_variation"
] = np.where(
    features[
        "hist_mean_completed_gap_days"
    ] > 0,

    features[
        "hist_std_completed_gap_days"
    ]
    /
    features[
        "hist_mean_completed_gap_days"
    ],

    np.nan,
)


# ============================================================
# 7. CURRENT GAP VS HISTORICAL CADENCE
# ============================================================
#
# These combine the current contact timing with prior cadence.
#
# They are PIT-safe but conceptually belong to:
#
#   current action timing × historical state
# ============================================================

features[
    "hist_current_gap_minus_mean_gap_days"
] = (
    current_gap_days
    -
    features[
        "hist_mean_completed_gap_days"
    ]
)


features[
    "hist_current_gap_to_mean_gap_ratio"
] = np.where(
    features[
        "hist_mean_completed_gap_days"
    ] > 0,

    current_gap_days
    /
    features[
        "hist_mean_completed_gap_days"
    ],

    np.nan,
)


# ============================================================
# 8. CADENCE ACCELERATION / DECELERATION
# ============================================================
#
# If:
#
# current gap < historical mean
#     -> contacts are accelerating
#
# current gap > historical mean
#     -> contacts are decelerating
#
# Require at least TWO completed historical gaps for a more
# meaningful comparison.
# ============================================================

enough_gap_history = (
    features[
        "hist_n_completed_contact_gaps"
    ] >= 2
)


features[
    "hist_cadence_accelerating"
] = (
    enough_gap_history
    &
    (
        current_gap_days
        <
        features[
            "hist_mean_completed_gap_days"
        ]
    )
).astype("int8")


features[
    "hist_cadence_decelerating"
] = (
    enough_gap_history
    &
    (
        current_gap_days
        >
        features[
            "hist_mean_completed_gap_days"
        ]
    )
).astype("int8")


features[
    "hist_cadence_near_historical_mean"
] = (
    enough_gap_history
    &
    (
        features[
            "hist_current_gap_to_mean_gap_ratio"
        ]
        .between(
            0.8,
            1.2,
            inclusive="both",
        )
    )
).astype("int8")


# ============================================================
# 9. STRONG ACCELERATION / DECELERATION
# ============================================================

features[
    "hist_cadence_strongly_accelerating"
] = (
    enough_gap_history
    &
    (
        features[
            "hist_current_gap_to_mean_gap_ratio"
        ] < 0.5
    )
).astype("int8")


features[
    "hist_cadence_strongly_decelerating"
] = (
    enough_gap_history
    &
    (
        features[
            "hist_current_gap_to_mean_gap_ratio"
        ] > 2.0
    )
).astype("int8")


# ============================================================
# 10. PREVIOUS GAP
# ============================================================
#
# Current event t:
#
# previous gap =
#   interval between t-2 and t-1
#
# This is fully historical at t.
# ============================================================

features[
    "hist_previous_completed_gap_days"
] = (
    current_gap_days
    .groupby(
        customer_key,
        sort=False,
    )
    .shift(1)
)


# ============================================================
# 11. CURRENT GAP VS PREVIOUS GAP
# ============================================================

previous_gap = (
    features[
        "hist_previous_completed_gap_days"
    ]
)


features[
    "hist_current_gap_minus_previous_gap_days"
] = (
    current_gap_days
    -
    previous_gap
)


features[
    "hist_current_gap_to_previous_gap_ratio"
] = np.where(
    previous_gap > 0,

    current_gap_days
    / previous_gap,

    np.nan,
)


features[
    "hist_gap_shorter_than_previous"
] = (
    previous_gap.notna()
    &
    (
        current_gap_days
        <
        previous_gap
    )
).astype("int8")


features[
    "hist_gap_longer_than_previous"
] = (
    previous_gap.notna()
    &
    (
        current_gap_days
        >
        previous_gap
    )
).astype("int8")


# ============================================================
# 12. HISTORICAL SHORT-GAP COUNTS
# ============================================================
#
# Count completed historical gaps before t satisfying:
#
# <= 1 day
# <= 2 days
# <= 3 days
# <= 7 days
#
# Current gap is excluded via shift.
# ============================================================

SHORT_GAP_THRESHOLDS = [
    1,
    2,
    3,
    7,
]


for days in SHORT_GAP_THRESHOLDS:

    current_short_gap = (
        current_gap_days
        .le(days)
        &
        current_gap_days.notna()
    ).astype("int16")

    cumulative_short_gap = (
        current_short_gap
        .groupby(
            customer_key,
            sort=False,
        )
        .cumsum()
    )

    features[
        f"hist_n_completed_gaps_le_{days}d"
    ] = (
        cumulative_short_gap
        .groupby(
            customer_key,
            sort=False,
        )
        .shift(1)
        .fillna(0)
        .astype("int16")
    )


# ============================================================
# 13. HISTORICAL SHORT-GAP SHARES
# ============================================================

completed_gap_n = (
    features[
        "hist_n_completed_contact_gaps"
    ]
)


for days in SHORT_GAP_THRESHOLDS:

    features[
        f"hist_share_completed_gaps_le_{days}d"
    ] = np.where(
        completed_gap_n > 0,

        features[
            f"hist_n_completed_gaps_le_{days}d"
        ]
        / completed_gap_n,

        np.nan,
    )


# ============================================================
# 14. HISTORICAL LONG-GAP COUNTS
# ============================================================

LONG_GAP_THRESHOLDS = [
    7,
    14,
]


for days in LONG_GAP_THRESHOLDS:

    current_long_gap = (
        current_gap_days
        .gt(days)
        &
        current_gap_days.notna()
    ).astype("int16")

    cumulative_long_gap = (
        current_long_gap
        .groupby(
            customer_key,
            sort=False,
        )
        .cumsum()
    )

    features[
        f"hist_n_completed_gaps_gt_{days}d"
    ] = (
        cumulative_long_gap
        .groupby(
            customer_key,
            sort=False,
        )
        .shift(1)
        .fillna(0)
        .astype("int16")
    )


# ============================================================
# 15. BURST HISTORY
# ============================================================
#
# Operational definitions:
#
# repeated short cadence:
#   >=2 historical gaps <=2d
#
# sustained pressure:
#   >=3 historical gaps <=3d
#
# high historical pressure:
#   >=5 historical gaps <=7d
#
# These are descriptive policy-exposure features,
# NOT causal treatment-effect features.
# ============================================================

features[
    "hist_has_repeated_short_cadence"
] = (
    features[
        "hist_n_completed_gaps_le_2d"
    ] >= 2
).astype("int8")


features[
    "hist_has_sustained_contact_pressure"
] = (
    features[
        "hist_n_completed_gaps_le_3d"
    ] >= 3
).astype("int8")


features[
    "hist_has_high_contact_pressure"
] = (
    features[
        "hist_n_completed_gaps_le_7d"
    ] >= 5
).astype("int8")


# ============================================================
# 16. RECENT PRESSURE VS LONGER-RUN PRESSURE
# ============================================================
#
# Already have:
#
# hist_n_messages_last_7d
# hist_n_messages_last_14d
# hist_n_messages_last_30d
#
# Normalize to comparable messages/day intensity.
# ============================================================

features[
    "hist_contact_rate_last_7d"
] = (
    features[
        "hist_n_messages_last_7d"
    ] / 7
)


features[
    "hist_contact_rate_last_14d"
] = (
    features[
        "hist_n_messages_last_14d"
    ] / 14
)


features[
    "hist_contact_rate_last_30d"
] = (
    features[
        "hist_n_messages_last_30d"
    ] / 30
)


# ============================================================
# 17. RECENT VS LONGER-RUN PRESSURE RATIOS
# ============================================================

features[
    "hist_contact_rate_7d_to_30d_ratio"
] = np.where(
    features[
        "hist_contact_rate_last_30d"
    ] > 0,

    features[
        "hist_contact_rate_last_7d"
    ]
    /
    features[
        "hist_contact_rate_last_30d"
    ],

    np.nan,
)


features[
    "hist_contact_rate_14d_to_30d_ratio"
] = np.where(
    features[
        "hist_contact_rate_last_30d"
    ] > 0,

    features[
        "hist_contact_rate_last_14d"
    ]
    /
    features[
        "hist_contact_rate_last_30d"
    ],

    np.nan,
)


# ============================================================
# 18. PRESSURE TREND FLAGS
# ============================================================
#
# >1:
# recent contact rate exceeds 30d average
#
# We use 1.25 / 0.75 to avoid classifying tiny differences
# as meaningful acceleration/deceleration.
# ============================================================

features[
    "hist_recent_pressure_increasing"
] = (
    features[
        "hist_contact_rate_7d_to_30d_ratio"
    ] > 1.25
).fillna(False).astype("int8")


features[
    "hist_recent_pressure_decreasing"
] = (
    features[
        "hist_contact_rate_7d_to_30d_ratio"
    ] < 0.75
).fillna(False).astype("int8")


# ============================================================
# 19. CURRENT GAP BUCKET
# ============================================================

features[
    "hist_current_contact_gap_bucket"
] = pd.cut(
    current_gap_days,
    bins=[
        -np.inf,
        1,
        2,
        3,
        7,
        14,
        30,
        np.inf,
    ],
    labels=[
        "gap_le_1d",
        "gap_1_2d",
        "gap_2_3d",
        "gap_3_7d",
        "gap_7_14d",
        "gap_14_30d",
        "gap_30d_plus",
    ],
)


# ============================================================
# 20. HISTORICAL CADENCE PROFILE
# ============================================================
#
# Descriptive summary only.
#
# first_contact:
#   no previous contact
#
# insufficient_history:
#   history exists but no completed historical gap
#
# compressed:
#   mean historical completed gap <= 2d
#
# moderate:
#   >2d and <=7d
#
# spaced:
#   >7d
# ============================================================

mean_gap = (
    features[
        "hist_mean_completed_gap_days"
    ]
)


features[
    "hist_cadence_profile"
] = np.select(
    [
        features[
            "hist_is_first_contact"
        ].eq(1),

        features[
            "hist_n_completed_contact_gaps"
        ].eq(0),

        mean_gap.le(2),

        mean_gap.le(7),
    ],
    [
        "first_contact",
        "insufficient_history",
        "compressed",
        "moderate",
    ],
    default="spaced",
)


# ============================================================
# 21. HARD PIT / CONSISTENCY AUDITS
# ============================================================

first_contact = (
    features[
        "hist_is_first_contact"
    ].eq(1)
)


# ------------------------------------------------------------
# First contact
# ------------------------------------------------------------

assert (
    features.loc[
        first_contact,
        "hist_n_completed_contact_gaps",
    ] == 0
).all()


assert (
    features.loc[
        first_contact,
        "hist_mean_completed_gap_days",
    ]
    .isna()
    .all()
)


assert (
    features.loc[
        first_contact,
        "hist_previous_completed_gap_days",
    ]
    .isna()
    .all()
)


# ------------------------------------------------------------
# Second contact:
# one previous message but zero completed HISTORICAL gaps
# ------------------------------------------------------------

second_contact = (
    features[
        "event_number"
    ].eq(2)
)


assert (
    features.loc[
        second_contact,
        "hist_n_completed_contact_gaps",
    ] == 0
).all()


assert (
    features.loc[
        second_contact,
        "hist_mean_completed_gap_days",
    ]
    .isna()
    .all()
)


# ------------------------------------------------------------
# Third contact:
# exactly one completed historical gap
# ------------------------------------------------------------

third_contact = (
    features[
        "event_number"
    ].eq(3)
)


assert (
    features.loc[
        third_contact,
        "hist_n_completed_contact_gaps",
    ] == 1
).all()


assert np.allclose(
    features.loc[
        third_contact,
        "hist_mean_completed_gap_days",
    ],
    features.loc[
        third_contact,
        "hist_previous_completed_gap_days",
    ],
    equal_nan=True,
)


# ------------------------------------------------------------
# Historical gap count formula
# ------------------------------------------------------------

expected_completed_gaps = (
    features[
        "hist_n_previous_messages"
    ]
    .sub(1)
    .clip(lower=0)
)


assert (
    features[
        "hist_n_completed_contact_gaps"
    ]
    ==
    expected_completed_gaps
).all()


# ------------------------------------------------------------
# Min <= mean <= max
# whenever historical gaps exist
# ------------------------------------------------------------

gap_history_mask = (
    features[
        "hist_n_completed_contact_gaps"
    ] > 0
)


assert (
    features.loc[
        gap_history_mask,
        "hist_min_completed_gap_days",
    ]
    <=
    features.loc[
        gap_history_mask,
        "hist_mean_completed_gap_days",
    ]
).all()


assert (
    features.loc[
        gap_history_mask,
        "hist_mean_completed_gap_days",
    ]
    <=
    features.loc[
        gap_history_mask,
        "hist_max_completed_gap_days",
    ]
).all()


# ------------------------------------------------------------
# Short-gap counts cannot exceed completed gaps
# ------------------------------------------------------------

for days in SHORT_GAP_THRESHOLDS:

    assert (
        features[
            f"hist_n_completed_gaps_le_{days}d"
        ]
        <=
        features[
            "hist_n_completed_contact_gaps"
        ]
    ).all()


# ------------------------------------------------------------
# Monotonic short-gap counts
# ------------------------------------------------------------

assert (
    features[
        "hist_n_completed_gaps_le_1d"
    ]
    <=
    features[
        "hist_n_completed_gaps_le_2d"
    ]
).all()


assert (
    features[
        "hist_n_completed_gaps_le_2d"
    ]
    <=
    features[
        "hist_n_completed_gaps_le_3d"
    ]
).all()


assert (
    features[
        "hist_n_completed_gaps_le_3d"
    ]
    <=
    features[
        "hist_n_completed_gaps_le_7d"
    ]
).all()


# ------------------------------------------------------------
# No negative historical gaps
# ------------------------------------------------------------

for col in [
    "hist_mean_completed_gap_days",
    "hist_std_completed_gap_days",
    "hist_min_completed_gap_days",
    "hist_max_completed_gap_days",
    "hist_previous_completed_gap_days",
]:

    non_null = (
        features[col]
        .dropna()
    )

    assert (
        non_null >= 0
    ).all()


# ------------------------------------------------------------
# No infinities in new numeric features
# ------------------------------------------------------------

new_columns = (
    features.columns[
        cols_before:
    ]
)


new_numeric_cols = (
    features[
        new_columns
    ]
    .select_dtypes(
        include=[
            np.number
        ]
    )
    .columns
)


assert ~np.isinf(
    features[
        new_numeric_cols
    ]
    .to_numpy()
).any()


print(
    "\nStrict PIT / cadence checks: PASSED"
)


# ============================================================
# 22. FINAL AUDIT
# ============================================================

cols_after = (
    features.shape[1]
)

n_new = (
    cols_after
    - cols_before
)


print("\n" + "=" * 100)
print("02D — HISTORICAL CADENCE / SEQUENCE STATE COMPLETE")
print("=" * 100)

print(
    f"Rows          : "
    f"{len(features):,}"
)

print(
    f"Customers     : "
    f"{features['customer_id'].nunique():,}"
)

print(
    f"Columns before: "
    f"{cols_before:,}"
)

print(
    f"Columns after : "
    f"{cols_after:,}"
)

print(
    f"New features  : "
    f"{n_new:,}"
)


# ============================================================
# 23. COMPLETED GAP DISTRIBUTION
# ============================================================

print(
    "\nNUMBER OF COMPLETED HISTORICAL GAPS"
)

display(
    features[
        "hist_n_completed_contact_gaps"
    ]
    .describe(
        percentiles=[
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame()
)


# ============================================================
# 24. HISTORICAL MEAN GAP
# ============================================================

print(
    "\nHISTORICAL MEAN COMPLETED GAP — DAYS"
)

display(
    features[
        "hist_mean_completed_gap_days"
    ]
    .describe(
        percentiles=[
            .10,
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame()
)


# ============================================================
# 25. HISTORICAL GAP VARIABILITY
# ============================================================

print(
    "\nHISTORICAL GAP COEFFICIENT OF VARIATION"
)

display(
    features[
        "hist_gap_coefficient_of_variation"
    ]
    .describe(
        percentiles=[
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame()
)


# ============================================================
# 26. CADENCE PROFILE
# ============================================================

print(
    "\nHISTORICAL CADENCE PROFILE"
)

display(
    features[
        "hist_cadence_profile"
    ]
    .value_counts(
        dropna=False
    )
    .to_frame("messages")
)


# ============================================================
# 27. CADENCE DIRECTION
# ============================================================

print(
    "\nCURRENT CADENCE VS HISTORICAL CADENCE"
)

cadence_direction = pd.DataFrame(
    {
        "accelerating":
            [
                features[
                    "hist_cadence_accelerating"
                ].sum()
            ],

        "decelerating":
            [
                features[
                    "hist_cadence_decelerating"
                ].sum()
            ],

        "near_mean":
            [
                features[
                    "hist_cadence_near_historical_mean"
                ].sum()
            ],

        "strong_acceleration":
            [
                features[
                    "hist_cadence_strongly_accelerating"
                ].sum()
            ],

        "strong_deceleration":
            [
                features[
                    "hist_cadence_strongly_decelerating"
                ].sum()
            ],
    }
)


display(
    cadence_direction
)


# ============================================================
# 28. PRESSURE TREND
# ============================================================

print(
    "\nRECENT CONTACT PRESSURE TREND"
)

pressure_trend = pd.DataFrame(
    {
        "increasing":
            [
                features[
                    "hist_recent_pressure_increasing"
                ].sum()
            ],

        "decreasing":
            [
                features[
                    "hist_recent_pressure_decreasing"
                ].sum()
            ],
    }
)


display(
    pressure_trend
)


# ============================================================
# 29. BURST / PRESSURE HISTORY
# ============================================================

print(
    "\nHISTORICAL BURST / PRESSURE FLAGS"
)

burst_summary = pd.DataFrame(
    {
        "repeated_short_cadence":
            [
                features[
                    "hist_has_repeated_short_cadence"
                ].sum()
            ],

        "sustained_contact_pressure":
            [
                features[
                    "hist_has_sustained_contact_pressure"
                ].sum()
            ],

        "high_contact_pressure":
            [
                features[
                    "hist_has_high_contact_pressure"
                ].sum()
            ],
    }
)


display(
    burst_summary
)


# ============================================================
# 30. SAMPLE CADENCE JOURNEY
# ============================================================

sample_customer = (
    features
    .groupby(
        "customer_id",
        sort=False,
    )
    .size()
    .idxmax()
)


print(
    f"\nSAMPLE CADENCE JOURNEY — "
    f"{sample_customer}"
)


display(
    features.loc[
        features[
            "customer_id"
        ].eq(
            sample_customer
        ),
        [
            "customer_id",
            "event_number",
            "sent_at",
            "current_template",
            "hist_n_previous_messages",
            "hist_days_since_previous_message",
            "hist_n_completed_contact_gaps",
            "hist_previous_completed_gap_days",
            "hist_mean_completed_gap_days",
            "hist_std_completed_gap_days",
            "hist_current_gap_to_mean_gap_ratio",
            "hist_cadence_accelerating",
            "hist_cadence_decelerating",
            "hist_n_messages_last_7d",
            "hist_n_messages_last_30d",
            "hist_contact_rate_7d_to_30d_ratio",
        ],
    ]
)


print("\n" + "=" * 100)
print("02D — HISTORICAL CADENCE / SEQUENCE STATE COMPLETE")
print("=" * 100)

02D — HISTORICAL CADENCE / SEQUENCE STATE
Starting shape: (75406, 360)

Strict PIT / cadence checks: PASSED

02D — HISTORICAL CADENCE / SEQUENCE STATE COMPLETE
Rows          : 75,406
Customers     : 11,724
Columns before: 360
Columns after : 400
New features  : 40

NUMBER OF COMPLETED HISTORICAL GAPS


,hist_n_completed_contact_gaps
count,"75,406.00"
mean,2.97
std,3.02
min,0.00
25%,0.00
50%,2.00
75%,5.00
90%,7.00
95%,9.00
99%,12.00



HISTORICAL MEAN COMPLETED GAP — DAYS


,hist_mean_completed_gap_days
count,"53,131.00"
mean,3.41
std,1.97
min,0.51
10%,1.39
25%,2.08
50%,3.01
75%,4.23
90%,5.78
95%,6.99



HISTORICAL GAP COEFFICIENT OF VARIATION


...


HISTORICAL BURST / PRESSURE FLAGS


,repeated_short_cadence,sustained_contact_pressure,high_contact_pressure
0,24900,21468,17534



SAMPLE CADENCE JOURNEY — C005915


,customer_id,event_number,sent_at,current_template,hist_n_previous_messages,hist_days_since_previous_message,hist_n_completed_contact_gaps,hist_previous_completed_gap_days,hist_mean_completed_gap_days,hist_std_completed_gap_days,hist_current_gap_to_mean_gap_ratio,hist_cadence_accelerating,hist_cadence_decelerating,hist_n_messages_last_7d,hist_n_messages_last_30d,hist_contact_rate_7d_to_30d_ratio
37464,C005915,1,2026-06-11 10:52:00,pix_link,0,NaN,0,NaN,NaN,NaN,NaN,0,0,0,0,NaN
37465,C005915,2,2026-06-12 10:02:00,pix_link,1,0.97,0,NaN,NaN,NaN,NaN,0,0,1,1,4.29
37466,C005915,3,2026-06-15 09:45:00,urgent_reminder,2,2.99,1,0.97,0.97,NaN,3.10,0,0,2,2,4.29
37467,C005915,4,2026-06-17 16:17:00,pix_link,3,2.27,2,2.99,1.98,1.43,1.15,0,1,3,3,4.29
37468,C005915,5,2026-06-19 12:18:00,urgent_reminder,4,1.83,3,2.27,2.08,1.03,0.88,1,0,2,4,2.14
37469,C005915,6,2026-06-21 11:52:00,urgent_reminder,5,1.98,4,1.83,2.01,0.85,0.98,1,0,3,5,2.57
37470,C005915,7,2026-06-22 16:15:00,pix_link,6,1.18,5,1.98,2.01,0.73,0.59,1,0,3,6,2.14
37471,C005915,8,2026-06-25 10:18:00,urgent_reminder,7,2.75,6,1.18,1.87,0.74,1.47,0,1,3,7,1.84
37472,C005915,9,2026-06-29 12:35:00,pix_link,8,4.10,7,2.75,2.00,0.75,2.05,0,1,2,8,1.07
37473,C005915,10,2026-06-30 14:22:00,urgent_reminder,9,1.07,8,4.10,2.26,1.02,0.48,1,0,2,9,0.95



02D — HISTORICAL CADENCE / SEQUENCE STATE COMPLETE


## ______________________________________________________

## ______________________________________________________

In [22]:
# ============================================================
# 03A — OUTCOME AVAILABILITY AUDIT
# ============================================================
#
# PURPOSE
# -------
# Audit outcome semantics BEFORE constructing historical
# response features.
#
# IMPORTANT:
#   This cell DOES NOT modify `features`.
#
# Outcomes:
#   1. event_delivery_status
#   2. event_interaction
#   3. event_paid_within_72h
#   4. event_amount_paid_brl
#
# Main question:
#
# At event t, which outcomes from previous events are actually
# mature / observable?
#
# Payment rule:
#
#   previous event j is mature at t iff
#
#       sent_at_j + 72h <= sent_at_t
#
# ============================================================

print("=" * 100)
print("03A — OUTCOME AVAILABILITY AUDIT")
print("=" * 100)


# ============================================================
# 0. CHECKPOINT
# ============================================================

assert features.shape == (75_406, 400)
assert features["customer_id"].nunique() == 11_724
assert features["event_id"].is_unique
assert features["message_id"].is_unique

print(f"Feature checkpoint : {features.shape}")


# ============================================================
# 1. MINIMAL OUTCOME AUDIT TABLE
# ============================================================
#
# Keep this narrow for memory efficiency.
# ============================================================

outcome_audit = features[
    [
        "event_id",
        "message_id",
        "customer_id",
        "event_number",
        "sent_at",
        "event_delivery_status",
        "event_interaction",
        "event_paid_within_72h",
        "event_amount_paid_brl",
    ]
].copy()


# ============================================================
# 2. BASIC VALIDITY
# ============================================================

print("\n" + "=" * 100)
print("A. BASIC OUTCOME VALIDITY")
print("=" * 100)

print(
    f"Rows       : {len(outcome_audit):,}"
)

print(
    f"Customers  : "
    f"{outcome_audit['customer_id'].nunique():,}"
)

print(
    f"Events     : "
    f"{outcome_audit['event_id'].nunique():,}"
)


print("\nMissingness:")

display(
    outcome_audit[
        [
            "event_delivery_status",
            "event_interaction",
            "event_paid_within_72h",
            "event_amount_paid_brl",
        ]
    ]
    .isna()
    .sum()
    .to_frame("missing")
)


# ============================================================
# 3. DELIVERY STATUS DISTRIBUTION
# ============================================================

print("\n" + "=" * 100)
print("B. DELIVERY STATUS")
print("=" * 100)

delivery_summary = (
    outcome_audit[
        "event_delivery_status"
    ]
    .value_counts(
        dropna=False
    )
    .rename("events")
    .to_frame()
)

delivery_summary["pct"] = (
    delivery_summary["events"]
    / len(outcome_audit)
    * 100
)

display(delivery_summary)


# ============================================================
# 4. INTERACTION DISTRIBUTION
# ============================================================

print("\n" + "=" * 100)
print("C. INTERACTION")
print("=" * 100)

interaction_summary = (
    outcome_audit[
        "event_interaction"
    ]
    .value_counts(
        dropna=False
    )
    .rename("events")
    .to_frame()
)

interaction_summary["pct"] = (
    interaction_summary["events"]
    / len(outcome_audit)
    * 100
)

display(interaction_summary)


# ============================================================
# 5. PAYMENT OUTCOME VALIDITY
# ============================================================

print("\n" + "=" * 100)
print("D. PAYMENT OUTCOME")
print("=" * 100)

payment_summary = (
    outcome_audit[
        "event_paid_within_72h"
    ]
    .value_counts(
        dropna=False
    )
    .rename("events")
    .to_frame()
)

payment_summary["pct"] = (
    payment_summary["events"]
    / len(outcome_audit)
    * 100
)

display(payment_summary)


print("\nPayment amount summary:")

display(
    outcome_audit[
        "event_amount_paid_brl"
    ]
    .describe(
        percentiles=[
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame()
)


# ============================================================
# 6. PAYMENT FLAG × AMOUNT CONSISTENCY
# ============================================================

paid_flag = (
    outcome_audit[
        "event_paid_within_72h"
    ].eq(1)
)

positive_amount = (
    outcome_audit[
        "event_amount_paid_brl"
    ] > 0
)


payment_consistency = pd.DataFrame(
    {
        "metric": [
            "paid_flag = 1",
            "amount > 0",
            "paid_flag = 1 AND amount > 0",
            "paid_flag = 1 AND amount <= 0",
            "paid_flag = 0 AND amount > 0",
        ],

        "events": [
            int(paid_flag.sum()),
            int(positive_amount.sum()),
            int(
                (
                    paid_flag
                    & positive_amount
                ).sum()
            ),
            int(
                (
                    paid_flag
                    & ~positive_amount
                ).sum()
            ),
            int(
                (
                    ~paid_flag
                    & positive_amount
                ).sum()
            ),
        ],
    }
)


print("\nPayment flag / amount consistency:")

display(payment_consistency)


# ============================================================
# 7. PAYMENT MATURITY TIMESTAMP
# ============================================================
#
# For a send at j:
#
# outcome_matures_at = sent_at_j + 72h
#
# Only after this timestamp can we safely know the complete
# binary outcome "paid within 72h".
# ============================================================

outcome_audit[
    "payment_outcome_matures_at"
] = (
    outcome_audit["sent_at"]
    + pd.Timedelta(hours=72)
)


# ============================================================
# 8. NEXT EVENT
# ============================================================
#
# This lets us answer:
#
# Did another action occur BEFORE the previous action's
# 72-hour payment window had matured?
# ============================================================

outcome_audit[
    "next_event_at"
] = (
    outcome_audit
    .groupby(
        "customer_id",
        sort=False,
    )["sent_at"]
    .shift(-1)
)


outcome_audit[
    "hours_to_next_event"
] = (
    (
        outcome_audit[
            "next_event_at"
        ]
        -
        outcome_audit[
            "sent_at"
        ]
    )
    .dt.total_seconds()
    / 3600
)


outcome_audit[
    "next_event_before_payment_maturity"
] = (
    outcome_audit[
        "next_event_at"
    ]
    .notna()
    &
    (
        outcome_audit[
            "next_event_at"
        ]
        <
        outcome_audit[
            "payment_outcome_matures_at"
        ]
    )
)


# ============================================================
# 9. HOW OFTEN DO WE ACT BEFORE 72H MATURITY?
# ============================================================

has_next_event = (
    outcome_audit[
        "next_event_at"
    ]
    .notna()
)


n_has_next = int(
    has_next_event.sum()
)


n_next_before_maturity = int(
    outcome_audit[
        "next_event_before_payment_maturity"
    ]
    .sum()
)


pct_next_before_maturity = (
    n_next_before_maturity
    / n_has_next
    * 100
    if n_has_next > 0
    else np.nan
)


print("\n" + "=" * 100)
print("E. 72H PAYMENT MATURITY VS NEXT ACTION")
print("=" * 100)

print(
    f"Events with a later send              : "
    f"{n_has_next:,}"
)

print(
    f"Next send occurs before 72h maturity  : "
    f"{n_next_before_maturity:,}"
)

print(
    f"Share of repeat opportunities         : "
    f"{pct_next_before_maturity:.2f}%"
)


# ============================================================
# 10. GAP TO NEXT EVENT DISTRIBUTION
# ============================================================

print("\nHours to next event:")

display(
    outcome_audit.loc[
        has_next_event,
        "hours_to_next_event"
    ]
    .describe(
        percentiles=[
            .01,
            .05,
            .10,
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame()
)


# ============================================================
# 11. IMPORTANT LEAKAGE TEST
# ============================================================
#
# Among events whose next action happens BEFORE the 72h window
# is mature:
#
# how many eventually have paid_within_72h = 1?
#
# If > 0, naive shift(payment flag) would leak future
# information into the next action.
# ============================================================

immature_before_next = (
    outcome_audit[
        "next_event_before_payment_maturity"
    ]
)


immature_positive_payment = (
    immature_before_next
    &
    paid_flag
)


print("\n" + "=" * 100)
print("F. POTENTIAL PAYMENT LEAKAGE")
print("=" * 100)

print(
    "Previous events still inside 72h window "
    f"when next send occurs : "
    f"{immature_before_next.sum():,}"
)

print(
    "Of those, eventually paid within 72h    : "
    f"{immature_positive_payment.sum():,}"
)


if immature_positive_payment.sum() > 0:

    print(
        "\nIMPORTANT:"
        "\nNaively shifting event_paid_within_72h would create "
        "future leakage for these events."
    )

else:

    print(
        "\nObserved dataset contains no positive-payment event "
        "whose next send occurred before its 72h maturity."
    )

    print(
        "We will STILL retain the maturity rule because it is "
        "the correct production-time semantics."
    )


# ============================================================
# 12. POSITIVE PAYMENT → NEXT SEND GAP
# ============================================================

paid_with_later_send = (
    paid_flag
    &
    has_next_event
)


print("\n" + "=" * 100)
print("G. POSITIVE PAYMENT EVENTS WITH LATER CONTACT")
print("=" * 100)

print(
    f"Positive payment events            : "
    f"{paid_flag.sum():,}"
)

print(
    f"Positive payment + later send      : "
    f"{paid_with_later_send.sum():,}"
)


if paid_with_later_send.any():

    display(
        outcome_audit.loc[
            paid_with_later_send,
            "hours_to_next_event"
        ]
        .describe(
            percentiles=[
                .01,
                .05,
                .10,
                .25,
                .50,
                .75,
                .90,
                .95,
                .99,
            ]
        )
        .to_frame()
    )


# ============================================================
# 13. DELIVERY × INTERACTION CONSISTENCY
# ============================================================
#
# Descriptive only.
#
# We want to understand whether interactions ever coexist
# with failure statuses.
# ============================================================

print("\n" + "=" * 100)
print("H. DELIVERY × INTERACTION CONSISTENCY")
print("=" * 100)


delivery_interaction = pd.crosstab(
    outcome_audit[
        "event_delivery_status"
    ],
    outcome_audit[
        "event_interaction"
    ],
    margins=True,
)


display(delivery_interaction)


# ============================================================
# 14. FAILURE STATUS × PAYMENT
# ============================================================
#
# Again descriptive only.
#
# This helps characterize persistent operational failure
# signals before we build their historical state.
# ============================================================

print("\n" + "=" * 100)
print("I. DELIVERY STATUS × PAYMENT")
print("=" * 100)


delivery_payment = pd.crosstab(
    outcome_audit[
        "event_delivery_status"
    ],
    outcome_audit[
        "event_paid_within_72h"
    ],
    margins=True,
)


display(delivery_payment)


# ============================================================
# 15. INTERACTION × PAYMENT
# ============================================================

print("\n" + "=" * 100)
print("J. INTERACTION × PAYMENT")
print("=" * 100)


interaction_payment = pd.crosstab(
    outcome_audit[
        "event_interaction"
    ],
    outcome_audit[
        "event_paid_within_72h"
    ],
    margins=True,
)


display(interaction_payment)


# ============================================================
# 16. CUSTOMER-LEVEL PAYMENT TIMING
# ============================================================
#
# For customers with any payment:
#
# first payment-attributed event and number of sends before it.
#
# Still descriptive; not yet a model feature.
# ============================================================

payer_events = (
    outcome_audit.loc[
        paid_flag
    ]
    .copy()
)


if len(payer_events) > 0:

    first_payment_event = (
        payer_events
        .sort_values(
            [
                "customer_id",
                "sent_at",
            ]
        )
        .groupby(
            "customer_id",
            sort=False,
        )
        .head(1)
    )

    print("\n" + "=" * 100)
    print("K. FIRST ATTRIBUTED PAYMENT EVENT")
    print("=" * 100)

    print(
        f"Payer customers : "
        f"{first_payment_event['customer_id'].nunique():,}"
    )

    print(
        "\nEvent number at first attributed payment:"
    )

    display(
        first_payment_event[
            "event_number"
        ]
        .describe(
            percentiles=[
                .25,
                .50,
                .75,
                .90,
                .95,
                .99,
            ]
        )
        .to_frame()
    )


# ============================================================
# 17. AVAILABILITY REGISTRY
# ============================================================
#
# This is NOT yet feature engineering.
#
# It documents the temporal status of each outcome.
# ============================================================

availability_registry = pd.DataFrame(
    [
        {
            "outcome":
                "event_delivery_status",

            "availability_rule":
                "timestamp not provided",

            "current_status":
                "requires temporal assumption",

            "planned_historical_use":
                "allowed only under documented availability assumption",
        },

        {
            "outcome":
                "event_interaction",

            "availability_rule":
                "interaction timestamp not provided",

            "current_status":
                "requires temporal assumption",

            "planned_historical_use":
                "conservative governance required",
        },

        {
            "outcome":
                "event_paid_within_72h",

            "availability_rule":
                "sent_at + 72h <= decision_time",

            "current_status":
                "explicit maturity rule",

            "planned_historical_use":
                "mature outcomes only",
        },

        {
            "outcome":
                "event_amount_paid_brl",

            "availability_rule":
                "sent_at + 72h <= decision_time",

            "current_status":
                "explicit maturity rule",

            "planned_historical_use":
                "mature outcomes only",
        },
    ]
)


print("\n" + "=" * 100)
print("L. OUTCOME AVAILABILITY REGISTRY")
print("=" * 100)

display(
    availability_registry
)


# ============================================================
# 18. FINAL
# ============================================================

print("\n" + "=" * 100)
print("03A — OUTCOME AVAILABILITY AUDIT COMPLETE")
print("=" * 100)

print(
    f"`features` remains unchanged: "
    f"{features.shape}"
)

assert features.shape == (75_406, 400)

print(
    "\nNo historical outcome features were created."
)

print("=" * 100)

03A — OUTCOME AVAILABILITY AUDIT
Feature checkpoint : (75406, 400)

A. BASIC OUTCOME VALIDITY
Rows       : 75,406
Customers  : 11,724
Events     : 75,406

Missingness:


,missing
event_delivery_status,0
event_interaction,0
event_paid_within_72h,0
event_amount_paid_brl,0



B. DELIVERY STATUS


,events,pct
event_delivery_status,,
delivered,63543,84.27
failed_blocked,5051,6.70
failed_invalid_number,4723,6.26
failed_unreachable,2089,2.77



C. INTERACTION


...


K. FIRST ATTRIBUTED PAYMENT EVENT
Payer customers : 4,893

Event number at first attributed payment:


,event_number
count,"4,893.00"
mean,3.82
std,2.81
min,1.00
25%,2.00
50%,3.00
75%,5.00
90%,8.00
95%,10.00
99%,12.00



L. OUTCOME AVAILABILITY REGISTRY


,outcome,availability_rule,current_status,planned_historical_use
0,event_delivery_status,timestamp not provided,requires temporal assumption,allowed only under documented availability ass...
1,event_interaction,interaction timestamp not provided,requires temporal assumption,conservative governance required
2,event_paid_within_72h,sent_at + 72h <= decision_time,explicit maturity rule,mature outcomes only
3,event_amount_paid_brl,sent_at + 72h <= decision_time,explicit maturity rule,mature outcomes only



03A — OUTCOME AVAILABILITY AUDIT COMPLETE
`features` remains unchanged: (75406, 400)

No historical outcome features were created.


In [23]:
# ============================================================
# 03B — CANONICAL MATURE PAYMENT HISTORY
# ============================================================
#
# PURPOSE
# -------
# Construct strict point-in-time historical payment features.
#
# At decision/event t, a previous event j contributes its
# payment outcome ONLY IF:
#
#       sent_at_j + 72h <= sent_at_t
#
# Therefore:
#
#   past event != necessarily known outcome
#
# IMPORTANT
# ---------
# This layer uses ONLY:
#
#   event_paid_within_72h
#   event_amount_paid_brl
#
# after explicit 72h maturity.
#
# Delivery and interaction history are NOT constructed here.
#
# Starting checkpoint:
#
#   75,406 × 400
#
# ============================================================

print("=" * 100)
print("03B — CANONICAL MATURE PAYMENT HISTORY")
print("=" * 100)


# ============================================================
# 0. STARTING CHECKPOINT
# ============================================================

rows_before = len(features)
cols_before = features.shape[1]

assert features.shape == (75_406, 400)
assert features["customer_id"].nunique() == 11_724
assert features["event_id"].is_unique
assert features["message_id"].is_unique

print(f"Starting shape: {features.shape}")


# ============================================================
# 1. NARROW PAYMENT EVENT TABLE
# ============================================================
#
# We intentionally do NOT work on the full 400-column frame.
# ============================================================

pay_events = features[
    [
        "event_id",
        "customer_id",
        "event_number",
        "sent_at",
        "event_paid_within_72h",
        "event_amount_paid_brl",
    ]
].copy()


pay_events["payment_matures_at"] = (
    pay_events["sent_at"]
    + pd.Timedelta(hours=72)
)


# ============================================================
# 2. BASIC PAYMENT CONSISTENCY
# ============================================================

paid_flag = (
    pay_events["event_paid_within_72h"]
    .eq(1)
)

positive_amount = (
    pay_events["event_amount_paid_brl"]
    .gt(0)
)


assert paid_flag.equals(positive_amount)

assert (
    pay_events.loc[
        ~paid_flag,
        "event_amount_paid_brl"
    ] == 0
).all()

assert (
    pay_events.loc[
        paid_flag,
        "event_amount_paid_brl"
    ] > 0
).all()


print("Payment flag / amount consistency: PASSED")


# ============================================================
# 3. BUILD MATURITY STREAM
# ============================================================
#
# Each source event creates an outcome that becomes available
# 72h later.
#
# Example:
#
# source send:
#   Jun 01 10:00
#
# outcome becomes observable:
#   Jun 04 10:00
#
# At every decision t we need all maturity events:
#
#   payment_matures_at <= sent_at_t
#
# ============================================================

maturity_stream = pay_events[
    [
        "event_id",
        "customer_id",
        "event_number",
        "sent_at",
        "payment_matures_at",
        "event_paid_within_72h",
        "event_amount_paid_brl",
    ]
].copy()


maturity_stream = maturity_stream.rename(
    columns={
        "event_id":
            "payment_source_event_id",

        "event_number":
            "payment_source_event_number",

        "sent_at":
            "payment_source_sent_at",

        "event_paid_within_72h":
            "mature_payment_flag",

        "event_amount_paid_brl":
            "mature_payment_amount_brl",
    }
)


# ============================================================
# 4. DECISION STREAM
# ============================================================

decision_stream = pay_events[
    [
        "event_id",
        "customer_id",
        "event_number",
        "sent_at",
    ]
].copy()


# ============================================================
# 5. SORT FOR SEARCH
# ============================================================
#
# We will process one customer at a time using numpy searchsorted.
#
# Dataset:
#   ~75k events
#
# This is fast, deterministic and avoids an expensive wide merge.
# ============================================================

customer_values = (
    decision_stream["customer_id"]
    .to_numpy(copy=False)
)

decision_times_ns = (
    decision_stream["sent_at"]
    .astype("int64")
    .to_numpy(copy=False)
)


maturity_times_ns = (
    maturity_stream["payment_matures_at"]
    .astype("int64")
    .to_numpy(copy=False)
)


payment_flags_np = (
    maturity_stream["mature_payment_flag"]
    .astype("int8")
    .to_numpy(copy=False)
)


payment_amount_np = (
    maturity_stream["mature_payment_amount_brl"]
    .astype("float64")
    .to_numpy(copy=False)
)


source_sent_ns = (
    maturity_stream["payment_source_sent_at"]
    .astype("int64")
    .to_numpy(copy=False)
)


# ============================================================
# 6. PREALLOCATE OUTPUT ARRAYS
# ============================================================

n = len(features)

n_mature_outcomes = np.zeros(
    n,
    dtype=np.int16,
)

n_mature_payments = np.zeros(
    n,
    dtype=np.int16,
)

n_mature_nonpayments = np.zeros(
    n,
    dtype=np.int16,
)

total_mature_amount = np.zeros(
    n,
    dtype=np.float64,
)

mean_mature_amount = np.full(
    n,
    np.nan,
    dtype=np.float64,
)

max_mature_amount = np.full(
    n,
    np.nan,
    dtype=np.float64,
)

last_payment_source_sent_ns = np.full(
    n,
    np.datetime64("NaT", "ns"),
    dtype="datetime64[ns]",
)

last_payment_matured_ns = np.full(
    n,
    np.datetime64("NaT", "ns"),
    dtype="datetime64[ns]",
)

last_mature_outcome_flag = np.full(
    n,
    -1,
    dtype=np.int8,
)

consecutive_mature_nonpayments = np.zeros(
    n,
    dtype=np.int16,
)


# ============================================================
# 7. CUSTOMER BOUNDARIES
# ============================================================
#
# Canonical events are already customer/time sorted.
# ============================================================

customer_change = np.r_[
    True,
    customer_values[1:]
    != customer_values[:-1],
]

group_starts = np.flatnonzero(
    customer_change
)

group_ends = np.r_[
    group_starts[1:],
    n,
]


# ============================================================
# 8. STRICT MATURITY ENGINE
# ============================================================
#
# For each decision time t:
#
# k = number of maturity timestamps <= t
#
# Only rows [0:k] are known at t.
#
# searchsorted(side="right") implements <=.
# ============================================================

for start, end in zip(
    group_starts,
    group_ends,
):

    decision_t = (
        decision_times_ns[
            start:end
        ]
    )

    maturity_t = (
        maturity_times_ns[
            start:end
        ]
    )

    flags = (
        payment_flags_np[
            start:end
        ]
    )

    amounts = (
        payment_amount_np[
            start:end
        ]
    )

    source_t = (
        source_sent_ns[
            start:end
        ]
    )

    # --------------------------------------------------------
    # Number of mature outcomes available at each decision
    # --------------------------------------------------------

    k_known = np.searchsorted(
        maturity_t,
        decision_t,
        side="right",
    )

    n_mature_outcomes[
        start:end
    ] = k_known


    # --------------------------------------------------------
    # Cumulative payment count
    # --------------------------------------------------------

    cumulative_payment_count = np.r_[
        0,
        np.cumsum(
            flags,
            dtype=np.int16,
        ),
    ]


    known_payment_count = (
        cumulative_payment_count[
            k_known
        ]
    )


    n_mature_payments[
        start:end
    ] = known_payment_count


    n_mature_nonpayments[
        start:end
    ] = (
        k_known
        -
        known_payment_count
    )


    # --------------------------------------------------------
    # Cumulative payment amount
    # --------------------------------------------------------

    cumulative_amount = np.r_[
        0.0,
        np.cumsum(
            amounts,
            dtype=np.float64,
        ),
    ]


    known_amount = (
        cumulative_amount[
            k_known
        ]
    )


    total_mature_amount[
        start:end
    ] = known_amount


    # --------------------------------------------------------
    # Mean amount AMONG historical payments
    # --------------------------------------------------------

    valid_payment_history = (
        known_payment_count > 0
    )


    local_mean_amount = np.full(
        end - start,
        np.nan,
        dtype=np.float64,
    )


    local_mean_amount[
        valid_payment_history
    ] = (
        known_amount[
            valid_payment_history
        ]
        /
        known_payment_count[
            valid_payment_history
        ]
    )


    mean_mature_amount[
        start:end
    ] = local_mean_amount


    # --------------------------------------------------------
    # Historical maximum payment
    # --------------------------------------------------------

    cumulative_max_amount = (
        np.maximum.accumulate(
            amounts
        )
    )


    local_max_amount = np.full(
        end - start,
        np.nan,
        dtype=np.float64,
    )


    has_known = (
        k_known > 0
    )


    local_max_amount[
        has_known
    ] = (
        cumulative_max_amount[
            k_known[
                has_known
            ] - 1
        ]
    )


    # If mature history exists but all outcomes are zero,
    # maximum historical payment amount should be 0.
    max_mature_amount[
        start:end
    ] = local_max_amount


    # --------------------------------------------------------
    # Last mature outcome flag
    # --------------------------------------------------------

    local_last_flag = np.full(
        end - start,
        -1,
        dtype=np.int8,
    )


    local_last_flag[
        has_known
    ] = (
        flags[
            k_known[
                has_known
            ] - 1
        ]
    )


    last_mature_outcome_flag[
        start:end
    ] = local_last_flag


    # --------------------------------------------------------
    # Last historical PAYMENT
    # --------------------------------------------------------

    payment_positions = np.flatnonzero(
        flags == 1
    )


    if len(payment_positions) > 0:

        payment_maturity_t = (
            maturity_t[
                payment_positions
            ]
        )

        # number of positive payment maturities <= decision t
        p_known = np.searchsorted(
            payment_maturity_t,
            decision_t,
            side="right",
        )

        has_payment = (
            p_known > 0
        )


        last_payment_position = np.full(
            end - start,
            -1,
            dtype=np.int32,
        )


        last_payment_position[
            has_payment
        ] = (
            payment_positions[
                p_known[
                    has_payment
                ] - 1
            ]
        )


        local_last_source = np.full(
            end - start,
            np.datetime64(
                "NaT",
                "ns",
            ),
            dtype="datetime64[ns]",
        )


        local_last_maturity = np.full(
            end - start,
            np.datetime64(
                "NaT",
                "ns",
            ),
            dtype="datetime64[ns]",
        )


        local_last_source[
            has_payment
        ] = (
            source_t[
                last_payment_position[
                    has_payment
                ]
            ]
            .astype(
                "datetime64[ns]"
            )
        )


        local_last_maturity[
            has_payment
        ] = (
            maturity_t[
                last_payment_position[
                    has_payment
                ]
            ]
            .astype(
                "datetime64[ns]"
            )
        )


        last_payment_source_sent_ns[
            start:end
        ] = local_last_source


        last_payment_matured_ns[
            start:end
        ] = local_last_maturity


    # --------------------------------------------------------
    # Consecutive mature non-payments
    # --------------------------------------------------------
    #
    # For every prefix of mature outcomes:
    #
    # [0, 0, 1, 0, 0]
    #
    # streaks:
    #
    # 1, 2, 0, 1, 2
    #
    # Then select streak at k_known.
    # --------------------------------------------------------

    streak_prefix = np.zeros(
        len(flags) + 1,
        dtype=np.int16,
    )


    running_streak = 0

    for j, flag in enumerate(
        flags,
        start=1,
    ):

        if flag == 0:
            running_streak += 1

        else:
            running_streak = 0

        streak_prefix[j] = (
            running_streak
        )


    consecutive_mature_nonpayments[
        start:end
    ] = (
        streak_prefix[
            k_known
        ]
    )


# ============================================================
# 9. ATTACH CANONICAL PAYMENT HISTORY FEATURES
# ============================================================

features[
    "hist_n_mature_payment_outcomes"
] = n_mature_outcomes


features[
    "hist_has_mature_payment_history"
] = (
    n_mature_outcomes > 0
).astype("int8")


features[
    "hist_no_mature_payment_history"
] = (
    n_mature_outcomes == 0
).astype("int8")


features[
    "hist_n_mature_payments"
] = n_mature_payments


features[
    "hist_n_mature_nonpayments"
] = n_mature_nonpayments


features[
    "hist_ever_paid_mature"
] = (
    n_mature_payments > 0
).astype("int8")


features[
    "hist_never_paid_among_mature"
] = (
    (n_mature_outcomes > 0)
    &
    (n_mature_payments == 0)
).astype("int8")


# ============================================================
# 10. HISTORICAL PAYMENT RATE
# ============================================================

features[
    "hist_mature_payment_rate"
] = np.where(
    n_mature_outcomes > 0,

    n_mature_payments
    / n_mature_outcomes,

    np.nan,
)


features[
    "hist_mature_nonpayment_rate"
] = np.where(
    n_mature_outcomes > 0,

    n_mature_nonpayments
    / n_mature_outcomes,

    np.nan,
)


# ============================================================
# 11. MONETARY HISTORY
# ============================================================

features[
    "hist_total_mature_amount_paid_brl"
] = total_mature_amount


features[
    "hist_mean_mature_payment_amount_brl"
] = mean_mature_amount


features[
    "hist_max_mature_payment_amount_brl"
] = max_mature_amount


features[
    "hist_log1p_total_mature_amount_paid"
] = np.log1p(
    total_mature_amount
)


# ============================================================
# 12. LAST PAYMENT TIMESTAMPS
# ============================================================

features[
    "hist_last_payment_source_sent_at"
] = pd.to_datetime(
    last_payment_source_sent_ns
)


features[
    "hist_last_payment_matured_at"
] = pd.to_datetime(
    last_payment_matured_ns
)


# ============================================================
# 13. PAYMENT RECENCY
# ============================================================

features[
    "hist_days_since_last_payment_source_send"
] = (
    (
        features["sent_at"]
        -
        features[
            "hist_last_payment_source_sent_at"
        ]
    )
    .dt.total_seconds()
    / 86400
)


features[
    "hist_days_since_last_payment_maturity"
] = (
    (
        features["sent_at"]
        -
        features[
            "hist_last_payment_matured_at"
        ]
    )
    .dt.total_seconds()
    / 86400
)


# ============================================================
# 14. LAST MATURE OUTCOME
# ============================================================
#
# -1 = no mature outcome yet
#  0 = last mature outcome was non-payment
#  1 = last mature outcome was payment
# ============================================================

features[
    "hist_last_mature_payment_outcome"
] = last_mature_outcome_flag


features[
    "hist_last_mature_outcome_was_payment"
] = (
    last_mature_outcome_flag == 1
).astype("int8")


features[
    "hist_last_mature_outcome_was_nonpayment"
] = (
    last_mature_outcome_flag == 0
).astype("int8")


# ============================================================
# 15. CONSECUTIVE MATURE NON-PAYMENTS
# ============================================================

features[
    "hist_consecutive_mature_nonpayments"
] = consecutive_mature_nonpayments


for threshold in [
    2,
    3,
    5,
    8,
]:

    features[
        f"hist_{threshold}plus_consecutive_mature_nonpayments"
    ] = (
        consecutive_mature_nonpayments
        >= threshold
    ).astype("int8")


# ============================================================
# 16. MATURE HISTORY COVERAGE
# ============================================================
#
# Important distinction:
#
# previous sends:
#   all prior actions
#
# mature outcomes:
#   prior actions whose 72h window has closed
#
# Difference:
#   previous actions still unresolved at decision time.
# ============================================================

previous_messages = (
    features[
        "hist_n_previous_messages"
    ]
    .to_numpy()
)


features[
    "hist_n_unmatured_previous_payment_outcomes"
] = (
    previous_messages
    -
    n_mature_outcomes
)


features[
    "hist_mature_payment_history_coverage"
] = np.where(
    previous_messages > 0,

    n_mature_outcomes
    / previous_messages,

    np.nan,
)


features[
    "hist_has_unmatured_previous_payment_outcome"
] = (
    (
        previous_messages
        -
        n_mature_outcomes
    ) > 0
).astype("int8")


# ============================================================
# 17. PAYMENT COUNT FLAGS
# ============================================================

for threshold in [
    1,
    2,
    3,
]:

    features[
        f"hist_{threshold}plus_mature_payments"
    ] = (
        n_mature_payments
        >= threshold
    ).astype("int8")


# ============================================================
# 18. PAYMENT RECENCY FLAGS
# ============================================================

days_since_payment_maturity = (
    features[
        "hist_days_since_last_payment_maturity"
    ]
)


for days in [
    3,
    7,
    14,
    30,
]:

    features[
        f"hist_payment_matured_within_{days}d"
    ] = (
        days_since_payment_maturity
        .le(days)
        &
        days_since_payment_maturity
        .notna()
    ).astype("int8")


# ============================================================
# 19. PAYMENT HISTORY PROFILE
# ============================================================

features[
    "hist_mature_payment_profile"
] = np.select(
    [
        n_mature_outcomes == 0,

        (
            (n_mature_outcomes > 0)
            &
            (n_mature_payments == 0)
        ),

        n_mature_payments == 1,

        n_mature_payments >= 2,
    ],
    [
        "no_mature_history",
        "mature_no_payment",
        "one_mature_payment",
        "multiple_mature_payments",
    ],
    default="unknown",
)


# ============================================================
# 20. HARD CONSISTENCY AUDITS
# ============================================================

print("\nRunning hard consistency checks...")


# ------------------------------------------------------------
# Counts
# ------------------------------------------------------------

assert (
    features[
        "hist_n_mature_payment_outcomes"
    ]
    ==
    (
        features[
            "hist_n_mature_payments"
        ]
        +
        features[
            "hist_n_mature_nonpayments"
        ]
    )
).all()


assert (
    features[
        "hist_n_mature_payment_outcomes"
    ]
    <=
    features[
        "hist_n_previous_messages"
    ]
).all()


assert (
    features[
        "hist_n_unmatured_previous_payment_outcomes"
    ]
    >= 0
).all()


assert (
    features[
        "hist_n_unmatured_previous_payment_outcomes"
    ]
    +
    features[
        "hist_n_mature_payment_outcomes"
    ]
    ==
    features[
        "hist_n_previous_messages"
    ]
).all()


# ------------------------------------------------------------
# First contacts
# ------------------------------------------------------------

first_contact = (
    features[
        "hist_is_first_contact"
    ].eq(1)
)


assert (
    features.loc[
        first_contact,
        "hist_n_mature_payment_outcomes"
    ] == 0
).all()


assert (
    features.loc[
        first_contact,
        "hist_n_mature_payments"
    ] == 0
).all()


assert (
    features.loc[
        first_contact,
        "hist_total_mature_amount_paid_brl"
    ] == 0
).all()


# ------------------------------------------------------------
# Payment rate bounds
# ------------------------------------------------------------

payment_rate_non_null = (
    features[
        "hist_mature_payment_rate"
    ]
    .dropna()
)


assert (
    payment_rate_non_null
    .between(
        0,
        1,
        inclusive="both",
    )
    .all()
)


# ------------------------------------------------------------
# Rate complements
# ------------------------------------------------------------

mature_history = (
    features[
        "hist_n_mature_payment_outcomes"
    ] > 0
)


assert np.allclose(
    (
        features.loc[
            mature_history,
            "hist_mature_payment_rate"
        ]
        +
        features.loc[
            mature_history,
            "hist_mature_nonpayment_rate"
        ]
    ),
    1.0,
)


# ------------------------------------------------------------
# No payment → zero total
# ------------------------------------------------------------

no_hist_payment = (
    features[
        "hist_n_mature_payments"
    ] == 0
)


assert (
    features.loc[
        no_hist_payment,
        "hist_total_mature_amount_paid_brl"
    ] == 0
).all()


assert (
    features.loc[
        no_hist_payment,
        "hist_mean_mature_payment_amount_brl"
    ]
    .isna()
    .all()
)


# ------------------------------------------------------------
# Historical payment timestamps must be in the past
# ------------------------------------------------------------

has_hist_payment = (
    features[
        "hist_ever_paid_mature"
    ].eq(1)
)


assert (
    features.loc[
        has_hist_payment,
        "hist_last_payment_source_sent_at"
    ]
    <
    features.loc[
        has_hist_payment,
        "sent_at"
    ]
).all()


assert (
    features.loc[
        has_hist_payment,
        "hist_last_payment_matured_at"
    ]
    <=
    features.loc[
        has_hist_payment,
        "sent_at"
    ]
).all()


assert (
    features.loc[
        has_hist_payment,
        "hist_days_since_last_payment_maturity"
    ] >= 0
).all()


# ------------------------------------------------------------
# No mature history => sentinel -1
# ------------------------------------------------------------

no_mature_history = (
    features[
        "hist_n_mature_payment_outcomes"
    ].eq(0)
)


assert (
    features.loc[
        no_mature_history,
        "hist_last_mature_payment_outcome"
    ] == -1
).all()


# ------------------------------------------------------------
# Mature history coverage
# ------------------------------------------------------------

coverage = (
    features[
        "hist_mature_payment_history_coverage"
    ]
    .dropna()
)


assert (
    coverage
    .between(
        0,
        1,
        inclusive="both",
    )
    .all()
)


# ------------------------------------------------------------
# No infinities in new numeric features
# ------------------------------------------------------------

new_columns = (
    features.columns[
        cols_before:
    ]
)


new_numeric = (
    features[
        new_columns
    ]
    .select_dtypes(
        include=[
            np.number
        ]
    )
)


assert ~np.isinf(
    new_numeric.to_numpy()
).any()


print("Hard consistency checks: PASSED")


# ============================================================
# 21. FINAL SHAPE
# ============================================================

cols_after = (
    features.shape[1]
)

n_new = (
    cols_after
    -
    cols_before
)


print("\n" + "=" * 100)
print("03B — CANONICAL MATURE PAYMENT HISTORY COMPLETE")
print("=" * 100)

print(
    f"Rows          : "
    f"{len(features):,}"
)

print(
    f"Customers     : "
    f"{features['customer_id'].nunique():,}"
)

print(
    f"Columns before: "
    f"{cols_before:,}"
)

print(
    f"Columns after : "
    f"{cols_after:,}"
)

print(
    f"New features  : "
    f"{n_new:,}"
)


# ============================================================
# 22. MATURE OUTCOME AVAILABILITY
# ============================================================

print(
    "\nMATURE PAYMENT OUTCOMES AVAILABLE AT DECISION"
)

display(
    features[
        "hist_n_mature_payment_outcomes"
    ]
    .describe(
        percentiles=[
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame()
)


# ============================================================
# 23. UNMATURED PREVIOUS OUTCOMES
# ============================================================

print(
    "\nUNMATURED PREVIOUS PAYMENT OUTCOMES"
)

display(
    features[
        "hist_n_unmatured_previous_payment_outcomes"
    ]
    .value_counts(
        dropna=False
    )
    .sort_index()
    .to_frame("events")
)


# ============================================================
# 24. PAYMENT HISTORY PROFILE
# ============================================================

print(
    "\nMATURE PAYMENT HISTORY PROFILE"
)

display(
    features[
        "hist_mature_payment_profile"
    ]
    .value_counts(
        dropna=False
    )
    .to_frame("events")
)


# ============================================================
# 25. HISTORICAL PAYMENT RATE
# ============================================================

print(
    "\nMATURE HISTORICAL PAYMENT RATE"
)

display(
    features[
        "hist_mature_payment_rate"
    ]
    .describe(
        percentiles=[
            .10,
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame()
)


# ============================================================
# 26. CONSECUTIVE MATURE NON-PAYMENTS
# ============================================================

print(
    "\nCONSECUTIVE MATURE NON-PAYMENTS"
)

display(
    features[
        "hist_consecutive_mature_nonpayments"
    ]
    .describe(
        percentiles=[
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame()
)


# ============================================================
# 27. HISTORICAL RECOVERED AMOUNT
# ============================================================

print(
    "\nTOTAL MATURE ATTRIBUTED PAYMENT AMOUNT"
)

display(
    features[
        "hist_total_mature_amount_paid_brl"
    ]
    .describe(
        percentiles=[
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame()
)


# ============================================================
# 28. PAYMENT RECENCY
# ============================================================

print(
    "\nDAYS SINCE LAST MATURE PAYMENT"
)

display(
    features[
        "hist_days_since_last_payment_maturity"
    ]
    .describe(
        percentiles=[
            .10,
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame()
)


# ============================================================
# 29. SAMPLE PAYMENT JOURNEY
# ============================================================
#
# Prefer a customer with multiple payments if one exists.
# ============================================================

customer_payment_counts = (
    pay_events.loc[
        pay_events[
            "event_paid_within_72h"
        ].eq(1)
    ]
    .groupby(
        "customer_id"
    )
    .size()
)


if (
    len(customer_payment_counts) > 0
):

    multi_payers = (
        customer_payment_counts[
            customer_payment_counts >= 2
        ]
    )


    if len(multi_payers) > 0:

        sample_customer = (
            multi_payers
            .sort_values(
                ascending=False
            )
            .index[0]
        )

    else:

        sample_customer = (
            customer_payment_counts
            .index[0]
        )


    print(
        f"\nSAMPLE PAYMENT JOURNEY — "
        f"{sample_customer}"
    )


    display(
        features.loc[
            features[
                "customer_id"
            ].eq(
                sample_customer
            ),
            [
                "customer_id",
                "event_number",
                "sent_at",
                "current_template",
                "event_paid_within_72h",
                "event_amount_paid_brl",
                "hist_n_previous_messages",
                "hist_n_mature_payment_outcomes",
                "hist_n_unmatured_previous_payment_outcomes",
                "hist_n_mature_payments",
                "hist_n_mature_nonpayments",
                "hist_mature_payment_rate",
                "hist_total_mature_amount_paid_brl",
                "hist_days_since_last_payment_maturity",
                "hist_consecutive_mature_nonpayments",
                "hist_mature_payment_profile",
            ],
        ]
    )


# ============================================================
# 30. IMPORTANT GOVERNANCE SUMMARY
# ============================================================

print("\n" + "=" * 100)
print("03B — TEMPORAL GOVERNANCE")
print("=" * 100)

print(
    """
PAYMENT HISTORY RULE

A previous WhatsApp event contributes to historical payment
features only when:

    previous_sent_at + 72h <= current_sent_at

Therefore:

    previous event
        !=
    necessarily observable payment outcome

Customers with:

    hist_n_mature_payment_outcomes == 0

are classified as:

    no_mature_history

NOT:

    non-payers

This distinction is required for strict point-in-time modeling.
"""
)


print("=" * 100)
print("03B — CANONICAL MATURE PAYMENT HISTORY COMPLETE")
print("=" * 100)

03B — CANONICAL MATURE PAYMENT HISTORY
Starting shape: (75406, 400)
Payment flag / amount consistency: PASSED

Running hard consistency checks...
Hard consistency checks: PASSED

03B — CANONICAL MATURE PAYMENT HISTORY COMPLETE
Rows          : 75,406
Customers     : 11,724
Columns before: 400
Columns after : 436
New features  : 36

MATURE PAYMENT OUTCOMES AVAILABLE AT DECISION


,hist_n_mature_payment_outcomes
count,"75,406.00"
mean,3.30
std,3.22
min,0.00
25%,0.00
50%,2.00
75%,5.00
90%,8.00
95%,10.00
99%,12.00



UNMATURED PREVIOUS PAYMENT OUTCOMES


,events
hist_n_unmatured_previous_payment_outcomes,
0,43926
1,25017
2,5958
3,505



MATURE PAYMENT HISTORY PROFILE


...


DAYS SINCE LAST MATURE PAYMENT


,hist_days_since_last_payment_maturity
count,"6,131.00"
mean,13.61
std,11.55
min,0.53
10%,1.99
25%,4.25
50%,10.18
75%,19.76
90%,31.02
95%,38.10



SAMPLE PAYMENT JOURNEY — C004058


,customer_id,event_number,sent_at,current_template,event_paid_within_72h,event_amount_paid_brl,hist_n_previous_messages,hist_n_mature_payment_outcomes,hist_n_unmatured_previous_payment_outcomes,hist_n_mature_payments,hist_n_mature_nonpayments,hist_mature_payment_rate,hist_total_mature_amount_paid_brl,hist_days_since_last_payment_maturity,hist_consecutive_mature_nonpayments,hist_mature_payment_profile
25949,C004058,1,2026-07-15 11:02:00,friendly_reminder,0,0.00,0,0,0,0,0,NaN,0.00,NaN,0,no_mature_history
25950,C004058,2,2026-07-21 15:39:00,friendly_reminder,0,0.00,1,1,0,0,1,0.00,0.00,NaN,1,mature_no_payment
25951,C004058,3,2026-07-23 13:05:00,pix_link,1,428.37,2,1,1,0,1,0.00,0.00,NaN,1,mature_no_payment
25952,C004058,4,2026-07-27 10:13:00,pix_link,1,188.79,3,3,0,1,2,0.33,428.37,0.88,0,one_mature_payment
25953,C004058,5,2026-07-31 09:29:00,urgent_reminder,0,0.00,4,4,0,2,2,0.50,617.16,0.97,0,multiple_mature_payments
25954,C004058,6,2026-08-02 15:33:00,pix_link,0,0.00,5,4,1,2,2,0.50,617.16,3.22,0,multiple_mature_payments
25955,C004058,7,2026-08-03 14:03:00,urgent_reminder,1,283.17,6,5,1,2,3,0.40,617.16,4.16,1,multiple_mature_payments
25956,C004058,8,2026-08-07 09:50:00,urgent_reminder,0,0.00,7,7,0,3,4,0.43,900.33,0.82,0,multiple_mature_payments
25957,C004058,9,2026-08-14 10:06:00,pix_link,1,78.86,8,8,0,3,5,0.38,900.33,7.84,1,multiple_mature_payments



03B — TEMPORAL GOVERNANCE

PAYMENT HISTORY RULE

A previous WhatsApp event contributes to historical payment
features only when:

    previous_sent_at + 72h <= current_sent_at

Therefore:

    previous event
        !=
    necessarily observable payment outcome

Customers with:

    hist_n_mature_payment_outcomes == 0

are classified as:

    no_mature_history

NOT:

    non-payers

This distinction is required for strict point-in-time modeling.

03B — CANONICAL MATURE PAYMENT HISTORY COMPLETE


In [24]:
# ============================================================
# 03C — HISTORICAL DELIVERY / FAILURE STATE
# ============================================================
#
# PURPOSE
# -------
# Build strict historical delivery-state features.
#
# TEMPORAL ASSUMPTION
# -------------------
# Unlike payment, the dataset does not provide a timestamp for
# delivery feedback.
#
# We therefore make the explicit operational assumption:
#
#   delivery_status from event j is available before the next
#   collection decision/event t.
#
# Under that assumption:
#
#   historical delivery features at t use ONLY events < t.
#
# IMPORTANT
# ---------
# Current event_delivery_status is NEVER used in its own
# historical features.
#
# Starting checkpoint:
#
#   75,406 × 436
#
# ============================================================

print("=" * 100)
print("03C — HISTORICAL DELIVERY / FAILURE STATE")
print("=" * 100)


# ============================================================
# 0. STARTING CHECKPOINT
# ============================================================

rows_before = len(features)
cols_before = features.shape[1]

assert features.shape == (75_406, 436)
assert features["customer_id"].nunique() == 11_724
assert features["event_id"].is_unique
assert features["message_id"].is_unique

print(f"Starting shape: {features.shape}")


# ============================================================
# 1. CANONICAL DELIVERY FLAGS — CURRENT EVENT
# ============================================================
#
# Temporary arrays only.
# They will NOT be attached as historical state until shifted.
# ============================================================

delivery = (
    features["event_delivery_status"]
    .astype(str)
)

customer = features["customer_id"]


is_delivered = (
    delivery.eq("delivered")
    .astype("int8")
)

is_blocked = (
    delivery.eq("failed_blocked")
    .astype("int8")
)

is_invalid = (
    delivery.eq("failed_invalid_number")
    .astype("int8")
)

is_unreachable = (
    delivery.eq("failed_unreachable")
    .astype("int8")
)

is_failed = (
    ~delivery.eq("delivered")
).astype("int8")


is_hard_failure = (
    delivery.isin(
        [
            "failed_blocked",
            "failed_invalid_number",
        ]
    )
).astype("int8")


# ------------------------------------------------------------
# Validate exhaustive delivery taxonomy
# ------------------------------------------------------------

assert (
    is_delivered
    + is_blocked
    + is_invalid
    + is_unreachable
    ==
    1
).all()

assert (
    is_failed
    ==
    (
        is_blocked
        + is_invalid
        + is_unreachable
    )
).all()

assert (
    is_hard_failure
    ==
    (
        is_blocked
        + is_invalid
    )
).all()

print("Delivery taxonomy: PASSED")


# ============================================================
# 2. HISTORICAL CUMULATIVE COUNTS
# ============================================================
#
# cumsum - current flag
#
# gives number observed STRICTLY BEFORE current event.
# ============================================================

def prior_cumsum(flag):

    cumulative = (
        flag
        .groupby(
            customer,
            sort=False,
        )
        .cumsum()
    )

    return (
        cumulative
        -
        flag
    )


hist_n_delivered = prior_cumsum(
    is_delivered
).astype("int16")

hist_n_failed = prior_cumsum(
    is_failed
).astype("int16")

hist_n_blocked = prior_cumsum(
    is_blocked
).astype("int16")

hist_n_invalid = prior_cumsum(
    is_invalid
).astype("int16")

hist_n_unreachable = prior_cumsum(
    is_unreachable
).astype("int16")

hist_n_hard_failures = prior_cumsum(
    is_hard_failure
).astype("int16")


features[
    "hist_n_delivered"
] = hist_n_delivered

features[
    "hist_n_failed"
] = hist_n_failed

features[
    "hist_n_failed_blocked"
] = hist_n_blocked

features[
    "hist_n_failed_invalid_number"
] = hist_n_invalid

features[
    "hist_n_failed_unreachable"
] = hist_n_unreachable

features[
    "hist_n_hard_delivery_failures"
] = hist_n_hard_failures


# ============================================================
# 3. EVER-SEEN FLAGS
# ============================================================

features[
    "hist_ever_delivered"
] = (
    hist_n_delivered > 0
).astype("int8")

features[
    "hist_ever_failed"
] = (
    hist_n_failed > 0
).astype("int8")

features[
    "hist_ever_failed_blocked"
] = (
    hist_n_blocked > 0
).astype("int8")

features[
    "hist_ever_failed_invalid_number"
] = (
    hist_n_invalid > 0
).astype("int8")

features[
    "hist_ever_failed_unreachable"
] = (
    hist_n_unreachable > 0
).astype("int8")

features[
    "hist_ever_hard_delivery_failure"
] = (
    hist_n_hard_failures > 0
).astype("int8")


# ============================================================
# 4. HISTORICAL DELIVERY / FAILURE RATES
# ============================================================

n_previous = (
    features[
        "hist_n_previous_messages"
    ]
)


features[
    "hist_delivery_rate"
] = np.where(
    n_previous > 0,
    hist_n_delivered / n_previous,
    np.nan,
)


features[
    "hist_failure_rate"
] = np.where(
    n_previous > 0,
    hist_n_failed / n_previous,
    np.nan,
)


features[
    "hist_blocked_rate"
] = np.where(
    n_previous > 0,
    hist_n_blocked / n_previous,
    np.nan,
)


features[
    "hist_invalid_number_rate"
] = np.where(
    n_previous > 0,
    hist_n_invalid / n_previous,
    np.nan,
)


features[
    "hist_unreachable_rate"
] = np.where(
    n_previous > 0,
    hist_n_unreachable / n_previous,
    np.nan,
)


features[
    "hist_hard_delivery_failure_rate"
] = np.where(
    n_previous > 0,
    hist_n_hard_failures / n_previous,
    np.nan,
)


# ============================================================
# 5. PREVIOUS DELIVERY STATUS
# ============================================================

features[
    "hist_previous_delivery_status"
] = (
    features
    .groupby(
        "customer_id",
        sort=False,
    )[
        "event_delivery_status"
    ]
    .shift(1)
)


features[
    "hist_previous_was_delivered"
] = (
    features[
        "hist_previous_delivery_status"
    ]
    .eq("delivered")
    .astype("int8")
)


features[
    "hist_previous_was_failed"
] = (
    features[
        "hist_previous_delivery_status"
    ]
    .isin(
        [
            "failed_blocked",
            "failed_invalid_number",
            "failed_unreachable",
        ]
    )
    .astype("int8")
)


features[
    "hist_previous_was_blocked"
] = (
    features[
        "hist_previous_delivery_status"
    ]
    .eq("failed_blocked")
    .astype("int8")
)


features[
    "hist_previous_was_invalid_number"
] = (
    features[
        "hist_previous_delivery_status"
    ]
    .eq("failed_invalid_number")
    .astype("int8")
)


features[
    "hist_previous_was_unreachable"
] = (
    features[
        "hist_previous_delivery_status"
    ]
    .eq("failed_unreachable")
    .astype("int8")
)


features[
    "hist_previous_was_hard_delivery_failure"
] = (
    features[
        "hist_previous_delivery_status"
    ]
    .isin(
        [
            "failed_blocked",
            "failed_invalid_number",
        ]
    )
    .astype("int8")
)


# ============================================================
# 6. LAST OCCURRENCE — FAILURE TYPES
# ============================================================
#
# Vectorized:
#
# current event time if flag == 1
# → forward-fill within customer
# → shift one event
#
# This ensures current outcome never leaks into current state.
# ============================================================

def historical_last_occurrence(flag):

    flagged_time = (
        features["sent_at"]
        .where(
            flag.eq(1)
        )
    )

    including_current = (
        flagged_time
        .groupby(
            customer,
            sort=False,
        )
        .ffill()
    )

    return (
        including_current
        .groupby(
            customer,
            sort=False,
        )
        .shift(1)
    )


last_failed_at = (
    historical_last_occurrence(
        is_failed
    )
)

last_blocked_at = (
    historical_last_occurrence(
        is_blocked
    )
)

last_invalid_at = (
    historical_last_occurrence(
        is_invalid
    )
)

last_unreachable_at = (
    historical_last_occurrence(
        is_unreachable
    )
)

last_hard_failure_at = (
    historical_last_occurrence(
        is_hard_failure
    )
)

last_delivered_at = (
    historical_last_occurrence(
        is_delivered
    )
)


features[
    "hist_last_failed_at"
] = last_failed_at

features[
    "hist_last_failed_blocked_at"
] = last_blocked_at

features[
    "hist_last_failed_invalid_number_at"
] = last_invalid_at

features[
    "hist_last_failed_unreachable_at"
] = last_unreachable_at

features[
    "hist_last_hard_delivery_failure_at"
] = last_hard_failure_at

features[
    "hist_last_delivered_at"
] = last_delivered_at


# ============================================================
# 7. RECENCY — DAYS SINCE LAST STATUS
# ============================================================

def days_since(timestamp_series):

    return (
        (
            features["sent_at"]
            -
            timestamp_series
        )
        .dt.total_seconds()
        / 86400
    )


features[
    "hist_days_since_last_failure"
] = days_since(
    last_failed_at
)

features[
    "hist_days_since_last_blocked"
] = days_since(
    last_blocked_at
)

features[
    "hist_days_since_last_invalid_number"
] = days_since(
    last_invalid_at
)

features[
    "hist_days_since_last_unreachable"
] = days_since(
    last_unreachable_at
)

features[
    "hist_days_since_last_hard_delivery_failure"
] = days_since(
    last_hard_failure_at
)

features[
    "hist_days_since_last_delivered"
] = days_since(
    last_delivered_at
)


# ============================================================
# 8. FAILURE RECENCY FLAGS
# ============================================================

for days in [
    1,
    3,
    7,
    14,
    30,
]:

    features[
        f"hist_failure_within_{days}d"
    ] = (
        features[
            "hist_days_since_last_failure"
        ]
        .le(days)
        &
        features[
            "hist_days_since_last_failure"
        ]
        .notna()
    ).astype("int8")


for days in [
    3,
    7,
    14,
    30,
]:

    features[
        f"hist_hard_failure_within_{days}d"
    ] = (
        features[
            "hist_days_since_last_hard_delivery_failure"
        ]
        .le(days)
        &
        features[
            "hist_days_since_last_hard_delivery_failure"
        ]
        .notna()
    ).astype("int8")


# ============================================================
# 9. CONSECUTIVE DELIVERY-STATUS STREAKS
# ============================================================
#
# Historical streak means streak ending at PREVIOUS event.
#
# Example:
#
# event statuses:
#
# delivered
# invalid
# invalid
# current event
#
# historical consecutive invalid streak at current = 2
# ============================================================

def historical_streak(flag):

    flag_np = (
        flag
        .to_numpy(
            dtype=np.int8,
            copy=False,
        )
    )

    customer_np = (
        customer
        .to_numpy(
            copy=False
        )
    )

    current_streak = np.zeros(
        len(flag_np),
        dtype=np.int16,
    )

    running = 0

    previous_customer = None

    for i in range(
        len(flag_np)
    ):

        current_customer = (
            customer_np[i]
        )

        if (
            i == 0
            or
            current_customer
            != previous_customer
        ):
            running = 0

        if flag_np[i] == 1:
            running += 1
        else:
            running = 0

        current_streak[i] = running

        previous_customer = (
            current_customer
        )

    # shift within customer
    historical = np.zeros(
        len(flag_np),
        dtype=np.int16,
    )

    same_customer_as_previous = np.r_[
        False,
        customer_np[1:]
        ==
        customer_np[:-1],
    ]

    historical[
        same_customer_as_previous
    ] = (
        current_streak[:-1][
            same_customer_as_previous[1:]
        ]
    )

    return historical


hist_failed_streak = (
    historical_streak(
        is_failed
    )
)

hist_hard_failure_streak = (
    historical_streak(
        is_hard_failure
    )
)

hist_invalid_streak = (
    historical_streak(
        is_invalid
    )
)

hist_blocked_streak = (
    historical_streak(
        is_blocked
    )
)

hist_unreachable_streak = (
    historical_streak(
        is_unreachable
    )
)

hist_delivered_streak = (
    historical_streak(
        is_delivered
    )
)


features[
    "hist_consecutive_failures"
] = hist_failed_streak

features[
    "hist_consecutive_hard_delivery_failures"
] = hist_hard_failure_streak

features[
    "hist_consecutive_invalid_number"
] = hist_invalid_streak

features[
    "hist_consecutive_blocked"
] = hist_blocked_streak

features[
    "hist_consecutive_unreachable"
] = hist_unreachable_streak

features[
    "hist_consecutive_delivered"
] = hist_delivered_streak


# ============================================================
# 10. FAILURE INTENSITY FLAGS
# ============================================================

for threshold in [
    2,
    3,
    5,
]:

    features[
        f"hist_{threshold}plus_consecutive_failures"
    ] = (
        hist_failed_streak
        >= threshold
    ).astype("int8")


for threshold in [
    2,
    3,
]:

    features[
        f"hist_{threshold}plus_consecutive_hard_failures"
    ] = (
        hist_hard_failure_streak
        >= threshold
    ).astype("int8")


# ============================================================
# 11. PERSISTENT HARD-FAILURE CANDIDATE STATE
# ============================================================
#
# IMPORTANT:
#
# These are predictive / operational candidate states.
#
# We are NOT yet declaring a causal stop rule.
#
# hard failure:
#   invalid_number OR blocked
#
# ============================================================

features[
    "hist_hard_failure_only_so_far"
] = (
    (n_previous > 0)
    &
    (hist_n_hard_failures == n_previous)
).astype("int8")


features[
    "hist_hard_failure_never_followed_by_delivery"
] = (
    (hist_n_hard_failures > 0)
    &
    (
        last_hard_failure_at.notna()
    )
    &
    (
        last_delivered_at.isna()
        |
        (
            last_delivered_at
            <
            last_hard_failure_at
        )
    )
).astype("int8")


# ============================================================
# 12. RECOVERY AFTER HISTORICAL FAILURE SIGNAL
# ============================================================
#
# Has the customer EVER had a delivered message after having
# previously experienced a given failure type?
#
# This is useful for distinguishing:
#
#   persistent failure
# vs
#   transient failure
#
# Still strictly historical at current event.
# ============================================================

def ever_delivered_after_failure(
    failure_flag
):

    failure_seen_before_current = (
        prior_cumsum(
            failure_flag
        ) > 0
    )

    # For each historical delivered event, ask whether a
    # failure had already happened before THAT delivered event.
    failure_before_event = (
        prior_cumsum(
            failure_flag
        ) > 0
    )

    delivered_after_failure_event = (
        (
            is_delivered.eq(1)
        )
        &
        failure_before_event
    ).astype("int8")

    historical_count = (
        prior_cumsum(
            delivered_after_failure_event
        )
    )

    return (
        failure_seen_before_current
        &
        (historical_count > 0)
    ).astype("int8")


features[
    "hist_ever_delivered_after_invalid"
] = (
    ever_delivered_after_failure(
        is_invalid
    )
)


features[
    "hist_ever_delivered_after_blocked"
] = (
    ever_delivered_after_failure(
        is_blocked
    )
)


features[
    "hist_ever_delivered_after_unreachable"
] = (
    ever_delivered_after_failure(
        is_unreachable
    )
)


features[
    "hist_ever_delivered_after_hard_failure"
] = (
    ever_delivered_after_failure(
        is_hard_failure
    )
)


# ============================================================
# 13. FAILURE PERSISTENCE STATE
# ============================================================
#
# Descriptive state only.
#
# This deliberately keeps unreachable separate from hard
# invalid/blocked signals.
# ============================================================

features[
    "hist_delivery_failure_profile"
] = np.select(
    [
        n_previous.eq(0),

        (
            hist_n_failed.eq(0)
            &
            n_previous.gt(0)
        ),

        (
            hist_n_hard_failures.gt(0)
            &
            features[
                "hist_ever_delivered_after_hard_failure"
            ].eq(0)
        ),

        (
            hist_n_hard_failures.gt(0)
            &
            features[
                "hist_ever_delivered_after_hard_failure"
            ].eq(1)
        ),

        (
            hist_n_unreachable.gt(0)
            &
            hist_n_hard_failures.eq(0)
        ),
    ],
    [
        "no_history",
        "all_delivered",
        "hard_failure_no_recovery",
        "hard_failure_then_recovered",
        "unreachable_only_failure",
    ],
    default="mixed_failure_history",
)


# ============================================================
# 14. HARD PIT / CONSISTENCY AUDITS
# ============================================================

print("\nRunning hard consistency checks...")


# ------------------------------------------------------------
# Historical status counts must sum to previous messages
# ------------------------------------------------------------

assert (
    hist_n_delivered
    +
    hist_n_blocked
    +
    hist_n_invalid
    +
    hist_n_unreachable
    ==
    n_previous
).all()


assert (
    hist_n_failed
    ==
    (
        hist_n_blocked
        +
        hist_n_invalid
        +
        hist_n_unreachable
    )
).all()


assert (
    hist_n_hard_failures
    ==
    (
        hist_n_blocked
        +
        hist_n_invalid
    )
).all()


# ------------------------------------------------------------
# First contact must have zero historical delivery state
# ------------------------------------------------------------

first_contact = (
    features[
        "hist_is_first_contact"
    ].eq(1)
)


count_columns = [
    "hist_n_delivered",
    "hist_n_failed",
    "hist_n_failed_blocked",
    "hist_n_failed_invalid_number",
    "hist_n_failed_unreachable",
    "hist_n_hard_delivery_failures",
]


assert (
    features.loc[
        first_contact,
        count_columns
    ]
    .eq(0)
    .all()
    .all()
)


assert (
    features.loc[
        first_contact,
        "hist_previous_delivery_status"
    ]
    .isna()
    .all()
)


# ------------------------------------------------------------
# Rates
# ------------------------------------------------------------

rate_columns = [
    "hist_delivery_rate",
    "hist_failure_rate",
    "hist_blocked_rate",
    "hist_invalid_number_rate",
    "hist_unreachable_rate",
    "hist_hard_delivery_failure_rate",
]


for col in rate_columns:

    non_null = (
        features[col]
        .dropna()
    )

    assert (
        non_null
        .between(
            0,
            1,
            inclusive="both",
        )
        .all()
    )


repeat_events = (
    n_previous > 0
)


assert np.allclose(
    (
        features.loc[
            repeat_events,
            "hist_delivery_rate"
        ]
        +
        features.loc[
            repeat_events,
            "hist_failure_rate"
        ]
    ),
    1.0,
)


# ------------------------------------------------------------
# Historical timestamps cannot be current/future events
# ------------------------------------------------------------

timestamp_columns = [
    "hist_last_failed_at",
    "hist_last_failed_blocked_at",
    "hist_last_failed_invalid_number_at",
    "hist_last_failed_unreachable_at",
    "hist_last_hard_delivery_failure_at",
    "hist_last_delivered_at",
]


for col in timestamp_columns:

    mask = (
        features[col]
        .notna()
    )

    assert (
        features.loc[
            mask,
            col
        ]
        <
        features.loc[
            mask,
            "sent_at"
        ]
    ).all()


# ------------------------------------------------------------
# Recencies nonnegative
# ------------------------------------------------------------

recency_columns = [
    "hist_days_since_last_failure",
    "hist_days_since_last_blocked",
    "hist_days_since_last_invalid_number",
    "hist_days_since_last_unreachable",
    "hist_days_since_last_hard_delivery_failure",
    "hist_days_since_last_delivered",
]


for col in recency_columns:

    assert (
        features[col]
        .dropna()
        .ge(0)
        .all()
    )


# ------------------------------------------------------------
# Streak cannot exceed historical count
# ------------------------------------------------------------

assert (
    features[
        "hist_consecutive_failures"
    ]
    <=
    features[
        "hist_n_failed"
    ]
).all()


assert (
    features[
        "hist_consecutive_hard_delivery_failures"
    ]
    <=
    features[
        "hist_n_hard_delivery_failures"
    ]
).all()


assert (
    features[
        "hist_consecutive_invalid_number"
    ]
    <=
    features[
        "hist_n_failed_invalid_number"
    ]
).all()


assert (
    features[
        "hist_consecutive_blocked"
    ]
    <=
    features[
        "hist_n_failed_blocked"
    ]
).all()


assert (
    features[
        "hist_consecutive_unreachable"
    ]
    <=
    features[
        "hist_n_failed_unreachable"
    ]
).all()


# ------------------------------------------------------------
# Recovered-after-failure implies historical failure exists
# ------------------------------------------------------------

assert (
    features.loc[
        features[
            "hist_ever_delivered_after_invalid"
        ].eq(1),
        "hist_n_failed_invalid_number"
    ] > 0
).all()


assert (
    features.loc[
        features[
            "hist_ever_delivered_after_blocked"
        ].eq(1),
        "hist_n_failed_blocked"
    ] > 0
).all()


assert (
    features.loc[
        features[
            "hist_ever_delivered_after_unreachable"
        ].eq(1),
        "hist_n_failed_unreachable"
    ] > 0
).all()


# ------------------------------------------------------------
# No infinities
# ------------------------------------------------------------

new_columns = (
    features.columns[
        cols_before:
    ]
)


new_numeric = (
    features[
        new_columns
    ]
    .select_dtypes(
        include=[
            np.number
        ]
    )
)


assert ~np.isinf(
    new_numeric.to_numpy()
).any()


print("Hard consistency checks: PASSED")


# ============================================================
# 15. FINAL SHAPE
# ============================================================

cols_after = features.shape[1]
n_new = cols_after - cols_before


print("\n" + "=" * 100)
print("03C — HISTORICAL DELIVERY / FAILURE STATE COMPLETE")
print("=" * 100)

print(
    f"Rows          : "
    f"{len(features):,}"
)

print(
    f"Customers     : "
    f"{features['customer_id'].nunique():,}"
)

print(
    f"Columns before: "
    f"{cols_before:,}"
)

print(
    f"Columns after : "
    f"{cols_after:,}"
)

print(
    f"New features  : "
    f"{n_new:,}"
)


# ============================================================
# 16. HISTORICAL FAILURE PREVALENCE
# ============================================================

print(
    "\nEVER EXPERIENCED DELIVERY STATUS BEFORE CURRENT EVENT"
)


prevalence = pd.DataFrame(
    {
        "events": [
            features[
                "hist_ever_failed"
            ].sum(),

            features[
                "hist_ever_failed_invalid_number"
            ].sum(),

            features[
                "hist_ever_failed_blocked"
            ].sum(),

            features[
                "hist_ever_failed_unreachable"
            ].sum(),

            features[
                "hist_ever_hard_delivery_failure"
            ].sum(),
        ],
    },
    index=[
        "any_failure",
        "invalid_number",
        "blocked",
        "unreachable",
        "hard_failure_invalid_or_blocked",
    ],
)


prevalence["pct_events"] = (
    prevalence["events"]
    / len(features)
    * 100
)


display(prevalence)


# ============================================================
# 17. DELIVERY FAILURE PROFILE
# ============================================================

print(
    "\nHISTORICAL DELIVERY FAILURE PROFILE"
)

display(
    features[
        "hist_delivery_failure_profile"
    ]
    .value_counts(
        dropna=False
    )
    .to_frame("events")
)


# ============================================================
# 18. FAILURE STREAKS
# ============================================================

print(
    "\nCONSECUTIVE HISTORICAL FAILURE STREAK"
)

display(
    features[
        "hist_consecutive_failures"
    ]
    .describe(
        percentiles=[
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame()
)


print(
    "\nCONSECUTIVE HISTORICAL HARD-FAILURE STREAK"
)

display(
    features[
        "hist_consecutive_hard_delivery_failures"
    ]
    .describe(
        percentiles=[
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame()
)


# ============================================================
# 19. RECOVERY AFTER FAILURE TYPE
# ============================================================

print(
    "\nDELIVERED AFTER PRIOR FAILURE — HISTORICAL STATE"
)


recovery_after_failure = pd.DataFrame(
    {
        "failure_events_with_history": [
            (
                features[
                    "hist_n_failed_invalid_number"
                ] > 0
            ).sum(),

            (
                features[
                    "hist_n_failed_blocked"
                ] > 0
            ).sum(),

            (
                features[
                    "hist_n_failed_unreachable"
                ] > 0
            ).sum(),

            (
                features[
                    "hist_n_hard_delivery_failures"
                ] > 0
            ).sum(),
        ],

        "later_delivery_observed": [
            features[
                "hist_ever_delivered_after_invalid"
            ].sum(),

            features[
                "hist_ever_delivered_after_blocked"
            ].sum(),

            features[
                "hist_ever_delivered_after_unreachable"
            ].sum(),

            features[
                "hist_ever_delivered_after_hard_failure"
            ].sum(),
        ],
    },
    index=[
        "invalid_number",
        "blocked",
        "unreachable",
        "hard_failure",
    ],
)


recovery_after_failure[
    "later_delivery_pct"
] = np.where(
    recovery_after_failure[
        "failure_events_with_history"
    ] > 0,

    recovery_after_failure[
        "later_delivery_observed"
    ]
    /
    recovery_after_failure[
        "failure_events_with_history"
    ]
    * 100,

    np.nan,
)


display(
    recovery_after_failure
)


# ============================================================
# 20. HARD FAILURE STATE
# ============================================================

print(
    "\nHARD FAILURE CANDIDATE STATES"
)


hard_failure_states = pd.DataFrame(
    {
        "events": [
            features[
                "hist_hard_failure_only_so_far"
            ].sum(),

            features[
                "hist_hard_failure_never_followed_by_delivery"
            ].sum(),

            features[
                "hist_2plus_consecutive_hard_failures"
            ].sum(),

            features[
                "hist_3plus_consecutive_hard_failures"
            ].sum(),
        ],
    },
    index=[
        "all_prior_messages_hard_failed",
        "hard_failure_not_followed_by_delivery",
        "2plus_consecutive_hard_failures",
        "3plus_consecutive_hard_failures",
    ],
)


hard_failure_states[
    "pct_events"
] = (
    hard_failure_states[
        "events"
    ]
    / len(features)
    * 100
)


display(
    hard_failure_states
)


# ============================================================
# 21. SAMPLE FAILURE JOURNEY
# ============================================================
#
# Pick customer with most historical hard failures.
# ============================================================

sample_customer = (
    features.loc[
        features[
            "hist_n_hard_delivery_failures"
        ].idxmax(),
        "customer_id",
    ]
)


print(
    f"\nSAMPLE FAILURE JOURNEY — "
    f"{sample_customer}"
)


display(
    features.loc[
        features[
            "customer_id"
        ].eq(
            sample_customer
        ),
        [
            "customer_id",
            "event_number",
            "sent_at",
            "current_template",
            "event_delivery_status",

            "hist_n_previous_messages",

            "hist_n_delivered",
            "hist_n_failed",
            "hist_n_failed_invalid_number",
            "hist_n_failed_blocked",
            "hist_n_failed_unreachable",

            "hist_previous_delivery_status",

            "hist_consecutive_failures",
            "hist_consecutive_hard_delivery_failures",

            "hist_days_since_last_hard_delivery_failure",

            "hist_ever_delivered_after_hard_failure",

            "hist_delivery_failure_profile",
        ],
    ]
)


# ============================================================
# 22. GOVERNANCE SUMMARY
# ============================================================

print("\n" + "=" * 100)
print("03C — TEMPORAL GOVERNANCE")
print("=" * 100)

print(
    """
DELIVERY HISTORY RULE

Historical delivery state uses only delivery outcomes from
events strictly before the current send.

TEMPORAL ASSUMPTION:

    delivery feedback from event j is available before the
    next collection decision.

The source dataset does not provide delivery timestamps, so
this is an explicit operational availability assumption.

BUSINESS INTERPRETATION:

    invalid_number / blocked
        → persistent hard-failure candidates

    unreachable
        → kept separate as potentially transient

No causal stop rule is being declared in this layer.
These are historical state features only.
"""
)


print("=" * 100)
print("03C — HISTORICAL DELIVERY / FAILURE STATE COMPLETE")
print("=" * 100)

03C — HISTORICAL DELIVERY / FAILURE STATE
Starting shape: (75406, 436)
Delivery taxonomy: PASSED

Running hard consistency checks...
Hard consistency checks: PASSED

03C — HISTORICAL DELIVERY / FAILURE STATE COMPLETE
Rows          : 75,406
Customers     : 11,724
Columns before: 436
Columns after : 500
New features  : 64

EVER EXPERIENCED DELIVERY STATUS BEFORE CURRENT EVENT


,events,pct_events
any_failure,15297,20.29
invalid_number,4115,5.46
blocked,4112,5.45
unreachable,7446,9.87
hard_failure_invalid_or_blocked,8227,10.91



HISTORICAL DELIVERY FAILURE PROFILE


,events
hist_delivery_failure_profile,
all_delivered,48385
no_history,11724
hard_failure_no_recovery,8227
unreachable_only_failure,7070



CONSECUTIVE HISTORICAL FAILURE STREAK


...


HARD FAILURE CANDIDATE STATES


,events,pct_events
all_prior_messages_hard_failed,4437,5.88
hard_failure_not_followed_by_delivery,8227,10.91
2plus_consecutive_hard_failures,6790,9.00
3plus_consecutive_hard_failures,5497,7.29



SAMPLE FAILURE JOURNEY — C001782


,customer_id,event_number,sent_at,current_template,event_delivery_status,hist_n_previous_messages,hist_n_delivered,hist_n_failed,hist_n_failed_invalid_number,hist_n_failed_blocked,hist_n_failed_unreachable,hist_previous_delivery_status,hist_consecutive_failures,hist_consecutive_hard_delivery_failures,hist_days_since_last_hard_delivery_failure,hist_ever_delivered_after_hard_failure,hist_delivery_failure_profile
11202,C001782,1,2026-06-20 12:41:00,pix_link,failed_invalid_number,0,0,0,0,0,0,NaN,0,0,NaN,0,no_history
11203,C001782,2,2026-06-21 11:10:00,pix_link,failed_invalid_number,1,0,1,1,0,0,failed_invalid_number,1,1,0.94,0,hard_failure_no_recovery
11204,C001782,3,2026-06-22 12:40:00,friendly_reminder,failed_invalid_number,2,0,2,2,0,0,failed_invalid_number,2,2,1.06,0,hard_failure_no_recovery
11205,C001782,4,2026-06-26 15:12:00,pix_link,failed_invalid_number,3,0,3,3,0,0,failed_invalid_number,3,3,4.11,0,hard_failure_no_recovery
11206,C001782,5,2026-06-29 10:11:00,pix_link,failed_invalid_number,4,0,4,4,0,0,failed_invalid_number,4,4,2.79,0,hard_failure_no_recovery
11207,C001782,6,2026-06-30 14:17:00,urgent_reminder,failed_invalid_number,5,0,5,5,0,0,failed_invalid_number,5,5,1.17,0,hard_failure_no_recovery
11208,C001782,7,2026-07-01 10:18:00,urgent_reminder,failed_invalid_number,6,0,6,6,0,0,failed_invalid_number,6,6,0.83,0,hard_failure_no_recovery
11209,C001782,8,2026-07-02 20:50:00,friendly_reminder,failed_invalid_number,7,0,7,7,0,0,failed_invalid_number,7,7,1.44,0,hard_failure_no_recovery
11210,C001782,9,2026-07-04 10:52:00,pix_link,failed_invalid_number,8,0,8,8,0,0,failed_invalid_number,8,8,1.58,0,hard_failure_no_recovery
11211,C001782,10,2026-07-09 11:43:00,pix_link,failed_invalid_number,9,0,9,9,0,0,failed_invalid_number,9,9,5.04,0,hard_failure_no_recovery



03C — TEMPORAL GOVERNANCE

DELIVERY HISTORY RULE

Historical delivery state uses only delivery outcomes from
events strictly before the current send.

TEMPORAL ASSUMPTION:

    delivery feedback from event j is available before the
    next collection decision.

The source dataset does not provide delivery timestamps, so
this is an explicit operational availability assumption.

BUSINESS INTERPRETATION:

    invalid_number / blocked
        → persistent hard-failure candidates

    unreachable
        → kept separate as potentially transient

No causal stop rule is being declared in this layer.
These are historical state features only.

03C — HISTORICAL DELIVERY / FAILURE STATE COMPLETE


In [25]:
# ============================================================
# 03D.1 — INTERACTION TEMPORAL RISK / SENSITIVITY AUDIT
# ============================================================
#
# PURPOSE
# -------
# Audit whether historical interaction outcomes can safely be
# treated as known before the next collection decision.
#
# IMPORTANT
# ---------
# Available in the source:
#
#   sent_at
#   interaction ∈ {
#       none,
#       read,
#       clicked_link,
#       replied
#   }
#
# NOT available:
#
#   read_at
#   clicked_at
#   replied_at
#
# Therefore we CANNOT prove exactly when engagement became
# observable.
#
# This layer:
#
#   - does NOT create model features
#   - does NOT modify the 500-column checkpoint
#   - quantifies sensitivity to different availability windows
#
# Starting checkpoint:
#
#   75,406 × 500
#
# ============================================================

print("=" * 100)
print("03D.1 — INTERACTION TEMPORAL RISK / SENSITIVITY AUDIT")
print("=" * 100)


# ============================================================
# 0. CHECKPOINT
# ============================================================

rows_before = len(features)
cols_before = features.shape[1]

assert features.shape == (75_406, 500)
assert features["customer_id"].nunique() == 11_724
assert features["event_id"].is_unique
assert features["message_id"].is_unique

print(f"Starting shape: {features.shape}")


# ============================================================
# 1. INTERACTION TAXONOMY
# ============================================================

interaction = (
    features["event_interaction"]
    .astype(str)
)


expected_interactions = {
    "none",
    "read",
    "clicked_link",
    "replied",
}


observed_interactions = set(
    interaction.unique()
)


assert observed_interactions.issubset(
    expected_interactions
)


print("\nINTERACTION DISTRIBUTION")

interaction_distribution = (
    interaction
    .value_counts(dropna=False)
    .to_frame("events")
)

interaction_distribution["pct_events"] = (
    interaction_distribution["events"]
    / len(features)
    * 100
)

display(interaction_distribution)


# ============================================================
# 2. ENGAGEMENT FLAGS
# ============================================================
#
# Temporary audit variables only.
# ============================================================

is_read = (
    interaction.eq("read")
)

is_clicked = (
    interaction.eq("clicked_link")
)

is_replied = (
    interaction.eq("replied")
)

is_engaged = (
    interaction.ne("none")
)


# ============================================================
# 3. NEXT EVENT TIME
# ============================================================
#
# For each source message j:
#
#     when was the NEXT collection decision?
#
# If the next decision is very soon after j, we have less
# confidence that an eventual interaction was already known.
# ============================================================

next_sent_at = (
    features
    .groupby(
        "customer_id",
        sort=False,
    )["sent_at"]
    .shift(-1)
)


hours_to_next_event = (
    (
        next_sent_at
        -
        features["sent_at"]
    )
    .dt.total_seconds()
    / 3600
)


has_next_event = (
    next_sent_at.notna()
)


assert (
    hours_to_next_event[
        has_next_event
    ] > 0
).all()


print("\nTIME FROM SOURCE MESSAGE TO NEXT COLLECTION DECISION")

display(
    hours_to_next_event[
        has_next_event
    ]
    .describe(
        percentiles=[
            .01,
            .05,
            .10,
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame("hours_to_next_event")
)


# ============================================================
# 4. HOW OFTEN DOES NEXT DECISION OCCUR QUICKLY?
# ============================================================

windows_hours = [
    1,
    3,
    6,
    12,
    24,
    48,
    72,
]


quick_next_rows = []

for h in windows_hours:

    count = (
        has_next_event
        &
        hours_to_next_event.lt(h)
    ).sum()

    quick_next_rows.append(
        {
            "next_decision_within_hours": h,
            "events": int(count),
            "pct_events_with_next_decision":
                count
                / has_next_event.sum()
                * 100,
        }
    )


quick_next = pd.DataFrame(
    quick_next_rows
)


print(
    "\nNEXT COLLECTION DECISION OCCURS BEFORE ASSUMED "
    "INTERACTION-AVAILABILITY WINDOW"
)

display(quick_next)


# ============================================================
# 5. ENGAGED EVENTS WITH A LATER COLLECTION DECISION
# ============================================================

engaged_with_next = (
    is_engaged
    &
    has_next_event
)


print(
    "\nENGAGED SOURCE EVENTS THAT HAVE A LATER DECISION"
)

print(
    f"Engaged source events with later send : "
    f"{engaged_with_next.sum():,}"
)


display(
    hours_to_next_event[
        engaged_with_next
    ]
    .describe(
        percentiles=[
            .01,
            .05,
            .10,
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame("hours_to_next_event")
)


# ============================================================
# 6. SENSITIVITY BY INTERACTION TYPE
# ============================================================
#
# We do NOT know interaction timestamps.
#
# So we test hypothetical availability delays:
#
#   1h
#   3h
#   6h
#   12h
#   24h
#   48h
#   72h
#
# Example:
#
# If next message occurs only 2h later, then under a
# conservative 6h availability assumption, previous
# interaction should NOT yet be used.
#
# ============================================================

interaction_types = [
    "read",
    "clicked_link",
    "replied",
]


sensitivity_rows = []


for interaction_type in interaction_types:

    type_mask = (
        interaction.eq(
            interaction_type
        )
        &
        has_next_event
    )

    n_type = int(
        type_mask.sum()
    )

    for h in windows_hours:

        potentially_unavailable = (
            type_mask
            &
            hours_to_next_event.lt(h)
        ).sum()

        sensitivity_rows.append(
            {
                "interaction_type":
                    interaction_type,

                "assumed_availability_hours":
                    h,

                "events_with_later_decision":
                    n_type,

                "potentially_unavailable_events":
                    int(
                        potentially_unavailable
                    ),

                "potentially_unavailable_pct":
                    (
                        potentially_unavailable
                        / n_type
                        * 100
                    )
                    if n_type > 0
                    else np.nan,
            }
        )


interaction_sensitivity = (
    pd.DataFrame(
        sensitivity_rows
    )
)


print(
    "\nINTERACTION AVAILABILITY SENSITIVITY"
)

display(
    interaction_sensitivity
)


# ============================================================
# 7. PIVOT — EASIER TO READ
# ============================================================

interaction_sensitivity_pivot = (
    interaction_sensitivity
    .pivot(
        index="interaction_type",
        columns="assumed_availability_hours",
        values="potentially_unavailable_pct",
    )
)


interaction_sensitivity_pivot.columns = [
    f"<{h}h"
    for h in
    interaction_sensitivity_pivot.columns
]


print(
    "\n% OF INTERACTION EVENTS WHOSE NEXT DECISION OCCURS "
    "BEFORE EACH ASSUMED AVAILABILITY WINDOW"
)

display(
    interaction_sensitivity_pivot
)


# ============================================================
# 8. VERY SHORT-GAP ENGAGEMENT EVENTS
# ============================================================
#
# These are the events with highest temporal ambiguity.
# ============================================================

short_gap_engagement = pd.DataFrame(
    {
        "interaction_type": [
            "read",
            "clicked_link",
            "replied",
            "any_engagement",
        ],

        "<1h": [
            (
                is_read
                &
                has_next_event
                &
                hours_to_next_event.lt(1)
            ).sum(),

            (
                is_clicked
                &
                has_next_event
                &
                hours_to_next_event.lt(1)
            ).sum(),

            (
                is_replied
                &
                has_next_event
                &
                hours_to_next_event.lt(1)
            ).sum(),

            (
                is_engaged
                &
                has_next_event
                &
                hours_to_next_event.lt(1)
            ).sum(),
        ],

        "<6h": [
            (
                is_read
                &
                has_next_event
                &
                hours_to_next_event.lt(6)
            ).sum(),

            (
                is_clicked
                &
                has_next_event
                &
                hours_to_next_event.lt(6)
            ).sum(),

            (
                is_replied
                &
                has_next_event
                &
                hours_to_next_event.lt(6)
            ).sum(),

            (
                is_engaged
                &
                has_next_event
                &
                hours_to_next_event.lt(6)
            ).sum(),
        ],

        "<12h": [
            (
                is_read
                &
                has_next_event
                &
                hours_to_next_event.lt(12)
            ).sum(),

            (
                is_clicked
                &
                has_next_event
                &
                hours_to_next_event.lt(12)
            ).sum(),

            (
                is_replied
                &
                has_next_event
                &
                hours_to_next_event.lt(12)
            ).sum(),

            (
                is_engaged
                &
                has_next_event
                &
                hours_to_next_event.lt(12)
            ).sum(),
        ],

        "<24h": [
            (
                is_read
                &
                has_next_event
                &
                hours_to_next_event.lt(24)
            ).sum(),

            (
                is_clicked
                &
                has_next_event
                &
                hours_to_next_event.lt(24)
            ).sum(),

            (
                is_replied
                &
                has_next_event
                &
                hours_to_next_event.lt(24)
            ).sum(),

            (
                is_engaged
                &
                has_next_event
                &
                hours_to_next_event.lt(24)
            ).sum(),
        ],
    }
)


print(
    "\nHIGH TEMPORAL-AMBIGUITY ENGAGEMENT EVENTS"
)

display(
    short_gap_engagement
)


# ============================================================
# 9. INTERACTION × DELIVERY CONSISTENCY
# ============================================================
#
# This does NOT establish timestamp availability.
#
# It only verifies structural consistency.
# ============================================================

print(
    "\nINTERACTION × DELIVERY STATUS"
)


interaction_delivery = pd.crosstab(
    features[
        "event_delivery_status"
    ],
    features[
        "event_interaction"
    ],
)


display(
    interaction_delivery
)


failed_delivery = (
    features[
        "event_delivery_status"
    ]
    .ne("delivered")
)


assert (
    features.loc[
        failed_delivery,
        "event_interaction"
    ]
    .eq("none")
    .all()
)


assert (
    features.loc[
        is_engaged,
        "event_delivery_status"
    ]
    .eq("delivered")
    .all()
)


print(
    "Delivery / interaction structural consistency: PASSED"
)


# ============================================================
# 10. INTERACTION × PAYMENT — DESCRIPTIVE ONLY
# ============================================================
#
# Useful for understanding the data, but NOT causal.
# ============================================================

interaction_payment = (
    features
    .groupby(
        "event_interaction",
        observed=True,
    )
    .agg(
        events=(
            "event_id",
            "size",
        ),

        payment_events=(
            "event_paid_within_72h",
            "sum",
        ),

        attributed_payment_brl=(
            "event_amount_paid_brl",
            "sum",
        ),
    )
)


interaction_payment[
    "payment_event_rate_pct"
] = (
    interaction_payment[
        "payment_events"
    ]
    /
    interaction_payment[
        "events"
    ]
    * 100
)


print(
    "\nINTERACTION × PAYMENT — DESCRIPTIVE ONLY"
)

display(
    interaction_payment
)


# ============================================================
# 11. WHAT WOULD NAIVE SHIFT DO?
# ============================================================
#
# This is NOT added to features.
#
# We calculate how many current decisions would receive a
# previous interaction simply because a previous row exists,
# even though the interaction timestamp itself is unknown.
# ============================================================

naive_previous_interaction = (
    features
    .groupby(
        "customer_id",
        sort=False,
    )[
        "event_interaction"
    ]
    .shift(1)
)


naive_previous_engaged = (
    naive_previous_interaction
    .isin(
        [
            "read",
            "clicked_link",
            "replied",
        ]
    )
)


print(
    "\nNAIVE SHIFT RISK"
)

print(
    f"Current decisions that would inherit a previous "
    f"engagement via naive shift : "
    f"{naive_previous_engaged.sum():,}"
)


# ============================================================
# 12. SENSITIVITY OF NAIVE PREVIOUS ENGAGEMENT
# ============================================================
#
# For the immediately previous message:
#
# current sent_at - previous sent_at
#
# We quantify how many inherited interactions would fail
# different hypothetical maturity assumptions.
# ============================================================

previous_sent_at = (
    features
    .groupby(
        "customer_id",
        sort=False,
    )[
        "sent_at"
    ]
    .shift(1)
)


hours_since_previous_send = (
    (
        features["sent_at"]
        -
        previous_sent_at
    )
    .dt.total_seconds()
    / 3600
)


naive_risk_rows = []


for h in windows_hours:

    inherited_too_soon = (
        naive_previous_engaged
        &
        hours_since_previous_send.lt(h)
    ).sum()

    total_inherited = (
        naive_previous_engaged.sum()
    )

    naive_risk_rows.append(
        {
            "assumed_availability_hours":
                h,

            "naive_previous_engagements":
                int(total_inherited),

            "would_be_temporally_unsafe":
                int(inherited_too_soon),

            "unsafe_pct":
                (
                    inherited_too_soon
                    / total_inherited
                    * 100
                )
                if total_inherited > 0
                else np.nan,
        }
    )


naive_shift_sensitivity = (
    pd.DataFrame(
        naive_risk_rows
    )
)


display(
    naive_shift_sensitivity
)


# ============================================================
# 13. FINAL CHECKPOINT
# ============================================================
#
# This audit MUST NOT alter features.
# ============================================================

assert len(features) == rows_before
assert features.shape[1] == cols_before
assert features.shape == (75_406, 500)


print("\n" + "=" * 100)
print("03D.1 — AUDIT COMPLETE")
print("=" * 100)

print(
    f"Rows          : "
    f"{len(features):,}"
)

print(
    f"Customers     : "
    f"{features['customer_id'].nunique():,}"
)

print(
    f"Columns before: "
    f"{cols_before:,}"
)

print(
    f"Columns after : "
    f"{features.shape[1]:,}"
)

print(
    "Features added: 0"
)


# ============================================================
# 14. GOVERNANCE SUMMARY
# ============================================================

print("\n" + "=" * 100)
print("03D.1 — TEMPORAL GOVERNANCE")
print("=" * 100)

print(
    """
INTERACTION AVAILABILITY

The source dataset records the final interaction category:

    none
    read
    clicked_link
    replied

but does NOT record:

    read_at
    clicked_at
    replied_at

Therefore the exact point-in-time availability of historical
interaction outcomes cannot be proven from the source data.

This audit does NOT authorize naive historical interaction
features.

Decision for 03D will depend on the sensitivity results:

    low short-gap exposure
        → operational assumption may be acceptable,
          with explicit governance flag

    material short-gap exposure
        → prefer conservative maturity window or exclude
          interaction history from canonical PIT feature set

Structural consistency with delivery does NOT prove temporal
availability.
"""
)


print("=" * 100)
print("03D.1 — INTERACTION TEMPORAL RISK AUDIT COMPLETE")
print("=" * 100)

03D.1 — INTERACTION TEMPORAL RISK / SENSITIVITY AUDIT
Starting shape: (75406, 500)

INTERACTION DISTRIBUTION


,events,pct_events
event_interaction,,
none,45201,59.94
read,17505,23.21
clicked_link,8145,10.80
replied,4555,6.04



TIME FROM SOURCE MESSAGE TO NEXT COLLECTION DECISION


,hours_to_next_event
count,"63,682.00"
mean,107.83
std,108.71
min,12.27
1%,15.47
5%,19.63
10%,22.70
25%,32.12
50%,72.52
75%,141.63



NEXT COLLECTION DECISION OCCURS BEFORE ASSUMED INTERACTION-AVAILABILITY WINDOW


...

Delivery / interaction structural consistency: PASSED

INTERACTION × PAYMENT — DESCRIPTIVE ONLY


,events,payment_events,attributed_payment_brl,payment_event_rate_pct
event_interaction,,,,
clicked_link,8145,1517,"957,599.93",18.62
none,45201,1988,"1,207,005.26",4.40
read,17505,1567,"960,239.67",8.95
replied,4555,554,"334,460.44",12.16



NAIVE SHIFT RISK
Current decisions that would inherit a previous engagement via naive shift : 25,095


,assumed_availability_hours,naive_previous_engagements,would_be_temporally_unsafe,unsafe_pct
0,1,25095,0,0.00
1,3,25095,0,0.00
2,6,25095,0,0.00
3,12,25095,0,0.00
4,24,25095,3570,14.23
5,48,25095,9088,36.21
6,72,25095,12846,51.19



03D.1 — AUDIT COMPLETE
Rows          : 75,406
Customers     : 11,724
Columns before: 500
Columns after : 500
Features added: 0

03D.1 — TEMPORAL GOVERNANCE

INTERACTION AVAILABILITY

The source dataset records the final interaction category:

    none
    read
    clicked_link
    replied

but does NOT record:

    read_at
    clicked_at
    replied_at

Therefore the exact point-in-time availability of historical
interaction outcomes cannot be proven from the source data.

This audit does NOT authorize naive historical interaction
features.

Decision for 03D will depend on the sensitivity results:

    low short-gap exposure
        → operational assumption may be acceptable,
          with explicit governance flag

    material short-gap exposure
        → prefer conservative maturity window or exclude
          interaction history from canonical PIT feature set

Structural consistency with delivery does NOT prove temporal
availability.

03D.1 — INTERACTION TEMPORAL RISK AUDIT COMPLE

In [26]:
# ============================================================
# 03D — HISTORICAL INTERACTION STATE
#       [TEMPORAL AVAILABILITY ASSUMPTION]
# ============================================================
#
# PURPOSE
# -------
# Build historical engagement / interaction features using
# events strictly before the current event.
#
# IMPORTANT GOVERNANCE
# --------------------
# The source contains final interaction category:
#
#   none
#   read
#   clicked_link
#   replied
#
# but does NOT contain:
#
#   read_at
#   clicked_at
#   replied_at
#
# Therefore these features are NOT classified as
# "proven canonical PIT".
#
# Operational assumption used here:
#
#   interaction outcome from event j is treated as available
#   before the next collection decision.
#
# These features must later be registered as:
#
#   temporal_assumption_interaction_timestamp_missing
#
# Starting checkpoint:
#
#   75,406 × 500
#
# ============================================================

print("=" * 100)
print("03D — HISTORICAL INTERACTION STATE [TEMPORAL ASSUMPTION]")
print("=" * 100)


# ============================================================
# 0. CHECKPOINT
# ============================================================

rows_before = len(features)
cols_before = features.shape[1]

assert features.shape == (75_406, 500)
assert features["customer_id"].nunique() == 11_724
assert features["event_id"].is_unique
assert features["message_id"].is_unique

print(f"Starting shape: {features.shape}")


# ============================================================
# 1. CURRENT EVENT INTERACTION FLAGS
# ============================================================
#
# Temporary only.
# Current event interaction NEVER enters its own history.
# ============================================================

interaction = (
    features["event_interaction"]
    .astype(str)
)

customer = features["customer_id"]


is_none = (
    interaction.eq("none")
    .astype("int8")
)

is_read = (
    interaction.eq("read")
    .astype("int8")
)

is_clicked = (
    interaction.eq("clicked_link")
    .astype("int8")
)

is_replied = (
    interaction.eq("replied")
    .astype("int8")
)

is_engaged = (
    interaction.ne("none")
    .astype("int8")
)


# Structural taxonomy

assert (
    is_none
    + is_read
    + is_clicked
    + is_replied
    ==
    1
).all()

assert (
    is_engaged
    ==
    (
        is_read
        + is_clicked
        + is_replied
    )
).all()

print("Interaction taxonomy: PASSED")


# ============================================================
# 2. STRICTLY HISTORICAL CUMULATIVE COUNTS
# ============================================================

def prior_cumsum_interaction(flag):

    return (
        flag
        .groupby(
            customer,
            sort=False,
        )
        .cumsum()
        -
        flag
    )


hist_n_none = (
    prior_cumsum_interaction(
        is_none
    )
    .astype("int16")
)

hist_n_read = (
    prior_cumsum_interaction(
        is_read
    )
    .astype("int16")
)

hist_n_clicked = (
    prior_cumsum_interaction(
        is_clicked
    )
    .astype("int16")
)

hist_n_replied = (
    prior_cumsum_interaction(
        is_replied
    )
    .astype("int16")
)

hist_n_engaged = (
    prior_cumsum_interaction(
        is_engaged
    )
    .astype("int16")
)


features[
    "hist_n_interaction_none"
] = hist_n_none

features[
    "hist_n_read"
] = hist_n_read

features[
    "hist_n_clicked_link"
] = hist_n_clicked

features[
    "hist_n_replied"
] = hist_n_replied

features[
    "hist_n_engaged"
] = hist_n_engaged


# ============================================================
# 3. EVER-SEEN FLAGS
# ============================================================

features[
    "hist_ever_read"
] = (
    hist_n_read > 0
).astype("int8")

features[
    "hist_ever_clicked_link"
] = (
    hist_n_clicked > 0
).astype("int8")

features[
    "hist_ever_replied"
] = (
    hist_n_replied > 0
).astype("int8")

features[
    "hist_ever_engaged"
] = (
    hist_n_engaged > 0
).astype("int8")

features[
    "hist_never_engaged_with_prior_messages"
] = (
    (
        features[
            "hist_n_previous_messages"
        ] > 0
    )
    &
    (
        hist_n_engaged == 0
    )
).astype("int8")


# ============================================================
# 4. HISTORICAL INTERACTION RATES
# ============================================================

n_previous = (
    features[
        "hist_n_previous_messages"
    ]
)


def historical_rate(
    numerator
):

    return np.where(
        n_previous > 0,
        numerator / n_previous,
        np.nan,
    )


features[
    "hist_no_interaction_rate"
] = historical_rate(
    hist_n_none
)

features[
    "hist_read_rate"
] = historical_rate(
    hist_n_read
)

features[
    "hist_click_rate"
] = historical_rate(
    hist_n_clicked
)

features[
    "hist_reply_rate"
] = historical_rate(
    hist_n_replied
)

features[
    "hist_engagement_rate"
] = historical_rate(
    hist_n_engaged
)


# ============================================================
# 5. PREVIOUS INTERACTION
# ============================================================

features[
    "hist_previous_interaction"
] = (
    features
    .groupby(
        "customer_id",
        sort=False,
    )[
        "event_interaction"
    ]
    .shift(1)
)


features[
    "hist_previous_was_none"
] = (
    features[
        "hist_previous_interaction"
    ]
    .eq("none")
    .astype("int8")
)

features[
    "hist_previous_was_read"
] = (
    features[
        "hist_previous_interaction"
    ]
    .eq("read")
    .astype("int8")
)

features[
    "hist_previous_was_clicked"
] = (
    features[
        "hist_previous_interaction"
    ]
    .eq("clicked_link")
    .astype("int8")
)

features[
    "hist_previous_was_replied"
] = (
    features[
        "hist_previous_interaction"
    ]
    .eq("replied")
    .astype("int8")
)

features[
    "hist_previous_was_engaged"
] = (
    features[
        "hist_previous_interaction"
    ]
    .isin(
        [
            "read",
            "clicked_link",
            "replied",
        ]
    )
    .astype("int8")
)


# ============================================================
# 6. HISTORICAL LAST OCCURRENCE
# ============================================================

def historical_last_interaction_occurrence(
    flag
):

    flagged_time = (
        features["sent_at"]
        .where(
            flag.eq(1)
        )
    )

    including_current = (
        flagged_time
        .groupby(
            customer,
            sort=False,
        )
        .ffill()
    )

    return (
        including_current
        .groupby(
            customer,
            sort=False,
        )
        .shift(1)
    )


last_engaged_at = (
    historical_last_interaction_occurrence(
        is_engaged
    )
)

last_read_at = (
    historical_last_interaction_occurrence(
        is_read
    )
)

last_clicked_at = (
    historical_last_interaction_occurrence(
        is_clicked
    )
)

last_replied_at = (
    historical_last_interaction_occurrence(
        is_replied
    )
)

last_none_at = (
    historical_last_interaction_occurrence(
        is_none
    )
)


features[
    "hist_last_engaged_message_at"
] = last_engaged_at

features[
    "hist_last_read_message_at"
] = last_read_at

features[
    "hist_last_clicked_message_at"
] = last_clicked_at

features[
    "hist_last_replied_message_at"
] = last_replied_at

features[
    "hist_last_no_interaction_message_at"
] = last_none_at


# ============================================================
# 7. RECENCY SINCE MESSAGE THAT GENERATED INTERACTION
# ============================================================
#
# NOTE:
#
# This is:
#
#   current decision time
#       -
#   SEND TIME of last historically engaged message
#
# It is NOT:
#
#   current time - actual engagement timestamp
#
# because engagement timestamps do not exist.
# ============================================================

def days_since_interaction_message(
    timestamp
):

    return (
        (
            features["sent_at"]
            -
            timestamp
        )
        .dt.total_seconds()
        / 86400
    )


features[
    "hist_days_since_last_engaged_message"
] = days_since_interaction_message(
    last_engaged_at
)

features[
    "hist_days_since_last_read_message"
] = days_since_interaction_message(
    last_read_at
)

features[
    "hist_days_since_last_clicked_message"
] = days_since_interaction_message(
    last_clicked_at
)

features[
    "hist_days_since_last_replied_message"
] = days_since_interaction_message(
    last_replied_at
)


# ============================================================
# 8. RECENT ENGAGEMENT FLAGS
# ============================================================

for days in [
    1,
    3,
    7,
    14,
    30,
]:

    recency = (
        features[
            "hist_days_since_last_engaged_message"
        ]
    )

    features[
        f"hist_engaged_message_within_{days}d"
    ] = (
        recency.notna()
        &
        recency.le(days)
    ).astype("int8")


for days in [
    3,
    7,
    14,
    30,
]:

    click_recency = (
        features[
            "hist_days_since_last_clicked_message"
        ]
    )

    features[
        f"hist_clicked_message_within_{days}d"
    ] = (
        click_recency.notna()
        &
        click_recency.le(days)
    ).astype("int8")


for days in [
    3,
    7,
    14,
    30,
]:

    reply_recency = (
        features[
            "hist_days_since_last_replied_message"
        ]
    )

    features[
        f"hist_replied_message_within_{days}d"
    ] = (
        reply_recency.notna()
        &
        reply_recency.le(days)
    ).astype("int8")


# ============================================================
# 9. CONSECUTIVE INTERACTION STREAKS
# ============================================================
#
# Historical streak ends at PREVIOUS event.
# ============================================================

def historical_interaction_streak(
    flag
):

    flag_np = (
        flag.to_numpy(
            dtype=np.int8,
            copy=False,
        )
    )

    customer_np = (
        customer.to_numpy(
            copy=False
        )
    )

    current_streak = np.zeros(
        len(flag_np),
        dtype=np.int16,
    )

    running = 0

    for i in range(
        len(flag_np)
    ):

        if (
            i == 0
            or
            customer_np[i]
            != customer_np[i - 1]
        ):
            running = 0

        if flag_np[i] == 1:
            running += 1
        else:
            running = 0

        current_streak[i] = running


    historical = np.zeros(
        len(flag_np),
        dtype=np.int16,
    )


    same_customer = np.r_[
        False,
        customer_np[1:]
        ==
        customer_np[:-1],
    ]


    historical[
        same_customer
    ] = (
        current_streak[:-1][
            same_customer[1:]
        ]
    )


    return historical


hist_engaged_streak = (
    historical_interaction_streak(
        is_engaged
    )
)

hist_none_streak = (
    historical_interaction_streak(
        is_none
    )
)

hist_read_streak = (
    historical_interaction_streak(
        is_read
    )
)

hist_click_streak = (
    historical_interaction_streak(
        is_clicked
    )
)

hist_reply_streak = (
    historical_interaction_streak(
        is_replied
    )
)


features[
    "hist_consecutive_engaged"
] = hist_engaged_streak

features[
    "hist_consecutive_no_interaction"
] = hist_none_streak

features[
    "hist_consecutive_read"
] = hist_read_streak

features[
    "hist_consecutive_clicked"
] = hist_click_streak

features[
    "hist_consecutive_replied"
] = hist_reply_streak


# ============================================================
# 10. NO-ENGAGEMENT PERSISTENCE
# ============================================================

for threshold in [
    2,
    3,
    5,
    8,
]:

    features[
        f"hist_{threshold}plus_consecutive_no_interaction"
    ] = (
        hist_none_streak
        >= threshold
    ).astype("int8")


for threshold in [
    2,
    3,
]:

    features[
        f"hist_{threshold}plus_consecutive_engaged"
    ] = (
        hist_engaged_streak
        >= threshold
    ).astype("int8")


# ============================================================
# 11. ENGAGEMENT DEPTH / STRONGEST HISTORICAL SIGNAL
# ============================================================
#
# This is a descriptive hierarchy:
#
# none < read < clicked < replied
#
# It does NOT claim replied is causally "better".
# ============================================================

features[
    "hist_strongest_interaction"
] = np.select(
    [
        hist_n_replied > 0,
        hist_n_clicked > 0,
        hist_n_read > 0,
        n_previous > 0,
    ],
    [
        "replied",
        "clicked_link",
        "read",
        "none",
    ],
    default="no_history",
)


features[
    "hist_interaction_depth_score"
] = np.select(
    [
        hist_n_replied > 0,
        hist_n_clicked > 0,
        hist_n_read > 0,
    ],
    [
        3,
        2,
        1,
    ],
    default=0,
).astype("int8")


# ============================================================
# 12. HISTORICAL INTERACTION DIVERSITY
# ============================================================

interaction_seen_matrix = np.column_stack(
    [
        hist_n_read > 0,
        hist_n_clicked > 0,
        hist_n_replied > 0,
    ]
)


features[
    "hist_n_distinct_engagement_types"
] = (
    interaction_seen_matrix
    .sum(axis=1)
    .astype("int8")
)


features[
    "hist_has_multiple_engagement_types"
] = (
    features[
        "hist_n_distinct_engagement_types"
    ]
    >= 2
).astype("int8")


# ============================================================
# 13. ENGAGEMENT AMONG DELIVERED MESSAGES
# ============================================================
#
# This separates:
#
#   "customer did not engage"
#
# from:
#
#   "message was not even delivered"
#
# denominator = historically delivered messages.
# ============================================================

hist_n_delivered = (
    features[
        "hist_n_delivered"
    ]
)


features[
    "hist_engagement_rate_given_delivered"
] = np.where(
    hist_n_delivered > 0,
    hist_n_engaged
    / hist_n_delivered,
    np.nan,
)


features[
    "hist_no_interaction_rate_given_delivered"
] = np.where(
    hist_n_delivered > 0,
    (
        hist_n_delivered
        -
        hist_n_engaged
    )
    / hist_n_delivered,
    np.nan,
)


features[
    "hist_click_rate_given_delivered"
] = np.where(
    hist_n_delivered > 0,
    hist_n_clicked
    / hist_n_delivered,
    np.nan,
)


features[
    "hist_reply_rate_given_delivered"
] = np.where(
    hist_n_delivered > 0,
    hist_n_replied
    / hist_n_delivered,
    np.nan,
)


# ============================================================
# 14. HISTORICAL ENGAGEMENT PROFILE
# ============================================================

features[
    "hist_interaction_profile"
] = np.select(
    [
        n_previous.eq(0),

        (
            n_previous.gt(0)
            &
            hist_n_delivered.eq(0)
        ),

        (
            hist_n_delivered.gt(0)
            &
            hist_n_engaged.eq(0)
        ),

        (
            hist_n_replied.gt(0)
        ),

        (
            hist_n_clicked.gt(0)
        ),

        (
            hist_n_read.gt(0)
        ),
    ],
    [
        "no_history",
        "no_historical_delivery",
        "delivered_never_engaged",
        "has_replied",
        "has_clicked_no_reply",
        "read_only",
    ],
    default="other",
)


# ============================================================
# 15. RECENT ENGAGEMENT COUNTS — CALENDAR WINDOWS
# ============================================================
#
# We already have the exact temporal machinery from Layer 02B.
#
# Here we count prior ENGAGED messages occurring on previous:
#
#   3 calendar dates
#   7 calendar dates
#   14 calendar dates
#   30 calendar dates
#
# This is historical exposure based on SOURCE SEND DATE.
#
# Still subject to interaction availability assumption.
# ============================================================

sent_date_np = (
    features["sent_at"]
    .dt.normalize()
    .to_numpy(
        dtype="datetime64[ns]",
        copy=False,
    )
)

customer_np = (
    features["customer_id"]
    .to_numpy(
        copy=False
    )
)

engaged_np = (
    is_engaged
    .to_numpy(
        dtype=np.int8,
        copy=False,
    )
)


# ------------------------------------------------------------
# Customer boundaries
# ------------------------------------------------------------

customer_change = np.r_[
    True,
    customer_np[1:]
    != customer_np[:-1],
]

starts = np.flatnonzero(
    customer_change
)

ends = np.r_[
    starts[1:],
    len(features),
]


recent_engagement_counts = {
    3: np.zeros(
        len(features),
        dtype=np.int16,
    ),
    7: np.zeros(
        len(features),
        dtype=np.int16,
    ),
    14: np.zeros(
        len(features),
        dtype=np.int16,
    ),
    30: np.zeros(
        len(features),
        dtype=np.int16,
    ),
}


one_day = np.timedelta64(
    1,
    "D",
)


for start, end in zip(
    starts,
    ends,
):

    dates = (
        sent_date_np[
            start:end
        ]
    )

    engaged_positions = np.flatnonzero(
        engaged_np[
            start:end
        ]
        == 1
    )

    if len(
        engaged_positions
    ) == 0:
        continue

    engaged_dates = (
        dates[
            engaged_positions
        ]
    )

    for window in [
        3,
        7,
        14,
        30,
    ]:

        lower_dates = (
            dates
            -
            window * one_day
        )

        left = np.searchsorted(
            engaged_dates,
            lower_dates,
            side="left",
        )

        right = np.searchsorted(
            engaged_dates,
            dates,
            side="left",
        )

        recent_engagement_counts[
            window
        ][
            start:end
        ] = (
            right
            -
            left
        )


for window in [
    3,
    7,
    14,
    30,
]:

    features[
        f"hist_n_engaged_messages_last_{window}_calendar_days"
    ] = (
        recent_engagement_counts[
            window
        ]
    )


# ============================================================
# 16. RECENT ENGAGEMENT SHARE
# ============================================================

features[
    "hist_recent_engagement_share_7d"
] = np.where(
    features[
        "hist_n_messages_last_7d"
    ] > 0,

    features[
        "hist_n_engaged_messages_last_7_calendar_days"
    ]
    /
    features[
        "hist_n_messages_last_7d"
    ],

    np.nan,
)


features[
    "hist_recent_engagement_share_14d"
] = np.where(
    features[
        "hist_n_messages_last_14d"
    ] > 0,

    features[
        "hist_n_engaged_messages_last_14_calendar_days"
    ]
    /
    features[
        "hist_n_messages_last_14d"
    ],

    np.nan,
)


features[
    "hist_recent_engagement_share_30d"
] = np.where(
    features[
        "hist_n_messages_last_30d"
    ] > 0,

    features[
        "hist_n_engaged_messages_last_30_calendar_days"
    ]
    /
    features[
        "hist_n_messages_last_30d"
    ],

    np.nan,
)


# ============================================================
# 17. ENGAGEMENT TREND
# ============================================================
#
# Compare recent engagement share against lifetime historical
# engagement rate.
#
# Descriptive only.
# ============================================================

features[
    "hist_engagement_rate_change_7d_vs_history"
] = (
    features[
        "hist_recent_engagement_share_7d"
    ]
    -
    features[
        "hist_engagement_rate"
    ]
)


features[
    "hist_engagement_rate_change_14d_vs_history"
] = (
    features[
        "hist_recent_engagement_share_14d"
    ]
    -
    features[
        "hist_engagement_rate"
    ]
)


features[
    "hist_recent_engagement_improving"
] = (
    features[
        "hist_engagement_rate_change_7d_vs_history"
    ] > 0
).astype("int8")


features[
    "hist_recent_engagement_declining"
] = (
    features[
        "hist_engagement_rate_change_7d_vs_history"
    ] < 0
).astype("int8")


# ============================================================
# 18. HARD CONSISTENCY / PIT-STRUCTURE AUDITS
# ============================================================

print("\nRunning hard consistency checks...")


# ------------------------------------------------------------
# Interaction counts sum exactly to previous messages
# ------------------------------------------------------------

assert (
    hist_n_none
    +
    hist_n_read
    +
    hist_n_clicked
    +
    hist_n_replied
    ==
    n_previous
).all()


assert (
    hist_n_engaged
    ==
    (
        hist_n_read
        +
        hist_n_clicked
        +
        hist_n_replied
    )
).all()


# ------------------------------------------------------------
# Historical engagement cannot exceed delivered history
# ------------------------------------------------------------

assert (
    hist_n_engaged
    <=
    hist_n_delivered
).all()


# ------------------------------------------------------------
# First contact = zero historical interaction
# ------------------------------------------------------------

first_contact = (
    features[
        "hist_is_first_contact"
    ].eq(1)
)


interaction_count_columns = [
    "hist_n_interaction_none",
    "hist_n_read",
    "hist_n_clicked_link",
    "hist_n_replied",
    "hist_n_engaged",
]


assert (
    features.loc[
        first_contact,
        interaction_count_columns
    ]
    .eq(0)
    .all()
    .all()
)


assert (
    features.loc[
        first_contact,
        "hist_previous_interaction"
    ]
    .isna()
    .all()
)


# ------------------------------------------------------------
# Rates bounded [0, 1]
# ------------------------------------------------------------

rate_columns = [
    "hist_no_interaction_rate",
    "hist_read_rate",
    "hist_click_rate",
    "hist_reply_rate",
    "hist_engagement_rate",
    "hist_engagement_rate_given_delivered",
    "hist_no_interaction_rate_given_delivered",
    "hist_click_rate_given_delivered",
    "hist_reply_rate_given_delivered",
    "hist_recent_engagement_share_7d",
    "hist_recent_engagement_share_14d",
    "hist_recent_engagement_share_30d",
]


for col in rate_columns:

    valid = (
        features[col]
        .dropna()
    )

    assert (
        valid
        .between(
            0,
            1,
            inclusive="both",
        )
        .all()
    ), col


# ------------------------------------------------------------
# Engagement + no-engagement among delivered = 1
# ------------------------------------------------------------

has_delivery_history = (
    hist_n_delivered > 0
)


assert np.allclose(
    (
        features.loc[
            has_delivery_history,
            "hist_engagement_rate_given_delivered"
        ]
        +
        features.loc[
            has_delivery_history,
            "hist_no_interaction_rate_given_delivered"
        ]
    ),
    1.0,
)


# ------------------------------------------------------------
# Historical timestamps strictly before current event
# ------------------------------------------------------------

timestamp_columns = [
    "hist_last_engaged_message_at",
    "hist_last_read_message_at",
    "hist_last_clicked_message_at",
    "hist_last_replied_message_at",
    "hist_last_no_interaction_message_at",
]


for col in timestamp_columns:

    mask = (
        features[col]
        .notna()
    )

    assert (
        features.loc[
            mask,
            col
        ]
        <
        features.loc[
            mask,
            "sent_at"
        ]
    ).all(), col


# ------------------------------------------------------------
# Recencies nonnegative
# ------------------------------------------------------------

recency_columns = [
    "hist_days_since_last_engaged_message",
    "hist_days_since_last_read_message",
    "hist_days_since_last_clicked_message",
    "hist_days_since_last_replied_message",
]


for col in recency_columns:

    assert (
        features[col]
        .dropna()
        .ge(0)
        .all()
    ), col


# ------------------------------------------------------------
# Streaks cannot exceed corresponding historical counts
# ------------------------------------------------------------

assert (
    features[
        "hist_consecutive_engaged"
    ]
    <=
    hist_n_engaged
).all()


assert (
    features[
        "hist_consecutive_no_interaction"
    ]
    <=
    hist_n_none
).all()


assert (
    features[
        "hist_consecutive_read"
    ]
    <=
    hist_n_read
).all()


assert (
    features[
        "hist_consecutive_clicked"
    ]
    <=
    hist_n_clicked
).all()


assert (
    features[
        "hist_consecutive_replied"
    ]
    <=
    hist_n_replied
).all()


# ------------------------------------------------------------
# Engagement diversity
# ------------------------------------------------------------

assert (
    features[
        "hist_n_distinct_engagement_types"
    ]
    .between(
        0,
        3,
        inclusive="both",
    )
    .all()
)


# ------------------------------------------------------------
# Recent engaged count cannot exceed corresponding historical
# message count.
# ------------------------------------------------------------

assert (
    features[
        "hist_n_engaged_messages_last_7_calendar_days"
    ]
    <=
    features[
        "hist_n_messages_last_7d"
    ]
).all()


assert (
    features[
        "hist_n_engaged_messages_last_14_calendar_days"
    ]
    <=
    features[
        "hist_n_messages_last_14d"
    ]
).all()


assert (
    features[
        "hist_n_engaged_messages_last_30_calendar_days"
    ]
    <=
    features[
        "hist_n_messages_last_30d"
    ]
).all()


# ------------------------------------------------------------
# No infinities
# ------------------------------------------------------------

new_columns = (
    features.columns[
        cols_before:
    ]
)


new_numeric = (
    features[
        new_columns
    ]
    .select_dtypes(
        include=[
            np.number
        ]
    )
)


assert ~np.isinf(
    new_numeric.to_numpy()
).any()


print("Hard consistency checks: PASSED")


# ============================================================
# 19. FINAL SHAPE
# ============================================================

cols_after = (
    features.shape[1]
)

n_new = (
    cols_after
    -
    cols_before
)


print("\n" + "=" * 100)
print("03D — HISTORICAL INTERACTION STATE COMPLETE")
print("=" * 100)

print(
    f"Rows          : "
    f"{len(features):,}"
)

print(
    f"Customers     : "
    f"{features['customer_id'].nunique():,}"
)

print(
    f"Columns before: "
    f"{cols_before:,}"
)

print(
    f"Columns after : "
    f"{cols_after:,}"
)

print(
    f"New features  : "
    f"{n_new:,}"
)


# ============================================================
# 20. HISTORICAL ENGAGEMENT PREVALENCE
# ============================================================

print(
    "\nHISTORICAL ENGAGEMENT PREVALENCE"
)


engagement_prevalence = pd.DataFrame(
    {
        "events": [
            features[
                "hist_ever_engaged"
            ].sum(),

            features[
                "hist_ever_read"
            ].sum(),

            features[
                "hist_ever_clicked_link"
            ].sum(),

            features[
                "hist_ever_replied"
            ].sum(),

            features[
                "hist_never_engaged_with_prior_messages"
            ].sum(),
        ],
    },
    index=[
        "ever_engaged",
        "ever_read",
        "ever_clicked",
        "ever_replied",
        "prior_history_but_never_engaged",
    ],
)


engagement_prevalence[
    "pct_events"
] = (
    engagement_prevalence[
        "events"
    ]
    /
    len(features)
    * 100
)


display(
    engagement_prevalence
)


# ============================================================
# 21. HISTORICAL INTERACTION PROFILE
# ============================================================

print(
    "\nHISTORICAL INTERACTION PROFILE"
)


display(
    features[
        "hist_interaction_profile"
    ]
    .value_counts(
        dropna=False
    )
    .to_frame("events")
)


# ============================================================
# 22. HISTORICAL ENGAGEMENT RATE
# ============================================================

print(
    "\nHISTORICAL ENGAGEMENT RATE"
)


display(
    features[
        "hist_engagement_rate"
    ]
    .describe(
        percentiles=[
            .10,
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame()
)


# ============================================================
# 23. ENGAGEMENT RATE GIVEN DELIVERY
# ============================================================

print(
    "\nENGAGEMENT RATE GIVEN HISTORICAL DELIVERY"
)


display(
    features[
        "hist_engagement_rate_given_delivered"
    ]
    .describe(
        percentiles=[
            .10,
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame()
)


# ============================================================
# 24. NO-INTERACTION STREAK
# ============================================================

print(
    "\nCONSECUTIVE HISTORICAL NO-INTERACTION STREAK"
)


display(
    features[
        "hist_consecutive_no_interaction"
    ]
    .describe(
        percentiles=[
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame()
)


# ============================================================
# 25. ENGAGEMENT DEPTH
# ============================================================

print(
    "\nSTRONGEST HISTORICAL INTERACTION"
)


display(
    features[
        "hist_strongest_interaction"
    ]
    .value_counts(
        dropna=False
    )
    .to_frame("events")
)


# ============================================================
# 26. RECENT ENGAGEMENT TREND
# ============================================================

print(
    "\nRECENT ENGAGEMENT TREND"
)


trend_summary = pd.DataFrame(
    {
        "events": [
            features[
                "hist_recent_engagement_improving"
            ].sum(),

            features[
                "hist_recent_engagement_declining"
            ].sum(),
        ],
    },
    index=[
        "recent_7d_above_historical_rate",
        "recent_7d_below_historical_rate",
    ],
)


trend_summary[
    "pct_events"
] = (
    trend_summary[
        "events"
    ]
    /
    len(features)
    * 100
)


display(
    trend_summary
)


# ============================================================
# 27. SAMPLE INTERACTION JOURNEY
# ============================================================
#
# Pick customer with high engagement diversity/history.
# ============================================================

sample_score = (
    features[
        "hist_n_engaged"
    ]
    +
    3
    *
    features[
        "hist_n_distinct_engagement_types"
    ]
)


sample_customer = (
    features.loc[
        sample_score.idxmax(),
        "customer_id",
    ]
)


print(
    f"\nSAMPLE INTERACTION JOURNEY — "
    f"{sample_customer}"
)


display(
    features.loc[
        features[
            "customer_id"
        ].eq(
            sample_customer
        ),
        [
            "customer_id",
            "event_number",
            "sent_at",
            "current_template",

            "event_interaction",

            "hist_n_previous_messages",

            "hist_n_engaged",
            "hist_n_read",
            "hist_n_clicked_link",
            "hist_n_replied",

            "hist_previous_interaction",

            "hist_engagement_rate",

            "hist_consecutive_no_interaction",
            "hist_consecutive_engaged",

            "hist_days_since_last_engaged_message",

            "hist_strongest_interaction",

            "hist_interaction_profile",
        ],
    ]
)


# ============================================================
# 28. GOVERNANCE SUMMARY
# ============================================================

print("\n" + "=" * 100)
print("03D — TEMPORAL GOVERNANCE")
print("=" * 100)

print(
    """
INTERACTION HISTORY AVAILABILITY CLASS

    temporal_assumption_interaction_timestamp_missing

The features created in 03D are historical with respect to
event ordering:

    source event j < current event t

but exact interaction-time availability cannot be proven
because the dataset does not contain:

    read_at
    clicked_at
    replied_at

Therefore:

    03B payment history
        → strict PIT via explicit +72h maturity

    03C delivery history
        → operational feedback availability assumption

    03D interaction history
        → TEMPORAL ASSUMPTION / sensitivity feature family

These interaction features should later be evaluated using:

    STRICT MODEL
        excludes 03D interaction history

    OPERATIONAL MODEL
        includes 03D interaction history

and compared out-of-time.

IMPORTANT:

    hist_days_since_last_*_message

measures time since the SEND that eventually generated the
interaction, NOT time since the actual read/click/reply.
"""
)


print("=" * 100)
print("03D — HISTORICAL INTERACTION STATE COMPLETE")
print("=" * 100)

03D — HISTORICAL INTERACTION STATE [TEMPORAL ASSUMPTION]
Starting shape: (75406, 500)
Interaction taxonomy: PASSED

Running hard consistency checks...


AssertionError: hist_recent_engagement_share_7d

In [27]:
# ============================================================
# 03D FIX — ALIGN RECENT ENGAGEMENT WINDOW SEMANTICS
# ============================================================
#
# PROBLEM
# -------
# Numerator:
#   engaged messages on previous N CALENDAR DATES
#
# Denominator previously used:
#   exact rolling N × 24h messages from 02B
#
# Those windows are NOT equivalent.
#
# FIX
# ---
# Build matching calendar-date denominators:
#
#   [D-N, D)
#
# where D = current calendar date.
#
# ============================================================

print("=" * 100)
print("03D FIX — CALENDAR-WINDOW DENOMINATOR ALIGNMENT")
print("=" * 100)

print("Current shape:", features.shape)


# ============================================================
# 1. BUILD CALENDAR-DATE MESSAGE COUNTS
# ============================================================

sent_date_np = (
    features["sent_at"]
    .dt.normalize()
    .to_numpy(
        dtype="datetime64[ns]",
        copy=False,
    )
)

customer_np = (
    features["customer_id"]
    .to_numpy(copy=False)
)


customer_change = np.r_[
    True,
    customer_np[1:] != customer_np[:-1],
]

starts = np.flatnonzero(customer_change)

ends = np.r_[
    starts[1:],
    len(features),
]


calendar_message_counts = {
    3: np.zeros(len(features), dtype=np.int16),
    7: np.zeros(len(features), dtype=np.int16),
    14: np.zeros(len(features), dtype=np.int16),
    30: np.zeros(len(features), dtype=np.int16),
}


one_day = np.timedelta64(1, "D")


for start, end in zip(starts, ends):

    dates = sent_date_np[start:end]

    for window in [3, 7, 14, 30]:

        lower_dates = (
            dates
            - window * one_day
        )

        left = np.searchsorted(
            dates,
            lower_dates,
            side="left",
        )

        # Exclude ALL messages on current calendar date.
        #
        # In this dataset there is max 1 message/customer/day,
        # but this formulation preserves the intended [D-N, D)
        # semantics explicitly.
        right = np.searchsorted(
            dates,
            dates,
            side="left",
        )

        calendar_message_counts[
            window
        ][start:end] = (
            right - left
        )


# ============================================================
# 2. ATTACH MATCHING DENOMINATORS
# ============================================================

for window in [3, 7, 14, 30]:

    features[
        f"hist_n_messages_last_{window}_calendar_days"
    ] = calendar_message_counts[
        window
    ]


# ============================================================
# 3. VALIDATE CALENDAR COUNTS
# ============================================================

assert (
    features[
        "hist_n_messages_last_3_calendar_days"
    ]
    <=
    features[
        "hist_n_previous_messages"
    ]
).all()

assert (
    features[
        "hist_n_messages_last_7_calendar_days"
    ]
    <=
    features[
        "hist_n_previous_messages"
    ]
).all()

assert (
    features[
        "hist_n_messages_last_14_calendar_days"
    ]
    <=
    features[
        "hist_n_previous_messages"
    ]
).all()

assert (
    features[
        "hist_n_messages_last_30_calendar_days"
    ]
    <=
    features[
        "hist_n_previous_messages"
    ]
).all()


# Engaged numerator must now be <= matching denominator.

for window in [3, 7, 14, 30]:

    assert (
        features[
            f"hist_n_engaged_messages_last_{window}_calendar_days"
        ]
        <=
        features[
            f"hist_n_messages_last_{window}_calendar_days"
        ]
    ).all(), window


print(
    "Calendar-window numerator/denominator alignment: PASSED"
)


# ============================================================
# 4. REPLACE THE THREE RECENT ENGAGEMENT SHARES
# ============================================================

for window in [7, 14, 30]:

    numerator = features[
        f"hist_n_engaged_messages_last_{window}_calendar_days"
    ]

    denominator = features[
        f"hist_n_messages_last_{window}_calendar_days"
    ]

    features[
        f"hist_recent_engagement_share_{window}d"
    ] = np.where(
        denominator > 0,
        numerator / denominator,
        np.nan,
    )


# ============================================================
# 5. REBUILD DERIVED TREND FEATURES
# ============================================================

features[
    "hist_engagement_rate_change_7d_vs_history"
] = (
    features[
        "hist_recent_engagement_share_7d"
    ]
    -
    features[
        "hist_engagement_rate"
    ]
)


features[
    "hist_engagement_rate_change_14d_vs_history"
] = (
    features[
        "hist_recent_engagement_share_14d"
    ]
    -
    features[
        "hist_engagement_rate"
    ]
)


features[
    "hist_recent_engagement_improving"
] = (
    features[
        "hist_engagement_rate_change_7d_vs_history"
    ] > 0
).astype("int8")


features[
    "hist_recent_engagement_declining"
] = (
    features[
        "hist_engagement_rate_change_7d_vs_history"
    ] < 0
).astype("int8")


# ============================================================
# 6. VALIDATE SHARES
# ============================================================

share_columns = [
    "hist_recent_engagement_share_7d",
    "hist_recent_engagement_share_14d",
    "hist_recent_engagement_share_30d",
]


for col in share_columns:

    valid = features[col].dropna()

    assert valid.between(
        0,
        1,
        inclusive="both",
    ).all(), col


print("Recent engagement shares bounded [0,1]: PASSED")


# ============================================================
# 7. DIAGNOSTIC — EXACT VS CALENDAR WINDOWS
# ============================================================

print(
    "\nEXACT 24H WINDOW vs CALENDAR-DATE WINDOW"
)


window_comparison = pd.DataFrame(
    {
        "window": [
            "7d",
            "14d",
            "30d",
        ],

        "exact_24h_total": [
            features[
                "hist_n_messages_last_7d"
            ].sum(),

            features[
                "hist_n_messages_last_14d"
            ].sum(),

            features[
                "hist_n_messages_last_30d"
            ].sum(),
        ],

        "calendar_date_total": [
            features[
                "hist_n_messages_last_7_calendar_days"
            ].sum(),

            features[
                "hist_n_messages_last_14_calendar_days"
            ].sum(),

            features[
                "hist_n_messages_last_30_calendar_days"
            ].sum(),
        ],
    }
)


window_comparison["difference"] = (
    window_comparison[
        "calendar_date_total"
    ]
    -
    window_comparison[
        "exact_24h_total"
    ]
)


display(window_comparison)


# ============================================================
# 8. MAX SHARE DIAGNOSTIC
# ============================================================

print(
    "\nRECENT ENGAGEMENT SHARE — POST-FIX"
)


display(
    features[
        share_columns
    ]
    .describe(
        percentiles=[
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
)


# ============================================================
# 9. RE-RUN ALL 03D HARD CONSISTENCY CHECKS
# ============================================================

print(
    "\nRe-running 03D hard consistency checks..."
)


n_previous = features[
    "hist_n_previous_messages"
]

hist_n_none = features[
    "hist_n_interaction_none"
]

hist_n_read = features[
    "hist_n_read"
]

hist_n_clicked = features[
    "hist_n_clicked_link"
]

hist_n_replied = features[
    "hist_n_replied"
]

hist_n_engaged = features[
    "hist_n_engaged"
]

hist_n_delivered = features[
    "hist_n_delivered"
]


# Counts

assert (
    hist_n_none
    + hist_n_read
    + hist_n_clicked
    + hist_n_replied
    ==
    n_previous
).all()


assert (
    hist_n_engaged
    ==
    (
        hist_n_read
        + hist_n_clicked
        + hist_n_replied
    )
).all()


assert (
    hist_n_engaged
    <=
    hist_n_delivered
).all()


# First contacts

first_contact = features[
    "hist_is_first_contact"
].eq(1)


interaction_count_columns = [
    "hist_n_interaction_none",
    "hist_n_read",
    "hist_n_clicked_link",
    "hist_n_replied",
    "hist_n_engaged",
]


assert (
    features.loc[
        first_contact,
        interaction_count_columns
    ]
    .eq(0)
    .all()
    .all()
)


assert (
    features.loc[
        first_contact,
        "hist_previous_interaction"
    ]
    .isna()
    .all()
)


# Rates

rate_columns = [
    "hist_no_interaction_rate",
    "hist_read_rate",
    "hist_click_rate",
    "hist_reply_rate",
    "hist_engagement_rate",
    "hist_engagement_rate_given_delivered",
    "hist_no_interaction_rate_given_delivered",
    "hist_click_rate_given_delivered",
    "hist_reply_rate_given_delivered",
    "hist_recent_engagement_share_7d",
    "hist_recent_engagement_share_14d",
    "hist_recent_engagement_share_30d",
]


for col in rate_columns:

    valid = features[col].dropna()

    assert valid.between(
        0,
        1,
        inclusive="both",
    ).all(), col


# Conditional engagement rates

has_delivery_history = (
    hist_n_delivered > 0
)


assert np.allclose(
    (
        features.loc[
            has_delivery_history,
            "hist_engagement_rate_given_delivered"
        ]
        +
        features.loc[
            has_delivery_history,
            "hist_no_interaction_rate_given_delivered"
        ]
    ),
    1.0,
)


# Historical timestamps

timestamp_columns = [
    "hist_last_engaged_message_at",
    "hist_last_read_message_at",
    "hist_last_clicked_message_at",
    "hist_last_replied_message_at",
    "hist_last_no_interaction_message_at",
]


for col in timestamp_columns:

    mask = features[col].notna()

    assert (
        features.loc[mask, col]
        <
        features.loc[mask, "sent_at"]
    ).all(), col


# Recencies

recency_columns = [
    "hist_days_since_last_engaged_message",
    "hist_days_since_last_read_message",
    "hist_days_since_last_clicked_message",
    "hist_days_since_last_replied_message",
]


for col in recency_columns:

    assert (
        features[col]
        .dropna()
        .ge(0)
        .all()
    ), col


# Streaks

assert (
    features[
        "hist_consecutive_engaged"
    ]
    <=
    hist_n_engaged
).all()


assert (
    features[
        "hist_consecutive_no_interaction"
    ]
    <=
    hist_n_none
).all()


assert (
    features[
        "hist_consecutive_read"
    ]
    <=
    hist_n_read
).all()


assert (
    features[
        "hist_consecutive_clicked"
    ]
    <=
    hist_n_clicked
).all()


assert (
    features[
        "hist_consecutive_replied"
    ]
    <=
    hist_n_replied
).all()


# Diversity

assert (
    features[
        "hist_n_distinct_engagement_types"
    ]
    .between(
        0,
        3,
        inclusive="both",
    )
    .all()
)


# Calendar-window consistency

for window in [3, 7, 14, 30]:

    assert (
        features[
            f"hist_n_engaged_messages_last_{window}_calendar_days"
        ]
        <=
        features[
            f"hist_n_messages_last_{window}_calendar_days"
        ]
    ).all(), window


# No infinities in all 03D features

interaction_feature_names = [
    col
    for col in features.columns
    if (
        col.startswith("hist_")
        and
        (
            "interaction" in col
            or
            "engag" in col
            or
            "read" in col
            or
            "click" in col
            or
            "repl" in col
        )
    )
]


interaction_numeric = (
    features[
        interaction_feature_names
    ]
    .select_dtypes(
        include=[np.number]
    )
)


assert ~np.isinf(
    interaction_numeric.to_numpy()
).any()


print(
    "03D hard consistency checks: PASSED"
)


# ============================================================
# 10. CURRENT CHECKPOINT
# ============================================================

print("\n" + "=" * 100)
print("03D FIX COMPLETE")
print("=" * 100)

print(
    f"Rows      : "
    f"{len(features):,}"
)

print(
    f"Customers : "
    f"{features['customer_id'].nunique():,}"
)

print(
    f"Columns   : "
    f"{features.shape[1]:,}"
)

03D FIX — CALENDAR-WINDOW DENOMINATOR ALIGNMENT
Current shape: (75406, 574)
Calendar-window numerator/denominator alignment: PASSED
Recent engagement shares bounded [0,1]: PASSED

EXACT 24H WINDOW vs CALENDAR-DATE WINDOW


,window,exact_24h_total,calendar_date_total,difference
0,7d,91663,98206,6543
1,14d,159283,163413,4130
2,30d,243427,244912,1485



RECENT ENGAGEMENT SHARE — POST-FIX


,hist_recent_engagement_share_7d,hist_recent_engagement_share_14d,hist_recent_engagement_share_30d
count,"53,341.00","61,066.00","63,555.00"
mean,0.42,0.42,0.44
std,0.42,0.38,0.35
min,0.00,0.00,0.00
50%,0.33,0.40,0.43
75%,1.00,0.67,0.67
90%,1.00,1.00,1.00
95%,1.00,1.00,1.00
99%,1.00,1.00,1.00
max,1.00,1.00,1.00



Re-running 03D hard consistency checks...
03D hard consistency checks: PASSED

03D FIX COMPLETE
Rows      : 75,406
Customers : 11,724
Columns   : 578


In [28]:
# ============================================================
# 03D — FINAL SUMMARY ONLY
# No features are created here.
# ============================================================

print("=" * 100)
print("03D — FINAL VALIDATED SUMMARY")
print("=" * 100)

assert features.shape == (75_406, 578)

print(f"Rows      : {len(features):,}")
print(f"Customers : {features['customer_id'].nunique():,}")
print(f"Columns   : {features.shape[1]:,}")


# ------------------------------------------------------------
# 1. Historical engagement prevalence
# ------------------------------------------------------------

print("\nHISTORICAL ENGAGEMENT PREVALENCE")

summary = pd.DataFrame(
    {
        "events": [
            features["hist_ever_engaged"].sum(),
            features["hist_ever_read"].sum(),
            features["hist_ever_clicked_link"].sum(),
            features["hist_ever_replied"].sum(),
            features["hist_never_engaged_with_prior_messages"].sum(),
        ]
    },
    index=[
        "ever_engaged",
        "ever_read",
        "ever_clicked",
        "ever_replied",
        "prior_history_but_never_engaged",
    ],
)

summary["pct_events"] = (
    summary["events"] / len(features) * 100
)

display(summary)


# ------------------------------------------------------------
# 2. Interaction profile
# ------------------------------------------------------------

print("\nHISTORICAL INTERACTION PROFILE")

display(
    features["hist_interaction_profile"]
    .value_counts(dropna=False)
    .to_frame("events")
)


# ------------------------------------------------------------
# 3. Engagement rate
# ------------------------------------------------------------

print("\nHISTORICAL ENGAGEMENT RATE")

display(
    features["hist_engagement_rate"]
    .describe(
        percentiles=[
            .10, .25, .50, .75, .90, .95, .99
        ]
    )
    .to_frame()
)


print("\nENGAGEMENT RATE GIVEN HISTORICAL DELIVERY")

display(
    features["hist_engagement_rate_given_delivered"]
    .describe(
        percentiles=[
            .10, .25, .50, .75, .90, .95, .99
        ]
    )
    .to_frame()
)


# ------------------------------------------------------------
# 4. No-interaction persistence
# ------------------------------------------------------------

print("\nCONSECUTIVE HISTORICAL NO-INTERACTION STREAK")

display(
    features["hist_consecutive_no_interaction"]
    .describe(
        percentiles=[
            .50, .75, .90, .95, .99
        ]
    )
    .to_frame()
)


# ------------------------------------------------------------
# 5. Strongest historical interaction
# ------------------------------------------------------------

print("\nSTRONGEST HISTORICAL INTERACTION")

display(
    features["hist_strongest_interaction"]
    .value_counts(dropna=False)
    .to_frame("events")
)


# ------------------------------------------------------------
# 6. Recent trend
# ------------------------------------------------------------

print("\nRECENT ENGAGEMENT TREND")

trend = pd.DataFrame(
    {
        "events": [
            features[
                "hist_recent_engagement_improving"
            ].sum(),

            features[
                "hist_recent_engagement_declining"
            ].sum(),
        ]
    },
    index=[
        "recent_7d_above_historical_rate",
        "recent_7d_below_historical_rate",
    ],
)

trend["pct_events"] = (
    trend["events"] / len(features) * 100
)

display(trend)


# ------------------------------------------------------------
# 7. Sample journey
# ------------------------------------------------------------

sample_score = (
    features["hist_n_engaged"]
    +
    3 * features[
        "hist_n_distinct_engagement_types"
    ]
)

sample_customer = features.loc[
    sample_score.idxmax(),
    "customer_id",
]

print(
    f"\nSAMPLE INTERACTION JOURNEY — "
    f"{sample_customer}"
)

display(
    features.loc[
        features["customer_id"].eq(sample_customer),
        [
            "customer_id",
            "event_number",
            "sent_at",
            "current_template",
            "event_interaction",

            "hist_n_previous_messages",
            "hist_n_engaged",
            "hist_n_read",
            "hist_n_clicked_link",
            "hist_n_replied",

            "hist_previous_interaction",
            "hist_engagement_rate",

            "hist_consecutive_no_interaction",
            "hist_consecutive_engaged",

            "hist_days_since_last_engaged_message",

            "hist_strongest_interaction",
            "hist_interaction_profile",
        ],
    ]
)


print("\n" + "=" * 100)
print("03D — GOVERNANCE")
print("=" * 100)

print("""
Availability class:
    temporal_assumption_interaction_timestamp_missing

03B Payment:
    STRICT PIT — explicit +72h maturity

03C Delivery:
    operational availability assumption

03D Interaction:
    temporal availability assumption
    read_at / clicked_at / replied_at unavailable

Current validated checkpoint:
    75,406 × 578
""")

print("=" * 100)
print("03D — CLOSED")
print("=" * 100)

03D — FINAL VALIDATED SUMMARY
Rows      : 75,406
Customers : 11,724
Columns   : 578

HISTORICAL ENGAGEMENT PREVALENCE


,events,pct_events
ever_engaged,49455,65.58
ever_read,39834,52.83
ever_clicked,22031,29.22
ever_replied,14506,19.24
prior_history_but_never_engaged,14227,18.87



HISTORICAL INTERACTION PROFILE


,events
hist_interaction_profile,
read_only,19142
has_clicked_no_reply,15807
has_replied,14506
no_history,11724
delivered_never_engaged,9474
no_historical_delivery,4753



HISTORICAL ENGAGEMENT RATE


...


RECENT ENGAGEMENT TREND


,events,pct_events
recent_7d_above_historical_rate,9495,12.59
recent_7d_below_historical_rate,15267,20.25



SAMPLE INTERACTION JOURNEY — C004243


,customer_id,event_number,sent_at,current_template,event_interaction,hist_n_previous_messages,hist_n_engaged,hist_n_read,hist_n_clicked_link,hist_n_replied,hist_previous_interaction,hist_engagement_rate,hist_consecutive_no_interaction,hist_consecutive_engaged,hist_days_since_last_engaged_message,hist_strongest_interaction,hist_interaction_profile
27143,C004243,1,2026-06-03 11:46:00,friendly_reminder,replied,0,0,0,0,0,NaN,NaN,0,0,NaN,no_history,no_history
27144,C004243,2,2026-06-04 19:44:00,pix_link,clicked_link,1,1,0,0,1,replied,1.00,0,1,1.33,replied,has_replied
27145,C004243,3,2026-06-10 12:44:00,pix_link,read,2,2,0,1,1,clicked_link,1.00,0,2,5.71,replied,has_replied
27146,C004243,4,2026-06-11 12:05:00,pix_link,read,3,3,1,1,1,read,1.00,0,3,0.97,replied,has_replied
27147,C004243,5,2026-06-12 09:57:00,urgent_reminder,none,4,4,2,1,1,read,1.00,0,4,0.91,replied,has_replied
27148,C004243,6,2026-06-14 13:53:00,urgent_reminder,none,5,4,2,1,1,none,0.80,1,0,3.08,replied,has_replied
27149,C004243,7,2026-06-15 10:39:00,pix_link,clicked_link,6,4,2,1,1,none,0.67,2,0,3.94,replied,has_replied
27150,C004243,8,2026-06-19 13:35:00,pix_link,clicked_link,7,5,2,2,1,clicked_link,0.71,0,1,4.12,replied,has_replied
27151,C004243,9,2026-06-20 13:51:00,pix_link,read,8,6,2,3,1,clicked_link,0.75,0,2,1.01,replied,has_replied
27152,C004243,10,2026-06-22 12:11:00,urgent_reminder,none,9,7,3,3,1,read,0.78,0,3,1.93,replied,has_replied



03D — GOVERNANCE

Availability class:
    temporal_assumption_interaction_timestamp_missing

03B Payment:
    STRICT PIT — explicit +72h maturity

03C Delivery:
    operational availability assumption

03D Interaction:
    temporal availability assumption
    read_at / clicked_at / replied_at unavailable

Current validated checkpoint:
    75,406 × 578

03D — CLOSED


##____________________________________________________________________________________________________________________________________________

##____________________________________________________________________________________________________________________________________________

In [29]:
# ============================================================
# 04A — BALANCE TRANSITION / PAYMENT ACCOUNTING AUDIT
# ============================================================
#
# PURPOSE
# -------
# Understand the accounting relationship between:
#
#   balance at event t
#   payment attributed after event t
#   balance observed at next event t+1
#
# BEFORE creating historical payment / balance journey features.
#
# IMPORTANT
# ---------
# This block is AUDIT ONLY.
#
# It does NOT create features.
# It does NOT use current-event payment as predictor.
# It does NOT assume every payment is visible in next balance.
#
# Discount semantics:
#
#   discount_offer settlement amount = 0.85 × balance
#
# Starting checkpoint:
#
#   75,406 × 578
#
# ============================================================

print("=" * 100)
print("04A — BALANCE TRANSITION / PAYMENT ACCOUNTING AUDIT")
print("=" * 100)


# ============================================================
# 0. CHECKPOINT
# ============================================================

assert features.shape == (75_406, 578)
assert features["customer_id"].nunique() == 11_724
assert features["event_id"].is_unique
assert features["message_id"].is_unique

rows_before = len(features)
cols_before = features.shape[1]

print(f"Starting shape: {features.shape}")


# ============================================================
# 1. MINIMAL TEMPORARY TRANSITION TABLE
# ============================================================
#
# Do NOT copy the 578-column table.
# ============================================================

audit = features[
    [
        "customer_id",
        "event_number",
        "sent_at",
        "current_template",
        "days_past_due",
        "outstanding_balance_brl",
        "event_paid_within_72h",
        "event_amount_paid_brl",
    ]
].copy()


audit = audit.rename(
    columns={
        "outstanding_balance_brl": "balance_t",
        "event_paid_within_72h": "paid72_t",
        "event_amount_paid_brl": "payment_t",
    }
)


# ============================================================
# 2. NEXT EVENT STATE
# ============================================================

g = audit.groupby(
    "customer_id",
    sort=False,
)


audit["next_sent_at"] = g["sent_at"].shift(-1)

audit["next_event_number"] = (
    g["event_number"].shift(-1)
)

audit["balance_next"] = (
    g["balance_t"].shift(-1)
)

audit["dpd_next"] = (
    g["days_past_due"].shift(-1)
)


audit["hours_to_next_event"] = (
    (
        audit["next_sent_at"]
        -
        audit["sent_at"]
    )
    .dt.total_seconds()
    / 3600
)


audit["has_next_event"] = (
    audit["next_sent_at"].notna()
)


# ============================================================
# 3. BASIC TRANSITION CHECKS
# ============================================================

transition = audit[
    audit["has_next_event"]
].copy()


assert len(transition) == (
    len(features)
    -
    features["customer_id"].nunique()
)

assert (
    transition["next_sent_at"]
    >
    transition["sent_at"]
).all()

assert (
    transition["next_event_number"]
    ==
    transition["event_number"] + 1
).all()


print(
    f"\nTransitions with next event : "
    f"{len(transition):,}"
)

print(
    f"Final observed events       : "
    f"{(~audit['has_next_event']).sum():,}"
)


# ============================================================
# 4. OBSERVED BALANCE CHANGE
# ============================================================

transition["observed_balance_reduction"] = (
    transition["balance_t"]
    -
    transition["balance_next"]
)


transition["balance_change_abs"] = (
    transition["observed_balance_reduction"]
    .abs()
)


transition["balance_decreased"] = (
    transition["observed_balance_reduction"]
    > 0.01
)

transition["balance_unchanged"] = (
    transition["observed_balance_reduction"]
    .abs()
    <= 0.01
)

transition["balance_increased"] = (
    transition["observed_balance_reduction"]
    < -0.01
)


print("\nBALANCE TRANSITION DIRECTION")

direction = pd.DataFrame(
    {
        "transitions": [
            transition["balance_decreased"].sum(),
            transition["balance_unchanged"].sum(),
            transition["balance_increased"].sum(),
        ]
    },
    index=[
        "balance_decreased",
        "balance_unchanged",
        "balance_increased",
    ],
)

direction["pct"] = (
    direction["transitions"]
    / len(transition)
    * 100
)

display(direction)


print("\nOBSERVED BALANCE REDUCTION")

display(
    transition[
        "observed_balance_reduction"
    ]
    .describe(
        percentiles=[
            .01,
            .05,
            .10,
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame()
)


# ============================================================
# 5. PAYMENT EVENT STRUCTURAL CONSISTENCY
# ============================================================

print("\nPAYMENT EVENT CONSISTENCY")


payment_flag = (
    audit["paid72_t"].eq(1)
)

positive_amount = (
    audit["payment_t"] > 0
)


payment_consistency = pd.DataFrame(
    {
        "events": [
            payment_flag.sum(),
            positive_amount.sum(),

            (
                payment_flag
                &
                positive_amount
            ).sum(),

            (
                payment_flag
                &
                ~positive_amount
            ).sum(),

            (
                ~payment_flag
                &
                positive_amount
            ).sum(),
        ]
    },
    index=[
        "paid72_flag_1",
        "positive_payment_amount",
        "flag_1_and_positive_amount",
        "flag_1_but_zero_amount",
        "flag_0_but_positive_amount",
    ],
)

display(payment_consistency)


assert (
    payment_flag
    ==
    positive_amount
).all()

print(
    "Payment flag / amount consistency: PASSED"
)


# ============================================================
# 6. PAYMENT EVENTS WITH VS WITHOUT NEXT OBSERVATION
# ============================================================

payment_events = audit[
    payment_flag
].copy()


payment_with_next = payment_events[
    payment_events["has_next_event"]
].copy()


payment_without_next = payment_events[
    ~payment_events["has_next_event"]
].copy()


print("\nPAYMENT OBSERVABILITY")

payment_observability = pd.DataFrame(
    {
        "payment_events": [
            len(payment_events),
            len(payment_with_next),
            len(payment_without_next),
        ],
        "payment_brl": [
            payment_events["payment_t"].sum(),
            payment_with_next["payment_t"].sum(),
            payment_without_next["payment_t"].sum(),
        ],
    },
    index=[
        "all_payment_events",
        "payment_with_next_observation",
        "payment_on_final_observed_event",
    ],
)


payment_observability[
    "pct_payment_events"
] = (
    payment_observability[
        "payment_events"
    ]
    /
    len(payment_events)
    * 100
)


payment_observability[
    "pct_payment_brl"
] = (
    payment_observability[
        "payment_brl"
    ]
    /
    payment_events[
        "payment_t"
    ].sum()
    * 100
)


display(payment_observability)


# ============================================================
# 7. ACCOUNTING GAP
# ============================================================
#
# If payment_t is the only thing changing the debt:
#
#     balance_next
#         ≈
#     balance_t - payment_t
#
# Therefore:
#
#     accounting_gap
#         =
#     balance_t - payment_t - balance_next
#
# gap ≈ 0:
#     payment exactly explains next observed balance.
#
# positive gap:
#     next balance fell MORE than payment_t explains.
#
# negative gap:
#     next balance fell LESS than payment_t explains.
#
# ============================================================

transition["expected_next_balance_from_payment"] = (
    transition["balance_t"]
    -
    transition["payment_t"]
)


transition["accounting_gap"] = (
    transition[
        "expected_next_balance_from_payment"
    ]
    -
    transition["balance_next"]
)


transition["accounting_gap_abs"] = (
    transition["accounting_gap"]
    .abs()
)


print("\nACCOUNTING GAP — ALL TRANSITIONS")

display(
    transition[
        "accounting_gap"
    ]
    .describe(
        percentiles=[
            .01,
            .05,
            .10,
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame()
)


# ============================================================
# 8. PAYMENT TRANSITIONS ONLY
# ============================================================

pay_trans = transition[
    transition["paid72_t"].eq(1)
].copy()


print(
    f"\nPayment transitions with next event: "
    f"{len(pay_trans):,}"
)


print(
    "\nACCOUNTING GAP — PAYMENT TRANSITIONS"
)

display(
    pay_trans[
        "accounting_gap"
    ]
    .describe(
        percentiles=[
            .01,
            .05,
            .10,
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame()
)


# ============================================================
# 9. MATCH TOLERANCES
# ============================================================

tolerances = [
    0.01,
    0.05,
    0.10,
    1.00,
    5.00,
]


tolerance_rows = []


for tol in tolerances:

    n_match = (
        pay_trans[
            "accounting_gap_abs"
        ]
        <= tol
    ).sum()

    tolerance_rows.append(
        {
            "tolerance_brl": tol,
            "payment_transitions": len(
                pay_trans
            ),
            "exact_or_near_match": n_match,
            "match_pct": (
                n_match
                /
                len(pay_trans)
                * 100
                if len(pay_trans)
                else np.nan
            ),
        }
    )


print(
    "\nPAYMENT → NEXT BALANCE MATCH RATE"
)

display(
    pd.DataFrame(
        tolerance_rows
    )
)


# ============================================================
# 10. NO-PAYMENT TRANSITIONS
# ============================================================

no_pay_trans = transition[
    transition["paid72_t"].eq(0)
].copy()


print(
    "\nBALANCE MOVEMENT WHEN CURRENT EVENT HAS NO PAYMENT"
)


no_pay_direction = pd.DataFrame(
    {
        "transitions": [
            (
                no_pay_trans[
                    "observed_balance_reduction"
                ] > 0.01
            ).sum(),

            (
                no_pay_trans[
                    "observed_balance_reduction"
                ]
                .abs()
                <= 0.01
            ).sum(),

            (
                no_pay_trans[
                    "observed_balance_reduction"
                ] < -0.01
            ).sum(),
        ]
    },
    index=[
        "balance_decreased",
        "balance_unchanged",
        "balance_increased",
    ],
)


no_pay_direction["pct"] = (
    no_pay_direction[
        "transitions"
    ]
    /
    len(no_pay_trans)
    * 100
)


display(no_pay_direction)


# ============================================================
# 11. PAYMENT SIZE RELATIVE TO BALANCE
# ============================================================

payment_events[
    "payment_to_balance"
] = (
    payment_events["payment_t"]
    /
    payment_events["balance_t"]
)


print(
    "\nPAYMENT / BALANCE RATIO"
)


display(
    payment_events[
        "payment_to_balance"
    ]
    .describe(
        percentiles=[
            .01,
            .05,
            .10,
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame()
)


# ============================================================
# 12. NOMINAL SETTLEMENT TARGET
# ============================================================
#
# friendly / urgent / pix:
#     nominal full = 100% of balance
#
# discount_offer:
#     nominal full = 85% of balance
#
# IMPORTANT:
# This is a semantic audit definition.
# We are not yet creating a historical full-payment feature.
# ============================================================

payment_events[
    "nominal_settlement_target"
] = np.where(
    payment_events[
        "current_template"
    ].eq(
        "discount_offer"
    ),

    0.85
    *
    payment_events[
        "balance_t"
    ],

    payment_events[
        "balance_t"
    ],
)


payment_events[
    "payment_to_settlement_target"
] = (
    payment_events["payment_t"]
    /
    payment_events[
        "nominal_settlement_target"
    ]
)


payment_events[
    "settlement_gap_brl"
] = (
    payment_events["payment_t"]
    -
    payment_events[
        "nominal_settlement_target"
    ]
)


print(
    "\nPAYMENT / NOMINAL SETTLEMENT TARGET"
)


display(
    payment_events[
        "payment_to_settlement_target"
    ]
    .describe(
        percentiles=[
            .01,
            .05,
            .10,
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame()
)


# ============================================================
# 13. EXACT / NEAR NOMINAL SETTLEMENT
# ============================================================

settlement_tolerance = 0.05


payment_events[
    "near_nominal_settlement"
] = (
    payment_events[
        "settlement_gap_brl"
    ]
    .abs()
    <= settlement_tolerance
)


print(
    "\nNOMINAL SETTLEMENT MATCH"
)


settlement_summary = (
    payment_events
    .groupby(
        "current_template",
        observed=True,
    )
    .agg(
        payment_events=(
            "payment_t",
            "size",
        ),
        near_nominal_settlement=(
            "near_nominal_settlement",
            "sum",
        ),
        payment_brl=(
            "payment_t",
            "sum",
        ),
    )
)


settlement_summary[
    "near_nominal_pct"
] = (
    settlement_summary[
        "near_nominal_settlement"
    ]
    /
    settlement_summary[
        "payment_events"
    ]
    * 100
)


display(
    settlement_summary
    .sort_values(
        "payment_events",
        ascending=False,
    )
)


# ============================================================
# 14. PAYMENT SIZE BANDS
# ============================================================

ratio = payment_events[
    "payment_to_settlement_target"
]


payment_events[
    "payment_size_band"
] = pd.cut(
    ratio,
    bins=[
        -np.inf,
        0.25,
        0.50,
        0.75,
        0.95,
        1.05,
        np.inf,
    ],
    labels=[
        "<=25% target",
        "25-50% target",
        "50-75% target",
        "75-95% target",
        "~100% target",
        ">105% target",
    ],
    include_lowest=True,
    right=True,
)


print(
    "\nPAYMENT SIZE RELATIVE TO SETTLEMENT TARGET"
)


display(
    payment_events[
        "payment_size_band"
    ]
    .value_counts(
        dropna=False
    )
    .to_frame(
        "payment_events"
    )
)


# ============================================================
# 15. DISCOUNT OFFER SPECIFIC AUDIT
# ============================================================

discount_pay = payment_events[
    payment_events[
        "current_template"
    ].eq(
        "discount_offer"
    )
].copy()


print(
    "\nDISCOUNT OFFER — PAYMENT EVENTS"
)

print(
    f"Payment events : "
    f"{len(discount_pay):,}"
)


if len(discount_pay):

    print(
        "\nDiscount payment / balance"
    )

    display(
        discount_pay[
            "payment_to_balance"
        ]
        .describe(
            percentiles=[
                .10,
                .25,
                .50,
                .75,
                .90,
                .95,
                .99,
            ]
        )
        .to_frame()
    )


    print(
        "\nDistance from 85% settlement target"
    )

    display(
        discount_pay[
            "settlement_gap_brl"
        ]
        .describe(
            percentiles=[
                .10,
                .25,
                .50,
                .75,
                .90,
                .95,
                .99,
            ]
        )
        .to_frame()
    )


    exact_85 = (
        discount_pay[
            "settlement_gap_brl"
        ]
        .abs()
        <= 0.05
    )


    print(
        f"\nNear exact 85% settlements "
        f"(±R$0.05): "
        f"{exact_85.sum():,} "
        f"({exact_85.mean()*100:.2f}%)"
    )


# ============================================================
# 16. TRANSITION CLASSIFICATION
# ============================================================

pay_trans[
    "transition_class"
] = np.select(
    [
        pay_trans[
            "accounting_gap_abs"
        ].le(0.05),

        pay_trans[
            "accounting_gap"
        ].gt(0.05),

        pay_trans[
            "accounting_gap"
        ].lt(-0.05),
    ],
    [
        "payment_exactly_explains_next_balance",
        "next_balance_lower_than_payment_explains",
        "next_balance_higher_than_payment_explains",
    ],
    default="other",
)


print(
    "\nPAYMENT TRANSITION CLASS"
)


display(
    pay_trans[
        "transition_class"
    ]
    .value_counts(
        dropna=False
    )
    .to_frame(
        "transitions"
    )
)


# ============================================================
# 17. DOES TIME TO NEXT SEND MATTER?
# ============================================================

pay_trans[
    "next_send_gap_band"
] = pd.cut(
    pay_trans[
        "hours_to_next_event"
    ],
    bins=[
        0,
        24,
        48,
        72,
        168,
        np.inf,
    ],
    labels=[
        "<=24h",
        "24-48h",
        "48-72h",
        "3-7d",
        ">7d",
    ],
    include_lowest=True,
)


print(
    "\nACCOUNTING MATCH BY TIME TO NEXT SEND"
)


time_match = (
    pay_trans
    .assign(
        accounting_match=(
            pay_trans[
                "accounting_gap_abs"
            ]
            <= 0.05
        )
    )
    .groupby(
        "next_send_gap_band",
        observed=True,
    )
    .agg(
        payment_transitions=(
            "payment_t",
            "size",
        ),
        accounting_matches=(
            "accounting_match",
            "sum",
        ),
        mean_gap_brl=(
            "accounting_gap",
            "mean",
        ),
        median_gap_brl=(
            "accounting_gap",
            "median",
        ),
    )
)


time_match[
    "match_pct"
] = (
    time_match[
        "accounting_matches"
    ]
    /
    time_match[
        "payment_transitions"
    ]
    * 100
)


display(time_match)


# ============================================================
# 18. EXAMPLE EXACT TRANSITIONS
# ============================================================

print(
    "\nEXAMPLES — PAYMENT EXACTLY EXPLAINS NEXT BALANCE"
)


exact_examples = (
    pay_trans[
        pay_trans[
            "accounting_gap_abs"
        ]
        <= 0.05
    ]
    .sort_values(
        "payment_t",
        ascending=False,
    )
    .head(15)
)


display(
    exact_examples[
        [
            "customer_id",
            "event_number",
            "sent_at",
            "current_template",
            "balance_t",
            "payment_t",
            "balance_next",
            "observed_balance_reduction",
            "accounting_gap",
            "hours_to_next_event",
        ]
    ]
)


# ============================================================
# 19. EXAMPLE NON-MATCHING TRANSITIONS
# ============================================================

print(
    "\nEXAMPLES — LARGEST ACCOUNTING GAPS"
)


gap_examples = (
    pay_trans
    .sort_values(
        "accounting_gap_abs",
        ascending=False,
    )
    .head(20)
)


display(
    gap_examples[
        [
            "customer_id",
            "event_number",
            "sent_at",
            "current_template",
            "balance_t",
            "payment_t",
            "balance_next",
            "observed_balance_reduction",
            "expected_next_balance_from_payment",
            "accounting_gap",
            "hours_to_next_event",
        ]
    ]
)


# ============================================================
# 20. MULTIPLE PAYMENT EVENTS PER CUSTOMER
# ============================================================

payment_customer = (
    payment_events
    .groupby(
        "customer_id",
        sort=False,
    )
    .agg(
        n_payment_events=(
            "payment_t",
            "size",
        ),
        total_attributed_payment_brl=(
            "payment_t",
            "sum",
        ),
    )
)


print(
    "\nPAYMENT EVENTS PER PAYER CUSTOMER"
)


display(
    payment_customer[
        "n_payment_events"
    ]
    .describe(
        percentiles=[
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame()
)


print(
    "\nSINGLE VS MULTIPLE PAYMENT EVENTS"
)


payer_behavior = pd.DataFrame(
    {
        "customers": [
            (
                payment_customer[
                    "n_payment_events"
                ]
                == 1
            ).sum(),

            (
                payment_customer[
                    "n_payment_events"
                ]
                > 1
            ).sum(),
        ]
    },
    index=[
        "single_payment_event",
        "multiple_payment_events",
    ],
)


payer_behavior[
    "pct_payers"
] = (
    payer_behavior[
        "customers"
    ]
    /
    len(payment_customer)
    * 100
)


display(payer_behavior)


# ============================================================
# 21. PAYMENT EVENT POSITION
# ============================================================

payment_events[
    "is_final_observed_event"
] = (
    ~payment_events[
        "has_next_event"
    ]
)


print(
    "\nPAYMENT EVENT POSITION"
)


display(
    payment_events[
        "is_final_observed_event"
    ]
    .value_counts()
    .rename(
        index={
            False: "payment_followed_by_later_send",
            True: "payment_on_final_observed_send",
        }
    )
    .to_frame(
        "payment_events"
    )
)


# ============================================================
# 22. FINAL AUDIT — FEATURES UNCHANGED
# ============================================================

assert features.shape == (
    rows_before,
    cols_before,
)

assert features.shape == (
    75_406,
    578,
)

assert features[
    "customer_id"
].nunique() == 11_724

assert features[
    "event_id"
].is_unique


print("\n" + "=" * 100)
print("04A — AUDIT COMPLETE")
print("=" * 100)

print(
    f"Rows          : "
    f"{len(features):,}"
)

print(
    f"Customers     : "
    f"{features['customer_id'].nunique():,}"
)

print(
    f"Columns before: "
    f"{cols_before:,}"
)

print(
    f"Columns after : "
    f"{features.shape[1]:,}"
)

print(
    "Features added: 0"
)


print("\n" + "=" * 100)
print("04A — WHAT THIS AUDIT WILL DECIDE")
print("=" * 100)

print(
    """
We will use these results to decide:

1. Whether amount_paid_brl can explain observed balance transitions.

2. Whether balance_next can be interpreted as:
       previous balance - payment

   or whether additional accounting processes exist.

3. How reliable the event-level payment amount is for reconstructing
   historical debt trajectory.

4. How to define:
       partial payment
       full payment
       settlement

   including the 85% discount_offer rule.

5. Whether payment events on the final observed message create
   right-censoring for balance reconciliation.

6. Which balance/payment features can safely become historical PIT
   state in the next layer.

NO feature engineering decision is made until this audit is reviewed.
"""
)

print("=" * 100)
print("04A — BALANCE TRANSITION / PAYMENT ACCOUNTING AUDIT COMPLETE")
print("=" * 100)

04A — BALANCE TRANSITION / PAYMENT ACCOUNTING AUDIT
Starting shape: (75406, 578)

Transitions with next event : 63,682
Final observed events       : 11,724

BALANCE TRANSITION DIRECTION


,transitions,pct
balance_decreased,1701,2.67
balance_unchanged,61981,97.33
balance_increased,0,0.00



OBSERVED BALANCE REDUCTION


,observed_balance_reduction
count,"63,682.00"
mean,10.44
std,77.30
min,0.00
1%,0.00
5%,0.00
10%,0.00
25%,0.00
50%,0.00
75%,0.00



PAYMENT EVENT CONSISTENCY


...


SINGLE VS MULTIPLE PAYMENT EVENTS


,customers,pct_payers
single_payment_event,4228,86.41
multiple_payment_events,665,13.59



PAYMENT EVENT POSITION


,payment_events
is_final_observed_event,
payment_on_final_observed_send,3925
payment_followed_by_later_send,1701



04A — AUDIT COMPLETE
Rows          : 75,406
Customers     : 11,724
Columns before: 578
Columns after : 578
Features added: 0

04A — WHAT THIS AUDIT WILL DECIDE

We will use these results to decide:

1. Whether amount_paid_brl can explain observed balance transitions.

2. Whether balance_next can be interpreted as:
       previous balance - payment

   or whether additional accounting processes exist.

3. How reliable the event-level payment amount is for reconstructing
   historical debt trajectory.

4. How to define:
       partial payment
       full payment
       settlement

   including the 85% discount_offer rule.

5. Whether payment events on the final observed message create
   right-censoring for balance reconciliation.

6. Which balance/payment features can safely become historical PIT
   state in the next layer.

NO feature engineering decision is made until this audit is reviewed.

04A — BALANCE TRANSITION / PAYMENT ACCOUNTING AUDIT COMPLETE


In [30]:
# ============================================================
# 04B — CANONICAL HISTORICAL PAYMENT JOURNEY
#       STRICT PIT — 72H MATURITY
# ============================================================
#
# PURPOSE
# -------
# Build historical payment-journey state available at each
# decision time t.
#
# STRICT TEMPORAL RULE
# --------------------
# A payment outcome from source event j becomes available only:
#
#       sent_at_j + 72h <= sent_at_t
#
# Therefore:
#
#   - current event outcome NEVER enters its own features
#   - a payment from the previous event may still be unavailable
#   - only MATURE payment signals enter historical state
#
# SETTLEMENT DEFINITION
# ---------------------
# nominal target:
#
#   discount_offer:
#       0.85 × balance_at_source_event
#
#   otherwise:
#       1.00 × balance_at_source_event
#
# payment >= target - R$0.05:
#       settlement/full
#
# 0 < payment < target - R$0.05:
#       partial
#
# IMPORTANT
# ---------
# We do NOT use balance_next here.
# We do NOT use future sends.
# We do NOT use current payment outcome as current-row predictor.
#
# Starting checkpoint:
#
#       75,406 × 578
#
# ============================================================

import numpy as np
import pandas as pd


print("=" * 100)
print("04B — CANONICAL HISTORICAL PAYMENT JOURNEY [STRICT PIT +72H]")
print("=" * 100)


# ============================================================
# 0. CHECKPOINT
# ============================================================

assert features.shape == (75_406, 578)
assert features["customer_id"].nunique() == 11_724
assert features["event_id"].is_unique
assert features["message_id"].is_unique

rows_before = len(features)
cols_before = features.shape[1]

print(f"Starting shape: {features.shape}")


# ============================================================
# 1. SOURCE EVENT PAYMENT SEMANTICS
# ============================================================
#
# These arrays describe the outcome of SOURCE events.
#
# They are TEMPORARY.
# They are NOT attached directly as predictive features.
#
# ============================================================

source_payment = (
    features["event_amount_paid_brl"]
    .fillna(0)
    .astype(float)
    .to_numpy(copy=False)
)

source_paid_flag = (
    features["event_paid_within_72h"]
    .fillna(0)
    .astype("int8")
    .to_numpy(copy=False)
)

source_balance = (
    features["outstanding_balance_brl"]
    .astype(float)
    .to_numpy(copy=False)
)

source_template = (
    features["current_template"]
    .astype(str)
    .to_numpy(copy=False)
)

source_sent_at = (
    features["sent_at"]
    .to_numpy(
        dtype="datetime64[ns]",
        copy=False,
    )
)


# Structural consistency from 04A

assert np.array_equal(
    source_paid_flag == 1,
    source_payment > 0,
)


# ============================================================
# 2. NOMINAL SETTLEMENT TARGET AT SOURCE EVENT
# ============================================================

source_is_discount = (
    source_template == "discount_offer"
)


source_settlement_target = np.where(
    source_is_discount,
    0.85 * source_balance,
    source_balance,
)


assert (
    source_settlement_target > 0
).all()


# ============================================================
# 3. SOURCE PAYMENT CLASSIFICATION
# ============================================================

SETTLEMENT_TOLERANCE_BRL = 0.05


source_is_payment = (
    source_payment > 0
)


source_is_settlement = (
    source_is_payment
    &
    (
        source_payment
        >=
        (
            source_settlement_target
            -
            SETTLEMENT_TOLERANCE_BRL
        )
    )
)


source_is_partial = (
    source_is_payment
    &
    ~source_is_settlement
)


assert np.array_equal(
    source_is_payment,
    source_is_partial | source_is_settlement,
)

assert not np.any(
    source_is_partial
    &
    source_is_settlement
)


print("\nSOURCE PAYMENT CLASSIFICATION")

source_classification = pd.DataFrame(
    {
        "events": [
            source_is_payment.sum(),
            source_is_partial.sum(),
            source_is_settlement.sum(),
        ]
    },
    index=[
        "payment",
        "partial_payment",
        "settlement_full",
    ],
)

source_classification["pct_payment_events"] = (
    source_classification["events"]
    /
    source_is_payment.sum()
    * 100
)

display(source_classification)


# Expected from 04A

assert source_is_payment.sum() == 5_626
assert source_is_settlement.sum() == 3_641
assert source_is_partial.sum() == 1_985

print(
    "Source payment classification reconciles with 04A: PASSED"
)


# ============================================================
# 4. MATURITY TIMESTAMP
# ============================================================
#
# Outcome availability timestamp:
#
#       source send + 72 hours
#
# Only payment events need to enter the payment ledger.
#
# ============================================================

source_maturity_at = (
    source_sent_at
    +
    np.timedelta64(72, "h")
)


# ============================================================
# 5. BUILD MINIMAL PAYMENT LEDGER
# ============================================================
#
# One row per POSITIVE payment event.
#
# This table is intentionally narrow.
#
# ============================================================

payment_idx = np.flatnonzero(
    source_is_payment
)


ledger = pd.DataFrame(
    {
        "customer_id":
            features["customer_id"]
            .to_numpy(copy=False)[payment_idx],

        "source_event_number":
            features["event_number"]
            .to_numpy(copy=False)[payment_idx],

        "source_sent_at":
            source_sent_at[payment_idx],

        "maturity_at":
            source_maturity_at[payment_idx],

        "payment_brl":
            source_payment[payment_idx],

        "settlement_target_brl":
            source_settlement_target[payment_idx],

        "is_partial":
            source_is_partial[payment_idx]
            .astype("int8"),

        "is_settlement":
            source_is_settlement[payment_idx]
            .astype("int8"),

        "source_template":
            source_template[payment_idx],
    }
)


ledger["payment_fraction_of_target"] = (
    ledger["payment_brl"]
    /
    ledger["settlement_target_brl"]
)


assert len(ledger) == 5_626
assert ledger["payment_brl"].gt(0).all()

assert (
    ledger["is_partial"]
    +
    ledger["is_settlement"]
    ==
    1
).all()


# ============================================================
# 6. OUTPUT ARRAYS
# ============================================================
#
# All arrays below represent information available BEFORE /
# AT current decision time.
#
# ============================================================

n = len(features)


hist_n_payments = np.zeros(
    n,
    dtype=np.int16,
)

hist_n_partial = np.zeros(
    n,
    dtype=np.int16,
)

hist_n_settlement = np.zeros(
    n,
    dtype=np.int16,
)


hist_total_paid = np.zeros(
    n,
    dtype=float,
)

hist_total_partial_paid = np.zeros(
    n,
    dtype=float,
)

hist_total_settlement_paid = np.zeros(
    n,
    dtype=float,
)


hist_last_payment_amount = np.full(
    n,
    np.nan,
)

hist_last_payment_fraction_target = np.full(
    n,
    np.nan,
)

hist_last_payment_maturity_at = np.full(
    n,
    np.datetime64("NaT"),
    dtype="datetime64[ns]",
)

hist_last_payment_source_sent_at = np.full(
    n,
    np.datetime64("NaT"),
    dtype="datetime64[ns]",
)


hist_last_partial_amount = np.full(
    n,
    np.nan,
)

hist_last_partial_maturity_at = np.full(
    n,
    np.datetime64("NaT"),
    dtype="datetime64[ns]",
)


hist_last_settlement_amount = np.full(
    n,
    np.nan,
)

hist_last_settlement_maturity_at = np.full(
    n,
    np.datetime64("NaT"),
    dtype="datetime64[ns]",
)


hist_max_payment_amount = np.full(
    n,
    np.nan,
)

hist_min_payment_amount = np.full(
    n,
    np.nan,
)


hist_last_payment_type = np.full(
    n,
    None,
    dtype=object,
)

hist_last_payment_template = np.full(
    n,
    None,
    dtype=object,
)


# ============================================================
# 7. CUSTOMER BOUNDARIES
# ============================================================

customer_np = (
    features["customer_id"]
    .to_numpy(copy=False)
)

decision_time_np = (
    features["sent_at"]
    .to_numpy(
        dtype="datetime64[ns]",
        copy=False,
    )
)


customer_change = np.r_[
    True,
    customer_np[1:] != customer_np[:-1],
]

starts = np.flatnonzero(
    customer_change
)

ends = np.r_[
    starts[1:],
    n,
]


# Payment ledger grouped once for lookup

ledger_groups = {
    customer_id: group
    for customer_id, group
    in ledger.groupby(
        "customer_id",
        sort=False,
    )
}


# ============================================================
# 8. STRICT-PIT LEDGER LOOKUP
# ============================================================
#
# For every decision t:
#
#   k = number of payment signals whose maturity_at <= t
#
# Only ledger rows [:k] are historically available.
#
# Because there are only 5,626 payment events and 11,724
# customers, this remains lightweight.
#
# ============================================================

for start, end in zip(
    starts,
    ends,
):

    customer_id = customer_np[start]

    if customer_id not in ledger_groups:
        continue

    lg = (
        ledger_groups[customer_id]
        .sort_values(
            [
                "maturity_at",
                "source_event_number",
            ]
        )
        .reset_index(drop=True)
    )


    maturity = (
        lg["maturity_at"]
        .to_numpy(
            dtype="datetime64[ns]",
            copy=False,
        )
    )

    payment = (
        lg["payment_brl"]
        .to_numpy(
            dtype=float,
            copy=False,
        )
    )

    partial = (
        lg["is_partial"]
        .to_numpy(
            dtype=np.int16,
            copy=False,
        )
    )

    settlement = (
        lg["is_settlement"]
        .to_numpy(
            dtype=np.int16,
            copy=False,
        )
    )

    fraction_target = (
        lg["payment_fraction_of_target"]
        .to_numpy(
            dtype=float,
            copy=False,
        )
    )

    source_time = (
        lg["source_sent_at"]
        .to_numpy(
            dtype="datetime64[ns]",
            copy=False,
        )
    )

    templates = (
        lg["source_template"]
        .astype(str)
        .to_numpy(copy=False)
    )


    # --------------------------------------------------------
    # Cumulative ledger statistics
    # --------------------------------------------------------

    cum_n_payment = np.arange(
        1,
        len(lg) + 1,
        dtype=np.int16,
    )

    cum_n_partial = np.cumsum(
        partial,
        dtype=np.int16,
    )

    cum_n_settlement = np.cumsum(
        settlement,
        dtype=np.int16,
    )

    cum_paid = np.cumsum(
        payment,
        dtype=float,
    )

    cum_partial_paid = np.cumsum(
        payment * partial,
        dtype=float,
    )

    cum_settlement_paid = np.cumsum(
        payment * settlement,
        dtype=float,
    )

    cum_max_payment = np.maximum.accumulate(
        payment
    )

    cum_min_payment = np.minimum.accumulate(
        payment
    )


    # --------------------------------------------------------
    # Decision timestamps for this customer
    # --------------------------------------------------------

    decision_times = (
        decision_time_np[start:end]
    )


    # Number of mature payment outcomes available at decision t.
    #
    # side="right":
    # maturity_at == decision_time IS available.
    #
    k = np.searchsorted(
        maturity,
        decision_times,
        side="right",
    )


    has_history = (
        k > 0
    )


    local_rows = np.arange(
        start,
        end,
    )

    target_rows = (
        local_rows[has_history]
    )

    idx = (
        k[has_history] - 1
    )


    # --------------------------------------------------------
    # Counts
    # --------------------------------------------------------

    hist_n_payments[
        target_rows
    ] = cum_n_payment[idx]

    hist_n_partial[
        target_rows
    ] = cum_n_partial[idx]

    hist_n_settlement[
        target_rows
    ] = cum_n_settlement[idx]


    # --------------------------------------------------------
    # Amounts
    # --------------------------------------------------------

    hist_total_paid[
        target_rows
    ] = cum_paid[idx]

    hist_total_partial_paid[
        target_rows
    ] = cum_partial_paid[idx]

    hist_total_settlement_paid[
        target_rows
    ] = cum_settlement_paid[idx]


    # --------------------------------------------------------
    # Last mature payment
    # --------------------------------------------------------

    hist_last_payment_amount[
        target_rows
    ] = payment[idx]

    hist_last_payment_fraction_target[
        target_rows
    ] = fraction_target[idx]

    hist_last_payment_maturity_at[
        target_rows
    ] = maturity[idx]

    hist_last_payment_source_sent_at[
        target_rows
    ] = source_time[idx]

    hist_last_payment_template[
        target_rows
    ] = templates[idx]


    payment_types = np.where(
        settlement[idx] == 1,
        "settlement",
        "partial",
    )

    hist_last_payment_type[
        target_rows
    ] = payment_types


    # --------------------------------------------------------
    # Historical max/min
    # --------------------------------------------------------

    hist_max_payment_amount[
        target_rows
    ] = cum_max_payment[idx]

    hist_min_payment_amount[
        target_rows
    ] = cum_min_payment[idx]


    # --------------------------------------------------------
    # Last PARTIAL mature signal
    # --------------------------------------------------------

    partial_positions = np.flatnonzero(
        partial == 1
    )

    if len(partial_positions):

        partial_maturity = (
            maturity[
                partial_positions
            ]
        )

        kp = np.searchsorted(
            partial_maturity,
            decision_times,
            side="right",
        )

        has_partial = (
            kp > 0
        )

        partial_rows = (
            local_rows[
                has_partial
            ]
        )

        partial_idx = (
            partial_positions[
                kp[has_partial] - 1
            ]
        )

        hist_last_partial_amount[
            partial_rows
        ] = payment[
            partial_idx
        ]

        hist_last_partial_maturity_at[
            partial_rows
        ] = maturity[
            partial_idx
        ]


    # --------------------------------------------------------
    # Last SETTLEMENT mature signal
    # --------------------------------------------------------

    settlement_positions = np.flatnonzero(
        settlement == 1
    )

    if len(settlement_positions):

        settlement_maturity = (
            maturity[
                settlement_positions
            ]
        )

        ks = np.searchsorted(
            settlement_maturity,
            decision_times,
            side="right",
        )

        has_settlement = (
            ks > 0
        )

        settlement_rows = (
            local_rows[
                has_settlement
            ]
        )

        settlement_idx = (
            settlement_positions[
                ks[has_settlement] - 1
            ]
        )

        hist_last_settlement_amount[
            settlement_rows
        ] = payment[
            settlement_idx
        ]

        hist_last_settlement_maturity_at[
            settlement_rows
        ] = maturity[
            settlement_idx
        ]


# ============================================================
# 9. ATTACH CANONICAL JOURNEY FEATURES
# ============================================================

features[
    "hist_journey_n_mature_payments"
] = hist_n_payments

features[
    "hist_journey_n_mature_partial_payments"
] = hist_n_partial

features[
    "hist_journey_n_mature_settlements"
] = hist_n_settlement


features[
    "hist_journey_total_mature_paid_brl"
] = hist_total_paid

features[
    "hist_journey_total_mature_partial_paid_brl"
] = hist_total_partial_paid

features[
    "hist_journey_total_mature_settlement_paid_brl"
] = hist_total_settlement_paid


features[
    "hist_journey_last_mature_payment_amount_brl"
] = hist_last_payment_amount

features[
    "hist_journey_last_mature_payment_fraction_target"
] = hist_last_payment_fraction_target


features[
    "hist_journey_last_mature_payment_at"
] = pd.to_datetime(
    hist_last_payment_maturity_at
)

features[
    "hist_journey_last_payment_source_sent_at"
] = pd.to_datetime(
    hist_last_payment_source_sent_at
)


features[
    "hist_journey_last_mature_partial_amount_brl"
] = hist_last_partial_amount

features[
    "hist_journey_last_mature_partial_at"
] = pd.to_datetime(
    hist_last_partial_maturity_at
)


features[
    "hist_journey_last_mature_settlement_amount_brl"
] = hist_last_settlement_amount

features[
    "hist_journey_last_mature_settlement_at"
] = pd.to_datetime(
    hist_last_settlement_maturity_at
)


features[
    "hist_journey_max_mature_payment_brl"
] = hist_max_payment_amount

features[
    "hist_journey_min_mature_payment_brl"
] = hist_min_payment_amount


features[
    "hist_journey_last_mature_payment_type"
] = hist_last_payment_type

features[
    "hist_journey_last_mature_payment_template"
] = hist_last_payment_template


# ============================================================
# 10. EVER / STATE FLAGS
# ============================================================

features[
    "hist_journey_ever_paid"
] = (
    features[
        "hist_journey_n_mature_payments"
    ] > 0
).astype("int8")


features[
    "hist_journey_ever_partial_paid"
] = (
    features[
        "hist_journey_n_mature_partial_payments"
    ] > 0
).astype("int8")


features[
    "hist_journey_ever_settled"
] = (
    features[
        "hist_journey_n_mature_settlements"
    ] > 0
).astype("int8")


features[
    "hist_journey_partial_without_settlement"
] = (
    features[
        "hist_journey_ever_partial_paid"
    ].eq(1)
    &
    features[
        "hist_journey_ever_settled"
    ].eq(0)
).astype("int8")


features[
    "hist_journey_partial_then_settlement"
] = (
    features[
        "hist_journey_ever_partial_paid"
    ].eq(1)
    &
    features[
        "hist_journey_ever_settled"
    ].eq(1)
).astype("int8")


features[
    "hist_journey_multiple_mature_payments"
] = (
    features[
        "hist_journey_n_mature_payments"
    ] >= 2
).astype("int8")


features[
    "hist_journey_multiple_partial_payments"
] = (
    features[
        "hist_journey_n_mature_partial_payments"
    ] >= 2
).astype("int8")


# ============================================================
# 11. PAYMENT MIX / MAGNITUDE
# ============================================================

n_pay = features[
    "hist_journey_n_mature_payments"
].astype(float)


features[
    "hist_journey_partial_payment_share"
] = np.where(
    n_pay > 0,

    features[
        "hist_journey_n_mature_partial_payments"
    ]
    /
    n_pay,

    np.nan,
)


features[
    "hist_journey_settlement_payment_share"
] = np.where(
    n_pay > 0,

    features[
        "hist_journey_n_mature_settlements"
    ]
    /
    n_pay,

    np.nan,
)


features[
    "hist_journey_avg_mature_payment_brl"
] = np.where(
    n_pay > 0,

    features[
        "hist_journey_total_mature_paid_brl"
    ]
    /
    n_pay,

    np.nan,
)


# ============================================================
# 12. RECENCY — AVAILABILITY CLOCK
# ============================================================
#
# IMPORTANT:
#
# Recency is measured from MATURITY/AVAILABILITY timestamp,
# NOT from source send time.
#
# This is the conservative PIT definition.
#
# ============================================================

features[
    "hist_journey_days_since_last_mature_payment"
] = (
    features["sent_at"]
    -
    features[
        "hist_journey_last_mature_payment_at"
    ]
).dt.total_seconds() / 86400


features[
    "hist_journey_days_since_last_mature_partial"
] = (
    features["sent_at"]
    -
    features[
        "hist_journey_last_mature_partial_at"
    ]
).dt.total_seconds() / 86400


features[
    "hist_journey_days_since_last_mature_settlement"
] = (
    features["sent_at"]
    -
    features[
        "hist_journey_last_mature_settlement_at"
    ]
).dt.total_seconds() / 86400


# ============================================================
# 13. RECENCY FLAGS
# ============================================================

for days in [
    1,
    3,
    7,
    14,
    30,
]:

    features[
        f"hist_journey_mature_payment_within_{days}d"
    ] = (
        features[
            "hist_journey_days_since_last_mature_payment"
        ]
        .le(days)
        &
        features[
            "hist_journey_days_since_last_mature_payment"
        ]
        .notna()
    ).astype("int8")


for days in [
    3,
    7,
    14,
    30,
]:

    features[
        f"hist_journey_mature_partial_within_{days}d"
    ] = (
        features[
            "hist_journey_days_since_last_mature_partial"
        ]
        .le(days)
        &
        features[
            "hist_journey_days_since_last_mature_partial"
        ]
        .notna()
    ).astype("int8")


# ============================================================
# 14. PAYMENT JOURNEY PROFILE
# ============================================================

conditions = [
    features[
        "hist_journey_n_mature_payments"
    ].eq(0),

    features[
        "hist_journey_partial_without_settlement"
    ].eq(1)
    &
    features[
        "hist_journey_n_mature_partial_payments"
    ].eq(1),

    features[
        "hist_journey_partial_without_settlement"
    ].eq(1)
    &
    features[
        "hist_journey_n_mature_partial_payments"
    ].ge(2),

    features[
        "hist_journey_ever_settled"
    ].eq(1)
    &
    features[
        "hist_journey_ever_partial_paid"
    ].eq(0),

    features[
        "hist_journey_partial_then_settlement"
    ].eq(1),
]


choices = [
    "no_mature_payment_history",
    "one_partial_no_settlement",
    "multiple_partials_no_settlement",
    "settlement_without_prior_partial",
    "partial_and_settlement_history",
]


features[
    "hist_journey_profile"
] = np.select(
    conditions,
    choices,
    default="other",
)


# ============================================================
# 15. CURRENT BALANCE RELATIVE TO MATURE PAYMENT HISTORY
# ============================================================
#
# Current balance is known at decision t.
# Mature historical payment total is also known at t.
#
# These are predictive state features, NOT causal quantities.
#
# ============================================================

current_balance = (
    features[
        "outstanding_balance_brl"
    ].astype(float)
)


features[
    "hist_journey_total_paid_to_current_balance_ratio"
] = (
    features[
        "hist_journey_total_mature_paid_brl"
    ]
    /
    current_balance
)


features[
    "hist_journey_last_payment_to_current_balance_ratio"
] = (
    features[
        "hist_journey_last_mature_payment_amount_brl"
    ]
    /
    current_balance
)


# ============================================================
# 16. CURRENT BALANCE VS HISTORICAL PAID VALUE
# ============================================================

features[
    "hist_journey_current_balance_plus_mature_paid_brl"
] = (
    current_balance
    +
    features[
        "hist_journey_total_mature_paid_brl"
    ]
)


features[
    "hist_journey_mature_paid_share_of_balance_plus_paid"
] = np.where(
    features[
        "hist_journey_current_balance_plus_mature_paid_brl"
    ] > 0,

    features[
        "hist_journey_total_mature_paid_brl"
    ]
    /
    features[
        "hist_journey_current_balance_plus_mature_paid_brl"
    ],

    np.nan,
)


# ============================================================
# 17. HARD PIT CONSISTENCY CHECKS
# ============================================================

print("\nRunning hard PIT consistency checks...")


# ------------------------------------------------------------
# 17A. Counts reconcile
# ------------------------------------------------------------

assert (
    features[
        "hist_journey_n_mature_payments"
    ]
    ==
    (
        features[
            "hist_journey_n_mature_partial_payments"
        ]
        +
        features[
            "hist_journey_n_mature_settlements"
        ]
    )
).all()


# ------------------------------------------------------------
# 17B. Amounts reconcile
# ------------------------------------------------------------

assert np.allclose(
    features[
        "hist_journey_total_mature_paid_brl"
    ],
    (
        features[
            "hist_journey_total_mature_partial_paid_brl"
        ]
        +
        features[
            "hist_journey_total_mature_settlement_paid_brl"
        ]
    ),
    atol=1e-8,
)


# ------------------------------------------------------------
# 17C. First contact has no mature payment history
# ------------------------------------------------------------

first_contact = (
    features["event_number"].eq(1)
)


assert (
    features.loc[
        first_contact,
        "hist_journey_n_mature_payments"
    ]
    .eq(0)
    .all()
)


assert (
    features.loc[
        first_contact,
        "hist_journey_total_mature_paid_brl"
    ]
    .eq(0)
    .all()
)


# ------------------------------------------------------------
# 17D. Availability timestamps cannot be in future
# ------------------------------------------------------------

for col in [
    "hist_journey_last_mature_payment_at",
    "hist_journey_last_mature_partial_at",
    "hist_journey_last_mature_settlement_at",
]:

    mask = features[col].notna()

    assert (
        features.loc[
            mask,
            col,
        ]
        <=
        features.loc[
            mask,
            "sent_at",
        ]
    ).all(), col


# ------------------------------------------------------------
# 17E. Source send timestamp must be strictly historical
# ------------------------------------------------------------

mask = features[
    "hist_journey_last_payment_source_sent_at"
].notna()


assert (
    features.loc[
        mask,
        "hist_journey_last_payment_source_sent_at",
    ]
    <
    features.loc[
        mask,
        "sent_at",
    ]
).all()


# ------------------------------------------------------------
# 17F. Strict 72h gap from source event
# ------------------------------------------------------------

mask = (
    features[
        "hist_journey_last_mature_payment_at"
    ].notna()
)


availability_gap_hours = (
    (
        features.loc[
            mask,
            "hist_journey_last_mature_payment_at",
        ]
        -
        features.loc[
            mask,
            "hist_journey_last_payment_source_sent_at",
        ]
    )
    .dt.total_seconds()
    /
    3600
)


assert np.allclose(
    availability_gap_hours,
    72.0,
)


# ------------------------------------------------------------
# 17G. Recencies nonnegative
# ------------------------------------------------------------

for col in [
    "hist_journey_days_since_last_mature_payment",
    "hist_journey_days_since_last_mature_partial",
    "hist_journey_days_since_last_mature_settlement",
]:

    assert (
        features[col]
        .dropna()
        .ge(0)
        .all()
    ), col


# ------------------------------------------------------------
# 17H. Shares bounded
# ------------------------------------------------------------

for col in [
    "hist_journey_partial_payment_share",
    "hist_journey_settlement_payment_share",
    "hist_journey_mature_paid_share_of_balance_plus_paid",
]:

    valid = (
        features[col]
        .dropna()
    )

    assert (
        valid
        .between(
            0,
            1,
            inclusive="both",
        )
        .all()
    ), col


# Partial + settlement shares = 1 when payment history exists

has_payment_history = (
    features[
        "hist_journey_n_mature_payments"
    ] > 0
)


assert np.allclose(
    (
        features.loc[
            has_payment_history,
            "hist_journey_partial_payment_share"
        ]
        +
        features.loc[
            has_payment_history,
            "hist_journey_settlement_payment_share"
        ]
    ),
    1.0,
)


# ------------------------------------------------------------
# 17I. Last payment type consistency
# ------------------------------------------------------------

has_hist = (
    features[
        "hist_journey_n_mature_payments"
    ] > 0
)


assert (
    features.loc[
        has_hist,
        "hist_journey_last_mature_payment_type"
    ]
    .isin(
        [
            "partial",
            "settlement",
        ]
    )
    .all()
)


assert (
    features.loc[
        ~has_hist,
        "hist_journey_last_mature_payment_type"
    ]
    .isna()
    .all()
)


# ------------------------------------------------------------
# 17J. No infinities
# ------------------------------------------------------------

journey_numeric_cols = [
    col
    for col in features.columns
    if (
        col.startswith(
            "hist_journey_"
        )
        and
        pd.api.types.is_numeric_dtype(
            features[col]
        )
    )
]


assert not np.isinf(
    features[
        journey_numeric_cols
    ].to_numpy()
).any()


print(
    "04B hard PIT consistency checks: PASSED"
)


# ============================================================
# 18. CRITICAL LEAKAGE AUDIT
# ============================================================
#
# Find payment source events that have another decision BEFORE
# their 72h maturity.
#
# At that next decision the payment MUST NOT yet appear.
#
# ============================================================

print(
    "\nCRITICAL LEAKAGE AUDIT — "
    "NEXT DECISION BEFORE PAYMENT MATURITY"
)


minimal = features[
    [
        "customer_id",
        "event_number",
        "sent_at",
        "event_paid_within_72h",
        "event_amount_paid_brl",
        "hist_journey_n_mature_payments",
        "hist_journey_total_mature_paid_brl",
    ]
].copy()


minimal[
    "next_sent_at"
] = (
    minimal
    .groupby(
        "customer_id",
        sort=False,
    )["sent_at"]
    .shift(-1)
)


minimal[
    "next_hist_n_mature_payments"
] = (
    minimal
    .groupby(
        "customer_id",
        sort=False,
    )[
        "hist_journey_n_mature_payments"
    ]
    .shift(-1)
)


minimal[
    "next_hist_total_mature_paid_brl"
] = (
    minimal
    .groupby(
        "customer_id",
        sort=False,
    )[
        "hist_journey_total_mature_paid_brl"
    ]
    .shift(-1)
)


minimal[
    "payment_maturity_at"
] = (
    minimal["sent_at"]
    +
    pd.Timedelta(
        hours=72
    )
)


unsafe_next = (
    minimal[
        "event_paid_within_72h"
    ].eq(1)
    &
    minimal[
        "next_sent_at"
    ].notna()
    &
    (
        minimal[
            "next_sent_at"
        ]
        <
        minimal[
            "payment_maturity_at"
        ]
    )
)


print(
    f"Payment events with next decision "
    f"before maturity: "
    f"{unsafe_next.sum():,}"
)


# For a robust audit, compare next-row mature history with
# current-row mature history.
#
# Since current payment is not mature yet, the next decision
# must NOT gain this payment from the current source event.
#
# Other previously mature payments are allowed, so the counts
# should remain unchanged between these two rows unless some
# OTHER prior payment matures in between.
#
# We therefore perform a direct ledger-based sample below,
# rather than making an invalid global equality assumption.


unsafe_examples = (
    minimal.loc[
        unsafe_next
    ]
    .head(15)
)


display(
    unsafe_examples[
        [
            "customer_id",
            "event_number",
            "sent_at",
            "event_amount_paid_brl",
            "payment_maturity_at",
            "next_sent_at",
            "hist_journey_n_mature_payments",
            "next_hist_n_mature_payments",
            "hist_journey_total_mature_paid_brl",
            "next_hist_total_mature_paid_brl",
        ]
    ]
)


# ============================================================
# 19. MATURITY COVERAGE AT DECISION TIME
# ============================================================

print(
    "\nMATURE PAYMENT HISTORY PREVALENCE"
)


history_prevalence = pd.DataFrame(
    {
        "events": [
            features[
                "hist_journey_ever_paid"
            ].sum(),

            features[
                "hist_journey_ever_partial_paid"
            ].sum(),

            features[
                "hist_journey_ever_settled"
            ].sum(),

            features[
                "hist_journey_partial_without_settlement"
            ].sum(),

            features[
                "hist_journey_partial_then_settlement"
            ].sum(),
        ]
    },
    index=[
        "ever_mature_payment",
        "ever_mature_partial",
        "ever_mature_settlement",
        "partial_without_settlement",
        "partial_and_settlement_history",
    ],
)


history_prevalence[
    "pct_events"
] = (
    history_prevalence[
        "events"
    ]
    /
    len(features)
    * 100
)


display(
    history_prevalence
)


# ============================================================
# 20. JOURNEY PROFILE
# ============================================================

print(
    "\nHISTORICAL PAYMENT JOURNEY PROFILE"
)


display(
    features[
        "hist_journey_profile"
    ]
    .value_counts(
        dropna=False
    )
    .to_frame(
        "events"
    )
)


# ============================================================
# 21. MATURE PAYMENT COUNT DISTRIBUTION
# ============================================================

print(
    "\nMATURE PAYMENT COUNT"
)


display(
    features[
        "hist_journey_n_mature_payments"
    ]
    .describe(
        percentiles=[
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame()
)


# ============================================================
# 22. MATURE PAYMENT AMOUNT DISTRIBUTION
# ============================================================

print(
    "\nTOTAL MATURE HISTORICAL PAYMENT — BRL"
)


display(
    features.loc[
        features[
            "hist_journey_ever_paid"
        ].eq(1),
        "hist_journey_total_mature_paid_brl",
    ]
    .describe(
        percentiles=[
            .10,
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame()
)


# ============================================================
# 23. PARTIAL WITHOUT SETTLEMENT — BUSINESS STATE
# ============================================================

partial_open_state = features[
    features[
        "hist_journey_partial_without_settlement"
    ].eq(1)
]


print(
    "\nPARTIAL PAYMENT WITHOUT MATURE SETTLEMENT"
)


print(
    f"Decision states : "
    f"{len(partial_open_state):,}"
)


if len(partial_open_state):

    display(
        partial_open_state[
            [
                "hist_journey_n_mature_partial_payments",
                "hist_journey_total_mature_partial_paid_brl",
                "hist_journey_days_since_last_mature_partial",
                "outstanding_balance_brl",
            ]
        ]
        .describe(
            percentiles=[
                .25,
                .50,
                .75,
                .90,
                .95,
                .99,
            ]
        )
    )


# ============================================================
# 24. SAMPLE PAYMENT JOURNEY
# ============================================================
#
# Prefer a customer with multiple mature payments.
#
# ============================================================

sample_candidates = features[
    features[
        "hist_journey_n_mature_payments"
    ].ge(2)
]


if len(sample_candidates):

    sample_idx = (
        sample_candidates[
            "hist_journey_n_mature_payments"
        ]
        .idxmax()
    )

else:

    sample_idx = (
        features[
            "hist_journey_n_mature_payments"
        ]
        .idxmax()
    )


sample_customer = features.loc[
    sample_idx,
    "customer_id",
]


print(
    f"\nSAMPLE PAYMENT JOURNEY — "
    f"{sample_customer}"
)


display(
    features.loc[
        features[
            "customer_id"
        ].eq(
            sample_customer
        ),
        [
            "customer_id",
            "event_number",
            "sent_at",
            "current_template",
            "outstanding_balance_brl",

            "event_paid_within_72h",
            "event_amount_paid_brl",

            "hist_journey_n_mature_payments",
            "hist_journey_n_mature_partial_payments",
            "hist_journey_n_mature_settlements",

            "hist_journey_total_mature_paid_brl",

            "hist_journey_last_mature_payment_type",
            "hist_journey_last_mature_payment_amount_brl",
            "hist_journey_days_since_last_mature_payment",

            "hist_journey_profile",
        ],
    ]
)


# ============================================================
# 25. FINAL CHECKPOINT
# ============================================================

assert len(features) == rows_before

assert (
    features[
        "customer_id"
    ].nunique()
    ==
    11_724
)

assert features[
    "event_id"
].is_unique


new_features = (
    features.shape[1]
    -
    cols_before
)


print("\n" + "=" * 100)
print("04B — FINAL CHECKPOINT")
print("=" * 100)

print(
    f"Rows          : "
    f"{len(features):,}"
)

print(
    f"Customers     : "
    f"{features['customer_id'].nunique():,}"
)

print(
    f"Columns before: "
    f"{cols_before:,}"
)

print(
    f"Columns after : "
    f"{features.shape[1]:,}"
)

print(
    f"Features added: "
    f"{new_features:,}"
)


print("\nTEMPORAL GOVERNANCE")

print(
    """
04B availability class:
    STRICT PIT — explicit +72h payment maturity

Current event payment:
    NEVER enters its own historical state

Payment becomes historical:
    source sent_at + 72h <= current sent_at

Settlement definition:
    regular templates:
        payment >= balance - R$0.05

    discount_offer:
        payment >= 0.85 × balance - R$0.05

Recency clock:
    measured from outcome AVAILABILITY timestamp,
    not from source-message timestamp

Important:
    historical payment behavior is predictive state.
    It must NOT be interpreted as causal evidence that
    a previous template/message caused the payment.
"""
)


print("=" * 100)
print("04B — CANONICAL HISTORICAL PAYMENT JOURNEY COMPLETE")
print("=" * 100)

04B — CANONICAL HISTORICAL PAYMENT JOURNEY [STRICT PIT +72H]
Starting shape: (75406, 578)

SOURCE PAYMENT CLASSIFICATION


,events,pct_payment_events
payment,5626,100.00
partial_payment,1985,35.28
settlement_full,3641,64.72


Source payment classification reconciles with 04A: PASSED

Running hard PIT consistency checks...
04B hard PIT consistency checks: PASSED

CRITICAL LEAKAGE AUDIT — NEXT DECISION BEFORE PAYMENT MATURITY
Payment events with next decision before maturity: 0


,customer_id,event_number,sent_at,event_amount_paid_brl,payment_maturity_at,next_sent_at,hist_journey_n_mature_payments,next_hist_n_mature_payments,hist_journey_total_mature_paid_brl,next_hist_total_mature_paid_brl



MATURE PAYMENT HISTORY PREVALENCE


...


PARTIAL PAYMENT WITHOUT MATURE SETTLEMENT
Decision states : 6,131


,hist_journey_n_mature_partial_payments,hist_journey_total_mature_partial_paid_brl,hist_journey_days_since_last_mature_partial,outstanding_balance_brl
count,"6,131.00","6,131.00","6,131.00","6,131.00"
mean,1.06,429.04,13.61,394.49
std,0.26,290.42,11.55,273.05
min,1.00,63.57,0.53,51.61
25%,1.00,208.91,4.25,189.68
50%,1.00,357.10,10.18,319.63
75%,1.00,570.33,19.76,525.64
90%,1.00,832.39,31.02,785.97
95%,2.00,"1,041.11",38.10,954.43
99%,2.00,"1,357.80",47.81,"1,340.83"



SAMPLE PAYMENT JOURNEY — C003527


,customer_id,event_number,sent_at,current_template,outstanding_balance_brl,event_paid_within_72h,event_amount_paid_brl,hist_journey_n_mature_payments,hist_journey_n_mature_partial_payments,hist_journey_n_mature_settlements,hist_journey_total_mature_paid_brl,hist_journey_last_mature_payment_type,hist_journey_last_mature_payment_amount_brl,hist_journey_days_since_last_mature_payment,hist_journey_profile
22523,C003527,1,2026-06-17 13:38:00,pix_link,"1,448.84",0,0.00,0,0,0,0.00,None,NaN,NaN,no_mature_payment_history
22524,C003527,2,2026-06-20 20:00:00,friendly_reminder,"1,448.84",0,0.00,0,0,0,0.00,None,NaN,NaN,no_mature_payment_history
22525,C003527,3,2026-06-23 09:30:00,pix_link,"1,448.84",0,0.00,0,0,0,0.00,None,NaN,NaN,no_mature_payment_history
22526,C003527,4,2026-06-24 11:25:00,urgent_reminder,"1,448.84",1,467.01,0,0,0,0.00,None,NaN,NaN,no_mature_payment_history
22527,C003527,5,2026-06-29 15:39:00,urgent_reminder,981.83,0,0.00,1,1,0,467.01,partial,467.01,2.18,one_partial_no_settlement
22528,C003527,6,2026-07-01 10:27:00,friendly_reminder,981.83,1,362.18,1,1,0,467.01,partial,467.01,3.96,one_partial_no_settlement
22529,C003527,7,2026-07-20 10:57:00,pix_link,619.65,0,0.00,2,2,0,829.19,partial,362.18,16.02,multiple_partials_no_settlement
22530,C003527,8,2026-07-21 15:08:00,urgent_reminder,619.65,1,249.19,2,2,0,829.19,partial,362.18,17.20,multiple_partials_no_settlement
22531,C003527,9,2026-07-29 09:26:00,pix_link,370.46,1,228.95,3,3,0,"1,078.38",partial,249.19,4.76,multiple_partials_no_settlement
22532,C003527,10,2026-08-03 11:18:00,urgent_reminder,141.51,0,0.00,4,4,0,"1,307.33",partial,228.95,2.08,multiple_partials_no_settlement



04B — FINAL CHECKPOINT
Rows          : 75,406
Customers     : 11,724
Columns before: 578
Columns after : 623
Features added: 45

TEMPORAL GOVERNANCE

04B availability class:
    STRICT PIT — explicit +72h payment maturity

Current event payment:
    NEVER enters its own historical state

Payment becomes historical:
    source sent_at + 72h <= current sent_at

Settlement definition:
    regular templates:
        payment >= balance - R$0.05

    discount_offer:
        payment >= 0.85 × balance - R$0.05

Recency clock:
    measured from outcome AVAILABILITY timestamp,
    not from source-message timestamp

Important:
    historical payment behavior is predictive state.
    It must NOT be interpreted as causal evidence that
    a previous template/message caused the payment.

04B — CANONICAL HISTORICAL PAYMENT JOURNEY COMPLETE


In [31]:
# ============================================================
# 04C — OBSERVED BALANCE TRAJECTORY [PIT]
# ============================================================
#
# PURPOSE
# -------
# Build the debt/balance trajectory known at each decision t.
#
# TEMPORAL SEMANTICS
# ------------------
# outstanding_balance_brl on the CURRENT row is assumed to be
# known at decision time t.
#
# Therefore it is valid current state.
#
# Historical balance features use only:
#
#       current balance
#       and balances observed at rows <= t
#
# NEVER:
#
#       balance_(t+1)
#       future balance reductions
#       future payment outcomes
#
#
# IMPORTANT TERMINOLOGY
# ---------------------
# "initial_observed_balance" means:
#
#   first balance observed in THIS WhatsApp history
#
# It does NOT necessarily mean:
#
#   original debt at collections entry
#
#
# Starting checkpoint:
#
#       75,406 × 623
#
# ============================================================

import numpy as np
import pandas as pd


print("=" * 100)
print("04C — OBSERVED BALANCE TRAJECTORY [PIT]")
print("=" * 100)


# ============================================================
# 0. CHECKPOINT
# ============================================================

assert features.shape == (75_406, 623)
assert features["customer_id"].nunique() == 11_724
assert features["event_id"].is_unique
assert features["message_id"].is_unique

rows_before = len(features)
cols_before = features.shape[1]

print(f"Starting shape: {features.shape}")


# ============================================================
# 1. BASE ARRAYS / SERIES
# ============================================================

customer = features["customer_id"]

balance = (
    features["outstanding_balance_brl"]
    .astype(float)
)

event_number = features["event_number"]

g_balance = balance.groupby(
    customer,
    sort=False,
)


assert balance.notna().all()
assert balance.gt(0).all()


# ============================================================
# 2. FIRST OBSERVED BALANCE
# ============================================================
#
# This is the first balance observed for the customer inside
# the WhatsApp dataset.
#
# It is NOT necessarily original debt.
#
# ============================================================

features[
    "hist_balance_initial_observed_brl"
] = (
    g_balance.transform("first")
)


# ============================================================
# 3. PREVIOUS OBSERVED BALANCE
# ============================================================
#
# Strictly previous event.
#
# ============================================================

features[
    "hist_balance_previous_observed_brl"
] = (
    g_balance.shift(1)
)


features[
    "hist_balance_has_previous_observation"
] = (
    features[
        "hist_balance_previous_observed_brl"
    ]
    .notna()
    .astype("int8")
)


# ============================================================
# 4. CURRENT BALANCE REDUCTION FROM FIRST OBSERVATION
# ============================================================
#
# Current balance is known at t, therefore:
#
#   first observed balance - current balance
#
# is known at t.
#
# ============================================================

features[
    "hist_balance_reduction_from_initial_brl"
] = (
    features[
        "hist_balance_initial_observed_brl"
    ]
    -
    balance
)


features[
    "hist_balance_reduction_from_initial_pct"
] = (
    features[
        "hist_balance_reduction_from_initial_brl"
    ]
    /
    features[
        "hist_balance_initial_observed_brl"
    ]
)


features[
    "hist_balance_current_to_initial_ratio"
] = (
    balance
    /
    features[
        "hist_balance_initial_observed_brl"
    ]
)


# ============================================================
# 5. CHANGE SINCE PREVIOUS OBSERVATION
# ============================================================
#
# Positive reduction:
#       previous balance > current balance
#
# Zero:
#       unchanged
#
# Negative:
#       balance increased
#
# 04A found no increases, but we do NOT hard-code that fact
# into the feature construction.
#
# ============================================================

features[
    "hist_balance_reduction_since_previous_brl"
] = (
    features[
        "hist_balance_previous_observed_brl"
    ]
    -
    balance
)


features[
    "hist_balance_reduction_since_previous_pct"
] = (
    features[
        "hist_balance_reduction_since_previous_brl"
    ]
    /
    features[
        "hist_balance_previous_observed_brl"
    ]
)


BALANCE_TOLERANCE_BRL = 0.05


features[
    "hist_balance_decreased_since_previous"
] = (
    features[
        "hist_balance_reduction_since_previous_brl"
    ]
    .gt(BALANCE_TOLERANCE_BRL)
    .astype("int8")
)


features[
    "hist_balance_unchanged_since_previous"
] = (
    features[
        "hist_balance_has_previous_observation"
    ].eq(1)
    &
    features[
        "hist_balance_reduction_since_previous_brl"
    ]
    .abs()
    .le(BALANCE_TOLERANCE_BRL)
).astype("int8")


features[
    "hist_balance_increased_since_previous"
] = (
    features[
        "hist_balance_reduction_since_previous_brl"
    ]
    .lt(-BALANCE_TOLERANCE_BRL)
    .astype("int8")
)


# ============================================================
# 6. HISTORICAL BALANCE REDUCTION EVENTS
# ============================================================
#
# IMPORTANT:
#
# A balance reduction becomes observable at the CURRENT event
# because current balance is known.
#
# Example:
#
# event 4:
#     balance = 1000
#
# payment after event 4
#
# event 5:
#     balance = 600
#
# At event 5 we now observe a R$400 reduction.
#
# This is valid PIT state at event 5.
#
# ============================================================

current_reduction_flag = (
    features[
        "hist_balance_decreased_since_previous"
    ]
    .astype("int16")
)


# Count reductions INCLUDING the reduction visible at current t.
features[
    "hist_balance_n_observed_reductions"
] = (
    current_reduction_flag
    .groupby(
        customer,
        sort=False,
    )
    .cumsum()
    .astype("int16")
)


# ============================================================
# 7. PRIOR REDUCTIONS STRICTLY BEFORE CURRENT OBSERVATION
# ============================================================
#
# Useful distinction:
#
# observed_reductions:
#     includes balance movement visible NOW
#
# prior_reductions:
#     only reductions already visible before this row
#
# ============================================================

features[
    "hist_balance_n_prior_reductions"
] = (
    features[
        "hist_balance_n_observed_reductions"
    ]
    -
    current_reduction_flag
).astype("int16")


features[
    "hist_balance_ever_reduced_before_current"
] = (
    features[
        "hist_balance_n_prior_reductions"
    ]
    .gt(0)
    .astype("int8")
)


features[
    "hist_balance_has_reduced_by_current"
] = (
    features[
        "hist_balance_n_observed_reductions"
    ]
    .gt(0)
    .astype("int8")
)


# ============================================================
# 8. CUMULATIVE OBSERVED REDUCTION
# ============================================================
#
# Sum of positive observed balance reductions up through t.
#
# With a monotonic balance this should reconcile with:
#
#   initial_observed_balance - current_balance
#
# We audit rather than assume it.
#
# ============================================================

positive_reduction = (
    features[
        "hist_balance_reduction_since_previous_brl"
    ]
    .fillna(0)
    .clip(lower=0)
)


features[
    "hist_balance_cumulative_observed_reduction_brl"
] = (
    positive_reduction
    .groupby(
        customer,
        sort=False,
    )
    .cumsum()
)


features[
    "hist_balance_cumulative_observed_reduction_pct_initial"
] = (
    features[
        "hist_balance_cumulative_observed_reduction_brl"
    ]
    /
    features[
        "hist_balance_initial_observed_brl"
    ]
)


# ============================================================
# 9. MAX SINGLE OBSERVED REDUCTION SO FAR
# ============================================================

features[
    "hist_balance_max_single_reduction_brl"
] = (
    positive_reduction
    .groupby(
        customer,
        sort=False,
    )
    .cummax()
)


features[
    "hist_balance_max_single_reduction_pct_initial"
] = (
    features[
        "hist_balance_max_single_reduction_brl"
    ]
    /
    features[
        "hist_balance_initial_observed_brl"
    ]
)


# ============================================================
# 10. NUMBER OF UNCHANGED BALANCE TRANSITIONS
# ============================================================

unchanged_flag = (
    features[
        "hist_balance_unchanged_since_previous"
    ]
    .astype("int16")
)


features[
    "hist_balance_n_unchanged_transitions"
] = (
    unchanged_flag
    .groupby(
        customer,
        sort=False,
    )
    .cumsum()
    .astype("int16")
)


# ============================================================
# 11. SHARE OF OBSERVED TRANSITIONS WITH REDUCTION
# ============================================================

n_transitions_observed = (
    event_number - 1
).astype(float)


features[
    "hist_balance_reduction_transition_rate"
] = np.where(
    n_transitions_observed > 0,

    features[
        "hist_balance_n_observed_reductions"
    ]
    /
    n_transitions_observed,

    np.nan,
)


features[
    "hist_balance_unchanged_transition_rate"
] = np.where(
    n_transitions_observed > 0,

    features[
        "hist_balance_n_unchanged_transitions"
    ]
    /
    n_transitions_observed,

    np.nan,
)


# ============================================================
# 12. TIME OF LAST OBSERVED BALANCE REDUCTION
# ============================================================
#
# The reduction becomes known at the CURRENT send timestamp.
#
# Therefore the timestamp attached to the reduction is the
# current decision timestamp where the lower balance is seen.
#
# ============================================================

reduction_observed_at = (
    features["sent_at"]
    .where(
        features[
            "hist_balance_decreased_since_previous"
        ].eq(1)
    )
)


features[
    "hist_balance_last_reduction_observed_at"
] = (
    reduction_observed_at
    .groupby(
        customer,
        sort=False,
    )
    .ffill()
)


features[
    "hist_balance_days_since_last_observed_reduction"
] = (
    features["sent_at"]
    -
    features[
        "hist_balance_last_reduction_observed_at"
    ]
).dt.total_seconds() / 86400


# ============================================================
# 13. BALANCE STAGNATION STREAK
# ============================================================
#
# Number of consecutive unchanged balance transitions ending
# at current event.
#
# Example:
#
# balances:
# 1000, 1000, 1000, 700, 700, 700
#
# streak:
#    0,    1,    2,   0,   1,   2
#
# ============================================================

unchanged_np = (
    features[
        "hist_balance_unchanged_since_previous"
    ]
    .to_numpy(
        dtype=np.int8,
        copy=False,
    )
)

customer_np = (
    features["customer_id"]
    .to_numpy(copy=False)
)

n = len(features)

stagnation_streak = np.zeros(
    n,
    dtype=np.int16,
)


customer_change = np.r_[
    True,
    customer_np[1:] != customer_np[:-1],
]

starts = np.flatnonzero(
    customer_change
)

ends = np.r_[
    starts[1:],
    n,
]


for start, end in zip(starts, ends):

    run = 0

    for i in range(start, end):

        if unchanged_np[i] == 1:
            run += 1
        else:
            run = 0

        stagnation_streak[i] = run


features[
    "hist_balance_consecutive_unchanged_transitions"
] = stagnation_streak


# ============================================================
# 14. STAGNATION FLAGS
# ============================================================

for threshold in [
    2,
    3,
    5,
    8,
    10,
]:

    features[
        f"hist_balance_{threshold}plus_consecutive_unchanged"
    ] = (
        features[
            "hist_balance_consecutive_unchanged_transitions"
        ]
        .ge(threshold)
        .astype("int8")
    )


# ============================================================
# 15. PROGRESS BANDS
# ============================================================
#
# Descriptive state based on proportion of first observed
# balance already removed.
#
# These are NOT causal and do NOT mean probability of recovery.
#
# ============================================================

progress = (
    features[
        "hist_balance_reduction_from_initial_pct"
    ]
    .clip(
        lower=0,
        upper=1,
    )
)


features[
    "hist_balance_progress_band"
] = pd.cut(
    progress,
    bins=[
        -np.inf,
        0.00,
        0.25,
        0.50,
        0.75,
        0.95,
        np.inf,
    ],
    labels=[
        "no_observed_reduction",
        "up_to_25pct",
        "25_to_50pct",
        "50_to_75pct",
        "75_to_95pct",
        "above_95pct",
    ],
    include_lowest=True,
)


# ============================================================
# 16. CURRENT BALANCE LEVEL RELATIVE TO INITIAL
# ============================================================

ratio = (
    features[
        "hist_balance_current_to_initial_ratio"
    ]
)


features[
    "hist_balance_remaining_band"
] = pd.cut(
    ratio,
    bins=[
        -np.inf,
        0.10,
        0.25,
        0.50,
        0.75,
        1.00,
        np.inf,
    ],
    labels=[
        "<=10pct_initial",
        "10_25pct_initial",
        "25_50pct_initial",
        "50_75pct_initial",
        "75_100pct_initial",
        ">100pct_initial",
    ],
    include_lowest=True,
)


# ============================================================
# 17. CURRENT BALANCE VS MATURE PAYMENT JOURNEY
# ============================================================
#
# Connect the 04B payment-memory state with current exposure.
#
# Both quantities are available at t:
#
#   current balance
#   mature historical payment total
#
# ============================================================

mature_paid = (
    features[
        "hist_journey_total_mature_paid_brl"
    ]
)


features[
    "hist_balance_current_to_mature_paid_ratio"
] = np.where(
    mature_paid > 0,

    balance
    /
    mature_paid,

    np.nan,
)


features[
    "hist_balance_mature_paid_minus_current_balance_brl"
] = (
    mature_paid
    -
    balance
)


features[
    "hist_balance_mature_paid_exceeds_current_balance"
] = (
    mature_paid
    .gt(balance)
    .astype("int8")
)


# ============================================================
# 18. PARTIAL-PAYER REMAINING EXPOSURE
# ============================================================

partial_open = (
    features[
        "hist_journey_partial_without_settlement"
    ].eq(1)
)


features[
    "hist_balance_partial_payer_current_balance_brl"
] = np.where(
    partial_open,
    balance,
    np.nan,
)


features[
    "hist_balance_partial_payer_remaining_vs_paid_ratio"
] = np.where(
    partial_open
    &
    mature_paid.gt(0),

    balance
    /
    mature_paid,

    np.nan,
)


features[
    "hist_balance_partial_payer_paid_share"
] = np.where(
    partial_open
    &
    (
        balance
        +
        mature_paid
    ).gt(0),

    mature_paid
    /
    (
        balance
        +
        mature_paid
    ),

    np.nan,
)


# ============================================================
# 19. BALANCE TRAJECTORY PROFILE
# ============================================================

conditions = [
    event_number.eq(1),

    features[
        "hist_balance_n_observed_reductions"
    ].eq(0),

    features[
        "hist_balance_n_observed_reductions"
    ].eq(1),

    features[
        "hist_balance_n_observed_reductions"
    ].ge(2),
]


choices = [
    "first_observation",
    "no_observed_reduction",
    "one_observed_reduction",
    "multiple_observed_reductions",
]


features[
    "hist_balance_trajectory_profile"
] = np.select(
    conditions,
    choices,
    default="other",
)


# ============================================================
# 20. HARD PIT / STRUCTURAL AUDITS
# ============================================================

print("\nRunning 04C hard consistency checks...")


# ------------------------------------------------------------
# 20A. First event
# ------------------------------------------------------------

first_event = (
    features[
        "event_number"
    ].eq(1)
)


assert (
    features.loc[
        first_event,
        "hist_balance_initial_observed_brl"
    ]
    .eq(
        features.loc[
            first_event,
            "outstanding_balance_brl"
        ]
    )
    .all()
)


assert (
    features.loc[
        first_event,
        "hist_balance_previous_observed_brl"
    ]
    .isna()
    .all()
)


assert (
    features.loc[
        first_event,
        "hist_balance_reduction_from_initial_brl"
    ]
    .abs()
    .le(1e-8)
    .all()
)


assert (
    features.loc[
        first_event,
        "hist_balance_n_observed_reductions"
    ]
    .eq(0)
    .all()
)


# ------------------------------------------------------------
# 20B. Current balance cannot exceed initial observed balance
#      beyond tolerance in this dataset
# ------------------------------------------------------------

assert (
    features[
        "hist_balance_reduction_from_initial_brl"
    ]
    >= -BALANCE_TOLERANCE_BRL
).all()


# ------------------------------------------------------------
# 20C. Transition categories reconcile
# ------------------------------------------------------------

has_previous = (
    features[
        "hist_balance_has_previous_observation"
    ].eq(1)
)


transition_class_sum = (
    features[
        "hist_balance_decreased_since_previous"
    ]
    +
    features[
        "hist_balance_unchanged_since_previous"
    ]
    +
    features[
        "hist_balance_increased_since_previous"
    ]
)


assert (
    transition_class_sum.loc[
        has_previous
    ]
    .eq(1)
    .all()
)


assert (
    transition_class_sum.loc[
        ~has_previous
    ]
    .eq(0)
    .all()
)


# ------------------------------------------------------------
# 20D. Reduction count cannot exceed transitions
# ------------------------------------------------------------

assert (
    features[
        "hist_balance_n_observed_reductions"
    ]
    <=
    (
        features[
            "event_number"
        ]
        -
        1
    )
).all()


# ------------------------------------------------------------
# 20E. Prior reduction count
# ------------------------------------------------------------

assert (
    features[
        "hist_balance_n_prior_reductions"
    ]
    >= 0
).all()


assert (
    features[
        "hist_balance_n_prior_reductions"
    ]
    <=
    features[
        "hist_balance_n_observed_reductions"
    ]
).all()


# ------------------------------------------------------------
# 20F. Cumulative observed reduction reconciles with
#      initial observed balance - current balance
#
# 04A found monotonic non-increasing balance.
# ------------------------------------------------------------

reconciliation_gap = (
    features[
        "hist_balance_cumulative_observed_reduction_brl"
    ]
    -
    features[
        "hist_balance_reduction_from_initial_brl"
    ]
)


assert (
    reconciliation_gap
    .abs()
    .le(BALANCE_TOLERANCE_BRL)
    .all()
)


# ------------------------------------------------------------
# 20G. Rates bounded
# ------------------------------------------------------------

for col in [
    "hist_balance_reduction_transition_rate",
    "hist_balance_unchanged_transition_rate",
    "hist_balance_reduction_from_initial_pct",
    "hist_balance_cumulative_observed_reduction_pct_initial",
]:

    valid = (
        features[col]
        .dropna()
    )

    assert (
        valid
        .between(
            -1e-8,
            1 + 1e-8,
            inclusive="both",
        )
        .all()
    ), col


# ------------------------------------------------------------
# 20H. Reduction + unchanged transition rates = 1
#      because 04A showed no balance increases.
# ------------------------------------------------------------

assert np.allclose(
    (
        features.loc[
            has_previous,
            "hist_balance_reduction_transition_rate"
        ]
        +
        features.loc[
            has_previous,
            "hist_balance_unchanged_transition_rate"
        ]
    ),
    1.0,
    atol=1e-8,
)


# ------------------------------------------------------------
# 20I. Reduction timestamp cannot be future
# ------------------------------------------------------------

mask = (
    features[
        "hist_balance_last_reduction_observed_at"
    ].notna()
)


assert (
    features.loc[
        mask,
        "hist_balance_last_reduction_observed_at"
    ]
    <=
    features.loc[
        mask,
        "sent_at"
    ]
).all()


assert (
    features[
        "hist_balance_days_since_last_observed_reduction"
    ]
    .dropna()
    .ge(0)
    .all()
)


# ------------------------------------------------------------
# 20J. Partial-payer share bounded
# ------------------------------------------------------------

valid = (
    features[
        "hist_balance_partial_payer_paid_share"
    ]
    .dropna()
)


assert (
    valid
    .between(
        0,
        1,
        inclusive="both",
    )
    .all()
)


# ------------------------------------------------------------
# 20K. No infinities
# ------------------------------------------------------------

balance_numeric_cols = [
    col
    for col in features.columns
    if (
        col.startswith(
            "hist_balance_"
        )
        and
        pd.api.types.is_numeric_dtype(
            features[col]
        )
    )
]


assert not np.isinf(
    features[
        balance_numeric_cols
    ].to_numpy()
).any()


print("04C hard consistency checks: PASSED")


# ============================================================
# 21. RECONCILIATION WITH 04A
# ============================================================

print("\nRECONCILIATION WITH 04A")


observed_transition_reductions = (
    features[
        "hist_balance_decreased_since_previous"
    ].sum()
)


print(
    f"Observed balance-reduction transitions : "
    f"{observed_transition_reductions:,}"
)

print(
    "Expected from 04A                     : "
    "1,701"
)


assert (
    observed_transition_reductions
    ==
    1_701
)


print(
    "04A / 04C reduction reconciliation: PASSED"
)


# ============================================================
# 22. BALANCE PROGRESS DISTRIBUTION
# ============================================================

print("\nBALANCE PROGRESS FROM FIRST OBSERVATION")


display(
    features[
        "hist_balance_reduction_from_initial_pct"
    ]
    .describe(
        percentiles=[
            .10,
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame()
)


print("\nBALANCE PROGRESS BAND")


display(
    features[
        "hist_balance_progress_band"
    ]
    .value_counts(
        dropna=False
    )
    .to_frame(
        "events"
    )
)


# ============================================================
# 23. BALANCE TRAJECTORY PROFILE
# ============================================================

print("\nBALANCE TRAJECTORY PROFILE")


display(
    features[
        "hist_balance_trajectory_profile"
    ]
    .value_counts(
        dropna=False
    )
    .to_frame(
        "events"
    )
)


# ============================================================
# 24. STAGNATION
# ============================================================

print("\nCONSECUTIVE UNCHANGED BALANCE TRANSITIONS")


display(
    features[
        "hist_balance_consecutive_unchanged_transitions"
    ]
    .describe(
        percentiles=[
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .to_frame()
)


print("\nHIGH BALANCE STAGNATION")


stagnation_summary = pd.DataFrame(
    {
        "events": [
            features[
                "hist_balance_2plus_consecutive_unchanged"
            ].sum(),

            features[
                "hist_balance_3plus_consecutive_unchanged"
            ].sum(),

            features[
                "hist_balance_5plus_consecutive_unchanged"
            ].sum(),

            features[
                "hist_balance_8plus_consecutive_unchanged"
            ].sum(),

            features[
                "hist_balance_10plus_consecutive_unchanged"
            ].sum(),
        ]
    },
    index=[
        "2+ unchanged transitions",
        "3+ unchanged transitions",
        "5+ unchanged transitions",
        "8+ unchanged transitions",
        "10+ unchanged transitions",
    ],
)


stagnation_summary[
    "pct_events"
] = (
    stagnation_summary[
        "events"
    ]
    /
    len(features)
    * 100
)


display(stagnation_summary)


# ============================================================
# 25. PARTIAL PAYER — REMAINING EXPOSURE
# ============================================================

partial_states = features[
    features[
        "hist_journey_partial_without_settlement"
    ].eq(1)
]


print("\nPARTIAL-PAYER REMAINING EXPOSURE")


print(
    f"Decision states : "
    f"{len(partial_states):,}"
)


if len(partial_states):

    display(
        partial_states[
            [
                "outstanding_balance_brl",
                "hist_journey_total_mature_paid_brl",
                "hist_balance_partial_payer_paid_share",
                "hist_balance_partial_payer_remaining_vs_paid_ratio",
                "hist_balance_reduction_from_initial_pct",
            ]
        ]
        .describe(
            percentiles=[
                .10,
                .25,
                .50,
                .75,
                .90,
                .95,
                .99,
            ]
        )
    )


# ============================================================
# 26. SAMPLE BALANCE JOURNEY
# ============================================================
#
# Choose customer with largest number of observed reductions.
#
# ============================================================

sample_idx = (
    features[
        "hist_balance_n_observed_reductions"
    ]
    .idxmax()
)


sample_customer = (
    features.loc[
        sample_idx,
        "customer_id",
    ]
)


print(
    f"\nSAMPLE BALANCE JOURNEY — "
    f"{sample_customer}"
)


display(
    features.loc[
        features[
            "customer_id"
        ].eq(
            sample_customer
        ),
        [
            "customer_id",
            "event_number",
            "sent_at",
            "current_template",

            "outstanding_balance_brl",

            "event_paid_within_72h",
            "event_amount_paid_brl",

            "hist_balance_initial_observed_brl",
            "hist_balance_previous_observed_brl",

            "hist_balance_reduction_since_previous_brl",
            "hist_balance_reduction_from_initial_brl",
            "hist_balance_reduction_from_initial_pct",

            "hist_balance_n_observed_reductions",
            "hist_balance_n_prior_reductions",

            "hist_balance_consecutive_unchanged_transitions",

            "hist_journey_n_mature_payments",
            "hist_journey_total_mature_paid_brl",

            "hist_balance_trajectory_profile",
        ],
    ]
)


# ============================================================
# 27. CURRENT-VS-PREVIOUS PIT AUDIT
# ============================================================
#
# Critical semantic validation:
#
# At event t:
#
# previous balance must equal balance from t-1.
#
# No balance from t+1 is used anywhere.
#
# ============================================================

sample_check = features[
    [
        "customer_id",
        "event_number",
        "outstanding_balance_brl",
        "hist_balance_previous_observed_brl",
    ]
].copy()


expected_previous = (
    sample_check
    .groupby(
        "customer_id",
        sort=False,
    )[
        "outstanding_balance_brl"
    ]
    .shift(1)
)


assert np.allclose(
    sample_check[
        "hist_balance_previous_observed_brl"
    ].fillna(-999999),
    expected_previous.fillna(-999999),
)


print(
    "\nCurrent-vs-previous balance PIT audit: PASSED"
)


# ============================================================
# 28. FINAL CHECKPOINT
# ============================================================

assert len(features) == rows_before

assert (
    features[
        "customer_id"
    ].nunique()
    ==
    11_724
)

assert features[
    "event_id"
].is_unique


new_features = (
    features.shape[1]
    -
    cols_before
)


print("\n" + "=" * 100)
print("04C — FINAL CHECKPOINT")
print("=" * 100)

print(
    f"Rows          : "
    f"{len(features):,}"
)

print(
    f"Customers     : "
    f"{features['customer_id'].nunique():,}"
)

print(
    f"Columns before: "
    f"{cols_before:,}"
)

print(
    f"Columns after : "
    f"{features.shape[1]:,}"
)

print(
    f"Features added: "
    f"{new_features:,}"
)


print("\nTEMPORAL GOVERNANCE")

print(
    """
04C availability class:
    PIT CURRENT STATE + HISTORICAL OBSERVED STATE

Current outstanding_balance_brl:
    known at decision time t

Previous balance:
    strictly from event t-1

Balance reduction at event t:
    previous observed balance - current observed balance
    therefore known at decision t

Initial observed balance:
    first balance seen in WhatsApp history
    NOT necessarily original debt

Future balance:
    NEVER used

Payment outcomes:
    governed separately by 04B +72h maturity

Interpretation:
    balance trajectory describes observed debt evolution.
    It is predictive state, not causal evidence about the
    effectiveness of prior collection actions.
"""
)


print("=" * 100)
print("04C — OBSERVED BALANCE TRAJECTORY COMPLETE")
print("=" * 100)

04C — OBSERVED BALANCE TRAJECTORY [PIT]
Starting shape: (75406, 623)

Running 04C hard consistency checks...
04C hard consistency checks: PASSED

RECONCILIATION WITH 04A
Observed balance-reduction transitions : 1,701
Expected from 04A                     : 1,701
04A / 04C reduction reconciliation: PASSED

BALANCE PROGRESS FROM FIRST OBSERVATION


,hist_balance_reduction_from_initial_pct
count,"75,406.00"
mean,0.04
std,0.15
min,0.00
10%,0.00
25%,0.00
50%,0.00
75%,0.00
90%,0.00
95%,0.46



BALANCE PROGRESS BAND


,events
hist_balance_progress_band,
no_observed_reduction,69275
50_to_75pct,3122
25_to_50pct,2787
75_to_95pct,222
up_to_25pct,0
above_95pct,0



BALANCE TRAJECTORY PROFILE


...


PARTIAL-PAYER REMAINING EXPOSURE
Decision states : 6,131


,outstanding_balance_brl,hist_journey_total_mature_paid_brl,hist_balance_partial_payer_paid_share,hist_balance_partial_payer_remaining_vs_paid_ratio,hist_balance_reduction_from_initial_pct
count,"6,131.00","6,131.00","6,131.00","6,131.00","6,131.00"
mean,394.49,429.04,0.52,1.12,0.52
std,273.05,290.42,0.15,0.72,0.15
min,51.61,63.57,0.25,0.08,0.25
10%,118.82,140.20,0.30,0.39,0.30
25%,189.68,208.91,0.39,0.53,0.39
50%,319.63,357.10,0.52,0.92,0.52
75%,525.64,570.33,0.65,1.55,0.65
90%,785.97,832.39,0.72,2.32,0.72
95%,954.43,"1,041.11",0.75,2.66,0.75



SAMPLE BALANCE JOURNEY — C003527


,customer_id,event_number,sent_at,current_template,outstanding_balance_brl,event_paid_within_72h,event_amount_paid_brl,hist_balance_initial_observed_brl,hist_balance_previous_observed_brl,hist_balance_reduction_since_previous_brl,hist_balance_reduction_from_initial_brl,hist_balance_reduction_from_initial_pct,hist_balance_n_observed_reductions,hist_balance_n_prior_reductions,hist_balance_consecutive_unchanged_transitions,hist_journey_n_mature_payments,hist_journey_total_mature_paid_brl,hist_balance_trajectory_profile
22523,C003527,1,2026-06-17 13:38:00,pix_link,"1,448.84",0,0.00,"1,448.84",NaN,NaN,0.00,0.00,0,0,0,0,0.00,first_observation
22524,C003527,2,2026-06-20 20:00:00,friendly_reminder,"1,448.84",0,0.00,"1,448.84","1,448.84",0.00,0.00,0.00,0,0,1,0,0.00,no_observed_reduction
22525,C003527,3,2026-06-23 09:30:00,pix_link,"1,448.84",0,0.00,"1,448.84","1,448.84",0.00,0.00,0.00,0,0,2,0,0.00,no_observed_reduction
22526,C003527,4,2026-06-24 11:25:00,urgent_reminder,"1,448.84",1,467.01,"1,448.84","1,448.84",0.00,0.00,0.00,0,0,3,0,0.00,no_observed_reduction
22527,C003527,5,2026-06-29 15:39:00,urgent_reminder,981.83,0,0.00,"1,448.84","1,448.84",467.01,467.01,0.32,1,0,0,1,467.01,one_observed_reduction
22528,C003527,6,2026-07-01 10:27:00,friendly_reminder,981.83,1,362.18,"1,448.84",981.83,0.00,467.01,0.32,1,1,1,1,467.01,one_observed_reduction
22529,C003527,7,2026-07-20 10:57:00,pix_link,619.65,0,0.00,"1,448.84",981.83,362.18,829.19,0.57,2,1,0,2,829.19,multiple_observed_reductions
22530,C003527,8,2026-07-21 15:08:00,urgent_reminder,619.65,1,249.19,"1,448.84",619.65,0.00,829.19,0.57,2,2,1,2,829.19,multiple_observed_reductions
22531,C003527,9,2026-07-29 09:26:00,pix_link,370.46,1,228.95,"1,448.84",619.65,249.19,"1,078.38",0.74,3,2,0,3,"1,078.38",multiple_observed_reductions
22532,C003527,10,2026-08-03 11:18:00,urgent_reminder,141.51,0,0.00,"1,448.84",370.46,228.95,"1,307.33",0.90,4,3,0,4,"1,307.33",multiple_observed_reductions



Current-vs-previous balance PIT audit: PASSED

04C — FINAL CHECKPOINT
Rows          : 75,406
Customers     : 11,724
Columns before: 623
Columns after : 662
Features added: 39

TEMPORAL GOVERNANCE

04C availability class:
    PIT CURRENT STATE + HISTORICAL OBSERVED STATE

Current outstanding_balance_brl:
    known at decision time t

Previous balance:
    strictly from event t-1

Balance reduction at event t:
    previous observed balance - current observed balance
    therefore known at decision t

Initial observed balance:
    first balance seen in WhatsApp history
    NOT necessarily original debt

Future balance:
    NEVER used

Payment outcomes:
    governed separately by 04B +72h maturity

Interpretation:
    balance trajectory describes observed debt evolution.
    It is predictive state, not causal evidence about the
    effectiveness of prior collection actions.

04C — OBSERVED BALANCE TRAJECTORY COMPLETE


In [32]:
# ============================================================
# 04D — PAYMENT JOURNEY × BALANCE STATE INTERACTIONS [PIT]
# ============================================================
#
# PURPOSE
# -------
# Combine two already validated PIT feature families:
#
#   04B — mature payment journey (+72h)
#   04C — observed balance trajectory
#
# We want to distinguish states such as:
#
#   - never paid + stagnant balance
#   - partial payer + meaningful progress
#   - partial payer + low remaining balance
#   - multiple partials + advanced recovery
#   - partial payer + long time since last payment
#   - high remaining exposure despite prior payment
#
# IMPORTANT
# ---------
# No new future information is introduced here.
#
# Every input used below was already validated as available
# at decision time t.
#
# Starting checkpoint:
#
#       75,406 × 662
#
# ============================================================

import numpy as np
import pandas as pd


print("=" * 100)
print("04D — PAYMENT JOURNEY × BALANCE STATE INTERACTIONS [PIT]")
print("=" * 100)


# ============================================================
# 0. CHECKPOINT
# ============================================================

assert features.shape == (75_406, 662)
assert features["customer_id"].nunique() == 11_724
assert features["event_id"].is_unique
assert features["message_id"].is_unique

rows_before = len(features)
cols_before = features.shape[1]

print(f"Starting shape: {features.shape}")


# ============================================================
# 1. BASE PIT STATES
# ============================================================

balance = features[
    "outstanding_balance_brl"
].astype(float)

initial_balance = features[
    "hist_balance_initial_observed_brl"
].astype(float)

balance_progress = features[
    "hist_balance_reduction_from_initial_pct"
].astype(float)

balance_remaining_ratio = features[
    "hist_balance_current_to_initial_ratio"
].astype(float)

n_reductions = features[
    "hist_balance_n_observed_reductions"
].astype(int)

stagnation = features[
    "hist_balance_consecutive_unchanged_transitions"
].astype(int)

mature_paid = features[
    "hist_journey_total_mature_paid_brl"
].astype(float)

n_mature_payments = features[
    "hist_journey_n_mature_payments"
].astype(int)

n_partial = features[
    "hist_journey_n_mature_partial_payments"
].astype(int)

days_since_payment = features[
    "hist_journey_days_since_last_mature_payment"
]

days_since_partial = features[
    "hist_journey_days_since_last_mature_partial"
]

has_mature_payment = features[
    "hist_journey_ever_paid"
].eq(1)

has_partial = features[
    "hist_journey_ever_partial_paid"
].eq(1)

partial_open = features[
    "hist_journey_partial_without_settlement"
].eq(1)


# ============================================================
# 2. NEVER-PAID × BALANCE STAGNATION
# ============================================================

features[
    "state_never_paid_balance_unchanged"
] = (
    ~has_mature_payment
    &
    features[
        "hist_balance_n_observed_reductions"
    ].eq(0)
).astype("int8")


for threshold in [
    2,
    3,
    5,
    8,
    10,
]:

    features[
        f"state_never_paid_{threshold}plus_balance_stagnation"
    ] = (
        ~has_mature_payment
        &
        stagnation.ge(threshold)
    ).astype("int8")


# ============================================================
# 3. PARTIAL PAYER × RECOVERY PROGRESS
# ============================================================
#
# These features describe how far through the observed debt
# journey the partial payer already is.
#
# ============================================================

features[
    "state_partial_payer_any_progress"
] = (
    partial_open
    &
    balance_progress.gt(0)
).astype("int8")


features[
    "state_partial_payer_25plus_pct_recovered"
] = (
    partial_open
    &
    balance_progress.ge(0.25)
).astype("int8")


features[
    "state_partial_payer_50plus_pct_recovered"
] = (
    partial_open
    &
    balance_progress.ge(0.50)
).astype("int8")


features[
    "state_partial_payer_75plus_pct_recovered"
] = (
    partial_open
    &
    balance_progress.ge(0.75)
).astype("int8")


features[
    "state_partial_payer_90plus_pct_recovered"
] = (
    partial_open
    &
    balance_progress.ge(0.90)
).astype("int8")


# ============================================================
# 4. PARTIAL PAYER × REMAINING EXPOSURE
# ============================================================

features[
    "state_partial_payer_remaining_le_75pct_initial"
] = (
    partial_open
    &
    balance_remaining_ratio.le(0.75)
).astype("int8")


features[
    "state_partial_payer_remaining_le_50pct_initial"
] = (
    partial_open
    &
    balance_remaining_ratio.le(0.50)
).astype("int8")


features[
    "state_partial_payer_remaining_le_25pct_initial"
] = (
    partial_open
    &
    balance_remaining_ratio.le(0.25)
).astype("int8")


features[
    "state_partial_payer_remaining_le_10pct_initial"
] = (
    partial_open
    &
    balance_remaining_ratio.le(0.10)
).astype("int8")


# ============================================================
# 5. PARTIAL PAYER × ABSOLUTE REMAINING BALANCE
# ============================================================
#
# Absolute exposure can matter independently from relative
# progress.
#
# These are fixed business thresholds, not quantile-derived.
#
# ============================================================

for threshold in [
    100,
    250,
    500,
    750,
    1000,
]:

    features[
        f"state_partial_payer_balance_le_{threshold}_brl"
    ] = (
        partial_open
        &
        balance.le(threshold)
    ).astype("int8")


# ============================================================
# 6. PARTIAL PAYER × PAYMENT FREQUENCY
# ============================================================

features[
    "state_partial_payer_single_mature_payment"
] = (
    partial_open
    &
    n_mature_payments.eq(1)
).astype("int8")


features[
    "state_partial_payer_multiple_mature_payments"
] = (
    partial_open
    &
    n_mature_payments.ge(2)
).astype("int8")


features[
    "state_partial_payer_2plus_partials"
] = (
    partial_open
    &
    n_partial.ge(2)
).astype("int8")


features[
    "state_partial_payer_3plus_partials"
] = (
    partial_open
    &
    n_partial.ge(3)
).astype("int8")


# ============================================================
# 7. MULTIPLE PARTIALS × RECOVERY PROGRESS
# ============================================================

features[
    "state_multiple_partials_50plus_pct_recovered"
] = (
    n_partial.ge(2)
    &
    balance_progress.ge(0.50)
).astype("int8")


features[
    "state_multiple_partials_75plus_pct_recovered"
] = (
    n_partial.ge(2)
    &
    balance_progress.ge(0.75)
).astype("int8")


features[
    "state_multiple_partials_balance_le_500_brl"
] = (
    n_partial.ge(2)
    &
    balance.le(500)
).astype("int8")


features[
    "state_multiple_partials_balance_le_250_brl"
] = (
    n_partial.ge(2)
    &
    balance.le(250)
).astype("int8")


# ============================================================
# 8. PARTIAL PAYER × PAYMENT RECENCY
# ============================================================

for days in [
    3,
    7,
    14,
    30,
]:

    features[
        f"state_partial_payer_last_payment_within_{days}d"
    ] = (
        partial_open
        &
        days_since_payment.notna()
        &
        days_since_payment.le(days)
    ).astype("int8")


features[
    "state_partial_payer_last_payment_gt_7d"
] = (
    partial_open
    &
    days_since_payment.gt(7)
).astype("int8")


features[
    "state_partial_payer_last_payment_gt_14d"
] = (
    partial_open
    &
    days_since_payment.gt(14)
).astype("int8")


features[
    "state_partial_payer_last_payment_gt_30d"
] = (
    partial_open
    &
    days_since_payment.gt(30)
).astype("int8")


# ============================================================
# 9. PARTIAL PAYER × POST-PAYMENT STAGNATION
# ============================================================
#
# This is particularly useful:
#
# customer has demonstrated willingness to pay,
# but balance has subsequently remained unchanged.
#
# ============================================================

for threshold in [
    1,
    2,
    3,
    5,
]:

    features[
        f"state_partial_payer_{threshold}plus_balance_stagnation"
    ] = (
        partial_open
        &
        stagnation.ge(threshold)
    ).astype("int8")


# ============================================================
# 10. NEVER-PAID × HIGH CONTACT / STAGNATION STATE
# ============================================================
#
# Still descriptive/predictive.
#
# DO NOT interpret these as:
#
#       "messages caused failure"
#
# High contact counts are confounded by historical policy.
#
# ============================================================

features[
    "state_never_paid_5plus_prior_messages"
] = (
    ~has_mature_payment
    &
    features[
        "hist_n_previous_messages"
    ].ge(5)
).astype("int8")


features[
    "state_never_paid_8plus_prior_messages"
] = (
    ~has_mature_payment
    &
    features[
        "hist_n_previous_messages"
    ].ge(8)
).astype("int8")


features[
    "state_never_paid_10plus_prior_messages"
] = (
    ~has_mature_payment
    &
    features[
        "hist_n_previous_messages"
    ].ge(10)
).astype("int8")


features[
    "state_never_paid_5plus_msgs_5plus_balance_stagnation"
] = (
    ~has_mature_payment
    &
    features[
        "hist_n_previous_messages"
    ].ge(5)
    &
    stagnation.ge(5)
).astype("int8")


features[
    "state_never_paid_8plus_msgs_8plus_balance_stagnation"
] = (
    ~has_mature_payment
    &
    features[
        "hist_n_previous_messages"
    ].ge(8)
    &
    stagnation.ge(8)
).astype("int8")


# ============================================================
# 11. MONETARY PROGRESS VS CONTACT PRESSURE
# ============================================================
#
# Again: predictive state only.
#
# ============================================================

prior_messages = features[
    "hist_n_previous_messages"
].astype(float)


features[
    "state_observed_reductions_per_prior_message"
] = np.where(
    prior_messages > 0,
    n_reductions / prior_messages,
    np.nan,
)


features[
    "state_recovered_pct_per_prior_message"
] = np.where(
    prior_messages > 0,
    balance_progress / prior_messages,
    np.nan,
)


features[
    "state_mature_paid_brl_per_prior_message"
] = np.where(
    prior_messages > 0,
    mature_paid / prior_messages,
    np.nan,
)


# ============================================================
# 12. PAID VALUE VS REMAINING VALUE
# ============================================================

features[
    "state_mature_paid_to_remaining_balance_ratio"
] = np.where(
    mature_paid > 0,
    mature_paid / balance,
    np.nan,
)


features[
    "state_remaining_balance_to_mature_paid_ratio"
] = np.where(
    mature_paid > 0,
    balance / mature_paid,
    np.nan,
)


features[
    "state_mature_paid_exceeds_remaining_balance"
] = (
    has_mature_payment
    &
    mature_paid.gt(balance)
).astype("int8")


features[
    "state_remaining_balance_exceeds_mature_paid"
] = (
    has_mature_payment
    &
    balance.gt(mature_paid)
).astype("int8")


# ============================================================
# 13. PAYMENT MOMENTUM
# ============================================================
#
# Combines:
#
#   payment recency
#   remaining exposure
#   progress
#
# ============================================================

features[
    "state_recent_partial_high_progress"
] = (
    partial_open
    &
    days_since_partial.le(7)
    &
    balance_progress.ge(0.50)
).astype("int8")


features[
    "state_recent_partial_low_remaining"
] = (
    partial_open
    &
    days_since_partial.le(7)
    &
    balance_remaining_ratio.le(0.50)
).astype("int8")


features[
    "state_stale_partial_high_progress"
] = (
    partial_open
    &
    days_since_partial.gt(14)
    &
    balance_progress.ge(0.50)
).astype("int8")


features[
    "state_stale_partial_low_remaining"
] = (
    partial_open
    &
    days_since_partial.gt(14)
    &
    balance_remaining_ratio.le(0.50)
).astype("int8")


# ============================================================
# 14. RECOVERY DEPTH
# ============================================================
#
# Continuous interaction:
#
#   number of reductions × proportion already recovered
#
# ============================================================

features[
    "state_recovery_depth"
] = (
    n_reductions
    *
    balance_progress
)


features[
    "state_partial_recovery_depth"
] = np.where(
    partial_open,
    n_partial * balance_progress,
    0.0,
)


# ============================================================
# 15. REMAINING EXPOSURE AFTER PAYMENT SIGNAL
# ============================================================

features[
    "state_remaining_balance_after_payment_signal_brl"
] = np.where(
    has_mature_payment,
    balance,
    np.nan,
)


features[
    "state_remaining_balance_after_partial_signal_brl"
] = np.where(
    partial_open,
    balance,
    np.nan,
)


features[
    "state_remaining_balance_share_after_partial"
] = np.where(
    partial_open,
    balance_remaining_ratio,
    np.nan,
)


# ============================================================
# 16. HIGH-LEVEL COLLECTIONS JOURNEY STATE
# ============================================================
#
# Mutually exclusive business-oriented state.
#
# IMPORTANT:
# settlement does not appear historically in this dataset
# because it behaves as terminal outcome.
#
# ============================================================

conditions = [

    # Partial payer, advanced recovery
    partial_open
    &
    balance_progress.ge(0.75),

    # Partial payer, meaningful recovery
    partial_open
    &
    balance_progress.ge(0.50),

    # Partial payer, earlier recovery
    partial_open,

    # Never-paid / no mature payment + strong stagnation
    ~has_mature_payment
    &
    stagnation.ge(5),

    # Never-paid / no mature payment + some history
    ~has_mature_payment
    &
    features[
        "hist_n_previous_messages"
    ].gt(0),

    # First/no prior message state
    ~has_mature_payment
    &
    features[
        "hist_n_previous_messages"
    ].eq(0),
]


choices = [
    "partial_advanced_recovery",
    "partial_mid_recovery",
    "partial_early_recovery",
    "no_payment_high_stagnation",
    "no_payment_contacted",
    "no_payment_first_contact",
]


features[
    "state_collections_journey"
] = np.select(
    conditions,
    choices,
    default="other",
)


# ============================================================
# 17. PARTIAL-PAYER JOURNEY STATE
# ============================================================

conditions = [

    partial_open
    &
    n_partial.ge(2)
    &
    balance_progress.ge(0.75),

    partial_open
    &
    n_partial.ge(2)
    &
    balance_progress.ge(0.50),

    partial_open
    &
    n_partial.ge(2),

    partial_open
    &
    n_partial.eq(1)
    &
    balance_progress.ge(0.75),

    partial_open
    &
    n_partial.eq(1)
    &
    balance_progress.ge(0.50),

    partial_open
    &
    n_partial.eq(1),
]


choices = [
    "multiple_partial_advanced",
    "multiple_partial_mid",
    "multiple_partial_early",
    "single_partial_advanced",
    "single_partial_mid",
    "single_partial_early",
]


features[
    "state_partial_payment_journey"
] = np.select(
    conditions,
    choices,
    default="not_partial_payer",
)


# ============================================================
# 18. HARD CONSISTENCY CHECKS
# ============================================================

print("\nRunning 04D hard consistency checks...")


# ------------------------------------------------------------
# 18A. Partial states must imply mature payment
# ------------------------------------------------------------

partial_state_cols = [
    col
    for col in features.columns
    if col.startswith(
        "state_partial_payer_"
    )
]


for col in partial_state_cols:

    if pd.api.types.is_numeric_dtype(
        features[col]
    ):

        mask = features[col].eq(1)

        assert (
            features.loc[
                mask,
                "hist_journey_ever_paid"
            ]
            .eq(1)
            .all()
        ), col


# ------------------------------------------------------------
# 18B. Multiple partials imply >= 2 partial payments
# ------------------------------------------------------------

mask = features[
    "state_partial_payer_2plus_partials"
].eq(1)

assert (
    features.loc[
        mask,
        "hist_journey_n_mature_partial_payments"
    ]
    .ge(2)
    .all()
)


# ------------------------------------------------------------
# 18C. 75% recovery implies 50% recovery
# ------------------------------------------------------------

assert (
    features[
        "state_partial_payer_75plus_pct_recovered"
    ]
    <=
    features[
        "state_partial_payer_50plus_pct_recovered"
    ]
).all()


# ------------------------------------------------------------
# 18D. 50% recovery implies 25% recovery
# ------------------------------------------------------------

assert (
    features[
        "state_partial_payer_50plus_pct_recovered"
    ]
    <=
    features[
        "state_partial_payer_25plus_pct_recovered"
    ]
).all()


# ------------------------------------------------------------
# 18E. Remaining balance hierarchy
# ------------------------------------------------------------

assert (
    features[
        "state_partial_payer_remaining_le_10pct_initial"
    ]
    <=
    features[
        "state_partial_payer_remaining_le_25pct_initial"
    ]
).all()


assert (
    features[
        "state_partial_payer_remaining_le_25pct_initial"
    ]
    <=
    features[
        "state_partial_payer_remaining_le_50pct_initial"
    ]
).all()


assert (
    features[
        "state_partial_payer_remaining_le_50pct_initial"
    ]
    <=
    features[
        "state_partial_payer_remaining_le_75pct_initial"
    ]
).all()


# ------------------------------------------------------------
# 18F. Never-paid states must have zero mature payments
# ------------------------------------------------------------

never_paid_cols = [
    col
    for col in features.columns
    if col.startswith(
        "state_never_paid_"
    )
]


for col in never_paid_cols:

    mask = features[col].eq(1)

    assert (
        features.loc[
            mask,
            "hist_journey_n_mature_payments"
        ]
        .eq(0)
        .all()
    ), col


# ------------------------------------------------------------
# 18G. Ratio reciprocity
# ------------------------------------------------------------

ratio_mask = (
    features[
        "state_mature_paid_to_remaining_balance_ratio"
    ].notna()
)


ratio_product = (
    features.loc[
        ratio_mask,
        "state_mature_paid_to_remaining_balance_ratio"
    ]
    *
    features.loc[
        ratio_mask,
        "state_remaining_balance_to_mature_paid_ratio"
    ]
)


assert np.allclose(
    ratio_product,
    1.0,
    atol=1e-8,
)


# ------------------------------------------------------------
# 18H. Partial payer state count reconciles with 04B
# ------------------------------------------------------------

assert (
    partial_open.sum()
    ==
    6_131
)


# ------------------------------------------------------------
# 18I. Collections journey must be fully populated
# ------------------------------------------------------------

assert (
    features[
        "state_collections_journey"
    ]
    .ne("other")
    .all()
)


# ------------------------------------------------------------
# 18J. No infinities
# ------------------------------------------------------------

state_numeric_cols = [
    col
    for col in features.columns
    if (
        col.startswith("state_")
        and
        pd.api.types.is_numeric_dtype(
            features[col]
        )
    )
]


assert not np.isinf(
    features[
        state_numeric_cols
    ].to_numpy()
).any()


print("04D hard consistency checks: PASSED")


# ============================================================
# 19. COLLECTIONS JOURNEY DISTRIBUTION
# ============================================================

print("\nCOLLECTIONS JOURNEY STATE")


journey_distribution = (
    features[
        "state_collections_journey"
    ]
    .value_counts()
    .to_frame("events")
)


journey_distribution[
    "pct_events"
] = (
    journey_distribution[
        "events"
    ]
    /
    len(features)
    * 100
)


display(
    journey_distribution
)


# ============================================================
# 20. PARTIAL-PAYER JOURNEY DISTRIBUTION
# ============================================================

print("\nPARTIAL-PAYER JOURNEY")


partial_journey = (
    features.loc[
        partial_open,
        "state_partial_payment_journey"
    ]
    .value_counts()
    .to_frame("events")
)


partial_journey[
    "pct_partial_states"
] = (
    partial_journey[
        "events"
    ]
    /
    partial_open.sum()
    * 100
)


display(
    partial_journey
)


# ============================================================
# 21. PARTIAL-PAYER RECOVERY DEPTH
# ============================================================

print("\nPARTIAL-PAYER RECOVERY DEPTH")


partial_summary = pd.DataFrame(
    {
        "events": [

            features[
                "state_partial_payer_25plus_pct_recovered"
            ].sum(),

            features[
                "state_partial_payer_50plus_pct_recovered"
            ].sum(),

            features[
                "state_partial_payer_75plus_pct_recovered"
            ].sum(),

            features[
                "state_partial_payer_90plus_pct_recovered"
            ].sum(),

            features[
                "state_partial_payer_multiple_mature_payments"
            ].sum(),

            features[
                "state_partial_payer_2plus_partials"
            ].sum(),
        ]
    },
    index=[
        "25%+ recovered",
        "50%+ recovered",
        "75%+ recovered",
        "90%+ recovered",
        "multiple mature payments",
        "2+ mature partials",
    ],
)


partial_summary[
    "pct_partial_states"
] = (
    partial_summary[
        "events"
    ]
    /
    partial_open.sum()
    * 100
)


display(
    partial_summary
)


# ============================================================
# 22. PARTIAL-PAYER RECENCY × PROGRESS
# ============================================================

print("\nPARTIAL-PAYER MOMENTUM")


momentum_summary = pd.DataFrame(
    {
        "events": [

            features[
                "state_recent_partial_high_progress"
            ].sum(),

            features[
                "state_recent_partial_low_remaining"
            ].sum(),

            features[
                "state_stale_partial_high_progress"
            ].sum(),

            features[
                "state_stale_partial_low_remaining"
            ].sum(),
        ]
    },
    index=[
        "recent partial + 50%+ recovered",
        "recent partial + <=50% remaining",
        "stale partial + 50%+ recovered",
        "stale partial + <=50% remaining",
    ],
)


momentum_summary[
    "pct_partial_states"
] = (
    momentum_summary[
        "events"
    ]
    /
    partial_open.sum()
    * 100
)


display(
    momentum_summary
)


# ============================================================
# 23. NEVER-PAID STAGNATION
# ============================================================

print("\nNO MATURE PAYMENT × STAGNATION")


never_paid_summary = pd.DataFrame(
    {
        "events": [

            features[
                "state_never_paid_balance_unchanged"
            ].sum(),

            features[
                "state_never_paid_3plus_balance_stagnation"
            ].sum(),

            features[
                "state_never_paid_5plus_balance_stagnation"
            ].sum(),

            features[
                "state_never_paid_8plus_balance_stagnation"
            ].sum(),

            features[
                "state_never_paid_10plus_balance_stagnation"
            ].sum(),

            features[
                "state_never_paid_8plus_msgs_8plus_balance_stagnation"
            ].sum(),
        ]
    },
    index=[
        "no mature payment + no observed reduction",
        "no mature payment + 3+ stagnation",
        "no mature payment + 5+ stagnation",
        "no mature payment + 8+ stagnation",
        "no mature payment + 10+ stagnation",
        "no mature payment + 8+ msgs + 8+ stagnation",
    ],
)


never_paid_summary[
    "pct_all_events"
] = (
    never_paid_summary[
        "events"
    ]
    /
    len(features)
    * 100
)


display(
    never_paid_summary
)


# ============================================================
# 24. BUSINESS STATE SUMMARY
# ============================================================
#
# Descriptive only.
#
# This does NOT compare causal effectiveness of actions.
#
# ============================================================

print("\nBUSINESS STATE SUMMARY")


business_summary = (
    features
    .groupby(
        "state_collections_journey",
        observed=True,
    )
    .agg(
        events=(
            "event_id",
            "size",
        ),

        customers=(
            "customer_id",
            "nunique",
        ),

        median_dpd=(
            "current_dpd",
            "median",
        ),

        median_balance_brl=(
            "outstanding_balance_brl",
            "median",
        ),

        median_prior_messages=(
            "hist_n_previous_messages",
            "median",
        ),

        median_balance_progress=(
            "hist_balance_reduction_from_initial_pct",
            "median",
        ),

        median_mature_paid_brl=(
            "hist_journey_total_mature_paid_brl",
            "median",
        ),
    )
    .sort_values(
        "events",
        ascending=False,
    )
)


business_summary[
    "pct_events"
] = (
    business_summary[
        "events"
    ]
    /
    len(features)
    * 100
)


display(
    business_summary
)


# ============================================================
# 25. SAMPLE — PARTIAL PAYER WITH ADVANCED RECOVERY
# ============================================================

advanced_partial = features[
    features[
        "state_partial_payer_75plus_pct_recovered"
    ].eq(1)
]


if len(advanced_partial):

    sample_idx = (
        advanced_partial[
            "hist_balance_reduction_from_initial_pct"
        ]
        .idxmax()
    )

    sample_customer = features.loc[
        sample_idx,
        "customer_id",
    ]

    print(
        "\nSAMPLE — ADVANCED PARTIAL-PAYER JOURNEY"
    )

    print(
        f"Customer: {sample_customer}"
    )

    display(
        features.loc[
            features[
                "customer_id"
            ].eq(sample_customer),
            [
                "customer_id",
                "event_number",
                "sent_at",
                "current_template",

                "outstanding_balance_brl",

                "hist_n_previous_messages",

                "hist_journey_n_mature_payments",
                "hist_journey_total_mature_paid_brl",
                "hist_journey_days_since_last_mature_payment",

                "hist_balance_reduction_from_initial_pct",
                "hist_balance_current_to_initial_ratio",
                "hist_balance_consecutive_unchanged_transitions",

                "state_partial_payment_journey",
                "state_collections_journey",
            ],
        ]
    )


# ============================================================
# 26. SAMPLE — NEVER PAID + HIGH STAGNATION
# ============================================================

stagnant_never_paid = features[
    features[
        "state_never_paid_8plus_msgs_8plus_balance_stagnation"
    ].eq(1)
]


if len(stagnant_never_paid):

    sample_idx = (
        stagnant_never_paid[
            "hist_n_previous_messages"
        ]
        .idxmax()
    )

    sample_customer = features.loc[
        sample_idx,
        "customer_id",
    ]

    print(
        "\nSAMPLE — NO MATURE PAYMENT + HIGH STAGNATION"
    )

    print(
        f"Customer: {sample_customer}"
    )

    display(
        features.loc[
            features[
                "customer_id"
            ].eq(sample_customer),
            [
                "customer_id",
                "event_number",
                "sent_at",
                "current_template",

                "current_dpd",
                "outstanding_balance_brl",

                "hist_n_previous_messages",

                "hist_journey_n_mature_payments",
                "hist_journey_total_mature_paid_brl",

                "hist_balance_n_observed_reductions",
                "hist_balance_consecutive_unchanged_transitions",

                "state_collections_journey",
            ],
        ]
    )


# ============================================================
# 27. FINAL CHECKPOINT
# ============================================================

assert len(features) == rows_before

assert (
    features[
        "customer_id"
    ].nunique()
    ==
    11_724
)

assert features[
    "event_id"
].is_unique


new_features = (
    features.shape[1]
    -
    cols_before
)


print("\n" + "=" * 100)
print("04D — FINAL CHECKPOINT")
print("=" * 100)

print(
    f"Rows          : "
    f"{len(features):,}"
)

print(
    f"Customers     : "
    f"{features['customer_id'].nunique():,}"
)

print(
    f"Columns before: "
    f"{cols_before:,}"
)

print(
    f"Columns after : "
    f"{features.shape[1]:,}"
)

print(
    f"Features added: "
    f"{new_features:,}"
)


print("\nTEMPORAL GOVERNANCE")

print(
    """
04D availability class:
    DERIVED PIT STATE

Inputs:
    04B mature payment journey
        explicit +72h availability

    04C observed balance trajectory
        balance known at current decision

No future outcome introduced.

Interpretation:
    These variables describe customer state at decision time.

    They may be strong predictive features.

    They must NOT be interpreted as causal evidence that
    higher contact pressure, a previous template, or a
    previous action caused payment/non-payment.

Important modeling implication:
    contact pressure variables partly encode historical
    collection policy and latent customer difficulty.
"""
)


print("=" * 100)
print("04D — PAYMENT × BALANCE STATE INTERACTIONS COMPLETE")
print("=" * 100)

04D — PAYMENT JOURNEY × BALANCE STATE INTERACTIONS [PIT]
Starting shape: (75406, 662)

Running 04D hard consistency checks...
04D hard consistency checks: PASSED

COLLECTIONS JOURNEY STATE


,events,pct_events
state_collections_journey,,
no_payment_contacted,33599,44.56
no_payment_high_stagnation,23952,31.76
no_payment_first_contact,11724,15.55
partial_mid_recovery,3122,4.14
partial_early_recovery,2787,3.70
partial_advanced_recovery,222,0.29



PARTIAL-PAYER JOURNEY


,events,pct_partial_states
state_partial_payment_journey,,
single_partial_mid,2997,48.88
single_partial_early,2787,45.46
multiple_partial_advanced,222,3.62
multiple_partial_mid,125,2.04



PARTIAL-PAYER RECOVERY DEPTH


...


SAMPLE — ADVANCED PARTIAL-PAYER JOURNEY
Customer: C007087


,customer_id,event_number,sent_at,current_template,outstanding_balance_brl,hist_n_previous_messages,hist_journey_n_mature_payments,hist_journey_total_mature_paid_brl,hist_journey_days_since_last_mature_payment,hist_balance_reduction_from_initial_pct,hist_balance_current_to_initial_ratio,hist_balance_consecutive_unchanged_transitions,state_partial_payment_journey,state_collections_journey
44961,C007087,1,2026-06-23 10:40:00,pix_link,"1,059.28",0,0,0.00,NaN,0.00,1.00,0,not_partial_payer,no_payment_first_contact
44962,C007087,2,2026-06-26 17:24:00,pix_link,"1,059.28",1,0,0.00,NaN,0.00,1.00,1,not_partial_payer,no_payment_contacted
44963,C007087,3,2026-06-29 11:39:00,friendly_reminder,"1,059.28",2,0,0.00,NaN,0.00,1.00,2,not_partial_payer,no_payment_contacted
44964,C007087,4,2026-06-30 16:56:00,pix_link,"1,059.28",3,0,0.00,NaN,0.00,1.00,3,not_partial_payer,no_payment_contacted
44965,C007087,5,2026-07-06 14:17:00,pix_link,"1,059.28",4,0,0.00,NaN,0.00,1.00,4,not_partial_payer,no_payment_contacted
44966,C007087,6,2026-07-28 14:17:00,urgent_reminder,"1,059.28",5,0,0.00,NaN,0.00,1.00,5,not_partial_payer,no_payment_high_stagnation
44967,C007087,7,2026-08-02 11:57:00,urgent_reminder,281.73,6,1,777.55,1.90,0.73,0.27,0,single_partial_mid,partial_mid_recovery
44968,C007087,8,2026-08-07 11:27:00,urgent_reminder,74.89,7,2,984.39,1.98,0.93,0.07,0,multiple_partial_advanced,partial_advanced_recovery



SAMPLE — NO MATURE PAYMENT + HIGH STAGNATION
Customer: C005915


,customer_id,event_number,sent_at,current_template,current_dpd,outstanding_balance_brl,hist_n_previous_messages,hist_journey_n_mature_payments,hist_journey_total_mature_paid_brl,hist_balance_n_observed_reductions,hist_balance_consecutive_unchanged_transitions,state_collections_journey
37464,C005915,1,2026-06-11 10:52:00,pix_link,5,827.79,0,0,0.00,0,0,no_payment_first_contact
37465,C005915,2,2026-06-12 10:02:00,pix_link,6,827.79,1,0,0.00,0,1,no_payment_contacted
37466,C005915,3,2026-06-15 09:45:00,urgent_reminder,9,827.79,2,0,0.00,0,2,no_payment_contacted
37467,C005915,4,2026-06-17 16:17:00,pix_link,11,827.79,3,0,0.00,0,3,no_payment_contacted
37468,C005915,5,2026-06-19 12:18:00,urgent_reminder,13,827.79,4,0,0.00,0,4,no_payment_contacted
37469,C005915,6,2026-06-21 11:52:00,urgent_reminder,15,827.79,5,0,0.00,0,5,no_payment_high_stagnation
37470,C005915,7,2026-06-22 16:15:00,pix_link,16,827.79,6,0,0.00,0,6,no_payment_high_stagnation
37471,C005915,8,2026-06-25 10:18:00,urgent_reminder,19,827.79,7,0,0.00,0,7,no_payment_high_stagnation
37472,C005915,9,2026-06-29 12:35:00,pix_link,23,827.79,8,0,0.00,0,8,no_payment_high_stagnation
37473,C005915,10,2026-06-30 14:22:00,urgent_reminder,24,827.79,9,0,0.00,0,9,no_payment_high_stagnation



04D — FINAL CHECKPOINT
Rows          : 75,406
Customers     : 11,724
Columns before: 662
Columns after : 724
Features added: 62

TEMPORAL GOVERNANCE

04D availability class:
    DERIVED PIT STATE

Inputs:
    04B mature payment journey
        explicit +72h availability

    04C observed balance trajectory
        balance known at current decision

No future outcome introduced.

Interpretation:
    These variables describe customer state at decision time.

    They may be strong predictive features.

    They must NOT be interpreted as causal evidence that
    higher contact pressure, a previous template, or a
    previous action caused payment/non-payment.

Important modeling implication:
    contact pressure variables partly encode historical
    collection policy and latent customer difficulty.

04D — PAYMENT × BALANCE STATE INTERACTIONS COMPLETE


In [33]:
# ============================================================
# 04E — HISTORICAL TEMPLATE × MATURE PAYMENT RESPONSE
#       [STRICT PIT +72H]
# ============================================================
#
# PURPOSE
# -------
# Reconstruct, at every decision t, the customer's historical
# payment response to EACH WhatsApp template using only
# exposures whose 72h payment outcome is already mature.
#
# Examples at decision t:
#
#   mature pix_link exposures before t
#   mature pix_link payment responses before t
#   historical pix_link payment response rate
#   historical pix_link amount paid
#
# Same for:
#   friendly_reminder
#   urgent_reminder
#   discount_offer
#
#
# CRITICAL TEMPORAL RULE
# ----------------------
#
# Source event j is eligible at decision t iff:
#
#       sent_at_j + 72h <= sent_at_t
#
# Therefore:
#
#   - current action NEVER enters its own history
#   - unmatured prior exposures do NOT enter denominator
#   - future outcomes are NEVER used
#
#
# CAUSAL WARNING
# --------------
# These are historical RESPONSE features.
#
# They do NOT estimate:
#
#       causal template effectiveness
#
# Historical template assignment is confounded by the
# collection policy, DPD, customer difficulty, prior response,
# etc.
#
#
# SPARSITY WARNING
# ----------------
# Raw rates such as:
#
#       1 payment / 1 mature exposure = 100%
#
# are intentionally retained here.
#
# We will NOT shrink them yet.
# Empirical Bayes / smoothing comes later.
#
#
# Starting checkpoint:
#
#       75,406 × 724
#
# ============================================================

import numpy as np
import pandas as pd


print("=" * 105)
print("04E — HISTORICAL TEMPLATE × MATURE PAYMENT RESPONSE [STRICT PIT +72H]")
print("=" * 105)


# ============================================================
# 0. CHECKPOINT
# ============================================================

assert features.shape == (75_406, 724)
assert features["customer_id"].nunique() == 11_724
assert features["event_id"].is_unique
assert features["message_id"].is_unique

rows_before = len(features)
cols_before = features.shape[1]

print(f"Starting shape: {features.shape}")


# ============================================================
# 1. TEMPLATE DEFINITIONS
# ============================================================

templates = [
    "friendly_reminder",
    "urgent_reminder",
    "discount_offer",
    "pix_link",
]


observed_templates = set(
    features["current_template"]
    .dropna()
    .unique()
)


assert observed_templates == set(templates), (
    observed_templates
)


# ============================================================
# 2. MINIMAL SOURCE TABLE
# ============================================================
#
# IMPORTANT FOR MEMORY:
# Do NOT sort/copy the 724-column feature table.
#
# We only need a narrow temporary table containing the fields
# required to reconstruct mature outcomes.
#
# ============================================================

tmp = features[
    [
        "customer_id",
        "event_number",
        "sent_at",
        "current_template",
        "event_paid_within_72h",
        "event_amount_paid_brl",
    ]
].copy()


tmp["maturity_at"] = (
    tmp["sent_at"]
    +
    pd.Timedelta(hours=72)
)


# ============================================================
# 3. SOURCE-EVENT PAYMENT CLASSIFICATION
# ============================================================
#
# Payment response:
#
#       event_paid_within_72h == 1
#
# Amount response:
#
#       event_amount_paid_brl
#
# We are NOT redefining settlement/partial here.
#
# ============================================================

tmp["source_payment_response"] = (
    tmp["event_paid_within_72h"]
    .astype("int8")
)


tmp["source_payment_amount_brl"] = (
    tmp["event_amount_paid_brl"]
    .astype(float)
)


assert (
    tmp["source_payment_response"].eq(
        tmp["source_payment_amount_brl"].gt(0).astype("int8")
    )
).all()


# ============================================================
# 4. CORE PIT ENGINE
# ============================================================
#
# We reconstruct historical mature exposures using
# customer-level searchsorted.
#
# For each customer:
#
#   source events are ordered by maturity_at
#
# At decision time t:
#
#   eligible sources =
#       maturity_at <= sent_at_t
#
#
# IMPORTANT:
# Current event cannot be eligible because:
#
#       sent_at_t + 72h > sent_at_t
#
# ============================================================

n = len(features)

customer_np = tmp["customer_id"].to_numpy(copy=False)
decision_time_np = (
    tmp["sent_at"]
    .to_numpy(dtype="datetime64[ns]")
)

maturity_np = (
    tmp["maturity_at"]
    .to_numpy(dtype="datetime64[ns]")
)

template_np = (
    tmp["current_template"]
    .astype(str)
    .to_numpy(copy=False)
)

payment_np = (
    tmp["source_payment_response"]
    .to_numpy(dtype=np.int8, copy=False)
)

amount_np = (
    tmp["source_payment_amount_brl"]
    .to_numpy(dtype=float, copy=False)
)


# Customer boundaries.
customer_change = np.r_[
    True,
    customer_np[1:] != customer_np[:-1],
]

starts = np.flatnonzero(
    customer_change
)

ends = np.r_[
    starts[1:],
    n,
]


# ============================================================
# 5. OUTPUT ARRAYS
# ============================================================

mature_exposure = {
    template: np.zeros(n, dtype=np.int16)
    for template in templates
}

mature_payment = {
    template: np.zeros(n, dtype=np.int16)
    for template in templates
}

mature_amount = {
    template: np.zeros(n, dtype=float)
    for template in templates
}

days_since_last_mature_exposure = {
    template: np.full(n, np.nan, dtype=float)
    for template in templates
}

days_since_last_mature_payment = {
    template: np.full(n, np.nan, dtype=float)
    for template in templates
}


# ============================================================
# 6. CUSTOMER-LEVEL STRICT PIT RECONSTRUCTION
# ============================================================
#
# Dataset is small enough at customer level for this narrow
# reconstruction and this avoids expanding the full feature
# table.
#
# ============================================================

NS_PER_DAY = 86_400_000_000_000


for start, end in zip(starts, ends):

    decision_times = decision_time_np[start:end]
    source_maturity = maturity_np[start:end]
    source_templates = template_np[start:end]
    source_payments = payment_np[start:end]
    source_amounts = amount_np[start:end]

    # Maturity time preserves source-event ordering because all
    # source events receive the same +72h shift.
    #
    # Number of source events whose outcomes are available at
    # each current decision.
    eligible_count = np.searchsorted(
        source_maturity,
        decision_times,
        side="right",
    )

    for template in templates:

        template_flag = (
            source_templates == template
        ).astype(np.int16)

        template_payment = (
            template_flag
            *
            source_payments
        ).astype(np.int16)

        template_amount = (
            template_flag
            *
            source_amounts
        )

        # Prefix sums with leading zero.
        exposure_cumsum = np.r_[
            0,
            np.cumsum(
                template_flag,
                dtype=np.int32,
            ),
        ]

        payment_cumsum = np.r_[
            0,
            np.cumsum(
                template_payment,
                dtype=np.int32,
            ),
        ]

        amount_cumsum = np.r_[
            0.0,
            np.cumsum(
                template_amount,
                dtype=float,
            ),
        ]

        mature_exposure[template][start:end] = (
            exposure_cumsum[
                eligible_count
            ]
        )

        mature_payment[template][start:end] = (
            payment_cumsum[
                eligible_count
            ]
        )

        mature_amount[template][start:end] = (
            amount_cumsum[
                eligible_count
            ]
        )

        # ----------------------------------------------------
        # Last mature exposure timestamp
        # ----------------------------------------------------
        #
        # Recency clock is based on OUTCOME AVAILABILITY:
        #
        #       maturity_at = sent_at_source + 72h
        #
        # not source sent_at.
        #
        # ----------------------------------------------------

        template_positions = np.flatnonzero(
            template_flag
        )

        if len(template_positions):

            template_maturity = (
                source_maturity[
                    template_positions
                ]
            )

            # Find last template maturity <= current decision.
            pos = np.searchsorted(
                template_maturity,
                decision_times,
                side="right",
            ) - 1

            valid = pos >= 0

            last_maturity = np.full(
                end - start,
                np.datetime64("NaT"),
                dtype="datetime64[ns]",
            )

            last_maturity[valid] = (
                template_maturity[
                    pos[valid]
                ]
            )

            delta = (
                decision_times
                -
                last_maturity
            )

            delta_days = (
                delta.astype(
                    "timedelta64[ns]"
                )
                .astype(np.float64)
                /
                NS_PER_DAY
            )

            delta_days[~valid] = np.nan

            days_since_last_mature_exposure[
                template
            ][start:end] = delta_days

        # ----------------------------------------------------
        # Last mature PAYMENT response timestamp
        # ----------------------------------------------------

        payment_positions = np.flatnonzero(
            template_payment
        )

        if len(payment_positions):

            payment_maturity = (
                source_maturity[
                    payment_positions
                ]
            )

            pos = np.searchsorted(
                payment_maturity,
                decision_times,
                side="right",
            ) - 1

            valid = pos >= 0

            last_payment_maturity = np

04E — HISTORICAL TEMPLATE × MATURE PAYMENT RESPONSE [STRICT PIT +72H]
Starting shape: (75406, 724)


In [34]:
# ============================================================
# RESTORE CHECKPOINT BEFORE 04E
# ============================================================

assert features.shape[0] == 75_406
assert features.shape[1] >= 724

features = features.iloc[:, :724].copy()

assert features.shape == (75_406, 724)

print("04D checkpoint restored:", features.shape)

04D checkpoint restored: (75406, 724)


In [35]:
# ============================================================
# 04E — HISTORICAL TEMPLATE × PAYMENT RESPONSE
#       [STRICT PIT +72H]
# ============================================================
#
# PURPOSE
# -------
# Reconstruct, at every decision t, the customer's historical
# mature payment response to EACH WhatsApp template.
#
# Examples:
#
#   Before current decision t:
#
#       how many mature pix_link exposures existed?
#       how many produced payment within 72h?
#       how much mature attributed payment came from pix_link?
#       what was the raw historical response rate?
#
# And:
#
#       if current action = pix_link,
#       what was this customer's known historical response
#       to pix_link BEFORE the current action?
#
#
# TEMPORAL GOVERNANCE
# -------------------
# Source outcome j is eligible only when:
#
#       sent_at_j + 72h <= sent_at_t
#
# Therefore:
#
#   current event NEVER enters its own history
#   immature prior events NEVER enter response denominators
#   mature no-payment events DO enter denominators
#
#
# CAUSAL WARNING
# --------------
# These are OBSERVATIONAL historical-response features.
#
# They describe:
#
#       "what happened historically after this template
#        for this customer"
#
# They DO NOT estimate:
#
#       "what would happen if I chose this template now"
#
# Historical template assignment is confounded by:
#   DPD
#   customer difficulty
#   previous response
#   historical policy
#   eligibility / positivity constraints
#
#
# SPARSITY
# --------
# Raw response rates are intentionally created here.
#
# DO NOT yet interpret:
#
#       1 payment / 1 mature exposure = true 100% propensity
#
# Shrinkage / Empirical Bayes comes later.
#
#
# Starting checkpoint:
#
#       75,406 × 724
#
# ============================================================

import numpy as np
import pandas as pd


print("=" * 105)
print("04E — HISTORICAL TEMPLATE × PAYMENT RESPONSE [STRICT PIT +72H]")
print("=" * 105)


# ============================================================
# 0. CHECKPOINT
# ============================================================

assert features.shape == (75_406, 724)
assert features["customer_id"].nunique() == 11_724
assert features["event_id"].is_unique
assert features["message_id"].is_unique

rows_before = len(features)
cols_before = features.shape[1]

print(f"Starting shape: {features.shape}")


# ============================================================
# 1. TEMPLATE DEFINITIONS
# ============================================================

TEMPLATES = [
    "friendly_reminder",
    "urgent_reminder",
    "discount_offer",
    "pix_link",
]


observed_templates = set(
    features["current_template"]
    .dropna()
    .unique()
)


assert observed_templates == set(TEMPLATES), (
    observed_templates
)


# ============================================================
# 2. BASE ARRAYS
# ============================================================

customer = features[
    "customer_id"
]

sent_at = features[
    "sent_at"
]

current_template = features[
    "current_template"
]

event_paid = (
    features[
        "event_paid_within_72h"
    ]
    .astype("int8")
)

event_amount = (
    features[
        "event_amount_paid_brl"
    ]
    .astype(float)
)


# Outcome availability timestamp.
outcome_available_at = (
    sent_at
    +
    pd.Timedelta(hours=72)
)


# ============================================================
# 3. MINIMAL TEMPORAL WORK TABLE
# ============================================================
#
# We deliberately do NOT copy the 724-column feature matrix.
#
# We only need:
#
#   customer
#   source event order
#   source template
#   source outcome
#   source outcome availability timestamp
#
# Then we process each customer chronologically.
#
# ============================================================

work = pd.DataFrame(
    {
        "customer_id": customer.to_numpy(copy=False),
        "sent_at": sent_at.to_numpy(copy=False),
        "template": current_template.to_numpy(copy=False),
        "paid": event_paid.to_numpy(copy=False),
        "amount": event_amount.to_numpy(copy=False),
        "available_at": outcome_available_at.to_numpy(copy=False),
    }
)


# Canonical table is already sorted by customer/sent_at.
# Audit rather than sorting the wide feature matrix.

assert work[
    ["customer_id", "sent_at"]
].equals(
    work[
        ["customer_id", "sent_at"]
    ].sort_values(
        ["customer_id", "sent_at"],
        kind="stable",
    )
)


# ============================================================
# 4. OUTPUT ARRAYS
# ============================================================
#
# For each template, at each current decision t:
#
#   mature_exposures
#   mature_payment_events
#   mature_no_payment_events
#   mature_amount_paid
#
# All strictly available before / at t.
#
# ============================================================

n = len(features)
k = len(TEMPLATES)

template_to_idx = {
    template: idx
    for idx, template in enumerate(TEMPLATES)
}


hist_exposures = np.zeros(
    (n, k),
    dtype=np.int16,
)

hist_payments = np.zeros(
    (n, k),
    dtype=np.int16,
)

hist_no_payments = np.zeros(
    (n, k),
    dtype=np.int16,
)

hist_amount = np.zeros(
    (n, k),
    dtype=np.float64,
)


# Last MATURE exposure availability timestamp by template.
last_mature_exposure_available = np.full(
    (n, k),
    np.datetime64("NaT"),
    dtype="datetime64[ns]",
)


# Last MATURE positive payment availability timestamp by template.
last_mature_payment_available = np.full(
    (n, k),
    np.datetime64("NaT"),
    dtype="datetime64[ns]",
)


# ============================================================
# 5. STRICT PIT RECONSTRUCTION
# ============================================================
#
# Algorithm:
#
# For each customer:
#
#   decision pointer = current event t
#   maturity pointer = prior source events whose
#                      sent_at + 72h <= current sent_at
#
# Before recording state for t:
#   add every newly matured source event to historical state
#
# Therefore current event cannot enter own history because:
#
#       sent_at_t + 72h > sent_at_t
#
# ============================================================

customer_np = work[
    "customer_id"
].to_numpy(copy=False)

sent_np = work[
    "sent_at"
].to_numpy(
    dtype="datetime64[ns]",
    copy=False,
)

template_np = work[
    "template"
].to_numpy(copy=False)

paid_np = work[
    "paid"
].to_numpy(
    dtype=np.int8,
    copy=False,
)

amount_np = work[
    "amount"
].to_numpy(
    dtype=float,
    copy=False,
)

available_np = work[
    "available_at"
].to_numpy(
    dtype="datetime64[ns]",
    copy=False,
)


customer_change = np.r_[
    True,
    customer_np[1:] != customer_np[:-1],
]

starts = np.flatnonzero(
    customer_change
)

ends = np.r_[
    starts[1:],
    n,
]


for start, end in zip(starts, ends):

    # Customer-level cumulative mature state.
    exposure_count = np.zeros(
        k,
        dtype=np.int16,
    )

    payment_count = np.zeros(
        k,
        dtype=np.int16,
    )

    no_payment_count = np.zeros(
        k,
        dtype=np.int16,
    )

    amount_sum = np.zeros(
        k,
        dtype=np.float64,
    )

    last_exposure_time = np.full(
        k,
        np.datetime64("NaT"),
        dtype="datetime64[ns]",
    )

    last_payment_time = np.full(
        k,
        np.datetime64("NaT"),
        dtype="datetime64[ns]",
    )

    mature_ptr = start

    for i in range(start, end):

        current_time = sent_np[i]

        # ----------------------------------------------------
        # Add all source events whose outcomes have matured
        # by current decision time.
        # ----------------------------------------------------

        while (
            mature_ptr < i
            and
            available_np[mature_ptr] <= current_time
        ):

            source_template = (
                template_np[mature_ptr]
            )

            template_idx = (
                template_to_idx[
                    source_template
                ]
            )

            exposure_count[
                template_idx
            ] += 1

            last_exposure_time[
                template_idx
            ] = available_np[mature_ptr]

            if paid_np[mature_ptr] == 1:

                payment_count[
                    template_idx
                ] += 1

                amount_sum[
                    template_idx
                ] += amount_np[mature_ptr]

                last_payment_time[
                    template_idx
                ] = available_np[mature_ptr]

            else:

                no_payment_count[
                    template_idx
                ] += 1

            mature_ptr += 1


        # ----------------------------------------------------
        # Record state available at current decision t.
        # ----------------------------------------------------

        hist_exposures[
            i,
            :
        ] = exposure_count

        hist_payments[
            i,
            :
        ] = payment_count

        hist_no_payments[
            i,
            :
        ] = no_payment_count

        hist_amount[
            i,
            :
        ] = amount_sum

        last_mature_exposure_available[
            i,
            :
        ] = last_exposure_time

        last_mature_payment_available[
            i,
            :
        ] = last_payment_time


# ============================================================
# 6. TEMPLATE-SPECIFIC HISTORICAL RESPONSE FEATURES
# ============================================================

for j, template in enumerate(TEMPLATES):

    prefix = (
        f"hist_response_{template}"
    )

    exposures = hist_exposures[:, j]
    payments = hist_payments[:, j]
    no_payments = hist_no_payments[:, j]
    amount = hist_amount[:, j]

    features[
        f"{prefix}_mature_exposures"
    ] = exposures

    features[
        f"{prefix}_mature_payment_events"
    ] = payments

    features[
        f"{prefix}_mature_no_payment_events"
    ] = no_payments

    features[
        f"{prefix}_mature_payment_rate"
    ] = np.where(
        exposures > 0,
        payments / exposures,
        np.nan,
    )

    features[
        f"{prefix}_mature_no_payment_rate"
    ] = np.where(
        exposures > 0,
        no_payments / exposures,
        np.nan,
    )

    features[
        f"{prefix}_mature_amount_paid_brl"
    ] = amount

    features[
        f"{prefix}_avg_amount_per_mature_exposure_brl"
    ] = np.where(
        exposures > 0,
        amount / exposures,
        np.nan,
    )

    features[
        f"{prefix}_avg_amount_per_payment_event_brl"
    ] = np.where(
        payments > 0,
        amount / payments,
        np.nan,
    )

    features[
        f"{prefix}_ever_mature_exposed"
    ] = (
        exposures > 0
    ).astype("int8")

    features[
        f"{prefix}_ever_mature_paid"
    ] = (
        payments > 0
    ).astype("int8")

    # --------------------------------------------------------
    # Recency from AVAILABILITY timestamp, not source send time.
    # --------------------------------------------------------

    last_exp = pd.Series(
        last_mature_exposure_available[:, j],
        index=features.index,
    )

    last_pay = pd.Series(
        last_mature_payment_available[:, j],
        index=features.index,
    )

    features[
        f"{prefix}_days_since_last_mature_exposure"
    ] = (
        sent_at
        -
        last_exp
    ).dt.total_seconds() / 86400

    features[
        f"{prefix}_days_since_last_mature_payment"
    ] = (
        sent_at
        -
        last_pay
    ).dt.total_seconds() / 86400


# ============================================================
# 7. TOTAL TEMPLATE-MATURE HISTORY RECONCILIATION
# ============================================================

total_template_exposures = (
    hist_exposures.sum(axis=1)
)

total_template_payments = (
    hist_payments.sum(axis=1)
)

total_template_no_payments = (
    hist_no_payments.sum(axis=1)
)

total_template_amount = (
    hist_amount.sum(axis=1)
)


features[
    "hist_response_total_template_mature_exposures"
] = total_template_exposures

features[
    "hist_response_total_template_mature_payment_events"
] = total_template_payments

features[
    "hist_response_total_template_mature_no_payment_events"
] = total_template_no_payments

features[
    "hist_response_total_template_mature_amount_paid_brl"
] = total_template_amount


# ============================================================
# 8. CURRENT-TEMPLATE HISTORICAL RESPONSE
# ============================================================
#
# This is a key NBA feature family.
#
# Example:
#
# current_template = pix_link
#
# Feature answers:
#
#   before choosing / sending this pix_link,
#   what mature response history did this customer have
#   to previous pix_link exposures?
#
# IMPORTANT:
# current event itself is excluded by construction.
#
# ============================================================

current_template_idx = np.array(
    [
        template_to_idx[x]
        for x in template_np
    ],
    dtype=np.int8,
)


row_idx = np.arange(
    n
)


current_hist_exposures = (
    hist_exposures[
        row_idx,
        current_template_idx,
    ]
)

current_hist_payments = (
    hist_payments[
        row_idx,
        current_template_idx,
    ]
)

current_hist_no_payments = (
    hist_no_payments[
        row_idx,
        current_template_idx,
    ]
)

current_hist_amount = (
    hist_amount[
        row_idx,
        current_template_idx,
    ]
)


features[
    "hist_current_template_mature_exposures"
] = current_hist_exposures

features[
    "hist_current_template_mature_payment_events"
] = current_hist_payments

features[
    "hist_current_template_mature_no_payment_events"
] = current_hist_no_payments

features[
    "hist_current_template_mature_payment_rate"
] = np.where(
    current_hist_exposures > 0,
    current_hist_payments
    /
    current_hist_exposures,
    np.nan,
)

features[
    "hist_current_template_mature_no_payment_rate"
] = np.where(
    current_hist_exposures > 0,
    current_hist_no_payments
    /
    current_hist_exposures,
    np.nan,
)

features[
    "hist_current_template_mature_amount_paid_brl"
] = current_hist_amount

features[
    "hist_current_template_avg_amount_per_exposure_brl"
] = np.where(
    current_hist_exposures > 0,
    current_hist_amount
    /
    current_hist_exposures,
    np.nan,
)

features[
    "hist_current_template_avg_amount_per_payment_brl"
] = np.where(
    current_hist_payments > 0,
    current_hist_amount
    /
    current_hist_payments,
    np.nan,
)

features[
    "hist_current_template_ever_mature_exposed"
] = (
    current_hist_exposures > 0
).astype("int8")

features[
    "hist_current_template_ever_mature_paid"
] = (
    current_hist_payments > 0
).astype("int8")


# ============================================================
# 9. CURRENT TEMPLATE RESPONSE PROFILE
# ============================================================

conditions = [

    current_hist_exposures == 0,

    (
        (current_hist_exposures == 1)
        &
        (current_hist_payments == 0)
    ),

    (
        (current_hist_exposures == 1)
        &
        (current_hist_payments == 1)
    ),

    (
        (current_hist_exposures >= 2)
        &
        (current_hist_payments == 0)
    ),

    (
        (current_hist_exposures >= 2)
        &
        (current_hist_payments > 0)
        &
        (current_hist_payments < current_hist_exposures)
    ),

    (
        (current_hist_exposures >= 2)
        &
        (current_hist_payments == current_hist_exposures)
    ),
]


choices = [
    "no_mature_history",
    "one_exposure_no_payment",
    "one_exposure_paid",
    "multiple_exposures_no_payment",
    "multiple_exposures_mixed_response",
    "multiple_exposures_all_paid",
]


features[
    "hist_current_template_response_profile"
] = np.select(
    conditions,
    choices,
    default="other",
)


# ============================================================
# 10. CURRENT TEMPLATE REPEATED NON-RESPONSE
# ============================================================
#
# Historical predictive state only.
#
# It does NOT prove that the template is ineffective.
#
# ============================================================

for threshold in [
    1,
    2,
    3,
    5,
]:

    features[
        f"hist_current_template_{threshold}plus_mature_nonresponses"
    ] = (
        current_hist_no_payments >= threshold
    ).astype("int8")


features[
    "hist_current_template_repeated_nonresponse"
] = (
    (current_hist_exposures >= 2)
    &
    (current_hist_payments == 0)
).astype("int8")


# ============================================================
# 11. CUSTOMER TEMPLATE BREADTH — MATURE HISTORY
# ============================================================

template_ever_exposed_matrix = (
    hist_exposures > 0
)

template_ever_paid_matrix = (
    hist_payments > 0
)


features[
    "hist_response_n_templates_mature_exposed"
] = (
    template_ever_exposed_matrix
    .sum(axis=1)
    .astype("int8")
)


features[
    "hist_response_n_templates_mature_paid"
] = (
    template_ever_paid_matrix
    .sum(axis=1)
    .astype("int8")
)


features[
    "hist_response_paid_multiple_templates"
] = (
    features[
        "hist_response_n_templates_mature_paid"
    ]
    .ge(2)
    .astype("int8")
)


# ============================================================
# 12. BEST OBSERVED RAW TEMPLATE RATE
# ============================================================
#
# Descriptive historical signal only.
#
# We intentionally preserve sparse rates here.
# Shrinkage comes later.
#
# ============================================================

raw_rates = np.divide(
    hist_payments,
    hist_exposures,
    out=np.full(
        hist_payments.shape,
        np.nan,
        dtype=float,
    ),
    where=hist_exposures > 0,
)


has_any_template_history = (
    hist_exposures.sum(axis=1) > 0
)


best_rate = np.full(
    n,
    np.nan,
    dtype=float,
)

worst_rate = np.full(
    n,
    np.nan,
    dtype=float,
)


if has_any_template_history.any():

    best_rate[
        has_any_template_history
    ] = np.nanmax(
        raw_rates[
            has_any_template_history
        ],
        axis=1,
    )

    worst_rate[
        has_any_template_history
    ] = np.nanmin(
        raw_rates[
            has_any_template_history
        ],
        axis=1,
    )


features[
    "hist_response_best_raw_template_payment_rate"
] = best_rate

features[
    "hist_response_worst_raw_template_payment_rate"
] = worst_rate

features[
    "hist_response_raw_template_rate_range"
] = (
    best_rate
    -
    worst_rate
)


# ============================================================
# 13. TEMPLATE WITH MOST MATURE EXPOSURES
# ============================================================

most_exposed_idx = (
    hist_exposures.argmax(
        axis=1
    )
)


most_exposed_template = np.array(
    TEMPLATES,
    dtype=object,
)[
    most_exposed_idx
]


features[
    "hist_response_most_exposed_template"
] = np.where(
    total_template_exposures > 0,
    most_exposed_template,
    "no_mature_history",
)


features[
    "hist_response_current_is_most_exposed_template"
] = (
    (
        total_template_exposures > 0
    )
    &
    (
        current_template.to_numpy(
            copy=False
        )
        ==
        most_exposed_template
    )
).astype("int8")


# ============================================================
# 14. TEMPLATE WITH MOST HISTORICAL PAYMENT EVENTS
# ============================================================

most_paid_idx = (
    hist_payments.argmax(
        axis=1
    )
)


most_paid_template = np.array(
    TEMPLATES,
    dtype=object,
)[
    most_paid_idx
]


features[
    "hist_response_most_paid_template"
] = np.where(
    total_template_payments > 0,
    most_paid_template,
    "no_mature_payment_history",
)


features[
    "hist_response_current_is_most_paid_template"
] = (
    (
        total_template_payments > 0
    )
    &
    (
        current_template.to_numpy(
            copy=False
        )
        ==
        most_paid_template
    )
).astype("int8")


# ============================================================
# 15. HARD CONSISTENCY CHECKS
# ============================================================

print("\nRunning 04E hard consistency checks...")


# ------------------------------------------------------------
# 15A. Exposure decomposition
# ------------------------------------------------------------

assert np.array_equal(
    hist_exposures,
    (
        hist_payments
        +
        hist_no_payments
    ),
)


# ------------------------------------------------------------
# 15B. Rates bounded
# ------------------------------------------------------------

for template in TEMPLATES:

    col = (
        f"hist_response_{template}"
        "_mature_payment_rate"
    )

    valid = (
        features[col]
        .dropna()
    )

    assert (
        valid
        .between(
            0,
            1,
            inclusive="both",
        )
        .all()
    )


# ------------------------------------------------------------
# 15C. Current-template response bounded
# ------------------------------------------------------------

valid = (
    features[
        "hist_current_template_mature_payment_rate"
    ]
    .dropna()
)

assert (
    valid
    .between(
        0,
        1,
        inclusive="both",
    )
    .all()
)


# ------------------------------------------------------------
# 15D. Current event cannot enter own history
#
# First events must have zero mature template exposure.
# ------------------------------------------------------------

first_event = (
    features[
        "event_number"
    ].eq(1)
)


assert (
    features.loc[
        first_event,
        "hist_response_total_template_mature_exposures"
    ]
    .eq(0)
    .all()
)


assert (
    features.loc[
        first_event,
        "hist_response_total_template_mature_payment_events"
    ]
    .eq(0)
    .all()
)


# ------------------------------------------------------------
# 15E. Reconcile with canonical mature payment history 03B/04B
# ------------------------------------------------------------

assert np.array_equal(
    total_template_payments,
    features[
        "hist_journey_n_mature_payments"
    ].to_numpy()
)


assert np.allclose(
    total_template_amount,
    features[
        "hist_journey_total_mature_paid_brl"
    ].to_numpy(),
    atol=0.01,
)


# ------------------------------------------------------------
# 15F. Mature exposures should reconcile with 03B maturity
#
# We expect this to match the canonical mature-history count
# built earlier.
# ------------------------------------------------------------

assert np.array_equal(
    total_template_exposures,
    features[
        "hist_n_mature_payment_outcomes"
    ].to_numpy()
)


# ------------------------------------------------------------
# 15G. Current-template extraction must equal matrix lookup
# ------------------------------------------------------------

assert np.array_equal(
    current_hist_exposures,
    hist_exposures[
        row_idx,
        current_template_idx,
    ],
)


assert np.array_equal(
    current_hist_payments,
    hist_payments[
        row_idx,
        current_template_idx,
    ],
)


# ------------------------------------------------------------
# 15H. Recency cannot be negative
# ------------------------------------------------------------

recency_cols = [
    col
    for col in features.columns
    if (
        col.startswith(
            "hist_response_"
        )
        and
        "days_since_last" in col
    )
]


for col in recency_cols:

    assert (
        features[col]
        .dropna()
        .ge(0)
        .all()
    ), col


# ------------------------------------------------------------
# 15I. No mature payment without mature exposure
# ------------------------------------------------------------

assert (
    hist_payments
    <=
    hist_exposures
).all()


# ------------------------------------------------------------
# 15J. No amount without payment event
# ------------------------------------------------------------

assert (
    hist_amount[
        hist_payments == 0
    ]
    ==
    0
).all()


# ------------------------------------------------------------
# 15K. Profile fully populated
# ------------------------------------------------------------

assert (
    features[
        "hist_current_template_response_profile"
    ]
    .ne("other")
    .all()
)


# ------------------------------------------------------------
# 15L. No infinities
# ------------------------------------------------------------

response_numeric_cols = [
    col
    for col in features.columns
    if (
        (
            col.startswith(
                "hist_response_"
            )
            or
            col.startswith(
                "hist_current_template_"
            )
        )
        and
        pd.api.types.is_numeric_dtype(
            features[col]
        )
    )
]


assert not np.isinf(
    features[
        response_numeric_cols
    ].to_numpy()
).any()


print("04E hard consistency checks: PASSED")


# ============================================================
# 16. MATURITY RECONCILIATION
# ============================================================

print("\nMATURITY RECONCILIATION")


print(
    "Maximum mature template exposures at a decision :",
    int(
        total_template_exposures.max()
    )
)

print(
    "Maximum mature payments at a decision          :",
    int(
        total_template_payments.max()
    )
)

print(
    "Decision states with any mature exposure        :",
    f"{(total_template_exposures > 0).sum():,}"
)

print(
    "Decision states with any mature payment         :",
    f"{(total_template_payments > 0).sum():,}"
)


# ============================================================
# 17. CURRENT-TEMPLATE HISTORY COVERAGE
# ============================================================

print("\nCURRENT-TEMPLATE MATURE HISTORY COVERAGE")


current_template_coverage = pd.DataFrame(
    {
        "events": [
            len(features),

            (
                current_hist_exposures
                > 0
            ).sum(),

            (
                current_hist_exposures
                >= 2
            ).sum(),

            (
                current_hist_exposures
                >= 3
            ).sum(),

            (
                current_hist_exposures
                >= 5
            ).sum(),
        ]
    },
    index=[
        "all decisions",
        "1+ mature same-template exposures",
        "2+ mature same-template exposures",
        "3+ mature same-template exposures",
        "5+ mature same-template exposures",
    ],
)


current_template_coverage[
    "pct_decisions"
] = (
    current_template_coverage[
        "events"
    ]
    /
    len(features)
    * 100
)


display(
    current_template_coverage
)


# ============================================================
# 18. CURRENT-TEMPLATE RESPONSE PROFILE
# ============================================================

print("\nCURRENT-TEMPLATE RESPONSE PROFILE")


profile = (
    features[
        "hist_current_template_response_profile"
    ]
    .value_counts()
    .to_frame("events")
)


profile[
    "pct_events"
] = (
    profile[
        "events"
    ]
    /
    len(features)
    * 100
)


display(
    profile
)


# ============================================================
# 19. RAW RESPONSE RATE BY HISTORY DEPTH
# ============================================================
#
# This is specifically to expose sparsity.
#
# ============================================================

print("\nRAW CURRENT-TEMPLATE RESPONSE RATE BY HISTORY DEPTH")


history_depth = pd.cut(
    current_hist_exposures,
    bins=[
        -np.inf,
        0,
        1,
        2,
        4,
        np.inf,
    ],
    labels=[
        "0",
        "1",
        "2",
        "3-4",
        "5+",
    ],
)


rate_depth_summary = pd.DataFrame(
    {
        "history_depth": history_depth,
        "raw_rate": features[
            "hist_current_template_mature_payment_rate"
        ],
    }
)


display(
    rate_depth_summary
    .groupby(
        "history_depth",
        observed=True,
    )
    .agg(
        decision_states=(
            "raw_rate",
            "size",
        ),

        states_with_defined_rate=(
            "raw_rate",
            "count",
        ),

        mean_raw_rate=(
            "raw_rate",
            "mean",
        ),

        median_raw_rate=(
            "raw_rate",
            "median",
        ),
    )
)


# ============================================================
# 20. TEMPLATE-SPECIFIC HISTORICAL COVERAGE
# ============================================================

print("\nTEMPLATE-SPECIFIC MATURE HISTORY")


template_summary_rows = []


for j, template in enumerate(TEMPLATES):

    exposures = hist_exposures[:, j]
    payments = hist_payments[:, j]
    amount = hist_amount[:, j]

    template_summary_rows.append(
        {
            "template": template,

            "decision_states_with_history": int(
                (
                    exposures > 0
                ).sum()
            ),

            "decision_states_with_2plus_exposures": int(
                (
                    exposures >= 2
                ).sum()
            ),

            "decision_states_with_prior_payment": int(
                (
                    payments > 0
                ).sum()
            ),

            "max_mature_exposures": int(
                exposures.max()
            ),

            "max_mature_payments": int(
                payments.max()
            ),

            "total_historical_mature_amount_across_states_brl": (
                amount.sum()
            ),
        }
    )


template_summary = pd.DataFrame(
    template_summary_rows
)


display(
    template_summary
)


# ============================================================
# 21. REPEATED SAME-TEMPLATE NON-RESPONSE
# ============================================================

print("\nCURRENT TEMPLATE — REPEATED MATURE NON-RESPONSE")


nonresponse_summary = pd.DataFrame(
    {
        "events": [

            features[
                "hist_current_template_1plus_mature_nonresponses"
            ].sum(),

            features[
                "hist_current_template_2plus_mature_nonresponses"
            ].sum(),

            features[
                "hist_current_template_3plus_mature_nonresponses"
            ].sum(),

            features[
                "hist_current_template_5plus_mature_nonresponses"
            ].sum(),

            features[
                "hist_current_template_repeated_nonresponse"
            ].sum(),
        ]
    },
    index=[
        "1+ mature same-template nonresponses",
        "2+ mature same-template nonresponses",
        "3+ mature same-template nonresponses",
        "5+ mature same-template nonresponses",
        "2+ exposures and zero payments",
    ],
)


nonresponse_summary[
    "pct_events"
] = (
    nonresponse_summary[
        "events"
    ]
    /
    len(features)
    * 100
)


display(
    nonresponse_summary
)


# ============================================================
# 22. SAMPLE — SAME TEMPLATE HISTORY
# ============================================================
#
# Select a customer/event with:
#
#   >= 3 mature exposures to current template
#
# to visually inspect strict PIT behavior.
#
# ============================================================

sample_candidates = features[
    features[
        "hist_current_template_mature_exposures"
    ].ge(3)
]


if len(sample_candidates):

    sample_idx = (
        sample_candidates[
            "hist_current_template_mature_exposures"
        ]
        .idxmax()
    )

    sample_customer = (
        features.loc[
            sample_idx,
            "customer_id",
        ]
    )

    print(
        "\nSAMPLE — HISTORICAL TEMPLATE RESPONSE"
    )

    print(
        f"Customer: {sample_customer}"
    )


    display(
        features.loc[
            features[
                "customer_id"
            ].eq(sample_customer),
            [
                "customer_id",
                "event_number",
                "sent_at",
                "current_template",

                "event_paid_within_72h",
                "event_amount_paid_brl",

                "hist_n_mature_payment_outcomes",
                "hist_journey_n_mature_payments",

                "hist_current_template_mature_exposures",
                "hist_current_template_mature_payment_events",
                "hist_current_template_mature_no_payment_events",
                "hist_current_template_mature_payment_rate",
                "hist_current_template_mature_amount_paid_brl",
                "hist_current_template_response_profile",
            ],
        ]
    )


# ============================================================
# 23. SAMPLE — CUSTOMER WITH PRIOR TEMPLATE PAYMENT
# ============================================================

payment_history_candidates = features[
    features[
        "hist_current_template_mature_payment_events"
    ].gt(0)
]


if len(payment_history_candidates):

    sample_idx = (
        payment_history_candidates[
            "hist_current_template_mature_payment_events"
        ]
        .idxmax()
    )

    sample_customer = (
        features.loc[
            sample_idx,
            "customer_id",
        ]
    )

    print(
        "\nSAMPLE — PRIOR PAYMENT TO CURRENT TEMPLATE"
    )

    print(
        f"Customer: {sample_customer}"
    )


    display(
        features.loc[
            features[
                "customer_id"
            ].eq(sample_customer),
            [
                "customer_id",
                "event_number",
                "sent_at",
                "current_template",

                "event_paid_within_72h",
                "event_amount_paid_brl",

                "hist_current_template_mature_exposures",
                "hist_current_template_mature_payment_events",
                "hist_current_template_mature_payment_rate",

                "hist_response_friendly_reminder_mature_payment_rate",
                "hist_response_urgent_reminder_mature_payment_rate",
                "hist_response_discount_offer_mature_payment_rate",
                "hist_response_pix_link_mature_payment_rate",

                "hist_response_n_templates_mature_exposed",
                "hist_response_n_templates_mature_paid",
            ],
        ]
    )


# ============================================================
# 24. FINAL CHECKPOINT
# ============================================================

assert len(features) == rows_before
assert features["customer_id"].nunique() == 11_724
assert features["event_id"].is_unique


new_features = (
    features.shape[1]
    -
    cols_before
)


print("\n" + "=" * 105)
print("04E — FINAL CHECKPOINT")
print("=" * 105)

print(
    f"Rows          : "
    f"{len(features):,}"
)

print(
    f"Customers     : "
    f"{features['customer_id'].nunique():,}"
)

print(
    f"Columns before: "
    f"{cols_before:,}"
)

print(
    f"Columns after : "
    f"{features.shape[1]:,}"
)

print(
    f"Features added: "
    f"{new_features:,}"
)


print("\nTEMPORAL GOVERNANCE")

print(
    """
04E availability class:
    STRICT PIT HISTORICAL RESPONSE

Outcome eligibility:
    source sent_at + 72h <= current sent_at

Denominator:
    only MATURE historical exposures

Numerator:
    only MATURE historical paid_within_72h outcomes

Current event:
    NEVER enters its own history

Immature prior event:
    NEVER enters denominator or numerator

Template response rates:
    OBSERVATIONAL historical response
    NOT causal template effectiveness

Current-template features:
    describe what was known about the customer's prior
    response to the action/template being used at t

Sparsity:
    raw rates intentionally retained
    shrinkage / Empirical Bayes deferred to later layer

Historical policy:
    template assignment is confounded and subject to
    positivity / eligibility constraints
"""
)


print("=" * 105)
print("04E — HISTORICAL TEMPLATE × PAYMENT RESPONSE COMPLETE")
print("=" * 105)

04E — HISTORICAL TEMPLATE × PAYMENT RESPONSE [STRICT PIT +72H]
Starting shape: (75406, 724)

Running 04E hard consistency checks...
04E hard consistency checks: PASSED

MATURITY RECONCILIATION
Maximum mature template exposures at a decision : 19
Maximum mature payments at a decision          : 4
Decision states with any mature exposure        : 55,707
Decision states with any mature payment         : 6,131

CURRENT-TEMPLATE MATURE HISTORY COVERAGE


,events,pct_decisions
all decisions,75406,100.00
1+ mature same-template exposures,34584,45.86
2+ mature same-template exposures,17803,23.61
3+ mature same-template exposures,8208,10.89
5+ mature same-template exposures,1168,1.55



CURRENT-TEMPLATE RESPONSE PROFILE


,events,pct_events
hist_current_template_response_profile,,
no_mature_history,40822,54.14
multiple_exposures_no_payment,16741,22.20
one_exposure_no_payment,16098,21.35
multiple_exposures_mixed_response,1045,1.39
one_exposure_paid,683,0.91
multiple_exposures_all_paid,17,0.02



RAW CURRENT-TEMPLATE RESPONSE RATE BY HISTORY DEPTH


...


SAMPLE — HISTORICAL TEMPLATE RESPONSE
Customer: C005915


,customer_id,event_number,sent_at,current_template,event_paid_within_72h,event_amount_paid_brl,hist_n_mature_payment_outcomes,hist_journey_n_mature_payments,hist_current_template_mature_exposures,hist_current_template_mature_payment_events,hist_current_template_mature_no_payment_events,hist_current_template_mature_payment_rate,hist_current_template_mature_amount_paid_brl,hist_current_template_response_profile
37464,C005915,1,2026-06-11 10:52:00,pix_link,0,0.00,0,0,0,0,0,NaN,0.00,no_mature_history
37465,C005915,2,2026-06-12 10:02:00,pix_link,0,0.00,0,0,0,0,0,NaN,0.00,no_mature_history
37466,C005915,3,2026-06-15 09:45:00,urgent_reminder,0,0.00,1,0,0,0,0,NaN,0.00,no_mature_history
37467,C005915,4,2026-06-17 16:17:00,pix_link,0,0.00,2,0,2,0,2,0.00,0.00,multiple_exposures_no_payment
37468,C005915,5,2026-06-19 12:18:00,urgent_reminder,0,0.00,3,0,1,0,1,0.00,0.00,one_exposure_no_payment
37469,C005915,6,2026-06-21 11:52:00,urgent_reminder,0,0.00,4,0,1,0,1,0.00,0.00,one_exposure_no_payment
37470,C005915,7,2026-06-22 16:15:00,pix_link,0,0.00,5,0,3,0,3,0.00,0.00,multiple_exposures_no_payment
37471,C005915,8,2026-06-25 10:18:00,urgent_reminder,0,0.00,6,0,3,0,3,0.00,0.00,multiple_exposures_no_payment
37472,C005915,9,2026-06-29 12:35:00,pix_link,0,0.00,8,0,4,0,4,0.00,0.00,multiple_exposures_no_payment
37473,C005915,10,2026-06-30 14:22:00,urgent_reminder,0,0.00,8,0,4,0,4,0.00,0.00,multiple_exposures_no_payment



SAMPLE — PRIOR PAYMENT TO CURRENT TEMPLATE
Customer: C000165


,customer_id,event_number,sent_at,current_template,event_paid_within_72h,event_amount_paid_brl,hist_current_template_mature_exposures,hist_current_template_mature_payment_events,hist_current_template_mature_payment_rate,hist_response_friendly_reminder_mature_payment_rate,hist_response_urgent_reminder_mature_payment_rate,hist_response_discount_offer_mature_payment_rate,hist_response_pix_link_mature_payment_rate,hist_response_n_templates_mature_exposed,hist_response_n_templates_mature_paid
990,C000165,1,2026-06-24 14:15:00,friendly_reminder,0,0.00,0,0,NaN,NaN,NaN,NaN,NaN,0,0
991,C000165,2,2026-06-25 16:05:00,urgent_reminder,1,"1,251.71",0,0,NaN,NaN,NaN,NaN,NaN,0,0
992,C000165,3,2026-06-30 10:03:00,urgent_reminder,1,388.59,1,1,1.00,0.00,1.00,NaN,NaN,2,1
993,C000165,4,2026-08-15 18:33:00,urgent_reminder,0,0.00,2,2,1.00,0.00,1.00,NaN,NaN,2,1



04E — FINAL CHECKPOINT
Rows          : 75,406
Customers     : 11,724
Columns before: 724
Columns after : 802
Features added: 78

TEMPORAL GOVERNANCE

04E availability class:
    STRICT PIT HISTORICAL RESPONSE

Outcome eligibility:
    source sent_at + 72h <= current sent_at

Denominator:
    only MATURE historical exposures

Numerator:
    only MATURE historical paid_within_72h outcomes

Current event:
    NEVER enters its own history

Immature prior event:
    NEVER enters denominator or numerator

Template response rates:
    OBSERVATIONAL historical response
    NOT causal template effectiveness

Current-template features:
    describe what was known about the customer's prior
    response to the action/template being used at t

Sparsity:
    raw rates intentionally retained
    shrinkage / Empirical Bayes deferred to later layer

Historical policy:
    template assignment is confounded and subject to
    positivity / eligibility constraints

04E — HISTORICAL TEMPLATE × PAYMENT RES

In [37]:
# ============================================================
# 04F.1 — MONETARY RESPONSE STATE
#         [STRICT PIT]
# ============================================================
#
# PURPOSE
# -------
# Consolidate already validated monetary information into
# interpretable customer states available at decision time.
#
# INPUTS
# ------
# 04B:
#   mature payment journey (+72h)
#
# 04C:
#   observed balance trajectory
#
# 04E:
#   mature historical outcome opportunities
#
# NO delivery or interaction information is used here.
#
# Therefore this block remains STRICT PIT.
#
# Starting checkpoint:
#       75,406 × 802
# ============================================================

import numpy as np
import pandas as pd


print("=" * 100)
print("04F.1 — MONETARY RESPONSE STATE [STRICT PIT]")
print("=" * 100)


# ============================================================
# 0. CHECKPOINT
# ============================================================

assert features.shape == (75_406, 802)
assert features["customer_id"].nunique() == 11_724
assert features["event_id"].is_unique
assert features["message_id"].is_unique

rows_before = len(features)
cols_before = features.shape[1]

print(f"Starting shape: {features.shape}")


# ============================================================
# 1. REQUIRED INPUTS
# ============================================================

required_cols = [

    # 04B
    "hist_journey_n_mature_payments",
    "hist_journey_n_mature_partial_payments",
    "hist_journey_total_mature_paid_brl",
    "hist_journey_days_since_last_mature_payment",

    # 04C
    "hist_balance_n_observed_reductions",
    "hist_balance_reduction_from_initial_pct",
    "hist_balance_consecutive_unchanged_transitions",

    # 04E
    "hist_response_total_template_mature_exposures",
    "hist_response_total_template_mature_payment_events",
]


missing = [
    col
    for col in required_cols
    if col not in features.columns
]

assert not missing, (
    f"Missing required columns: {missing}"
)

print("Required inputs: PASSED")


# ============================================================
# 2. BASE ARRAYS
# ============================================================

n_payments = (
    features[
        "hist_journey_n_mature_payments"
    ]
    .fillna(0)
    .to_numpy()
)


n_partials = (
    features[
        "hist_journey_n_mature_partial_payments"
    ]
    .fillna(0)
    .to_numpy()
)


total_paid = (
    features[
        "hist_journey_total_mature_paid_brl"
    ]
    .fillna(0)
    .to_numpy()
)


days_since_payment = (
    features[
        "hist_journey_days_since_last_mature_payment"
    ]
    .to_numpy()
)


n_reductions = (
    features[
        "hist_balance_n_observed_reductions"
    ]
    .fillna(0)
    .to_numpy()
)


progress = (
    features[
        "hist_balance_reduction_from_initial_pct"
    ]
    .fillna(0)
    .to_numpy()
)


stagnation = (
    features[
        "hist_balance_consecutive_unchanged_transitions"
    ]
    .fillna(0)
    .to_numpy()
)


mature_exposures = (
    features[
        "hist_response_total_template_mature_exposures"
    ]
    .fillna(0)
    .to_numpy()
)


mature_payment_events = (
    features[
        "hist_response_total_template_mature_payment_events"
    ]
    .fillna(0)
    .to_numpy()
)


mature_nonpayments = (
    mature_exposures
    -
    mature_payment_events
)


has_payment = (
    n_payments > 0
)


has_partial = (
    n_partials > 0
)


has_progress = (
    n_reductions > 0
)


# ============================================================
# 3. MONETARY RESPONSE STATE
# ============================================================

features[
    "state_monetary_response"
] = np.select(
    [
        n_payments >= 2,
        n_payments == 1,
    ],
    [
        "multiple_mature_payments",
        "one_mature_payment",
    ],
    default="no_mature_payment",
)


features[
    "state_monetary_ever_paid"
] = (
    has_payment
).astype("int8")


features[
    "state_monetary_multiple_payments"
] = (
    n_payments >= 2
).astype("int8")


features[
    "state_monetary_ever_partial"
] = (
    has_partial
).astype("int8")


# ============================================================
# 4. RECOVERY PROGRESS STATE
# ============================================================

features[
    "state_monetary_recovery_progress"
] = np.select(
    [
        progress >= 0.75,
        progress >= 0.50,
        progress >= 0.25,
        has_progress,
    ],
    [
        "advanced_75plus",
        "mid_50_75",
        "early_25_50",
        "progress_below_25",
    ],
    default="no_observed_balance_reduction",
)


features[
    "state_monetary_50plus_recovered"
] = (
    progress >= 0.50
).astype("int8")


features[
    "state_monetary_75plus_recovered"
] = (
    progress >= 0.75
).astype("int8")


# ============================================================
# 5. PAYMENT RECENCY STATE
# ============================================================

features[
    "state_monetary_payment_recency"
] = np.select(
    [
        ~has_payment,

        (
            has_payment
            &
            (days_since_payment <= 3)
        ),

        (
            has_payment
            &
            (days_since_payment <= 7)
        ),

        (
            has_payment
            &
            (days_since_payment <= 14)
        ),

        (
            has_payment
            &
            (days_since_payment <= 30)
        ),
    ],
    [
        "no_mature_payment",
        "payment_within_3d",
        "payment_4_7d",
        "payment_8_14d",
        "payment_15_30d",
    ],
    default="payment_over_30d",
)


# ============================================================
# 6. MATURE NON-PAYMENT COUNT
# ============================================================

features[
    "state_monetary_n_mature_nonpayments"
] = mature_nonpayments


for threshold in [
    3,
    5,
    8,
    10,
]:

    features[
        f"state_monetary_no_payment_{threshold}plus_nonresponses"
    ] = (
        (~has_payment)
        &
        (
            mature_nonpayments >= threshold
        )
    ).astype("int8")


# ============================================================
# 7. PAYMENT × BALANCE CONSISTENCY
# ============================================================

features[
    "state_monetary_paid_with_balance_progress"
] = (
    has_payment
    &
    has_progress
).astype("int8")


features[
    "state_monetary_payment_without_observed_progress"
] = (
    has_payment
    &
    ~has_progress
).astype("int8")


features[
    "state_monetary_progress_without_mature_payment"
] = (
    has_progress
    &
    ~has_payment
).astype("int8")


# ============================================================
# 8. STAGNATION × MONETARY RESPONSE
# ============================================================

features[
    "state_monetary_high_stagnation_no_payment"
] = (
    (stagnation >= 5)
    &
    ~has_payment
).astype("int8")


features[
    "state_monetary_high_stagnation_5plus_nonresponses"
] = (
    (stagnation >= 5)
    &
    (~has_payment)
    &
    (mature_nonpayments >= 5)
).astype("int8")


features[
    "state_monetary_extreme_stagnation_8plus_nonresponses"
] = (
    (stagnation >= 8)
    &
    (~has_payment)
    &
    (mature_nonpayments >= 8)
).astype("int8")


# ============================================================
# 9. HARD CONSISTENCY CHECKS
# ============================================================

print("\nRunning 04F.1 hard consistency checks...")


# Mature exposure accounting

assert (
    mature_nonpayments >= 0
).all()


assert np.array_equal(
    mature_exposures,
    (
        mature_payment_events
        +
        mature_nonpayments
    ),
)


# 04E ↔ 04B reconciliation

assert np.array_equal(
    mature_payment_events,
    n_payments,
)


# Payment flag consistency

assert np.array_equal(
    features[
        "state_monetary_ever_paid"
    ].to_numpy(),
    has_payment.astype("int8"),
)


# Multiple payment consistency

assert (
    features.loc[
        features[
            "state_monetary_multiple_payments"
        ].eq(1),
        "hist_journey_n_mature_payments",
    ]
    .ge(2)
    .all()
)


# Progress hierarchy

assert (
    features[
        "state_monetary_75plus_recovered"
    ]
    <=
    features[
        "state_monetary_50plus_recovered"
    ]
).all()


# No payment without mature exposure

assert not (
    has_payment
    &
    (mature_exposures == 0)
).any()


# First event must have no mature monetary history

first_event = (
    features[
        "event_number"
    ].eq(1)
)


assert (
    features.loc[
        first_event,
        "state_monetary_ever_paid"
    ]
    .eq(0)
    .all()
)


assert (
    features.loc[
        first_event,
        "state_monetary_n_mature_nonpayments"
    ]
    .eq(0)
    .all()
)


print("04F.1 hard consistency checks: PASSED")


# ============================================================
# 10. MONETARY RESPONSE DISTRIBUTION
# ============================================================

print("\nMONETARY RESPONSE STATE")


summary = (
    features[
        "state_monetary_response"
    ]
    .value_counts()
    .to_frame("events")
)


summary[
    "pct_events"
] = (
    summary["events"]
    /
    len(features)
    * 100
)


display(summary)


# ============================================================
# 11. RECOVERY PROGRESS DISTRIBUTION
# ============================================================

print("\nRECOVERY PROGRESS STATE")


summary = (
    features[
        "state_monetary_recovery_progress"
    ]
    .value_counts()
    .to_frame("events")
)


summary[
    "pct_events"
] = (
    summary["events"]
    /
    len(features)
    * 100
)


display(summary)


# ============================================================
# 12. PAYMENT RECENCY
# ============================================================

print("\nPAYMENT RECENCY STATE")


summary = (
    features[
        "state_monetary_payment_recency"
    ]
    .value_counts()
    .to_frame("events")
)


summary[
    "pct_events"
] = (
    summary["events"]
    /
    len(features)
    * 100
)


display(summary)


# ============================================================
# 13. MATURE NON-PAYMENT DISTRIBUTION
# ============================================================

print("\nMATURE NON-PAYMENT DISTRIBUTION")


nonpayment_summary = pd.DataFrame(
    {
        "events": [

            (
                (~has_payment)
                &
                (mature_nonpayments >= 1)
            ).sum(),

            (
                (~has_payment)
                &
                (mature_nonpayments >= 3)
            ).sum(),

            (
                (~has_payment)
                &
                (mature_nonpayments >= 5)
            ).sum(),

            (
                (~has_payment)
                &
                (mature_nonpayments >= 8)
            ).sum(),

            (
                (~has_payment)
                &
                (mature_nonpayments >= 10)
            ).sum(),
        ]
    },
    index=[
        "no payment + 1+ mature nonresponses",
        "no payment + 3+ mature nonresponses",
        "no payment + 5+ mature nonresponses",
        "no payment + 8+ mature nonresponses",
        "no payment + 10+ mature nonresponses",
    ],
)


nonpayment_summary[
    "pct_events"
] = (
    nonpayment_summary[
        "events"
    ]
    /
    len(features)
    * 100
)


display(nonpayment_summary)


# ============================================================
# 14. PAYMENT × BALANCE RECONCILIATION
# ============================================================

print("\nPAYMENT × BALANCE STATE")


payment_balance_summary = pd.DataFrame(
    {
        "events": [

            features[
                "state_monetary_paid_with_balance_progress"
            ].sum(),

            features[
                "state_monetary_payment_without_observed_progress"
            ].sum(),

            features[
                "state_monetary_progress_without_mature_payment"
            ].sum(),

            features[
                "state_monetary_high_stagnation_no_payment"
            ].sum(),

            features[
                "state_monetary_high_stagnation_5plus_nonresponses"
            ].sum(),

            features[
                "state_monetary_extreme_stagnation_8plus_nonresponses"
            ].sum(),
        ]
    },
    index=[
        "mature payment + observed balance progress",
        "mature payment without observed progress",
        "observed progress without mature payment",
        "5+ balance stagnation + no payment",
        "5+ stagnation + 5+ mature nonresponses",
        "8+ stagnation + 8+ mature nonresponses",
    ],
)


payment_balance_summary[
    "pct_events"
] = (
    payment_balance_summary[
        "events"
    ]
    /
    len(features)
    * 100
)


display(payment_balance_summary)


# ============================================================
# 15. SAMPLE — PAYMENT STATE TRANSITION
# ============================================================

sample_candidates = features[
    features[
        "hist_journey_n_mature_payments"
    ].ge(2)
]


if len(sample_candidates):

    sample_idx = (
        sample_candidates[
            "hist_journey_n_mature_payments"
        ]
        .idxmax()
    )

    sample_customer = (
        features.loc[
            sample_idx,
            "customer_id",
        ]
    )

    print("\nSAMPLE — MONETARY STATE TRANSITION")
    print(f"Customer: {sample_customer}")


    display(
        features.loc[
            features[
                "customer_id"
            ].eq(sample_customer),
            [
                "customer_id",
                "event_number",
                "sent_at",
                "current_template",

                "event_paid_within_72h",
                "event_amount_paid_brl",

                "hist_response_total_template_mature_exposures",
                "hist_journey_n_mature_payments",
                "hist_journey_total_mature_paid_brl",

                "hist_balance_reduction_from_initial_pct",
                "hist_balance_consecutive_unchanged_transitions",

                "state_monetary_response",
                "state_monetary_recovery_progress",
                "state_monetary_payment_recency",
            ],
        ]
    )


# ============================================================
# 16. FINAL CHECKPOINT
# ============================================================

assert len(features) == rows_before
assert features["customer_id"].nunique() == 11_724
assert features["event_id"].is_unique
assert features["message_id"].is_unique


features_added = (
    features.shape[1]
    -
    cols_before
)


print("\n" + "=" * 100)
print("04F.1 — FINAL CHECKPOINT")
print("=" * 100)

print(f"Rows          : {len(features):,}")
print(
    f"Customers     : "
    f"{features['customer_id'].nunique():,}"
)
print(f"Columns before: {cols_before:,}")
print(f"Columns after : {features.shape[1]:,}")
print(f"Features added: {features_added:,}")


print(
    """
TEMPORAL GOVERNANCE

04F.1 availability class:
    STRICT PIT DERIVED STATE

Inputs:
    04B mature payment history (+72h)
    04C observed balance trajectory
    04E mature outcome opportunities

No delivery history used.
No interaction history used.
No future outcome introduced.

Interpretation:
    predictive customer monetary state
    NOT causal treatment effect.
"""
)


print("=" * 100)
print("04F.1 — MONETARY RESPONSE STATE COMPLETE")
print("=" * 100)

04F.1 — MONETARY RESPONSE STATE [STRICT PIT]
Starting shape: (75406, 802)
Required inputs: PASSED

Running 04F.1 hard consistency checks...
04F.1 hard consistency checks: PASSED

MONETARY RESPONSE STATE


,events,pct_events
state_monetary_response,,
no_mature_payment,69275,91.87
one_mature_payment,5784,7.67
multiple_mature_payments,347,0.46



RECOVERY PROGRESS STATE


,events,pct_events
state_monetary_recovery_progress,,
no_observed_balance_reduction,69275,91.87
mid_50_75,3122,4.14
early_25_50,2787,3.70
advanced_75plus,222,0.29



PAYMENT RECENCY STATE


...


PAYMENT × BALANCE STATE


,events,pct_events
mature payment + observed balance progress,6131,8.13
mature payment without observed progress,0,0.00
observed progress without mature payment,0,0.00
5+ balance stagnation + no payment,23952,31.76
5+ stagnation + 5+ mature nonresponses,20611,27.33
8+ stagnation + 8+ mature nonresponses,7978,10.58



SAMPLE — MONETARY STATE TRANSITION
Customer: C003527


,customer_id,event_number,sent_at,current_template,event_paid_within_72h,event_amount_paid_brl,hist_response_total_template_mature_exposures,hist_journey_n_mature_payments,hist_journey_total_mature_paid_brl,hist_balance_reduction_from_initial_pct,hist_balance_consecutive_unchanged_transitions,state_monetary_response,state_monetary_recovery_progress,state_monetary_payment_recency
22523,C003527,1,2026-06-17 13:38:00,pix_link,0,0.00,0,0,0.00,0.00,0,no_mature_payment,no_observed_balance_reduction,no_mature_payment
22524,C003527,2,2026-06-20 20:00:00,friendly_reminder,0,0.00,1,0,0.00,0.00,1,no_mature_payment,no_observed_balance_reduction,no_mature_payment
22525,C003527,3,2026-06-23 09:30:00,pix_link,0,0.00,1,0,0.00,0.00,2,no_mature_payment,no_observed_balance_reduction,no_mature_payment
22526,C003527,4,2026-06-24 11:25:00,urgent_reminder,1,467.01,2,0,0.00,0.00,3,no_mature_payment,no_observed_balance_reduction,no_mature_payment
22527,C003527,5,2026-06-29 15:39:00,urgent_reminder,0,0.00,4,1,467.01,0.32,0,one_mature_payment,early_25_50,payment_within_3d
22528,C003527,6,2026-07-01 10:27:00,friendly_reminder,1,362.18,4,1,467.01,0.32,1,one_mature_payment,early_25_50,payment_4_7d
22529,C003527,7,2026-07-20 10:57:00,pix_link,0,0.00,6,2,829.19,0.57,0,multiple_mature_payments,mid_50_75,payment_15_30d
22530,C003527,8,2026-07-21 15:08:00,urgent_reminder,1,249.19,6,2,829.19,0.57,1,multiple_mature_payments,mid_50_75,payment_15_30d
22531,C003527,9,2026-07-29 09:26:00,pix_link,1,228.95,8,3,"1,078.38",0.74,0,multiple_mature_payments,mid_50_75,payment_4_7d
22532,C003527,10,2026-08-03 11:18:00,urgent_reminder,0,0.00,9,4,"1,307.33",0.90,0,multiple_mature_payments,advanced_75plus,payment_within_3d



04F.1 — FINAL CHECKPOINT
Rows          : 75,406
Customers     : 11,724
Columns before: 802
Columns after : 821
Features added: 19

TEMPORAL GOVERNANCE

04F.1 availability class:
    STRICT PIT DERIVED STATE

Inputs:
    04B mature payment history (+72h)
    04C observed balance trajectory
    04E mature outcome opportunities

No delivery history used.
No interaction history used.
No future outcome introduced.

Interpretation:
    predictive customer monetary state
    NOT causal treatment effect.

04F.1 — MONETARY RESPONSE STATE COMPLETE


In [39]:
%pip install pyarrow

  Using cached pyarrow-25.0.1-cp313-cp313-win_amd64.whl.metadata (3.0 kB)
Using cached pyarrow-25.0.1-cp313-cp313-win_amd64.whl (27.9 MB)
Note: you may need to restart the kernel to use updated packages.


In [41]:
from pathlib import Path

CHECKPOINT_DIR = Path("../data/processed")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

PICKLE_PATH = (
    CHECKPOINT_DIR
    / "features_checkpoint_04F1_TEMP.pkl"
)

# ============================================================
# VALIDATE
# ============================================================

assert features.shape == (75_406, 821)
assert features["customer_id"].nunique() == 11_724
assert features["event_id"].is_unique
assert features["message_id"].is_unique

print("Pre-save checks: PASSED")


# ============================================================
# EMERGENCY CHECKPOINT
# ============================================================

features.to_pickle(
    PICKLE_PATH
)

print("Pickle saved.")


# ============================================================
# READ-BACK VALIDATION
# ============================================================

checkpoint_test = pd.read_pickle(
    PICKLE_PATH
)

assert checkpoint_test.shape == (75_406, 821)
assert checkpoint_test["customer_id"].nunique() == 11_724
assert checkpoint_test["event_id"].is_unique
assert checkpoint_test["message_id"].is_unique

print("\n" + "=" * 80)
print("TEMPORARY CHECKPOINT SAVED AND VALIDATED")
print("=" * 80)
print(f"Path      : {PICKLE_PATH.resolve()}")
print(f"Shape     : {checkpoint_test.shape}")
print(
    f"Customers : "
    f"{checkpoint_test['customer_id'].nunique():,}"
)
print(
    f"File size : "
    f"{PICKLE_PATH.stat().st_size / 1024**2:,.2f} MB"
)
print("=" * 80)

del checkpoint_test

Pre-save checks: PASSED
Pickle saved.

TEMPORARY CHECKPOINT SAVED AND VALIDATED
Path      : C:\Users\beelt\Documents\collections_case_candidate\data\processed\features_checkpoint_04F1_TEMP.pkl
Shape     : (75406, 821)
Customers : 11,724
File size : 227.63 MB


In [44]:
import os

print(os.getcwd())

C:\Users\beelt\Documents\collections_case_candidate\notebooks


In [46]:
from pathlib import Path

project_root = Path("..")

for f in project_root.rglob("*.pkl"):
    print(f)

..\data\processed\features_checkpoint_04F1_TEMP.pkl
..\.venv\Lib\site-packages\statsmodels\tsa\statespace\tests\results\sm-0.9-sarimax.pkl
..\.venv\Lib\site-packages\numpy\_core\tests\data\astype_copy.pkl
..\.venv\Lib\site-packages\joblib\test\data\joblib_0.10.0_pickle_py27_np17.pkl
..\.venv\Lib\site-packages\joblib\test\data\joblib_0.10.0_pickle_py33_np18.pkl
..\.venv\Lib\site-packages\joblib\test\data\joblib_0.10.0_pickle_py34_np19.pkl
..\.venv\Lib\site-packages\joblib\test\data\joblib_0.10.0_pickle_py35_np19.pkl
..\.venv\Lib\site-packages\joblib\test\data\joblib_0.11.0_pickle_py36_np111.pkl
..\.venv\Lib\site-packages\joblib\test\data\joblib_0.9.2_pickle_py27_np16.pkl
..\.venv\Lib\site-packages\joblib\test\data\joblib_0.9.2_pickle_py27_np17.pkl
..\.venv\Lib\site-packages\joblib\test\data\joblib_0.9.2_pickle_py33_np18.pkl
..\.venv\Lib\site-packages\joblib\test\data\joblib_0.9.2_pickle_py34_np19.pkl
..\.venv\Lib\site-packages\joblib\test\data\joblib_0.9.2_pickle_py35_np19.pkl


In [47]:
import pandas as pd

df = pd.read_pickle(
    "../data/processed/features_checkpoint_04F1_TEMP.pkl"
)

print("Shape:", df.shape)
print("Número de colunas:", df.shape[1])

Shape: (75406, 821)
Número de colunas: 821


In [48]:
df.columns.tolist()

['message_id',
 'customer_id',
 'sent_at',
 'template',
 'n_msgs_last_14d',
 'days_past_due',
 'outstanding_balance_brl',
 'monthly_salary_brl',
 'payday_day_of_month',
 'n_prior_transactions',
 'account_age_months',
 'days_since_last_app_login',
 'state_uf',
 'event_delivery_status',
 'event_interaction',
 'event_paid_within_72h',
 'event_amount_paid_brl',
 '_source_row',
 'event_number',
 'event_id',
 'sent_date',
 'current_template',
 'send_year',
 'send_month',
 'send_day',
 'send_day_of_year',
 'send_weekday',
 'send_weekday_name',
 'send_week_of_year',
 'send_quarter',
 'is_weekend',
 'is_weekday',
 'is_monday',
 'is_tuesday',
 'is_wednesday',
 'is_thursday',
 'is_friday',
 'is_saturday',
 'is_sunday',
 'send_hour',
 'send_minute',
 'send_minutes_since_midnight',
 'send_hour_decimal',
 'send_time_of_day',
 'is_business_hours_09_18',
 'is_morning_09_11',
 'is_lunch_12_13',
 'is_afternoon_14_17',
 'is_evening_18_plus',
 'send_hour_sin',
 'send_hour_cos',
 'send_weekday_sin',
 'send